<a href="https://colab.research.google.com/github/Vishesh-Goyal7/QBioAuth/blob/main/Q_Bio_Auth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Bio Auth System Logs**

## **The Grand Idea**
In traditional biometric auth systems, we, as the user, have to share the biometric template which poses risk of leaking of the data. So, instead of the traditional system, we propose that on the authentication terminal itself, we reconstruct a dilithium private key for the user and verify it against the user's public key. Once the terminal verifies the user, the terminal initiates a QKD session with the server.

<br/>
<br/>

## **The structure outline**
1) Terminal, server and user exist.
2) User approaches the terminal and gives a communication request. The terminal uses a hardware secured dilithium private key to verify itself to the server. The server authenticates using terminal's public key. Then server verifies itself against the terminal in return using Dilithium key.
3) The user gives a unique id like email. The terminal sends (metadata + UUID) to the server. Server responds with user's public key and helper data to terminal.
4) The terminal scans the user's biometric data, uses helper data and reconstructs the user's secret key. Then, terminal uses secret key to verify user's private key. Once verified, the terminal declares to the user that the person is who they are claiming to be.
5) Once the trust in user, server and terminal is established, the server initiates a QKD session with the terminal.
6) The QKD session creates AES key for AES communication channel.

<br/>
<br/>

## **Instructions to Run (Mac)**

1) Clone the git repository :
```
git clone https://github.com/Vishesh-Goyal7/QBioAuth
```
2) Open the qbioauth-frontend and download the necessary node modules (Ensure that you have the Node and NPM installed on your system)
```
cd qbioauth-frontend && npm i
```
3) Come back to this notebook and run all the cells in this notebook till the Additional Requirement Phase 5 GUI cell block.
4) Copy the Public API URL string output from the last code cell : Download cloudflared binary
5) Go back to the terminal and run :
```
npm run dev
```
6) Navigate to http://localhost:5173.
7) The system asks you to input the FastAPI URL. Paste the Public API URL copied in step 4 and hit enter.

<br/>
<br/>

## **The Phase Plan**
1) Phase 0 : Install and import libraries. Prelim setup.

2) Phase 1 : Create terminal, server, users and generate Dilihtium key pairs.

3) Phase 2 : Terminal signs authentication request, server verifies terminal, server signs authentication response, terminal verifies server, mutual authentication succeeds

4) Phase 3 : Simulate enrolled iris, simulate fresh noisy iris scan, reconstruct biometric secret, unlock user Dilithium private key, sign fresh user challenge, verify user signature locally, create QAuth token, terminal signs QAuth

5) Phase 4 : Server verifies QAuth, Authorized BB84 session begins, QBER is estimated, error correction and privacy amplification occur, final QKD key is produced, AES-GCM key is derived, encrypted communication begins

6) Phase 5 : Replace the simulated binary iris codes with real iris scan database. (Nullified. We will make assumption that the iris scanner exists)

7) Additional Requirement Phase 5 : Create a GUI for the end to end pipeline.

<br/><br/>

## **In Project Logs**
### <b>27-07-2026T13:35:53 </b>

Phase 0, 1, and 2 were preliminary setup. Things happened pretty smooth.

Now comes the real challenging part of making synthetic iris binary code, making user unrollment and then recreation logic of synthetic iris.
If this phase happens peacyfully, the Phase 4 would be very easy.

Current major challenge in this phase expected at creating synthetic iris codes to simulate real ones. Next challenge expected at recreating the user secret from noisy samples.

Next major challenge after this phase expected at recreating this Phase 3 for real iris scan. The major part would be for iris scans instead of binary synthetic codes.

<b>I am wondering if iris is a right choice or not. Fingerprint can work as well but then user inconvenience problem in the other system of continuous auth system. And if we go for face, face can be easily recreated. So, obvious security threat. Need to simulate a high power iris scanner that can capture iris scan from a distance of 1 feet like at airport. Seems like the only possible solution at this moment.</b>

<br/><br/>

###<b>28-07-2026T02:45:22</b>

Phase 3A has been successfuly completed till now. The Phase 3A is concerned with enrolling new users into the system. You give a name, you give an email and the system automatically creates a synthetic binary iris code for the user.

From my theoretical calculation, we can tolerate a 3, maximum 4% of binary scanner noise. I don't know if that is a good thing or should we improvise to make the system more noise tolerant.

<br/><br/>

###<b>28-07-2026T22:22:17</b>

Phase 3B has been successfully completed. The system can successfully identify real, fake and noisy scans. All of the enrolled users get logged in a temporary JSON file.

According to the Reed Solomon error correction and helper data based biometric reconstruction on synthetic iris codes, currently, we can support around 4% of noisy iris scans. This is fairly low and should be improvised. Let's see it in the continuity. TBD in meeting.

<br/><br/>

###<b>29-07-2026T03:04:26</b>

Phase 4 is successfully completed. Everything is successful. AES session is happening, QKD session is happening, biometric reconstruction is happening. I think everything is working in the script right now.

I think the system is ready to be loaded up and called in a systematic pipeline so that we can start with stress testing the system by varying the parameters.

<br/><br/>

###<b>29-07-2026T13:25:26</b>

So.... the big news. An End to End pipeline is ready. Hehehehe. Now, ready for the meeting and to stress test the system and to test the noise and everything else.

<br/><br/>

###<b>31-07-2026T11:21:54</b>

Beginning the testing protocol to evaluate the effects of different parameters and noise and everything on the entire system.

<br/><br/>

###<b>04-08-2026T12:11:32</b>

OFAT study concluded. Graphs made successfully for all the trials

<br/><br/>

###<b>04-08-2026T13:56:24</b>

Beginning the creation of Gradio based GUI

<br/><br/>

###<b>07-08-2026T23:37:16</b>

Pipeline marked complete on my end as of now.

<br/><br/>

## Phase 0 : 27-07-2026T00:25:27 - 27-07-2026T00:48:54

In [3]:
#@title Basic Dependencies

!pip install "pqcrypto==0.3.1" cryptography numpy pandas bitarray galois qiskit qiskit-aer reedsolo xlsxwriter

!pip install --upgrade "gradio>=6.0,<7"

In [4]:
#@title Basic Imports and Functions

from __future__ import annotations

import base64
import hashlib
import hmac
import json
import os
import platform
import secrets
import sys
import time
import uuid

from copy import deepcopy
from dataclasses import asdict, dataclass, field
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np
import pandas as pd

from bitarray import bitarray

from cryptography.exceptions import InvalidTag
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

from pqcrypto.sign.ml_dsa_65 import (
    generate_keypair as ml_dsa_generate_keypair,
    sign as ml_dsa_sign,
    verify as ml_dsa_verify,
)

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

print("All imports completed successfully.")

All imports completed successfully.


In [5]:
#@title Global Config

PROTOCOL_NAME = "QBioAuth"
PROTOCOL_VERSION = "0.1.0"

# Standardised Dilithium successor
SIGNATURE_ALGORITHM = "ML-DSA-65"

# Biometric configuration
IRIS_CODE_LENGTH = 2048
IRIS_CODE_FORMAT = "binary"
IRIS_FINGERPRINT_HASH = "SHA3-256"

# Symmetric cryptography
AES_KEY_LENGTH_BYTES = 32
AES_GCM_NONCE_LENGTH_BYTES = 12
HKDF_HASH_LENGTH_BYTES = 32

# Freshness and replay protection
MESSAGE_FRESHNESS_SECONDS = 60
QAUTH_LIFETIME_SECONDS = 60

# QKD settings will be refined in Phase 4
DEFAULT_QKD_RAW_BITS = 512
DEFAULT_QKD_CHANNEL_ERROR_RATE = 0.01
DEFAULT_QKD_ABORT_QBER = 0.11

# Simulation settings
SIMULATED_LIVENESS_STATUS = "SIMULATED_PASS"

print("Protocol configuration loaded.")
print(f"Protocol: {PROTOCOL_NAME} {PROTOCOL_VERSION}")
print(f"Signature algorithm: {SIGNATURE_ALGORITHM}")
print(f"Iris-code length: {IRIS_CODE_LENGTH} bits")

Protocol configuration loaded.
Protocol: QBioAuth 0.1.0
Signature algorithm: ML-DSA-65
Iris-code length: 2048 bits


In [6]:
#@title Utilities

def utc_now() -> datetime:
    """Return the current timezone-aware UTC datetime."""
    return datetime.now(timezone.utc)


def utc_timestamp() -> str:
    """Return a canonical ISO-8601 UTC timestamp."""
    return utc_now().isoformat(timespec="microseconds")


def generate_uuid() -> str:
    """Generate a random UUID string."""
    return str(uuid.uuid4())


def generate_nonce(length: int = 32) -> bytes:
    """Generate a cryptographically secure random nonce."""
    if length <= 0:
        raise ValueError("Nonce length must be positive.")
    return secrets.token_bytes(length)


def canonical_json(data: Dict[str, Any]) -> bytes:
    """
    Convert a dictionary into deterministic JSON bytes.

    sort_keys=True prevents different key orderings from producing
    different signatures.
    """
    return json.dumps(
        data,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode("utf-8")


def bytes_to_b64(data: bytes) -> str:
    """Encode bytes as a Base64 string."""
    return base64.b64encode(data).decode("ascii")


def b64_to_bytes(data: str) -> bytes:
    """Decode a Base64 string into bytes."""
    return base64.b64decode(data.encode("ascii"), validate=True)


def bytes_to_hex(data: bytes) -> str:
    """Encode bytes as hexadecimal."""
    return data.hex()


def hex_to_bytes(data: str) -> bytes:
    """Decode hexadecimal text."""
    return bytes.fromhex(data)


def sha3_256(data: bytes) -> bytes:
    """Return a SHA3-256 digest."""
    return hashlib.sha3_256(data).digest()


def sha3_256_hex(data: bytes) -> str:
    """Return a hexadecimal SHA3-256 digest."""
    return hashlib.sha3_256(data).hexdigest()


def fingerprint(data: bytes, visible_chars: int = 20) -> str:
    """Create a short display fingerprint without exposing the full value."""
    digest = sha3_256_hex(data)
    return f"{digest[:visible_chars]}..."


print("Encoding and serialisation utilities ready.")

Encoding and serialisation utilities ready.


In [7]:
#@title Dilithium Wrapper

class MLDSA65:
    """
    Small protocol wrapper around pqcrypto's ML-DSA-65 implementation.
    """

    name = "ML-DSA-65"

    @staticmethod
    def generate_keypair() -> Tuple[bytes, bytes]:
        """
        Generate:
            public_key, private_key
        """
        public_key, private_key = ml_dsa_generate_keypair()

        if not public_key or not private_key:
            raise RuntimeError("ML-DSA key generation returned empty keys.")

        return public_key, private_key

    @staticmethod
    def sign(private_key: bytes, message: bytes) -> bytes:
        """Sign a byte message using an ML-DSA private key."""
        if not isinstance(private_key, bytes) or not private_key:
            raise ValueError("A non-empty private key is required.")

        if not isinstance(message, bytes):
            raise TypeError("Message must be bytes.")

        signature = ml_dsa_sign(private_key, message)

        if not signature:
            raise RuntimeError("ML-DSA signing returned an empty signature.")

        return signature

    @staticmethod
    def verify(
        public_key: bytes,
        message: bytes,
        signature: bytes,
    ) -> bool:
        """Safely verify an ML-DSA signature."""
        try:
            result = ml_dsa_verify(public_key, message, signature)

            # Different wrappers may return True or None on success.
            return result is True or result is None

        except Exception:
            return False


print("ML-DSA-65 wrapper ready.")

ML-DSA-65 wrapper ready.


In [8]:
#@title Dilithium Sanity Test

test_public_key, test_private_key = MLDSA65.generate_keypair()

test_message = b"QBioAuth ML-DSA baseline self-test"
test_signature = MLDSA65.sign(test_private_key, test_message)

valid_signature = MLDSA65.verify(
    test_public_key,
    test_message,
    test_signature,
)

tampered_signature_valid = MLDSA65.verify(
    test_public_key,
    b"Tampered message",
    test_signature,
)

assert valid_signature is True
assert tampered_signature_valid is False

print("ML-DSA SELF-TEST")
print("-" * 50)
print(f"Public-key size : {len(test_public_key):,} bytes")
print(f"Private-key size: {len(test_private_key):,} bytes")
print(f"Signature size  : {len(test_signature):,} bytes")
print(f"Valid message   : {valid_signature}")
print(f"Tampered message: {tampered_signature_valid}")
print("-" * 50)
print("ML-DSA baseline: PASS")

ML-DSA SELF-TEST
--------------------------------------------------
Public-key size : 1,952 bytes
Private-key size: 4,032 bytes
Signature size  : 3,309 bytes
Valid message   : True
Tampered message: False
--------------------------------------------------
ML-DSA baseline: PASS


In [9]:
#@title HKDF to derive different keys from one secret QKD derived key

def derive_key_hkdf(
    input_key_material: bytes,
    *,
    salt: Optional[bytes],
    info: bytes,
    length: int = AES_KEY_LENGTH_BYTES,
) -> bytes:
    """
    Derive cryptographic key material using HKDF-SHA-256.
    """
    if not input_key_material:
        raise ValueError("Input key material cannot be empty.")

    if not isinstance(info, bytes) or not info:
        raise ValueError("HKDF info must be non-empty bytes.")

    if length <= 0:
        raise ValueError("Derived-key length must be positive.")

    hkdf = HKDF(
        algorithm=hashes.SHA256(),
        length=length,
        salt=salt,
        info=info,
    )

    return hkdf.derive(input_key_material)


test_ikm = secrets.token_bytes(32)
test_salt = secrets.token_bytes(32)

derived_key_1 = derive_key_hkdf(
    test_ikm,
    salt=test_salt,
    info=b"QBioAuth|BaselineTest",
)

derived_key_2 = derive_key_hkdf(
    test_ikm,
    salt=test_salt,
    info=b"QBioAuth|BaselineTest",
)

assert hmac.compare_digest(derived_key_1, derived_key_2)
assert len(derived_key_1) == AES_KEY_LENGTH_BYTES

print("HKDF baseline: PASS")
print(f"Derived-key length: {len(derived_key_1)} bytes")

HKDF baseline: PASS
Derived-key length: 32 bytes


In [10]:
#@title AES Wrapper Function

@dataclass(frozen=True)
class AESGCMEnvelope:
    nonce_b64: str
    ciphertext_b64: str
    associated_data_b64: str


def aes_gcm_encrypt(
    key: bytes,
    plaintext: bytes,
    associated_data: bytes,
) -> AESGCMEnvelope:
    """
    Encrypt plaintext with AES-256-GCM.

    cryptography's AESGCM output contains:
        ciphertext || authentication tag
    """
    if len(key) != AES_KEY_LENGTH_BYTES:
        raise ValueError("AES-256-GCM requires a 32-byte key.")

    nonce = generate_nonce(AES_GCM_NONCE_LENGTH_BYTES)
    aes = AESGCM(key)

    encrypted = aes.encrypt(
        nonce,
        plaintext,
        associated_data,
    )

    return AESGCMEnvelope(
        nonce_b64=bytes_to_b64(nonce),
        ciphertext_b64=bytes_to_b64(encrypted),
        associated_data_b64=bytes_to_b64(associated_data),
    )


def aes_gcm_decrypt(
    key: bytes,
    envelope: AESGCMEnvelope,
) -> bytes:
    """
    Verify and decrypt an AES-GCM envelope.

    Raises InvalidTag when the key, nonce, ciphertext,
    tag or associated data is incorrect.
    """
    if len(key) != AES_KEY_LENGTH_BYTES:
        raise ValueError("AES-256-GCM requires a 32-byte key.")

    aes = AESGCM(key)

    nonce = b64_to_bytes(envelope.nonce_b64)
    encrypted = b64_to_bytes(envelope.ciphertext_b64)
    associated_data = b64_to_bytes(envelope.associated_data_b64)

    return aes.decrypt(
        nonce,
        encrypted,
        associated_data,
    )


print("AES-256-GCM wrapper ready.")

AES-256-GCM wrapper ready.


In [11]:
#@title AES Sanity Test

test_aes_key = secrets.token_bytes(AES_KEY_LENGTH_BYTES)
test_plaintext = b"Quantum-secure biometric authentication test"
test_aad = b"QBioAuth|Session|TEST-001"

test_envelope = aes_gcm_encrypt(
    test_aes_key,
    test_plaintext,
    test_aad,
)

recovered_plaintext = aes_gcm_decrypt(
    test_aes_key,
    test_envelope,
)

assert recovered_plaintext == test_plaintext

tampered_ciphertext = bytearray(
    b64_to_bytes(test_envelope.ciphertext_b64)
)
tampered_ciphertext[0] ^= 1

tampered_envelope = AESGCMEnvelope(
    nonce_b64=test_envelope.nonce_b64,
    ciphertext_b64=bytes_to_b64(bytes(tampered_ciphertext)),
    associated_data_b64=test_envelope.associated_data_b64,
)

tampering_detected = False

try:
    aes_gcm_decrypt(test_aes_key, tampered_envelope)
except InvalidTag:
    tampering_detected = True

assert tampering_detected is True

print("AES-GCM SELF-TEST")
print("-" * 50)
print(f"Original plaintext : {test_plaintext.decode()}")
print(f"Recovered plaintext: {recovered_plaintext.decode()}")
print(f"Tampering detected : {tampering_detected}")
print("-" * 50)
print("AES-GCM baseline: PASS")

AES-GCM SELF-TEST
--------------------------------------------------
Original plaintext : Quantum-secure biometric authentication test
Recovered plaintext: Quantum-secure biometric authentication test
Tampering detected : True
--------------------------------------------------
AES-GCM baseline: PASS


In [12]:
#@title Qiskit Simulator Verification

simulator = AerSimulator()

test_circuit = QuantumCircuit(1, 1)
test_circuit.h(0)
test_circuit.measure(0, 0)

compiled_circuit = transpile(
    test_circuit,
    simulator,
)

job = simulator.run(
    compiled_circuit,
    shots=256,
)

result = job.result()
counts = result.get_counts()

assert set(counts.keys()).issubset({"0", "1"})
assert sum(counts.values()) == 256

print("QISKIT AER SELF-TEST")
print("-" * 50)
print(f"Measurement counts: {counts}")
print("-" * 50)
print("Qiskit Aer baseline: PASS")

QISKIT AER SELF-TEST
--------------------------------------------------
Measurement counts: {'0': 128, '1': 128}
--------------------------------------------------
Qiskit Aer baseline: PASS


In [13]:
#@title Synthetic Iris

@dataclass
class IrisSample:
    """
    Standardised output of any iris-acquisition source.

    In Phase 3:
        SyntheticIrisSource -> IrisSample

    In Phase 6:
        Iris image extractor -> IrisSample
    """

    iris_code: np.ndarray
    reliability_mask: np.ndarray
    quality_score: float
    sample_id: str
    capture_timestamp: str
    source_type: str
    metadata: Dict[str, Any] = field(default_factory=dict)

    def validate(self) -> None:
        if not isinstance(self.iris_code, np.ndarray):
            raise TypeError("iris_code must be a NumPy array.")

        if not isinstance(self.reliability_mask, np.ndarray):
            raise TypeError("reliability_mask must be a NumPy array.")

        if self.iris_code.ndim != 1:
            raise ValueError("iris_code must be one-dimensional.")

        if self.reliability_mask.ndim != 1:
            raise ValueError("reliability_mask must be one-dimensional.")

        if len(self.iris_code) != len(self.reliability_mask):
            raise ValueError(
                "iris_code and reliability_mask lengths must match."
            )

        if len(self.iris_code) != IRIS_CODE_LENGTH:
            raise ValueError(
                f"Iris code must contain {IRIS_CODE_LENGTH} bits."
            )

        if not np.all(np.isin(self.iris_code, [0, 1])):
            raise ValueError("iris_code must contain only binary values.")

        if not np.all(np.isin(self.reliability_mask, [0, 1])):
            raise ValueError(
                "reliability_mask must contain only binary values."
            )

        if not 0.0 <= self.quality_score <= 1.0:
            raise ValueError(
                "quality_score must lie between 0.0 and 1.0."
            )


def binary_array_to_string(bits: np.ndarray) -> str:
    """Convert a NumPy binary array into a copy-ready bit string."""
    if not np.all(np.isin(bits, [0, 1])):
        raise ValueError("Input array must contain only zeroes and ones.")

    return "".join(str(int(bit)) for bit in bits)


def binary_string_to_array(code: str) -> np.ndarray:
    """Convert a raw binary iris-code string to a NumPy array."""
    cleaned = "".join(code.split())

    if not cleaned:
        raise ValueError("Iris code cannot be empty.")

    if any(character not in {"0", "1"} for character in cleaned):
        raise ValueError("Iris code must contain only 0 and 1.")

    return np.fromiter(
        (int(character) for character in cleaned),
        dtype=np.uint8,
    )


def binary_array_to_bytes(bits: np.ndarray) -> bytes:
    """Pack a binary NumPy array into bytes."""
    return np.packbits(bits.astype(np.uint8)).tobytes()


def iris_fingerprint(bits: np.ndarray) -> str:
    """Return the complete SHA3-256 fingerprint of an iris code."""
    packed = binary_array_to_bytes(bits)
    return sha3_256_hex(packed)


print("Iris data interface ready.")

Iris data interface ready.


In [14]:
#@title Protocol Message Envelope

@dataclass(frozen=True)
class ProtocolEnvelope:
    message_type: str
    sender_id: str
    receiver_id: str
    session_id: str
    message_id: str
    issued_at: str
    payload: Dict[str, Any]
    signature_b64: Optional[str] = None

    def unsigned_dictionary(self) -> Dict[str, Any]:
        return {
            "message_type": self.message_type,
            "sender_id": self.sender_id,
            "receiver_id": self.receiver_id,
            "session_id": self.session_id,
            "message_id": self.message_id,
            "issued_at": self.issued_at,
            "payload": self.payload,
        }

    def unsigned_bytes(self) -> bytes:
        return canonical_json(self.unsigned_dictionary())

    def to_dictionary(self) -> Dict[str, Any]:
        result = self.unsigned_dictionary()
        result["signature_b64"] = self.signature_b64
        return result

    def to_json(self) -> str:
        return json.dumps(
            self.to_dictionary(),
            sort_keys=True,
            indent=2,
        )


print("Protocol message envelope ready.")

Protocol message envelope ready.


In [15]:
#@title Phase 0 Final Check

baseline_report = {
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "protocol": f"{PROTOCOL_NAME} {PROTOCOL_VERSION}",
    "signature_algorithm": SIGNATURE_ALGORITHM,
    "iris_code_length_bits": IRIS_CODE_LENGTH,
    "aes_algorithm": "AES-256-GCM",
    "kdf": "HKDF-SHA-256",
    "qiskit_aer": "Operational",
    "mldsa_self_test": "PASS",
    "aes_gcm_self_test": "PASS",
    "hkdf_self_test": "PASS",
}

display(pd.DataFrame(
    baseline_report.items(),
    columns=["Component", "Status"],
))

print("\n" + "=" * 65)
print("PHASE 0 COMPLETED SUCCESSFULLY")
print("=" * 65)
print("The environment is ready for terminal/server association.")

,Component,Status
0,python_version,3.12.13
1,platform,Linux-6.6.122+-x86_64-with-glibc2.35
2,protocol,QBioAuth 0.1.0
3,signature_algorithm,ML-DSA-65
4,iris_code_length_bits,2048
5,aes_algorithm,AES-256-GCM
6,kdf,HKDF-SHA-256
7,qiskit_aer,Operational
8,mldsa_self_test,PASS
9,aes_gcm_self_test,PASS



PHASE 0 COMPLETED SUCCESSFULLY
The environment is ready for terminal/server association.


## Phase 1 : 27-07-2026T00:52:17 - 27-07-2026T12:25:46

In [16]:
#@title Declaring Node Roles and Associating them Identities

from enum import Enum


class NodeRole(str, Enum):
    TERMINAL = "TRUSTED_TERMINAL"
    SERVER = "AUTHENTICATION_SERVER"


@dataclass(frozen=True)
class PublicIdentityCard:
    """
    Public information that a node may safely disclose.

    The private key is deliberately absent.
    """

    node_id: str
    role: str
    algorithm: str
    public_key_b64: str
    public_key_fingerprint: str
    credential_version: int
    created_at: str

    def public_key_bytes(self) -> bytes:
        return b64_to_bytes(self.public_key_b64)

    def to_dictionary(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class TrustedPeerRecord:
    """
    Public-key record pinned in a node's local trust registry.
    """

    peer_id: str
    peer_role: str
    algorithm: str
    public_key_b64: str
    public_key_fingerprint: str
    credential_version: int
    trust_status: str
    registered_at: str

    def public_key_bytes(self) -> bytes:
        return b64_to_bytes(self.public_key_b64)


print("Node-role and public-identity structures ready.")

Node-role and public-identity structures ready.


In [17]:
#@title Dilithium Signing Vault : The Server and Terminal are going to have hardware backed keys. This process does a similar thing. Now, we can't access the dilithium keys directly, only ask for the signature from them.

def create_mldsa_signing_vault(
    private_key: bytes,
) -> Tuple[Any, Any]:
    """
    Create a closure-backed private-key vault.

    Returned functions:
        sign_function(message)
        metadata_function()

    No function is provided to export the private key.

    This provides logical isolation for the Colab prototype.
    It is not equivalent to an HSM or separate operating-system process.
    """

    if not isinstance(private_key, bytes) or not private_key:
        raise ValueError("A valid ML-DSA private key is required.")

    protected_private_key = bytes(private_key)

    def sign_function(message: bytes) -> bytes:
        if not isinstance(message, bytes):
            raise TypeError("The message to be signed must be bytes.")

        return MLDSA65.sign(
            protected_private_key,
            message,
        )

    def metadata_function() -> Dict[str, Any]:
        return {
            "algorithm": MLDSA65.name,
            "private_key_loaded": True,
            "private_key_size_bytes": len(protected_private_key),
            "private_key_exportable": False,
        }

    return sign_function, metadata_function


print("Closure-backed signing vault ready.")

Closure-backed signing vault ready.


In [18]:
#@title Isolated Node class. Server and terminal would be objects of this class.

class IsolatedNode:
    """
    Base class for terminal and server nodes.

    Each node owns:
        - its own ML-DSA key pair,
        - a private signing vault,
        - a trusted-peer public-key registry,
        - a replay cache,
        - active-session state,
        - a local event log.

    No node stores another node's private key.
    """

    def __init__(
        self,
        node_id: str,
        role: NodeRole,
        credential_version: int = 1,
    ) -> None:

        if not node_id or not isinstance(node_id, str):
            raise ValueError("node_id must be a non-empty string.")

        if not isinstance(role, NodeRole):
            raise TypeError("role must be a NodeRole value.")

        self.node_id = node_id
        self.role = role
        self.algorithm = MLDSA65.name
        self.credential_version = credential_version
        self.created_at = utc_timestamp()

        # Generate this node's independent ML-DSA identity.
        public_key, private_key = MLDSA65.generate_keypair()

        self._public_key = bytes(public_key)

        # The private key is placed inside the closure-backed vault.
        self._sign, self._vault_metadata = create_mldsa_signing_vault(private_key)

        # Remove the constructor's ordinary local reference.
        del private_key

        # Local trust and protocol state.
        self._trusted_peers: Dict[str, TrustedPeerRecord] = {}
        self._used_nonces: Set[str] = set()
        self._active_sessions: Dict[str, Dict[str, Any]] = {}
        self._event_log: List[Dict[str, Any]] = []

        self._log_event(
            event_type="NODE_CREATED",
            result="SUCCESS",
            details={
                "node_id": self.node_id,
                "role": self.role.value,
                "algorithm": self.algorithm,
            },
        )

    # Internal logging

    def _log_event(
        self,
        event_type: str,
        result: str,
        details: Optional[Dict[str, Any]] = None,
    ) -> None:

        self._event_log.append({
            "timestamp": utc_timestamp(),
            "node_id": self.node_id,
            "event_type": event_type,
            "result": result,
            "details": deepcopy(details or {}),
        })

    # Creating an id card for showing the public key as a face card.

    def public_identity_card(self) -> PublicIdentityCard:
        """
        Export only this node's public identity information.
        """

        return PublicIdentityCard(
            node_id=self.node_id,
            role=self.role.value,
            algorithm=self.algorithm,
            public_key_b64=bytes_to_b64(self._public_key),
            public_key_fingerprint=sha3_256_hex(self._public_key),
            credential_version=self.credential_version,
            created_at=self.created_at,
        )

    def public_key_bytes(self) -> bytes:
        """
        Return a copy of the public key.
        """

        return bytes(self._public_key)


    # Private Signing Operations

    def sign_bytes(self, message: bytes) -> bytes:
        """
        Sign bytes using the node's private vault.

        The private key itself is never returned.
        """

        signature = self._sign(message)

        self._log_event(
            event_type="DATA_SIGNED",
            result="SUCCESS",
            details={
                "message_hash": sha3_256_hex(message),
                "signature_size_bytes": len(signature),
            },
        )

        return signature

    # Trust Registry : Log of public keys of all the other users which have been verified once.

    def register_trusted_peer(
        self,
        identity_card: PublicIdentityCard,
        expected_role: NodeRole,
    ) -> None:
        """
        Pin another node's public identity into the local trust registry.

        This models secure provisioning performed before online
        mutual authentication.
        """

        if identity_card.node_id == self.node_id:
            raise ValueError(
                "A node cannot register itself as a trusted peer."
            )

        if identity_card.role != expected_role.value:
            raise ValueError(
                f"Unexpected peer role. Expected {expected_role.value}, "
                f"received {identity_card.role}."
            )

        if identity_card.algorithm != MLDSA65.name:
            raise ValueError(
                f"Unsupported peer algorithm: {identity_card.algorithm}"
            )

        public_key = identity_card.public_key_bytes()
        calculated_fingerprint = sha3_256_hex(public_key)

        if not hmac.compare_digest(
            calculated_fingerprint,
            identity_card.public_key_fingerprint,
        ):
            raise ValueError(
                "Public-key fingerprint validation failed."
            )

        existing = self._trusted_peers.get(identity_card.node_id)

        if existing is not None:
            if not hmac.compare_digest(
                existing.public_key_fingerprint,
                identity_card.public_key_fingerprint,
            ):
                raise ValueError(
                    "A different key is already registered for this peer."
                )

            return

        self._trusted_peers[identity_card.node_id] = TrustedPeerRecord(
            peer_id=identity_card.node_id,
            peer_role=identity_card.role,
            algorithm=identity_card.algorithm,
            public_key_b64=identity_card.public_key_b64,
            public_key_fingerprint=identity_card.public_key_fingerprint,
            credential_version=identity_card.credential_version,
            trust_status="TRUSTED",
            registered_at=utc_timestamp(),
        )

        self._log_event(
            event_type="TRUSTED_PEER_REGISTERED",
            result="SUCCESS",
            details={
                "peer_id": identity_card.node_id,
                "peer_role": identity_card.role,
                "peer_public_key_fingerprint":
                    identity_card.public_key_fingerprint,
            },
        )

    def is_peer_trusted(self, peer_id: str) -> bool:
        record = self._trusted_peers.get(peer_id)

        return (
            record is not None
            and record.trust_status == "TRUSTED"
        )

    def trusted_peer_public_key(
        self,
        peer_id: str,
    ) -> bytes:
        """
        Retrieve a trusted peer's public key.

        Only public keys are stored in this registry.
        """

        record = self._trusted_peers.get(peer_id)

        if record is None:
            raise KeyError(
                f"{peer_id} is not present in {self.node_id}'s trust registry."
            )

        if record.trust_status != "TRUSTED":
            raise PermissionError(
                f"{peer_id} does not currently have trusted status."
            )

        return record.public_key_bytes()

    def verify_trusted_peer_signature(
        self,
        peer_id: str,
        message: bytes,
        signature: bytes,
    ) -> bool:
        """
        Verify a signature using a public key pinned in the local registry.
        """

        try:
            public_key = self.trusted_peer_public_key(peer_id)

            valid = MLDSA65.verify(
                public_key,
                message,
                signature,
            )

        except Exception:
            valid = False

        self._log_event(
            event_type="PEER_SIGNATURE_VERIFICATION",
            result="SUCCESS" if valid else "FAILURE",
            details={
                "peer_id": peer_id,
                "message_hash": sha3_256_hex(message),
            },
        )

        return valid

    # Cache replay to see the recent system developments

    def has_seen_nonce(self, nonce_b64: str) -> bool:
        return nonce_b64 in self._used_nonces

    def consume_nonce(self, nonce_b64: str) -> None:
        if nonce_b64 in self._used_nonces:
            raise ValueError("Replay detected: nonce has already been used.")

        self._used_nonces.add(nonce_b64)

        self._log_event(
            event_type="NONCE_CONSUMED",
            result="SUCCESS",
            details={
                "nonce_fingerprint": fingerprint(
                    b64_to_bytes(nonce_b64)
                )
            },
        )

    # Session Logs

    def create_local_session(
        self,
        session_id: Optional[str] = None,
        initial_state: Optional[Dict[str, Any]] = None,
    ) -> str:

        resolved_session_id = session_id or generate_uuid()

        if resolved_session_id in self._active_sessions:
            raise ValueError(
                f"Session {resolved_session_id} already exists."
            )

        self._active_sessions[resolved_session_id] = deepcopy(
            initial_state or {}
        )

        self._active_sessions[resolved_session_id].update({
            "session_id": resolved_session_id,
            "created_at": utc_timestamp(),
            "owner_node_id": self.node_id,
        })

        self._log_event(
            event_type="SESSION_CREATED",
            result="SUCCESS",
            details={
                "session_id": resolved_session_id,
            },
        )

        return resolved_session_id

    def get_local_session(
        self,
        session_id: str,
    ) -> Dict[str, Any]:

        if session_id not in self._active_sessions:
            raise KeyError(
                f"Unknown local session: {session_id}"
            )

        return deepcopy(self._active_sessions[session_id])

    def update_local_session(
        self,
        session_id: str,
        updates: Dict[str, Any],
    ) -> None:

        if session_id not in self._active_sessions:
            raise KeyError(
                f"Unknown local session: {session_id}"
            )

        self._active_sessions[session_id].update(
            deepcopy(updates)
        )

    # Diagnostic check

    def safe_snapshot(self) -> Dict[str, Any]:
        """
        Return diagnostic state without exposing secret key material.
        """

        return {
            "node_id": self.node_id,
            "role": self.role.value,
            "algorithm": self.algorithm,
            "public_key_fingerprint":
                sha3_256_hex(self._public_key),
            "public_key_size_bytes":
                len(self._public_key),
            "credential_version":
                self.credential_version,
            "trusted_peer_ids":
                sorted(self._trusted_peers.keys()),
            "trusted_peer_count":
                len(self._trusted_peers),
            "used_nonce_count":
                len(self._used_nonces),
            "active_session_count":
                len(self._active_sessions),
            "vault":
                self._vault_metadata(),
        }

    def event_log(self) -> List[Dict[str, Any]]:
        return deepcopy(self._event_log)


print("Base isolated-node class ready.")

Base isolated-node class ready.


In [19]:
# @title Terminal and Server Nodes as extentsions of IsolatedNode class

class TerminalNode(IsolatedNode):
    """
    Trusted biometric terminal.

    Future responsibilities:
        - authenticate the server,
        - perform local biometric enrollment,
        - perform local biometric authentication,
        - unlock the user's ML-DSA credential,
        - generate and sign QAuth,
        - operate one QKD endpoint.
    """

    def __init__(
        self,
        terminal_id: str,
        terminal_location: str,
        credential_version: int = 1,
    ) -> None:

        super().__init__(
            node_id=terminal_id,
            role=NodeRole.TERMINAL,
            credential_version=credential_version,
        )

        self.terminal_location = terminal_location

        self._local_user_records: Dict[str, Dict[str, Any]] = {}
        self._pending_authentications: Dict[str, Dict[str, Any]] = {}

        self._log_event(
            event_type="TERMINAL_INITIALISED",
            result="SUCCESS",
            details={
                "terminal_location": terminal_location,
            },
        )

    def safe_snapshot(self) -> Dict[str, Any]:
        snapshot = super().safe_snapshot()

        snapshot.update({
            "terminal_location": self.terminal_location,
            "local_user_record_count":
                len(self._local_user_records),
            "pending_authentication_count":
                len(self._pending_authentications),
        })

        return snapshot


class ServerNode(IsolatedNode):
    """
    Central authentication server.

    Future responsibilities:
        - authenticate terminals,
        - verify signed QAuth tokens,
        - maintain registered user public keys,
        - authorise QKD sessions,
        - operate the server-side QKD endpoint.
    """

    def __init__(
        self,
        server_id: str,
        service_name: str,
        credential_version: int = 1,
    ) -> None:

        super().__init__(
            node_id=server_id,
            role=NodeRole.SERVER,
            credential_version=credential_version,
        )

        self.service_name = service_name

        self._registered_users: Dict[str, Dict[str, Any]] = {}
        self._accepted_qauth_tokens: Dict[str, Dict[str, Any]] = {}

        self._log_event(
            event_type="SERVER_INITIALISED",
            result="SUCCESS",
            details={
                "service_name": service_name,
            },
        )

    def safe_snapshot(self) -> Dict[str, Any]:
        snapshot = super().safe_snapshot()

        snapshot.update({
            "service_name": self.service_name,
            "registered_user_count":
                len(self._registered_users),
            "accepted_qauth_token_count":
                len(self._accepted_qauth_tokens),
        })

        return snapshot


print("TerminalNode and ServerNode classes ready.")

TerminalNode and ServerNode classes ready.


In [20]:
#@title Message Sending Protocol so that the nodes are interacting with each other's message packets and not using each other's data dictionaries to ensure complete node isolation.

@dataclass(frozen=True)
class TransportPacket:
    packet_id: str
    sender_id: str
    receiver_id: str
    sent_at: str
    serialized_message: str


class InMemoryMessageTransport:
    """
    Simulated network transport.

    Important property:
    Messages cross the boundary as serialized JSON text.

    The receiver does not obtain a direct reference to the
    sender's original Python object.
    """

    def __init__(self) -> None:
        self._queues: Dict[str, List[TransportPacket]] = {}
        self._delivery_log: List[Dict[str, Any]] = []

    def register_endpoint(self, node_id: str) -> None:
        if node_id in self._queues:
            raise ValueError(
                f"Endpoint {node_id} has already been registered."
            )

        self._queues[node_id] = []

    def send(
        self,
        sender_id: str,
        receiver_id: str,
        message: Dict[str, Any],
    ) -> str:

        if sender_id not in self._queues:
            raise KeyError(
                f"Unknown sender endpoint: {sender_id}"
            )

        if receiver_id not in self._queues:
            raise KeyError(
                f"Unknown receiver endpoint: {receiver_id}"
            )

        if sender_id == receiver_id:
            raise ValueError(
                "Sender and receiver cannot be identical."
            )

        # Canonical serialization creates a boundary between nodes.
        serialized_message = canonical_json(message).decode("utf-8")

        packet = TransportPacket(
            packet_id=generate_uuid(),
            sender_id=sender_id,
            receiver_id=receiver_id,
            sent_at=utc_timestamp(),
            serialized_message=serialized_message,
        )

        self._queues[receiver_id].append(packet)

        self._delivery_log.append({
            "packet_id": packet.packet_id,
            "sender_id": sender_id,
            "receiver_id": receiver_id,
            "sent_at": packet.sent_at,
            "message_hash": sha3_256_hex(
                serialized_message.encode("utf-8")
            ),
        })

        return packet.packet_id

    def receive(
        self,
        receiver_id: str,
    ) -> Optional[Tuple[TransportPacket, Dict[str, Any]]]:

        if receiver_id not in self._queues:
            raise KeyError(
                f"Unknown receiver endpoint: {receiver_id}"
            )

        if not self._queues[receiver_id]:
            return None

        packet = self._queues[receiver_id].pop(0)

        # JSON decoding creates a new object owned by the receiver.
        message = json.loads(packet.serialized_message)

        return packet, message

    def pending_count(self, receiver_id: str) -> int:
        if receiver_id not in self._queues:
            raise KeyError(
                f"Unknown endpoint: {receiver_id}"
            )

        return len(self._queues[receiver_id])

    def delivery_log(self) -> List[Dict[str, Any]]:
        return deepcopy(self._delivery_log)


print("Serialized in-memory transport ready.")

Serialized in-memory transport ready.


In [21]:
#@title Terminal and Server Node Creation

TERMINAL_ID = "TERMINAL-QBA-001"
SERVER_ID = "SERVER-QBA-PRIMARY-001"

terminal = TerminalNode(
    terminal_id=TERMINAL_ID,
    terminal_location="Loc1",
)

server = ServerNode(
    server_id=SERVER_ID,
    service_name="Loc2",
)

network = InMemoryMessageTransport()

network.register_endpoint(terminal.node_id)
network.register_endpoint(server.node_id)

print("Nodes created successfully.")
print()
print(f"Terminal ID : {terminal.node_id}")
print(f"Server ID   : {server.node_id}")
print()
print("Independent ML-DSA-65 key pairs generated.")
print("Serialized communication endpoints registered.")

Nodes created successfully.

Terminal ID : TERMINAL-QBA-001
Server ID   : SERVER-QBA-PRIMARY-001

Independent ML-DSA-65 key pairs generated.
Serialized communication endpoints registered.


In [22]:
#@title Display Public Cards of the Terminal and Server nodes

terminal_identity = terminal.public_identity_card()
server_identity = server.public_identity_card()

identity_table = pd.DataFrame([
    {
        "Node ID": terminal_identity.node_id,
        "Role": terminal_identity.role,
        "Algorithm": terminal_identity.algorithm,
        "Public-Key Size": len(
            terminal_identity.public_key_bytes()
        ),
        "Public-Key Fingerprint":
            terminal_identity.public_key_fingerprint,
        "Credential Version":
            terminal_identity.credential_version,
    },
    {
        "Node ID": server_identity.node_id,
        "Role": server_identity.role,
        "Algorithm": server_identity.algorithm,
        "Public-Key Size": len(
            server_identity.public_key_bytes()
        ),
        "Public-Key Fingerprint":
            server_identity.public_key_fingerprint,
        "Credential Version":
            server_identity.credential_version,
    },
])

display(identity_table)

assert (
    terminal_identity.public_key_fingerprint
    != server_identity.public_key_fingerprint
)

print("The terminal and server have distinct ML-DSA identities.")

,Node ID,Role,Algorithm,Public-Key Size,Public-Key Fingerprint,Credential Version
0,TERMINAL-QBA-001,TRUSTED_TERMINAL,ML-DSA-65,1952,bde791397eb32adb7c0f24aee8762628c3d32d63aceac6...,1
1,SERVER-QBA-PRIMARY-001,AUTHENTICATION_SERVER,ML-DSA-65,1952,4ca04fb83c255f1c4ad0d283bc052254f1248c3d337707...,1


The terminal and server have distinct ML-DSA identities.


In [23]:
#@title Adding the server's public key to terminal's trusted peer registry and vice versa

# Terminal pins the server's public identity.
terminal.register_trusted_peer(
    identity_card=server_identity,
    expected_role=NodeRole.SERVER,
)

# Server pins the terminal's public identity.
server.register_trusted_peer(
    identity_card=terminal_identity,
    expected_role=NodeRole.TERMINAL,
)

assert terminal.is_peer_trusted(SERVER_ID)
assert server.is_peer_trusted(TERMINAL_ID)

print("Mutual public-key association completed.")
print()
print(f"{terminal.node_id} trusts public key of {server.node_id}")
print(f"{server.node_id} trusts public key of {terminal.node_id}")
print()

Mutual public-key association completed.

TERMINAL-QBA-001 trusts public key of SERVER-QBA-PRIMARY-001
SERVER-QBA-PRIMARY-001 trusts public key of TERMINAL-QBA-001



In [24]:
#@title Terminal and Server Checking for Dilithium

terminal_test_message = (b"Terminal independent signing identity test")

server_test_message = (b"Server independent signing identity test")

terminal_test_signature = terminal.sign_bytes(terminal_test_message)

server_test_signature = server.sign_bytes(server_test_message)

# Correct verifications
server_accepts_terminal_signature = (
    server.verify_trusted_peer_signature(
        peer_id=TERMINAL_ID,
        message=terminal_test_message,
        signature=terminal_test_signature,
    )
)

terminal_accepts_server_signature = (
    terminal.verify_trusted_peer_signature(
        peer_id=SERVER_ID,
        message=server_test_message,
        signature=server_test_signature,
    )
)

# Wrong-key cross-checks
server_signature_as_terminal = (
    server.verify_trusted_peer_signature(
        peer_id=TERMINAL_ID,
        message=server_test_message,
        signature=server_test_signature,
    )
)

terminal_signature_as_server = (
    terminal.verify_trusted_peer_signature(
        peer_id=SERVER_ID,
        message=terminal_test_message,
        signature=terminal_test_signature,
    )
)

assert server_accepts_terminal_signature is True
assert terminal_accepts_server_signature is True

assert server_signature_as_terminal is False
assert terminal_signature_as_server is False

print("PHASE 1 CRYPTOGRAPHIC IDENTITY TEST")
print("-" * 65)
print("Server verifies Terminal's signature :", server_accepts_terminal_signature)
print("Terminal verifies Server's signature :", terminal_accepts_server_signature)
print("Server key accepted as Terminal key   :", server_signature_as_terminal,)
print("Terminal key accepted as Server key   :", terminal_signature_as_server,)
print("-" * 65)
print("Independent-node identity test: PASS")

PHASE 1 CRYPTOGRAPHIC IDENTITY TEST
-----------------------------------------------------------------
Server verifies Terminal's signature : True
Terminal verifies Server's signature : True
Server key accepted as Terminal key   : False
Terminal key accepted as Server key   : False
-----------------------------------------------------------------
Independent-node identity test: PASS


In [25]:
#@title Message packet sharing test

original_message = {
    "message_type": "TRANSPORT_SELF_TEST",
    "sender_id": TERMINAL_ID,
    "receiver_id": SERVER_ID,
    "payload": {
        "value": 42,
        "status": "ORIGINAL",
    },
}

packet_id = network.send(
    sender_id=TERMINAL_ID,
    receiver_id=SERVER_ID,
    message=original_message,
)

# Mutate the sender's original dictionary after transmission.
original_message["payload"]["value"] = 999
original_message["payload"]["status"] = "MUTATED_AFTER_SEND"

received = network.receive(SERVER_ID)

assert received is not None

received_packet, received_message = received

# Receiver must see the state that existed when send() was called.
assert received_message["payload"]["value"] == 42
assert received_message["payload"]["status"] == "ORIGINAL"

print("SERIALIZED TRANSPORT TEST")
print("-" * 65)
print(f"Packet ID       : {packet_id}")
print(f"Sender          : {received_packet.sender_id}")
print(f"Receiver        : {received_packet.receiver_id}")
print(
    "Received value :",
    received_message["payload"]["value"],
)
print(
    "Received status:",
    received_message["payload"]["status"],
)
print("-" * 65)
print("Object-reference isolation test: PASS")

SERIALIZED TRANSPORT TEST
-----------------------------------------------------------------
Packet ID       : 697e6b14-2e42-44a7-8ff8-30a30ba9bee9
Sender          : TERMINAL-QBA-001
Receiver        : SERVER-QBA-PRIMARY-001
Received value : 42
Received status: ORIGINAL
-----------------------------------------------------------------
Object-reference isolation test: PASS


In [26]:
#@title Snapshot creation for Debug

terminal_snapshot = terminal.safe_snapshot()
server_snapshot = server.safe_snapshot()

state_report = pd.DataFrame([
    {
        "Node": terminal_snapshot["node_id"],
        "Role": terminal_snapshot["role"],
        "Public Key Fingerprint" : terminal_snapshot["public_key_fingerprint"],
        "Public Key Size" : terminal_snapshot["public_key_size_bytes"],
        "Trusted Peers" : terminal_snapshot["trusted_peer_count"],
        "Active Sessions" : terminal_snapshot["active_session_count"],
        "Private Key Loaded" : terminal_snapshot["vault"]["private_key_loaded"],
        "Private Key Exportable" : terminal_snapshot["vault"]["private_key_exportable"],
    },
    {
        "Node": server_snapshot["node_id"],
        "Role": server_snapshot["role"],
        "Public Key Fingerprint" : server_snapshot["public_key_fingerprint"],
        "Public Key Size" : server_snapshot["public_key_size_bytes"],
        "Trusted Peers" : server_snapshot["trusted_peer_count"],
        "Active Sessions" : server_snapshot["active_session_count"],
        "Private Key Loaded" : server_snapshot["vault"]["private_key_loaded"],
        "Private Key Exportable" : server_snapshot["vault"]["private_key_exportable"],
    },
])

display(state_report)

print("No private-key bytes are included in the exported snapshots.")

,Node,Role,Public Key Fingerprint,Public Key Size,Trusted Peers,Active Sessions,Private Key Loaded,Private Key Exportable
0,TERMINAL-QBA-001,TRUSTED_TERMINAL,bde791397eb32adb7c0f24aee8762628c3d32d63aceac6...,1952,1,0,True,False
1,SERVER-QBA-PRIMARY-001,AUTHENTICATION_SERVER,4ca04fb83c255f1c4ad0d283bc052254f1248c3d337707...,1952,1,0,True,False


No private-key bytes are included in the exported snapshots.


## Phase 2 : 27-07-2026T12:42:05 - 27-07-2026T13:38:42

In [27]:
#@title Phase 2 Utilities for Dilithium Verification between Server and Terminal

class ProtocolValidationError(Exception):
    """Raised when a protocol message fails validation."""


class AuthenticationFailure(Exception):
    """Raised when mutual authentication cannot be completed."""


def parse_utc_timestamp(timestamp: str) -> datetime:
    """
    Parse an ISO-8601 timestamp and ensure that it is timezone-aware.
    """
    try:
        parsed = datetime.fromisoformat(timestamp)
    except (TypeError, ValueError) as exc:
        raise ProtocolValidationError(
            "Invalid ISO-8601 timestamp."
        ) from exc

    if parsed.tzinfo is None:
        raise ProtocolValidationError(
            "Timestamp must contain timezone information."
        )

    return parsed.astimezone(timezone.utc)


def validate_timestamp_freshness(
    timestamp: str,
    *,
    maximum_age_seconds: int = MESSAGE_FRESHNESS_SECONDS,
    maximum_future_skew_seconds: int = 5,
) -> None:
    """
    Reject stale messages and messages dated too far into the future.
    """
    issued_at = parse_utc_timestamp(timestamp)
    current_time = utc_now()

    age_seconds = (current_time - issued_at).total_seconds()

    if age_seconds > maximum_age_seconds:
        raise ProtocolValidationError(
            f"Message is stale by {age_seconds:.2f} seconds."
        )

    if age_seconds < -maximum_future_skew_seconds:
        raise ProtocolValidationError(
            "Message timestamp is too far in the future."
        )


def envelope_from_dictionary(
    data: Dict[str, Any],
) -> ProtocolEnvelope:
    """
    Reconstruct and validate a ProtocolEnvelope received over transport.
    """

    if not isinstance(data, dict):
        raise ProtocolValidationError(
            "Received protocol message must be a dictionary."
        )

    required_fields = {
        "message_type",
        "sender_id",
        "receiver_id",
        "session_id",
        "message_id",
        "issued_at",
        "payload",
        "signature_b64",
    }

    missing_fields = required_fields - set(data.keys())

    if missing_fields:
        raise ProtocolValidationError(
            f"Missing envelope fields: {sorted(missing_fields)}"
        )

    if not isinstance(data["payload"], dict):
        raise ProtocolValidationError(
            "Envelope payload must be a dictionary."
        )

    string_fields = [
        "message_type",
        "sender_id",
        "receiver_id",
        "session_id",
        "message_id",
        "issued_at",
    ]

    for field_name in string_fields:
        value = data[field_name]

        if not isinstance(value, str) or not value:
            raise ProtocolValidationError(
                f"{field_name} must be a non-empty string."
            )

    signature_b64 = data["signature_b64"]

    if not isinstance(signature_b64, str) or not signature_b64:
        raise ProtocolValidationError(
            "A non-empty Base64 signature is required."
        )

    try:
        b64_to_bytes(signature_b64)
    except Exception as exc:
        raise ProtocolValidationError(
            "Envelope signature is not valid Base64."
        ) from exc

    return ProtocolEnvelope(
        message_type=data["message_type"],
        sender_id=data["sender_id"],
        receiver_id=data["receiver_id"],
        session_id=data["session_id"],
        message_id=data["message_id"],
        issued_at=data["issued_at"],
        payload=deepcopy(data["payload"]),
        signature_b64=signature_b64,
    )


def sign_protocol_envelope(
    node: IsolatedNode,
    unsigned_envelope: ProtocolEnvelope,
) -> ProtocolEnvelope:
    """
    Sign the canonical unsigned representation of an envelope.
    """

    if unsigned_envelope.signature_b64 is not None:
        raise ValueError(
            "Envelope must be unsigned before signing."
        )

    signature = node.sign_bytes(
        unsigned_envelope.unsigned_bytes()
    )

    return ProtocolEnvelope(
        message_type=unsigned_envelope.message_type,
        sender_id=unsigned_envelope.sender_id,
        receiver_id=unsigned_envelope.receiver_id,
        session_id=unsigned_envelope.session_id,
        message_id=unsigned_envelope.message_id,
        issued_at=unsigned_envelope.issued_at,
        payload=deepcopy(unsigned_envelope.payload),
        signature_b64=bytes_to_b64(signature),
    )


def verify_received_envelope_signature(
    receiver: IsolatedNode,
    envelope: ProtocolEnvelope,
) -> bool:
    """
    Verify a received envelope using the sender's pinned public key.
    """

    if envelope.signature_b64 is None:
        return False

    try:
        signature = b64_to_bytes(envelope.signature_b64)
    except Exception:
        return False

    return receiver.verify_trusted_peer_signature(
        peer_id=envelope.sender_id,
        message=envelope.unsigned_bytes(),
        signature=signature,
    )


def local_session_exists(
    node: IsolatedNode,
    session_id: str,
) -> bool:
    try:
        node.get_local_session(session_id)
        return True
    except KeyError:
        return False


print("Phase 2 validation utilities ready.")

Phase 2 validation utilities ready.


In [28]:
#@title Mutual Authentication Structure

@dataclass(frozen=True)
class MutualAuthenticationResult:
    success: bool
    session_id: str
    terminal_id: str
    server_id: str
    terminal_authenticated_server: bool
    server_authenticated_terminal: bool
    request_hash: str
    response_hash: str
    transcript_hash: str
    completed_at: str
    failure_reason: Optional[str] = None

    def to_dictionary(self) -> Dict[str, Any]:
        return asdict(self)

print("Mutual-authentication result structure ready.")

Mutual-authentication result structure ready.


In [29]:
#@title Actual Mutual Authentication Protocol Engine

class MutualAuthenticationProtocol:
    """
    Implements a signed terminal-server challenge-response protocol.

    Message flow:

        TerminalAuthRequest
                ↓
        ML-DSA terminal signature
                ↓
        Server verification
                ↓
        ServerAuthResponse
                ↓
        ML-DSA server signature
                ↓
        Terminal verification
    """

    TERMINAL_REQUEST_TYPE = "TERMINAL_AUTH_REQUEST"
    SERVER_RESPONSE_TYPE = "SERVER_AUTH_RESPONSE"

    def __init__(
        self,
        terminal: TerminalNode,
        server: ServerNode,
        transport: InMemoryMessageTransport,
    ) -> None:

        self.terminal = terminal
        self.server = server
        self.transport = transport

    # TERMINAL : Create Request

    def terminal_create_authentication_request(
        self,
    ) -> ProtocolEnvelope:

        if not self.terminal.is_peer_trusted(
            self.server.node_id
        ):
            raise AuthenticationFailure(
                "Terminal does not trust the server public key."
            )

        session_id = generate_uuid()
        terminal_nonce_b64 = bytes_to_b64(
            generate_nonce(32)
        )

        self.terminal.create_local_session(
            session_id=session_id,
            initial_state={
                "protocol_phase": "MUTUAL_AUTHENTICATION",
                "authentication_state" : "AWAITING_SERVER_RESPONSE",
                "expected_server_id" : self.server.node_id,
                "terminal_nonce_b64" : terminal_nonce_b64,
                "terminal_authenticated_server" : False,
                "server_authenticated_terminal" : False,
            },
        )

        unsigned_request = ProtocolEnvelope(
            message_type=self.TERMINAL_REQUEST_TYPE,
            sender_id=self.terminal.node_id,
            receiver_id=self.server.node_id,
            session_id=session_id,
            message_id=generate_uuid(),
            issued_at=utc_timestamp(),
            payload={
                "protocol_name": PROTOCOL_NAME,
                "protocol_version": PROTOCOL_VERSION,
                "authentication_purpose":
                    "TERMINAL_SERVER_MUTUAL_AUTH",
                "terminal_nonce_b64":
                    terminal_nonce_b64,
                "requested_server_id":
                    self.server.node_id,
                "terminal_credential_version":
                    self.terminal.credential_version,
            },
            signature_b64=None,
        )

        signed_request = sign_protocol_envelope(
            self.terminal,
            unsigned_request,
        )

        request_hash = sha3_256_hex(
            signed_request.unsigned_bytes()
        )

        self.terminal.update_local_session(
            session_id,
            {
                "request_message_id" : signed_request.message_id,
                "request_hash" : request_hash,
                "signed_request_dictionary" : signed_request.to_dictionary(),
            },
        )

        return signed_request

    def terminal_send_authentication_request(
        self,
        request: ProtocolEnvelope,
    ) -> str:

        return self.transport.send(
            sender_id=self.terminal.node_id,
            receiver_id=self.server.node_id,
            message=request.to_dictionary(),
        )

    # SERVER : Verify Request

    def server_process_authentication_request(
        self,
    ) -> ProtocolEnvelope:

        received = self.transport.receive(
            self.server.node_id
        )

        if received is None:
            raise AuthenticationFailure(
                "Server has no pending authentication request."
            )

        packet, received_dictionary = received

        try:
            request = envelope_from_dictionary(
                received_dictionary
            )

            # Transport-level sender and receiver binding.
            if packet.sender_id != request.sender_id:
                raise ProtocolValidationError(
                    "Packet sender does not match envelope sender."
                )

            if packet.receiver_id != request.receiver_id:
                raise ProtocolValidationError(
                    "Packet receiver does not match envelope receiver."
                )

            if request.message_type != self.TERMINAL_REQUEST_TYPE:
                raise ProtocolValidationError(
                    "Unexpected authentication message type."
                )

            if request.sender_id != self.terminal.node_id:
                raise ProtocolValidationError(
                    "Authentication request came from an "
                    "unexpected terminal."
                )

            if request.receiver_id != self.server.node_id:
                raise ProtocolValidationError(
                    "Authentication request targets another server."
                )

            if not self.server.is_peer_trusted(
                request.sender_id
            ):
                raise ProtocolValidationError(
                    "Requesting terminal is not trusted."
                )

            validate_timestamp_freshness(
                request.issued_at
            )

            required_payload_fields = {
                "protocol_name",
                "protocol_version",
                "authentication_purpose",
                "terminal_nonce_b64",
                "requested_server_id",
                "terminal_credential_version",
            }

            missing_payload_fields = (
                required_payload_fields
                - set(request.payload.keys())
            )

            if missing_payload_fields:
                raise ProtocolValidationError(
                    "Missing request payload fields: "
                    f"{sorted(missing_payload_fields)}"
                )

            if (
                request.payload["protocol_name"]
                != PROTOCOL_NAME
            ):
                raise ProtocolValidationError(
                    "Unsupported protocol name."
                )

            if (
                request.payload["protocol_version"]
                != PROTOCOL_VERSION
            ):
                raise ProtocolValidationError(
                    "Unsupported protocol version."
                )

            if (
                request.payload["authentication_purpose"]
                != "TERMINAL_SERVER_MUTUAL_AUTH"
            ):
                raise ProtocolValidationError(
                    "Invalid authentication purpose."
                )

            if (
                request.payload["requested_server_id"]
                != self.server.node_id
            ):
                raise ProtocolValidationError(
                    "Payload server ID does not match "
                    "the receiving server."
                )

            terminal_nonce_b64 = request.payload[
                "terminal_nonce_b64"
            ]

            try:
                terminal_nonce = b64_to_bytes(
                    terminal_nonce_b64
                )
            except Exception as exc:
                raise ProtocolValidationError(
                    "Terminal nonce is not valid Base64."
                ) from exc

            if len(terminal_nonce) != 32:
                raise ProtocolValidationError(
                    "Terminal nonce must contain 32 bytes."
                )

            if self.server.has_seen_nonce(
                terminal_nonce_b64
            ):
                raise ProtocolValidationError(
                    "Replay detected: terminal nonce "
                    "has already been consumed."
                )

            if local_session_exists(
                self.server,
                request.session_id,
            ):
                raise ProtocolValidationError(
                    "Session ID has already been used."
                )

            signature_valid = (
                verify_received_envelope_signature(
                    self.server,
                    request,
                )
            )

            if not signature_valid:
                raise ProtocolValidationError(
                    "Terminal ML-DSA signature is invalid."
                )

            # Consume the nonce only after all checks and signature verification pass.
            self.server.consume_nonce(
                terminal_nonce_b64
            )

            request_hash = sha3_256_hex(
                request.unsigned_bytes()
            )

            server_nonce_b64 = bytes_to_b64(
                generate_nonce(32)
            )

            self.server.create_local_session(
                session_id=request.session_id,
                initial_state={
                    "protocol_phase" : "MUTUAL_AUTHENTICATION",
                    "authentication_state" : "TERMINAL_AUTHENTICATED",
                    "terminal_id" : request.sender_id,
                    "terminal_nonce_b64" : terminal_nonce_b64,
                    "server_nonce_b64" : server_nonce_b64,
                    "request_hash" : request_hash,
                    "request_message_id" : request.message_id,
                    "server_authenticated_terminal" : True,
                    "terminal_authenticated_server" : False,
                },
            )

            unsigned_response = ProtocolEnvelope(
                message_type=self.SERVER_RESPONSE_TYPE,
                sender_id=self.server.node_id,
                receiver_id=self.terminal.node_id,
                session_id=request.session_id,
                message_id=generate_uuid(),
                issued_at=utc_timestamp(),
                payload={
                    "protocol_name" : PROTOCOL_NAME,
                    "protocol_version" : PROTOCOL_VERSION,
                    "authentication_result" : "TERMINAL_ACCEPTED",
                    "server_nonce_b64" : server_nonce_b64,
                    "echoed_terminal_nonce_b64" : terminal_nonce_b64,
                    "request_hash" : request_hash,
                    "request_message_id" : request.message_id,
                    "server_credential_version" : self.server.credential_version,
                },
                signature_b64=None,
            )

            signed_response = sign_protocol_envelope(
                self.server,
                unsigned_response,
            )

            response_hash = sha3_256_hex(
                signed_response.unsigned_bytes()
            )

            self.server.update_local_session(
                request.session_id,
                {
                    "response_hash" : response_hash,
                    "response_message_id" : signed_response.message_id,
                    "signed_response_dictionary" : signed_response.to_dictionary(),
                    "authentication_state" : "AWAITING_TERMINAL_CONFIRMATION",
                },
            )

            return signed_response

        except Exception as exc:
            self.server._log_event(
                event_type=
                    "TERMINAL_AUTHENTICATION_REQUEST",
                result="FAILURE",
                details={
                    "reason": str(exc),
                    "packet_id": packet.packet_id,
                },
            )

            if isinstance(
                exc,
                (
                    ProtocolValidationError,
                    AuthenticationFailure,
                ),
            ):
                raise AuthenticationFailure(
                    str(exc)
                ) from exc

            raise AuthenticationFailure(
                f"Unexpected request-processing failure: {exc}"
            ) from exc

    def server_send_authentication_response(
        self,
        response: ProtocolEnvelope,
    ) -> str:

        return self.transport.send(
            sender_id=self.server.node_id,
            receiver_id=self.terminal.node_id,
            message=response.to_dictionary(),
        )

    # TERMINAL : Rcv and verify server resp

    def terminal_process_authentication_response(
        self,
    ) -> MutualAuthenticationResult:

        received = self.transport.receive(
            self.terminal.node_id
        )

        if received is None:
            raise AuthenticationFailure(
                "Terminal has no pending server response."
            )

        packet, received_dictionary = received

        response: Optional[ProtocolEnvelope] = None

        try:
            response = envelope_from_dictionary(
                received_dictionary
            )

            if packet.sender_id != response.sender_id:
                raise ProtocolValidationError(
                    "Packet sender does not match "
                    "response sender."
                )

            if packet.receiver_id != response.receiver_id:
                raise ProtocolValidationError(
                    "Packet receiver does not match "
                    "response receiver."
                )

            if response.message_type != self.SERVER_RESPONSE_TYPE:
                raise ProtocolValidationError(
                    "Unexpected server response type."
                )

            if response.sender_id != self.server.node_id:
                raise ProtocolValidationError(
                    "Response came from an unexpected server."
                )

            if response.receiver_id != self.terminal.node_id:
                raise ProtocolValidationError(
                    "Server response targets another terminal."
                )

            if not self.terminal.is_peer_trusted(
                response.sender_id
            ):
                raise ProtocolValidationError(
                    "Responding server is not trusted."
                )

            validate_timestamp_freshness(
                response.issued_at
            )

            if not local_session_exists(
                self.terminal,
                response.session_id,
            ):
                raise ProtocolValidationError(
                    "Response references an unknown session."
                )

            terminal_session = (
                self.terminal.get_local_session(
                    response.session_id
                )
            )

            if (
                terminal_session["authentication_state"]
                != "AWAITING_SERVER_RESPONSE"
            ):
                raise ProtocolValidationError(
                    "Terminal session is not expecting "
                    "a server response."
                )

            required_payload_fields = {
                "protocol_name",
                "protocol_version",
                "authentication_result",
                "server_nonce_b64",
                "echoed_terminal_nonce_b64",
                "request_hash",
                "request_message_id",
                "server_credential_version",
            }

            missing_payload_fields = (
                required_payload_fields
                - set(response.payload.keys())
            )

            if missing_payload_fields:
                raise ProtocolValidationError(
                    "Missing response payload fields: "
                    f"{sorted(missing_payload_fields)}"
                )

            if (
                response.payload["protocol_name"]
                != PROTOCOL_NAME
            ):
                raise ProtocolValidationError(
                    "Unexpected protocol name."
                )

            if (
                response.payload["protocol_version"]
                != PROTOCOL_VERSION
            ):
                raise ProtocolValidationError(
                    "Unexpected protocol version."
                )

            if (
                response.payload["authentication_result"]
                != "TERMINAL_ACCEPTED"
            ):
                raise ProtocolValidationError(
                    "Server did not accept the terminal."
                )

            if (
                response.payload[
                    "echoed_terminal_nonce_b64"
                ]
                != terminal_session[
                    "terminal_nonce_b64"
                ]
            ):
                raise ProtocolValidationError(
                    "Server did not correctly echo "
                    "the terminal nonce."
                )

            if (
                response.payload["request_hash"]
                != terminal_session["request_hash"]
            ):
                raise ProtocolValidationError(
                    "Server response is not bound to "
                    "the original terminal request."
                )

            if (
                response.payload["request_message_id"]
                != terminal_session[
                    "request_message_id"
                ]
            ):
                raise ProtocolValidationError(
                    "Server echoed the wrong request message ID."
                )

            server_nonce_b64 = response.payload[
                "server_nonce_b64"
            ]

            try:
                server_nonce = b64_to_bytes(
                    server_nonce_b64
                )
            except Exception as exc:
                raise ProtocolValidationError(
                    "Server nonce is not valid Base64."
                ) from exc

            if len(server_nonce) != 32:
                raise ProtocolValidationError(
                    "Server nonce must contain 32 bytes."
                )

            if self.terminal.has_seen_nonce(
                server_nonce_b64
            ):
                raise ProtocolValidationError(
                    "Replay detected: server nonce "
                    "has already been consumed."
                )

            signature_valid = (
                verify_received_envelope_signature(
                    self.terminal,
                    response,
                )
            )

            if not signature_valid:
                raise ProtocolValidationError(
                    "Server ML-DSA signature is invalid."
                )

            self.terminal.consume_nonce(
                server_nonce_b64
            )

            response_hash = sha3_256_hex(
                response.unsigned_bytes()
            )

            transcript_data = canonical_json({
                "protocol_name": PROTOCOL_NAME,
                "protocol_version": PROTOCOL_VERSION,
                "session_id": response.session_id,
                "terminal_id": self.terminal.node_id,
                "server_id": self.server.node_id,
                "request_hash":
                    terminal_session["request_hash"],
                "response_hash":
                    response_hash,
                "terminal_nonce_b64":
                    terminal_session[
                        "terminal_nonce_b64"
                    ],
                "server_nonce_b64":
                    server_nonce_b64,
            })

            transcript_hash = sha3_256_hex(
                transcript_data
            )

            self.terminal.update_local_session(
                response.session_id,
                {
                    "authentication_state" : "MUTUALLY_AUTHENTICATED",
                    "terminal_authenticated_server" : True,
                    "server_authenticated_terminal" : True,
                    "server_nonce_b64" : server_nonce_b64,
                    "response_hash" : response_hash,
                    "response_message_id" : response.message_id,
                    "transcript_hash" : transcript_hash,
                    "mutual_auth_completed_at" : utc_timestamp(),
                },
            )

            # The server had already authenticated the terminal.
            # Record successful completion of the same transcript.
            self.server.update_local_session(
                response.session_id,
                {
                    "authentication_state" : "MUTUALLY_AUTHENTICATED",
                    "terminal_authenticated_server" : True,
                    "server_authenticated_terminal" : True,
                    "transcript_hash" : transcript_hash,
                    "mutual_auth_completed_at" : utc_timestamp(),
                },
            )

            return MutualAuthenticationResult(
                success=True,
                session_id=response.session_id,
                terminal_id=self.terminal.node_id,
                server_id=self.server.node_id,
                terminal_authenticated_server=True,
                server_authenticated_terminal=True,
                request_hash=
                    terminal_session["request_hash"],
                response_hash=response_hash,
                transcript_hash=transcript_hash,
                completed_at=utc_timestamp(),
                failure_reason=None,
            )

        except Exception as exc:
            session_id = (
                response.session_id
                if response is not None
                else "UNKNOWN"
            )

            self.terminal._log_event(
                event_type=
                    "SERVER_AUTHENTICATION_RESPONSE",
                result="FAILURE",
                details={
                    "reason": str(exc),
                    "session_id": session_id,
                    "packet_id": packet.packet_id,
                },
            )

            if isinstance(
                exc,
                (
                    ProtocolValidationError,
                    AuthenticationFailure,
                ),
            ):
                raise AuthenticationFailure(
                    str(exc)
                ) from exc

            raise AuthenticationFailure(
                f"Unexpected response-processing failure: {exc}"
            ) from exc

    # Completing the mutual authentication request for one handshake

    def execute(self) -> MutualAuthenticationResult:
        """
        Execute the complete mutual-authentication exchange.
        """

        request = (
            self.terminal_create_authentication_request()
        )

        self.terminal_send_authentication_request(
            request
        )

        response = (
            self.server_process_authentication_request()
        )

        self.server_send_authentication_response(
            response
        )

        return (
            self.terminal_process_authentication_response()
        )


print("Mutual-authentication protocol engine ready.")

Mutual-authentication protocol engine ready.


In [30]:
# @title Live Authentication

mutual_auth_protocol = MutualAuthenticationProtocol(
    terminal=terminal,
    server=server,
    transport=network,
)

mutual_auth_result = mutual_auth_protocol.execute()

assert mutual_auth_result.success is True
assert (
    mutual_auth_result.terminal_authenticated_server
    is True
)
assert (
    mutual_auth_result.server_authenticated_terminal
    is True
)

print("=" * 72)
print("TERMINAL–SERVER MUTUAL AUTHENTICATION")
print("=" * 72)
print(f"Session ID        : {mutual_auth_result.session_id}")
print(f"Terminal          : {mutual_auth_result.terminal_id}")
print(f"Server            : {mutual_auth_result.server_id}")
print()
print(
    "Server authenticated Terminal :",
    mutual_auth_result.server_authenticated_terminal,
)
print(
    "Terminal authenticated Server :",
    mutual_auth_result.terminal_authenticated_server,
)
print()
print(
    "Request hash    :",
    mutual_auth_result.request_hash,
)
print(
    "Response hash   :",
    mutual_auth_result.response_hash,
)
print(
    "Transcript hash :",
    mutual_auth_result.transcript_hash,
)
print("=" * 72)
print("MUTUAL AUTHENTICATION: PASS")

TERMINAL–SERVER MUTUAL AUTHENTICATION
Session ID        : 3dbc43dc-b8da-4466-86d0-8a3fcdbdd0e5
Terminal          : TERMINAL-QBA-001
Server            : SERVER-QBA-PRIMARY-001

Server authenticated Terminal : True
Terminal authenticated Server : True

Request hash    : 3dc08f107e7fe699542180cab0f710a0423714883e6ddb692195cf912a68798d
Response hash   : 323df6752e86a3670a83754ae6cd7bbb35bb7d7f6cdcba17184c492e798aa19e
Transcript hash : c5d9f5e6465d2684da5e6358ee859d5d553c7e7b431d91b40411cdf124da7744
MUTUAL AUTHENTICATION: PASS


In [31]:
# @title Session state check

authenticated_session_id = (
    mutual_auth_result.session_id
)

terminal_auth_session = terminal.get_local_session(
    authenticated_session_id
)

server_auth_session = server.get_local_session(
    authenticated_session_id
)

session_comparison = pd.DataFrame([
    {
        "Property": "Session ID",
        "Terminal View":
            terminal_auth_session["session_id"],
        "Server View":
            server_auth_session["session_id"],
        "Match":
            terminal_auth_session["session_id"]
            == server_auth_session["session_id"],
    },
    {
        "Property": "Authentication State",
        "Terminal View":
            terminal_auth_session[
                "authentication_state"
            ],
        "Server View":
            server_auth_session[
                "authentication_state"
            ],
        "Match":
            terminal_auth_session[
                "authentication_state"
            ]
            == server_auth_session[
                "authentication_state"
            ],
    },
    {
        "Property": "Request Hash",
        "Terminal View":
            terminal_auth_session["request_hash"],
        "Server View":
            server_auth_session["request_hash"],
        "Match":
            terminal_auth_session["request_hash"]
            == server_auth_session["request_hash"],
    },
    {
        "Property": "Response Hash",
        "Terminal View":
            terminal_auth_session["response_hash"],
        "Server View":
            server_auth_session["response_hash"],
        "Match":
            terminal_auth_session["response_hash"]
            == server_auth_session["response_hash"],
    },
    {
        "Property": "Transcript Hash",
        "Terminal View":
            terminal_auth_session["transcript_hash"],
        "Server View":
            server_auth_session["transcript_hash"],
        "Match":
            terminal_auth_session["transcript_hash"]
            == server_auth_session["transcript_hash"],
    },
])

display(session_comparison)

assert session_comparison["Match"].all()

assert (
    terminal_auth_session["authentication_state"]
    == "MUTUALLY_AUTHENTICATED"
)

assert (
    server_auth_session["authentication_state"]
    == "MUTUALLY_AUTHENTICATED"
)

print("Terminal and Server agree on the authenticated transcript.")

,Property,Terminal View,Server View,Match
0,Session ID,3dbc43dc-b8da-4466-86d0-8a3fcdbdd0e5,3dbc43dc-b8da-4466-86d0-8a3fcdbdd0e5,True
1,Authentication State,MUTUALLY_AUTHENTICATED,MUTUALLY_AUTHENTICATED,True
2,Request Hash,3dc08f107e7fe699542180cab0f710a0423714883e6ddb...,3dc08f107e7fe699542180cab0f710a0423714883e6ddb...,True
3,Response Hash,323df6752e86a3670a83754ae6cd7bbb35bb7d7f6cdcba...,323df6752e86a3670a83754ae6cd7bbb35bb7d7f6cdcba...,True
4,Transcript Hash,c5d9f5e6465d2684da5e6358ee859d5d553c7e7b431d91...,c5d9f5e6465d2684da5e6358ee859d5d553c7e7b431d91...,True


Terminal and Server agree on the authenticated transcript.


In [32]:
#@title Negative tests

def create_phase_2_test_environment():
    test_terminal = TerminalNode(
        terminal_id=f"TEST-TERMINAL-{generate_uuid()}",
        terminal_location="Phase 2 Test Environment",
    )

    test_server = ServerNode(
        server_id=f"TEST-SERVER-{generate_uuid()}",
        service_name="Phase 2 Test Server",
    )

    test_network = InMemoryMessageTransport()

    test_network.register_endpoint(
        test_terminal.node_id
    )
    test_network.register_endpoint(
        test_server.node_id
    )

    test_terminal.register_trusted_peer(
        test_server.public_identity_card(),
        expected_role=NodeRole.SERVER,
    )

    test_server.register_trusted_peer(
        test_terminal.public_identity_card(),
        expected_role=NodeRole.TERMINAL,
    )

    test_protocol = MutualAuthenticationProtocol(
        terminal=test_terminal,
        server=test_server,
        transport=test_network,
    )

    return (
        test_terminal,
        test_server,
        test_network,
        test_protocol,
    )


print("Negative-test environment factory ready.")

Negative-test environment factory ready.


In [33]:
#@title Tampered Terminal Request Test

(
    tamper_terminal,
    tamper_server,
    tamper_network,
    tamper_protocol,
) = create_phase_2_test_environment()

valid_request = (
    tamper_protocol
    .terminal_create_authentication_request()
)

tampered_request_dictionary = (
    valid_request.to_dictionary()
)

# Modify signed data without generating a new signature.
tampered_request_dictionary["payload"][
    "protocol_version"
] = "ATTACKER-MODIFIED-VERSION"

tamper_network.send(
    sender_id=tamper_terminal.node_id,
    receiver_id=tamper_server.node_id,
    message=tampered_request_dictionary,
)

tampered_request_rejected = False
tampered_request_reason = ""

try:
    tamper_protocol.server_process_authentication_request()
except AuthenticationFailure as exc:
    tampered_request_rejected = True
    tampered_request_reason = str(exc)

assert tampered_request_rejected is True

print("TAMPERED REQUEST TEST")
print("-" * 70)
print("Rejected:", tampered_request_rejected)
print("Reason  :", tampered_request_reason)
print("-" * 70)
print("Tampered terminal request: REJECTED")

TAMPERED REQUEST TEST
----------------------------------------------------------------------
Rejected: True
Reason  : Unsupported protocol version.
----------------------------------------------------------------------
Tampered terminal request: REJECTED


In [34]:
#@title Replay attack Test

(
    replay_terminal,
    replay_server,
    replay_network,
    replay_protocol,
) = create_phase_2_test_environment()

replay_request = (
    replay_protocol
    .terminal_create_authentication_request()
)

# First delivery: valid.
replay_protocol.terminal_send_authentication_request(
    replay_request
)

replay_response = (
    replay_protocol
    .server_process_authentication_request()
)

# Do not need to complete the terminal side for this test.
# The server has already consumed the terminal nonce.

# Replay the exact same signed request.
replay_network.send(
    sender_id=replay_terminal.node_id,
    receiver_id=replay_server.node_id,
    message=replay_request.to_dictionary(),
)

replay_rejected = False
replay_reason = ""

try:
    replay_protocol.server_process_authentication_request()
except AuthenticationFailure as exc:
    replay_rejected = True
    replay_reason = str(exc)

assert replay_rejected is True

print("REPLAY ATTACK TEST")
print("-" * 70)
print("First request accepted : True")
print("Replay rejected        :", replay_rejected)
print("Reason                 :", replay_reason)
print("-" * 70)
print("Replayed request: REJECTED")

REPLAY ATTACK TEST
----------------------------------------------------------------------
First request accepted : True
Replay rejected        : True
Reason                 : Replay detected: terminal nonce has already been consumed.
----------------------------------------------------------------------
Replayed request: REJECTED


In [35]:
# @title Tampered Server Reuqest Test

(
    response_terminal,
    response_server,
    response_network,
    response_protocol,
) = create_phase_2_test_environment()

response_request = (
    response_protocol
    .terminal_create_authentication_request()
)

response_protocol.terminal_send_authentication_request(
    response_request
)

valid_server_response = (
    response_protocol
    .server_process_authentication_request()
)

tampered_response_dictionary = (
    valid_server_response.to_dictionary()
)

# Replace the request binding while retaining the old signature.
tampered_response_dictionary["payload"][
    "request_hash"
] = "0" * 64

response_network.send(
    sender_id=response_server.node_id,
    receiver_id=response_terminal.node_id,
    message=tampered_response_dictionary,
)

tampered_response_rejected = False
tampered_response_reason = ""

try:
    response_protocol.terminal_process_authentication_response()
except AuthenticationFailure as exc:
    tampered_response_rejected = True
    tampered_response_reason = str(exc)

assert tampered_response_rejected is True

print("TAMPERED SERVER RESPONSE TEST")
print("-" * 70)
print("Rejected:", tampered_response_rejected)
print("Reason  :", tampered_response_reason)
print("-" * 70)
print("Tampered server response: REJECTED")

TAMPERED SERVER RESPONSE TEST
----------------------------------------------------------------------
Rejected: True
Reason  : Server response is not bound to the original terminal request.
----------------------------------------------------------------------
Tampered server response: REJECTED


In [36]:
#@title Stale message test

(
    stale_terminal,
    stale_server,
    stale_network,
    stale_protocol,
) = create_phase_2_test_environment()

fresh_request = (
    stale_protocol
    .terminal_create_authentication_request()
)

stale_timestamp = (
    utc_now()
    - timedelta(
        seconds=MESSAGE_FRESHNESS_SECONDS + 30
    )
).isoformat(timespec="microseconds")

# Build a correctly signed but intentionally stale request.
stale_unsigned_request = ProtocolEnvelope(
    message_type=fresh_request.message_type,
    sender_id=fresh_request.sender_id,
    receiver_id=fresh_request.receiver_id,
    session_id=fresh_request.session_id,
    message_id=fresh_request.message_id,
    issued_at=stale_timestamp,
    payload=deepcopy(fresh_request.payload),
    signature_b64=None,
)

stale_signed_request = sign_protocol_envelope(
    stale_terminal,
    stale_unsigned_request,
)

stale_network.send(
    sender_id=stale_terminal.node_id,
    receiver_id=stale_server.node_id,
    message=stale_signed_request.to_dictionary(),
)

stale_request_rejected = False
stale_request_reason = ""

try:
    stale_protocol.server_process_authentication_request()
except AuthenticationFailure as exc:
    stale_request_rejected = True
    stale_request_reason = str(exc)

assert stale_request_rejected is True

print("STALE MESSAGE TEST")
print("-" * 70)
print("Rejected:", stale_request_rejected)
print("Reason  :", stale_request_reason)
print("-" * 70)
print("Correctly signed but stale request: REJECTED")

STALE MESSAGE TEST
----------------------------------------------------------------------
Rejected: True
Reason  : Message is stale by 90.00 seconds.
----------------------------------------------------------------------
Correctly signed but stale request: REJECTED


In [37]:
#@title Phase 2 Final Test

terminal_final_session = terminal.get_local_session(
    mutual_auth_result.session_id
)

server_final_session = server.get_local_session(
    mutual_auth_result.session_id
)

phase_2_checks = {
    "Terminal request signed with ML-DSA":
        bool(mutual_auth_result.request_hash),

    "Server verified Terminal identity":
        mutual_auth_result
        .server_authenticated_terminal,

    "Server response signed with ML-DSA":
        bool(mutual_auth_result.response_hash),

    "Terminal verified Server identity":
        mutual_auth_result
        .terminal_authenticated_server,

    "Terminal nonce echoed correctly":
        (
            terminal_final_session[
                "terminal_nonce_b64"
            ]
            == server_final_session[
                "terminal_nonce_b64"
            ]
        ),

    "Request-response binding established":
        (
            terminal_final_session["request_hash"]
            == server_final_session["request_hash"]
        ),

    "Transcript hashes agree":
        (
            terminal_final_session[
                "transcript_hash"
            ]
            == server_final_session[
                "transcript_hash"
            ]
        ),

    "Terminal session mutually authenticated":
        (
            terminal_final_session[
                "authentication_state"
            ]
            == "MUTUALLY_AUTHENTICATED"
        ),

    "Server session mutually authenticated":
        (
            server_final_session[
                "authentication_state"
            ]
            == "MUTUALLY_AUTHENTICATED"
        ),

    "Tampered terminal request rejected":
        tampered_request_rejected,

    "Replayed request rejected":
        replay_rejected,

    "Tampered server response rejected":
        tampered_response_rejected,

    "Stale signed request rejected":
        stale_request_rejected,
}

phase_2_report = pd.DataFrame([
    {
        "Security Property": property_name,
        "Status": "PASS" if result else "FAIL",
    }
    for property_name, result
    in phase_2_checks.items()
])

display(phase_2_report)

assert all(phase_2_checks.values())

print("\n" + "=" * 74)
print("PHASE 2 COMPLETED SUCCESSFULLY")
print("=" * 74)
print("The Server has verified:")
print("  • Terminal identity")
print("  • Terminal ML-DSA private-key possession")
print("  • Request freshness")
print("  • Terminal nonce uniqueness")
print()
print("The Terminal has verified:")
print("  • Server identity")
print("  • Server ML-DSA private-key possession")
print("  • Response freshness")
print("  • Server nonce uniqueness")
print("  • Binding to the original request")
print()
print("Both nodes agree on:")
print("  • Session ID")
print("  • Request hash")
print("  • Response hash")
print("  • Mutual-authentication transcript hash")
print()
print("Terminal–Server mutual authentication is established.")

,Security Property,Status
0,Terminal request signed with ML-DSA,PASS
1,Server verified Terminal identity,PASS
2,Server response signed with ML-DSA,PASS
3,Terminal verified Server identity,PASS
4,Terminal nonce echoed correctly,PASS
5,Request-response binding established,PASS
6,Transcript hashes agree,PASS
7,Terminal session mutually authenticated,PASS
8,Server session mutually authenticated,PASS
9,Tampered terminal request rejected,PASS



PHASE 2 COMPLETED SUCCESSFULLY
The Server has verified:
  • Terminal identity
  • Terminal ML-DSA private-key possession
  • Request freshness
  • Terminal nonce uniqueness

The Terminal has verified:
  • Server identity
  • Server ML-DSA private-key possession
  • Response freshness
  • Server nonce uniqueness
  • Binding to the original request

Both nodes agree on:
  • Session ID
  • Request hash
  • Response hash
  • Mutual-authentication transcript hash

Terminal–Server mutual authentication is established.


## Phase 3A : 27-07-2026T13:42:26 - 28-07-2026T02:43:27

Phase 3A exists to enroll preople in the system. Run the second last cell as many times as the number of users. Currently, the system creates synthetic iris codes.

In [38]:
#@title Phase 3 Imports and Config

from pathlib import Path

from reedsolo import RSCodec, ReedSolomonError

# Existing iris length from Phase 0:
assert IRIS_CODE_LENGTH == 2048

IRIS_CODE_LENGTH_BYTES = IRIS_CODE_LENGTH // 8

# The 256-byte iris is divided into two 128-byte blocks.
FUZZY_BLOCK_COUNT = 2

# Each block contains:
#   64 bytes of random secret
#   64 Reed–Solomon parity bytes
#
# Final encoded block length = 128 bytes.
FUZZY_SECRET_BYTES_PER_BLOCK = 64
FUZZY_PARITY_BYTES_PER_BLOCK = 64
FUZZY_CODEWORD_BYTES_PER_BLOCK = (
    FUZZY_SECRET_BYTES_PER_BLOCK
    + FUZZY_PARITY_BYTES_PER_BLOCK
)

FUZZY_SECRET_LENGTH_BYTES = (
    FUZZY_SECRET_BYTES_PER_BLOCK
    * FUZZY_BLOCK_COUNT
)

FUZZY_HELPER_LENGTH_BYTES = IRIS_CODE_LENGTH_BYTES

# A Reed–Solomon code with 64 parity symbols can correct at most 32 erroneous byte symbols in each block.
FUZZY_MAX_SYMBOL_ERRORS_PER_BLOCK = (
    FUZZY_PARITY_BYTES_PER_BLOCK // 2
)

# Rotation compensation is expressed in iris-code bits.
DEFAULT_MAX_ROTATION_BITS = 8

BIOMETRIC_CREDENTIAL_VERSION = 1

# Exported synthetic test codes.
SYNTHETIC_IRIS_EXPORT_PATH = Path(
    "/content/synthetic_iris_codes.json"
)

print("Phase 3 biometric configuration loaded.")
print("-" * 65)
print(f"Iris-code length              : {IRIS_CODE_LENGTH} bits")
print(f"Iris-code storage             : {IRIS_CODE_LENGTH_BYTES} bytes")
print(f"Fuzzy secret length           : {FUZZY_SECRET_LENGTH_BYTES} bytes")
print(f"Helper-data length            : {FUZZY_HELPER_LENGTH_BYTES} bytes")
print(
    "Maximum RS symbol errors/block:",
    FUZZY_MAX_SYMBOL_ERRORS_PER_BLOCK,
)

Phase 3 biometric configuration loaded.
-----------------------------------------------------------------
Iris-code length              : 2048 bits
Iris-code storage             : 256 bytes
Fuzzy secret length           : 128 bytes
Helper-data length            : 256 bytes
Maximum RS symbol errors/block: 32


In [39]:
#@title Iris Code Validation

def validate_binary_iris_string(
    iris_code: str,
    *,
    expected_length: int = IRIS_CODE_LENGTH,
) -> str:
    """
    Validate and normalise a binary iris-code string.
    """

    if not isinstance(iris_code, str):
        raise TypeError("Iris code must be supplied as a string.")

    cleaned = "".join(iris_code.split())

    if len(cleaned) != expected_length:
        raise ValueError(
            f"Iris code must contain exactly {expected_length} bits; "
            f"received {len(cleaned)}."
        )

    if any(character not in {"0", "1"} for character in cleaned):
        raise ValueError(
            "Iris code may contain only binary values 0 and 1."
        )

    return cleaned


def iris_string_to_bits(iris_code: str) -> np.ndarray:
    """
    Convert a validated iris string into a NumPy bit array.
    """

    cleaned = validate_binary_iris_string(iris_code)

    return np.fromiter(
        (int(character) for character in cleaned),
        dtype=np.uint8,
        count=IRIS_CODE_LENGTH,
    )


def iris_bits_to_string(bits: np.ndarray) -> str:
    """
    Convert a 2048-bit NumPy array to a copy-ready string.
    """

    bits = np.asarray(bits, dtype=np.uint8).reshape(-1)

    if len(bits) != IRIS_CODE_LENGTH:
        raise ValueError(
            f"Expected {IRIS_CODE_LENGTH} bits; received {len(bits)}."
        )

    if not np.all(np.isin(bits, [0, 1])):
        raise ValueError("Iris array contains non-binary values.")

    return "".join(str(int(bit)) for bit in bits)


def iris_bits_to_bytes(bits: np.ndarray) -> bytes:
    """
    Pack a 2048-bit array into exactly 256 bytes.
    """

    bits = np.asarray(bits, dtype=np.uint8).reshape(-1)

    if len(bits) != IRIS_CODE_LENGTH:
        raise ValueError(
            f"Expected {IRIS_CODE_LENGTH} iris bits."
        )

    if not np.all(np.isin(bits, [0, 1])):
        raise ValueError("Iris array contains non-binary values.")

    packed = np.packbits(bits, bitorder="big").tobytes()

    if len(packed) != IRIS_CODE_LENGTH_BYTES:
        raise RuntimeError("Unexpected packed iris-code length.")

    return packed


def iris_bytes_to_bits(data: bytes) -> np.ndarray:
    """
    Unpack exactly 256 bytes into a 2048-bit iris array.
    """

    if len(data) != IRIS_CODE_LENGTH_BYTES:
        raise ValueError(
            f"Expected {IRIS_CODE_LENGTH_BYTES} iris bytes."
        )

    return np.unpackbits(
        np.frombuffer(data, dtype=np.uint8),
        bitorder="big",
    ).astype(np.uint8)


def iris_string_to_hex(iris_code: str) -> str:
    """
    Convert a binary iris code to hexadecimal.
    """

    bits = iris_string_to_bits(iris_code)
    return iris_bits_to_bytes(bits).hex()


def iris_code_fingerprint_from_string(
    iris_code: str,
) -> str:
    """
    Compute a SHA3-256 fingerprint of a binary iris code.
    """

    bits = iris_string_to_bits(iris_code)
    return sha3_256_hex(iris_bits_to_bytes(bits))


print("Iris conversion utilities ready.")

Iris conversion utilities ready.


In [40]:
#@title Synthetic Iris Factory for User Profiling. Part of Demo, will be replaced by real iris scan from images later on.

@dataclass(frozen=True)
class SyntheticIrisProfile:
    profile_id: str
    canonical_iris_code: str
    iris_code_hex: str
    iris_fingerprint: str
    created_at: str
    code_length_bits: int

    def safe_summary(self) -> Dict[str, Any]:
        return {
            "profile_id": self.profile_id,
            "iris_code_preview":
                f"{self.canonical_iris_code[:32]}..."
                f"{self.canonical_iris_code[-32:]}",
            "iris_code_hex_preview":
                f"{self.iris_code_hex[:24]}..."
                f"{self.iris_code_hex[-24:]}",
            "iris_fingerprint": self.iris_fingerprint,
            "created_at": self.created_at,
            "code_length_bits": self.code_length_bits,
        }


class SyntheticIrisFactory:
    """
    Generates canonical synthetic iris codes.

    The generated 2048 bits come from cryptographically secure
    operating-system randomness.
    """

    @staticmethod
    def create_profile() -> SyntheticIrisProfile:
        raw_iris_bytes = secrets.token_bytes(
            IRIS_CODE_LENGTH_BYTES
        )

        iris_bits = iris_bytes_to_bits(raw_iris_bytes)
        iris_code = iris_bits_to_string(iris_bits)

        return SyntheticIrisProfile(
            profile_id=f"IRIS-{generate_uuid()}",
            canonical_iris_code=iris_code,
            iris_code_hex=raw_iris_bytes.hex(),
            iris_fingerprint=sha3_256_hex(raw_iris_bytes),
            created_at=utc_timestamp(),
            code_length_bits=IRIS_CODE_LENGTH,
        )


# This registry exists only for the demonstration notebook.
#
# In Phase 4, raw biometric codes will not be exported this way.
SYNTHETIC_IRIS_DEMO_REGISTRY: Dict[
    str,
    SyntheticIrisProfile,
] = {}


print("Synthetic iris factory ready.")

Synthetic iris factory ready.


In [41]:
#@title Phase 3 Specialised Data Structures for Iris Scanning

@dataclass(frozen=True)
class PublicBiometricHelperPackage:
    """
    Public fuzzy-commitment information.

    It does not contain:
        - the canonical iris code,
        - the fuzzy secret,
        - the ML-DSA private key,
        - the key-encryption key.
    """

    scheme: str
    scheme_version: int
    helper_data_b64: str
    fuzzy_salt_b64: str
    secret_commitment_hex: str
    iris_code_length_bits: int
    block_count: int
    secret_bytes_per_block: int
    parity_bytes_per_block: int
    maximum_rotation_bits: int
    created_at: str

    def helper_data_bytes(self) -> bytes:
        return b64_to_bytes(self.helper_data_b64)

    def fuzzy_salt_bytes(self) -> bytes:
        return b64_to_bytes(self.fuzzy_salt_b64)

    def to_dictionary(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass(frozen=True)
class ProtectedUserCredential:
    """
    Terminal-side protected user credential.
    """

    user_id: str
    user_name: str
    user_email: str
    user_public_key_b64: str
    public_key_fingerprint: str
    encrypted_private_key: AESGCMEnvelope
    credential_version: int
    biometric_scheme: str
    created_at: str
    account_status: str

    def public_key_bytes(self) -> bytes:
        return b64_to_bytes(self.user_public_key_b64)


@dataclass(frozen=True)
class UserEnrollmentResult:
    user_id: str
    user_name: str
    user_email: str
    iris_profile_id: str
    raw_iris_code: str
    raw_iris_hex: str
    iris_fingerprint: str
    public_key_fingerprint: str
    helper_data_fingerprint: str
    enrollment_timestamp: str


@dataclass(frozen=True)
class FuzzyReconstructionResult:
    success: bool
    recovered_secret: Optional[bytes]
    selected_rotation_bits: Optional[int]
    failure_reason: Optional[str]
    attempts: int


print("Phase 3 data structures ready.")

Phase 3 data structures ready.


In [42]:
#@title Reed Solomon Utilities

def create_rs_codec() -> RSCodec:
    """
    Create a fresh codec instance.

    A new instance is used for each encoding/decoding operation
    to avoid accidental mutable decoder state.
    """

    return RSCodec(FUZZY_PARITY_BYTES_PER_BLOCK)


def rs_encode_secret(secret: bytes) -> bytes:
    """
    Encode a 128-byte random secret into a 256-byte codeword.

    Layout:
        secret block 1: 64 bytes -> codeword block 1: 128 bytes
        secret block 2: 64 bytes -> codeword block 2: 128 bytes
    """

    if len(secret) != FUZZY_SECRET_LENGTH_BYTES:
        raise ValueError(
            f"Fuzzy secret must contain exactly "
            f"{FUZZY_SECRET_LENGTH_BYTES} bytes."
        )

    encoded_blocks: List[bytes] = []

    for block_index in range(FUZZY_BLOCK_COUNT):
        start = (
            block_index
            * FUZZY_SECRET_BYTES_PER_BLOCK
        )
        end = start + FUZZY_SECRET_BYTES_PER_BLOCK

        secret_block = secret[start:end]

        codec = create_rs_codec()
        encoded_block = bytes(codec.encode(secret_block))

        if len(encoded_block) != FUZZY_CODEWORD_BYTES_PER_BLOCK:
            raise RuntimeError(
                "Unexpected Reed–Solomon codeword length."
            )

        encoded_blocks.append(encoded_block)

    complete_codeword = b"".join(encoded_blocks)

    if len(complete_codeword) != IRIS_CODE_LENGTH_BYTES:
        raise RuntimeError(
            "Encoded codeword does not match iris-code length."
        )

    return complete_codeword


def _extract_decoded_message(
    decoder_output: Any,
) -> bytes:
    """
    Handle reedsolo versions that return either decoded data
    directly or a tuple whose first element is decoded data.
    """

    if isinstance(decoder_output, tuple):
        if not decoder_output:
            raise ReedSolomonError(
                "Decoder returned an empty tuple."
            )

        decoded_message = decoder_output[0]
    else:
        decoded_message = decoder_output

    return bytes(decoded_message)


def rs_decode_codeword(codeword: bytes) -> bytes:
    """
    Decode a potentially corrupted 256-byte codeword and
    reconstruct the original 128-byte random secret.

    Raises ReedSolomonError when corruption exceeds decoder
    capability.
    """

    if len(codeword) != IRIS_CODE_LENGTH_BYTES:
        raise ValueError(
            f"Codeword must contain exactly "
            f"{IRIS_CODE_LENGTH_BYTES} bytes."
        )

    recovered_blocks: List[bytes] = []

    for block_index in range(FUZZY_BLOCK_COUNT):
        start = (
            block_index
            * FUZZY_CODEWORD_BYTES_PER_BLOCK
        )
        end = start + FUZZY_CODEWORD_BYTES_PER_BLOCK

        encoded_block = bytearray(codeword[start:end])

        codec = create_rs_codec()
        decoder_output = codec.decode(encoded_block)

        decoded_block = _extract_decoded_message(
            decoder_output
        )

        if len(decoded_block) != FUZZY_SECRET_BYTES_PER_BLOCK:
            raise ReedSolomonError(
                "Decoder returned an unexpected secret length."
            )

        recovered_blocks.append(decoded_block)

    recovered_secret = b"".join(recovered_blocks)

    if len(recovered_secret) != FUZZY_SECRET_LENGTH_BYTES:
        raise RuntimeError(
            "Recovered fuzzy secret has an invalid length."
        )

    return recovered_secret


print("Reed–Solomon fuzzy-code utilities ready.")

Reed–Solomon fuzzy-code utilities ready.


In [43]:
#@title Fuzzy Commitment

FUZZY_COMMITMENT_DOMAIN = (
    b"QBioAuth|FuzzyCommitment|SecretVerification|v1"
)

BIOMETRIC_WRAP_DOMAIN = (
    b"QBioAuth|BiometricCredentialWrapping|ML-DSA-65|v1"
)


def xor_equal_length(
    left: bytes,
    right: bytes,
) -> bytes:
    """
    XOR two byte strings of equal length.
    """

    if len(left) != len(right):
        raise ValueError(
            "XOR operands must have equal lengths."
        )

    return bytes(
        left_byte ^ right_byte
        for left_byte, right_byte
        in zip(left, right)
    )


def calculate_secret_commitment(
    fuzzy_secret: bytes,
    fuzzy_salt: bytes,
    user_id: str,
) -> str:
    """
    Calculate a domain-separated commitment used to verify that
    reconstruction produced the correct secret.
    """

    commitment_input = (
        FUZZY_COMMITMENT_DOMAIN
        + b"|"
        + user_id.encode("utf-8")
        + b"|"
        + fuzzy_salt
        + b"|"
        + fuzzy_secret
    )

    return sha3_256_hex(commitment_input)


def derive_biometric_wrapping_key(
    fuzzy_secret: bytes,
    fuzzy_salt: bytes,
    user_id: str,
    credential_version: int,
) -> bytes:
    """
    Derive the AES-256 key that protects the user's ML-DSA
    private key.
    """

    hkdf_info = (
        BIOMETRIC_WRAP_DOMAIN
        + b"|"
        + user_id.encode("utf-8")
        + b"|VERSION="
        + str(credential_version).encode("ascii")
    )

    return derive_key_hkdf(
        fuzzy_secret,
        salt=fuzzy_salt,
        info=hkdf_info,
        length=AES_KEY_LENGTH_BYTES,
    )


def create_fuzzy_commitment(
    canonical_iris_code: str,
    user_id: str,
) -> Tuple[PublicBiometricHelperPackage, bytes]:
    """
    Enrol an iris code into a fuzzy commitment.

    Returns:
        public helper package,
        random fuzzy secret

    The caller uses the secret temporarily to protect the
    user's ML-DSA private key.
    """

    canonical_iris_code = validate_binary_iris_string(
        canonical_iris_code
    )

    iris_bits = iris_string_to_bits(
        canonical_iris_code
    )
    iris_bytes = iris_bits_to_bytes(iris_bits)

    # The secret is independently random. It is not an iris hash.
    fuzzy_secret = secrets.token_bytes(
        FUZZY_SECRET_LENGTH_BYTES
    )

    codeword = rs_encode_secret(fuzzy_secret)

    # Public fuzzy-commitment helper.
    helper_data = xor_equal_length(
        iris_bytes,
        codeword,
    )

    fuzzy_salt = secrets.token_bytes(32)

    secret_commitment = calculate_secret_commitment(
        fuzzy_secret=fuzzy_secret,
        fuzzy_salt=fuzzy_salt,
        user_id=user_id,
    )

    package = PublicBiometricHelperPackage(
        scheme="RS-FUZZY-COMMITMENT",
        scheme_version=1,
        helper_data_b64=bytes_to_b64(helper_data),
        fuzzy_salt_b64=bytes_to_b64(fuzzy_salt),
        secret_commitment_hex=secret_commitment,
        iris_code_length_bits=IRIS_CODE_LENGTH,
        block_count=FUZZY_BLOCK_COUNT,
        secret_bytes_per_block=
            FUZZY_SECRET_BYTES_PER_BLOCK,
        parity_bytes_per_block=
            FUZZY_PARITY_BYTES_PER_BLOCK,
        maximum_rotation_bits=
            DEFAULT_MAX_ROTATION_BITS,
        created_at=utc_timestamp(),
    )

    return package, fuzzy_secret


print("Fuzzy-commitment enrollment logic ready.")

Fuzzy-commitment enrollment logic ready.


In [44]:
#@title Fuzzy Secret Reconstruction to allow slightly noisy iris samples

def rotate_iris_bits(
    iris_bits: np.ndarray,
    shift_bits: int,
) -> np.ndarray:
    """
    Circularly rotate an iris code.

    This models and compensates for small eye/head rotation.
    """

    iris_bits = np.asarray(
        iris_bits,
        dtype=np.uint8,
    ).reshape(-1)

    if len(iris_bits) != IRIS_CODE_LENGTH:
        raise ValueError(
            f"Expected {IRIS_CODE_LENGTH} iris bits. Got {iris_bits} bits."
        )

    return np.roll(iris_bits, shift_bits)


def reconstruction_shift_order(
    maximum_rotation_bits: int,
) -> List[int]:
    """
    Try zero rotation first, then alternating positive and
    negative shifts.
    """

    shifts = [0]

    for magnitude in range(
        1,
        maximum_rotation_bits + 1,
    ):
        shifts.extend([magnitude, -magnitude])

    return shifts


def reconstruct_fuzzy_secret(
    candidate_iris_code: str,
    helper_package: PublicBiometricHelperPackage,
    user_id: str,
) -> FuzzyReconstructionResult:
    """
    Recover the random fuzzy secret from a nearby iris code.

    Success requires:
        1. Reed–Solomon decoding succeeds.
        2. Recovered secret matches the stored commitment.
    """

    try:
        candidate_iris_code = validate_binary_iris_string(
            candidate_iris_code
        )
    except Exception as exc:
        return FuzzyReconstructionResult(
            success=False,
            recovered_secret=None,
            selected_rotation_bits=None,
            failure_reason=str(exc),
            attempts=0,
        )

    if (
        helper_package.iris_code_length_bits
        != IRIS_CODE_LENGTH
    ):
        return FuzzyReconstructionResult(
            success=False,
            recovered_secret=None,
            selected_rotation_bits=None,
            failure_reason=(
                "Helper package has an unsupported "
                "iris-code length."
            ),
            attempts=0,
        )

    helper_data = helper_package.helper_data_bytes()
    fuzzy_salt = helper_package.fuzzy_salt_bytes()

    if len(helper_data) != IRIS_CODE_LENGTH_BYTES:
        return FuzzyReconstructionResult(
            success=False,
            recovered_secret=None,
            selected_rotation_bits=None,
            failure_reason="Helper data has an invalid length.",
            attempts=0,
        )

    candidate_bits = iris_string_to_bits(
        candidate_iris_code
    )

    attempts = 0
    last_error = "No valid reconstruction candidate."

    for shift in reconstruction_shift_order(
        helper_package.maximum_rotation_bits
    ):
        attempts += 1

        aligned_bits = rotate_iris_bits(
            candidate_bits,
            shift,
        )

        aligned_bytes = iris_bits_to_bytes(
            aligned_bits
        )

        noisy_codeword = xor_equal_length(
            aligned_bytes,
            helper_data,
        )

        try:
            recovered_secret = rs_decode_codeword(
                noisy_codeword
            )

        except ReedSolomonError as exc:
            last_error = (
                f"Reed–Solomon decoding failed at "
                f"rotation {shift}: {exc}"
            )
            continue

        except Exception as exc:
            last_error = (
                f"Decoder failure at rotation {shift}: {exc}"
            )
            continue

        calculated_commitment = (
            calculate_secret_commitment(
                fuzzy_secret=recovered_secret,
                fuzzy_salt=fuzzy_salt,
                user_id=user_id,
            )
        )

        commitment_valid = hmac.compare_digest(
            calculated_commitment,
            helper_package.secret_commitment_hex,
        )

        if commitment_valid:
            return FuzzyReconstructionResult(
                success=True,
                recovered_secret=recovered_secret,
                selected_rotation_bits=shift,
                failure_reason=None,
                attempts=attempts,
            )

        last_error = (
            f"Decoded candidate at rotation {shift}, "
            "but the secret commitment did not match."
        )

    return FuzzyReconstructionResult(
        success=False,
        recovered_secret=None,
        selected_rotation_bits=None,
        failure_reason=last_error,
        attempts=attempts,
    )


print("Fuzzy reconstruction logic ready.")

Fuzzy reconstruction logic ready.


In [45]:
#@title User Record to Identify and Enroll Users. Email is the UUID to identify people.

def normalise_email(email: str) -> str:
    """
    Perform basic normalisation for the prototype.
    """

    if not isinstance(email, str):
        raise TypeError("Email must be a string.")

    normalised = email.strip().lower()

    if not normalised:
        raise ValueError("Email cannot be empty.")

    if "@" not in normalised:
        raise ValueError(
            "The unique identifier must resemble an email address."
        )

    return normalised


def create_user_id_from_email(email: str) -> str:
    """
    Produce a privacy-preserving internal identifier from email.

    The email remains available in the prototype record, while the
    protocol-facing ID is a stable hash-based identifier.
    """

    normalised = normalise_email(email)

    digest = sha3_256_hex(
        b"QBioAuth|UserID|"
        + normalised.encode("utf-8")
    )

    return f"USR-{digest[:24].upper()}"


def terminal_store_protected_credential(
    terminal_node: TerminalNode,
    credential: ProtectedUserCredential,
) -> None:
    """
    Store the protected private-key record at the terminal.
    """

    if credential.user_id in terminal_node._local_user_records:
        raise ValueError(
            f"Terminal already contains user "
            f"{credential.user_id}."
        )

    terminal_node._local_user_records[
        credential.user_id
    ] = {
        "user_id": credential.user_id,
        "user_name": credential.user_name,
        "user_email": credential.user_email,
        "user_public_key_b64":
            credential.user_public_key_b64,
        "public_key_fingerprint":
            credential.public_key_fingerprint,
        "encrypted_private_key":
            asdict(credential.encrypted_private_key),
        "credential_version":
            credential.credential_version,
        "biometric_scheme":
            credential.biometric_scheme,
        "created_at":
            credential.created_at,
        "account_status":
            credential.account_status,
    }


def server_store_user_registration(
    server_node: ServerNode,
    *,
    user_id: str,
    user_name: str,
    user_email: str,
    user_public_key: bytes,
    helper_package: PublicBiometricHelperPackage,
    credential_version: int,
) -> None:
    """
    Store the server-side public registration record.
    """

    if user_id in server_node._registered_users:
        raise ValueError(
            f"Server already contains user {user_id}."
        )

    server_node._registered_users[user_id] = {
        "user_id": user_id,
        "user_name": user_name,
        "user_email": user_email,
        "user_public_key_b64":
            bytes_to_b64(user_public_key),
        "public_key_fingerprint":
            sha3_256_hex(user_public_key),
        "helper_package":
            helper_package.to_dictionary(),
        "credential_version":
            credential_version,
        "account_status":
            "ACTIVE",
        "created_at":
            utc_timestamp(),
    }


def server_get_public_biometric_package(
    server_node: ServerNode,
    *,
    user_id: str,
    authenticated_session_id: str,
) -> PublicBiometricHelperPackage:
    """
    Return the public helper package only after the terminal-server
    mutual-authentication session has been validated.
    """

    session = server_node.get_local_session(
        authenticated_session_id
    )

    if (
        session.get("authentication_state")
        != "MUTUALLY_AUTHENTICATED"
    ):
        raise PermissionError(
            "A mutually authenticated terminal-server session "
            "is required."
        )

    record = server_node._registered_users.get(user_id)

    if record is None:
        raise KeyError("User is not registered.")

    if record["account_status"] != "ACTIVE":
        raise PermissionError(
            "User account is not active."
        )

    helper = record["helper_package"]

    return PublicBiometricHelperPackage(**deepcopy(helper))


def terminal_get_protected_credential(
    terminal_node: TerminalNode,
    user_id: str,
) -> ProtectedUserCredential:
    """
    Load a protected terminal-side credential.
    """

    record = terminal_node._local_user_records.get(
        user_id
    )

    if record is None:
        raise KeyError(
            "No protected credential exists for this user."
        )

    envelope = AESGCMEnvelope(
        **record["encrypted_private_key"]
    )

    return ProtectedUserCredential(
        user_id=record["user_id"],
        user_name=record["user_name"],
        user_email=record["user_email"],
        user_public_key_b64=
            record["user_public_key_b64"],
        public_key_fingerprint=
            record["public_key_fingerprint"],
        encrypted_private_key=envelope,
        credential_version=
            record["credential_version"],
        biometric_scheme=
            record["biometric_scheme"],
        created_at=record["created_at"],
        account_status=record["account_status"],
    )


print("Terminal and server user-record storage ready.")

Terminal and server user-record storage ready.


In [46]:
#@title Store Synthetic Iris codes for giving inputs and verifying users later on.

def export_synthetic_iris_registry(
    path: Path = SYNTHETIC_IRIS_EXPORT_PATH,
) -> Path:
    """
    Export copy-ready synthetic iris codes for notebook testing.

    WARNING:
    This is a simulator-only operation. A production biometric
    system must not export raw biometric templates.
    """

    export_data = {
        user_id: {
            "profile_id": profile.profile_id,
            "canonical_iris_code":
                profile.canonical_iris_code,
            "canonical_iris_hex":
                profile.iris_code_hex,
            "iris_fingerprint":
                profile.iris_fingerprint,
            "created_at":
                profile.created_at,
        }
        for user_id, profile
        in SYNTHETIC_IRIS_DEMO_REGISTRY.items()
    }

    path.write_text(
        json.dumps(
            export_data,
            indent=2,
            sort_keys=True,
        ),
        encoding="utf-8",
    )

    return path


def display_synthetic_iris_registry() -> None:
    """
    Display summaries and complete copy-ready binary codes.
    """

    if not SYNTHETIC_IRIS_DEMO_REGISTRY:
        print("No synthetic users have been enrolled.")
        return

    summary_rows = []

    for user_id, profile in (
        SYNTHETIC_IRIS_DEMO_REGISTRY.items()
    ):
        summary_rows.append({
            "User ID": user_id,
            "Profile ID": profile.profile_id,
            "Iris Preview":
                f"{profile.canonical_iris_code[:24]}..."
                f"{profile.canonical_iris_code[-24:]}",
            "Fingerprint":
                profile.iris_fingerprint,
        })

    display(pd.DataFrame(summary_rows))

    print("\nCOPY-READY RAW SYNTHETIC IRIS CODES")
    print("=" * 80)

    for user_id, profile in (
        SYNTHETIC_IRIS_DEMO_REGISTRY.items()
    ):
        print(f"\n{user_id}")
        print("-" * 80)
        print(profile.canonical_iris_code)

    exported_path = export_synthetic_iris_registry()

    print("\n" + "=" * 80)
    print(f"Registry saved to: {exported_path}")


print("Synthetic iris export utilities ready.")

Synthetic iris export utilities ready.


In [47]:
#@title Creating User Dilithium key pairs according to the iris scan

def build_user_private_key_aad(
    *,
    user_id: str,
    user_email: str,
    public_key_fingerprint: str,
    credential_version: int,
) -> bytes:
    """
    Construct associated data that binds the encrypted private key
    to the user's identity and public credential.
    """

    return canonical_json({
        "protocol_name": PROTOCOL_NAME,
        "record_type":
            "BIOMETRICALLY_PROTECTED_ML_DSA_PRIVATE_KEY",
        "user_id": user_id,
        "user_email": user_email,
        "public_key_fingerprint":
            public_key_fingerprint,
        "credential_version":
            credential_version,
    })


def enrol_synthetic_user(
    *,
    terminal_node: TerminalNode,
    server_node: ServerNode,
    user_name: str,
    user_email: str,
) -> UserEnrollmentResult:
    """
    Complete synthetic biometric enrollment.

    Steps:
        1. Validate identity.
        2. Generate synthetic iris.
        3. Create fuzzy commitment.
        4. Generate random ML-DSA key pair.
        5. Derive biometric wrapping key.
        6. Encrypt ML-DSA private key.
        7. Store protected terminal record.
        8. Store public server record.
        9. Export raw synthetic iris for testing.
    """

    user_name = user_name.strip()

    if not user_name:
        raise ValueError("User name cannot be empty.")

    user_email = normalise_email(user_email)
    user_id = create_user_id_from_email(user_email)

    if user_id in server_node._registered_users:
        raise ValueError(
            f"A user with email {user_email} is already enrolled."
        )

    if user_id in terminal_node._local_user_records:
        raise ValueError(
            "The terminal already contains this user."
        )

    # --------------------------------------------------------
    # 1. Generate the synthetic canonical iris.
    # --------------------------------------------------------

    iris_profile = SyntheticIrisFactory.create_profile()

    # --------------------------------------------------------
    # 2. Create fuzzy commitment and temporary random secret.
    # --------------------------------------------------------

    helper_package, fuzzy_secret = (
        create_fuzzy_commitment(
            canonical_iris_code=
                iris_profile.canonical_iris_code,
            user_id=user_id,
        )
    )

    # --------------------------------------------------------
    # 3. Generate a normal independent ML-DSA key pair.
    # --------------------------------------------------------

    user_public_key, user_private_key = (
        MLDSA65.generate_keypair()
    )

    public_key_fingerprint = sha3_256_hex(
        user_public_key
    )

    # --------------------------------------------------------
    # 4. Derive the wrapping key from the recoverable secret.
    # --------------------------------------------------------

    wrapping_key = derive_biometric_wrapping_key(
        fuzzy_secret=fuzzy_secret,
        fuzzy_salt=helper_package.fuzzy_salt_bytes(),
        user_id=user_id,
        credential_version=
            BIOMETRIC_CREDENTIAL_VERSION,
    )

    private_key_aad = build_user_private_key_aad(
        user_id=user_id,
        user_email=user_email,
        public_key_fingerprint=
            public_key_fingerprint,
        credential_version=
            BIOMETRIC_CREDENTIAL_VERSION,
    )

    encrypted_private_key = aes_gcm_encrypt(
        key=wrapping_key,
        plaintext=user_private_key,
        associated_data=private_key_aad,
    )

    protected_credential = ProtectedUserCredential(
        user_id=user_id,
        user_name=user_name,
        user_email=user_email,
        user_public_key_b64=
            bytes_to_b64(user_public_key),
        public_key_fingerprint=
            public_key_fingerprint,
        encrypted_private_key=
            encrypted_private_key,
        credential_version=
            BIOMETRIC_CREDENTIAL_VERSION,
        biometric_scheme=
            helper_package.scheme,
        created_at=utc_timestamp(),
        account_status="ACTIVE",
    )

    # --------------------------------------------------------
    # 5. Store separate terminal/server records.
    # --------------------------------------------------------

    terminal_store_protected_credential(
        terminal_node,
        protected_credential,
    )

    server_store_user_registration(
        server_node,
        user_id=user_id,
        user_name=user_name,
        user_email=user_email,
        user_public_key=user_public_key,
        helper_package=helper_package,
        credential_version=
            BIOMETRIC_CREDENTIAL_VERSION,
    )

    # Simulator-only raw iris registry.
    SYNTHETIC_IRIS_DEMO_REGISTRY[
        user_id
    ] = iris_profile

    export_synthetic_iris_registry()

    # Remove ordinary references to sensitive temporary values.
    del user_private_key
    del wrapping_key
    del fuzzy_secret

    terminal_node._log_event(
        event_type="USER_ENROLLED",
        result="SUCCESS",
        details={
            "user_id": user_id,
            "public_key_fingerprint":
                public_key_fingerprint,
            "iris_fingerprint":
                iris_profile.iris_fingerprint,
        },
    )

    server_node._log_event(
        event_type="USER_REGISTERED",
        result="SUCCESS",
        details={
            "user_id": user_id,
            "public_key_fingerprint":
                public_key_fingerprint,
            "biometric_scheme":
                helper_package.scheme,
        },
    )

    return UserEnrollmentResult(
        user_id=user_id,
        user_name=user_name,
        user_email=user_email,
        iris_profile_id=iris_profile.profile_id,
        raw_iris_code=
            iris_profile.canonical_iris_code,
        raw_iris_hex=
            iris_profile.iris_code_hex,
        iris_fingerprint=
            iris_profile.iris_fingerprint,
        public_key_fingerprint=
            public_key_fingerprint,
        helper_data_fingerprint=
            sha3_256_hex(
                helper_package.helper_data_bytes()
            ),
        enrollment_timestamp=utc_timestamp(),
    )


print("Complete synthetic-user enrollment workflow ready.")

Complete synthetic-user enrollment workflow ready.


In [48]:
#@title Dynamic, Interactive Enrollment. Run once for each new user

# entered_name = input(
#     "Enter the user's full name: "
# ).strip()

# entered_email = input(
#     "Enter the user's unique email identifier: "
# ).strip()

# enrollment_result = enrol_synthetic_user(
#     terminal_node=terminal,
#     server_node=server,
#     user_name=entered_name,
#     user_email=entered_email,
# )

# print("\n" + "=" * 80)
# print("USER ENROLLMENT SUCCESSFUL")
# print("=" * 80)
# print(f"User name              : {enrollment_result.user_name}")
# print(f"User email             : {enrollment_result.user_email}")
# print(f"Internal user ID       : {enrollment_result.user_id}")
# print(f"Synthetic profile ID   : {enrollment_result.iris_profile_id}")
# print(f"Iris fingerprint       : {enrollment_result.iris_fingerprint}")
# print(
#     f"ML-DSA key fingerprint : "
#     f"{enrollment_result.public_key_fingerprint}"
# )
# print(
#     f"Helper-data fingerprint: "
#     f"{enrollment_result.helper_data_fingerprint}"
# )
# print()
# print("COPY-READY RAW SYNTHETIC IRIS CODE")
# print("-" * 80)
# print(enrollment_result.raw_iris_code)
# print("-" * 80)
# print(
#     f"All synthetic iris codes are also stored at:\n"
#     f"{SYNTHETIC_IRIS_EXPORT_PATH}"
# )

In [49]:
#@title Display all synthetic iris codes registered till now

display_synthetic_iris_registry()

No synthetic users have been enrolled.


## Phase 3B : 28-07-2026T10:26:29 - 28-07-2026T22:20:52

In [50]:
#@title Give the raw iris sample to generate a synthetic binary iris code

@dataclass(frozen=True)
class SyntheticScanResult:
    source_iris_code: str
    noisy_iris_code: str
    requested_bit_error_rate: float
    actual_changed_bits: int
    actual_bit_error_rate: float
    applied_rotation_bits: int
    changed_byte_symbols_block_1: int
    changed_byte_symbols_block_2: int


def count_changed_byte_symbols(
    original_bits: np.ndarray,
    candidate_bits: np.ndarray,
) -> Tuple[int, int]:
    """
    Count changed byte symbols in each RS block.

    Reed–Solomon corrects erroneous bytes, not individual bits.
    """

    original_bytes = np.frombuffer(
        iris_bits_to_bytes(original_bits),
        dtype=np.uint8,
    )

    candidate_bytes = np.frombuffer(
        iris_bits_to_bytes(candidate_bits),
        dtype=np.uint8,
    )

    changed = original_bytes != candidate_bytes

    block_1_errors = int(
        np.sum(
            changed[
                :FUZZY_CODEWORD_BYTES_PER_BLOCK
            ]
        )
    )

    block_2_errors = int(
        np.sum(
            changed[
                FUZZY_CODEWORD_BYTES_PER_BLOCK:
            ]
        )
    )

    return block_1_errors, block_2_errors


def simulate_noisy_iris_scan(
    source_iris_code: str,
    *,
    bit_error_rate: float = 0.01,
    rotation_bits: int = 0,
) -> SyntheticScanResult:
    """
    Generate a fresh noisy scan from an input canonical iris code.

    Noise positions are selected without replacement.

    Rotation is applied after noise and will be compensated by the
    reconstruction algorithm.
    """

    source_iris_code = validate_binary_iris_string(
        source_iris_code
    )

    if not 0.0 <= bit_error_rate <= 1.0:
        raise ValueError(
            "bit_error_rate must lie between 0 and 1."
        )

    if abs(rotation_bits) > DEFAULT_MAX_ROTATION_BITS:
        raise ValueError(
            f"Rotation must lie between "
            f"-{DEFAULT_MAX_ROTATION_BITS} and "
            f"+{DEFAULT_MAX_ROTATION_BITS} bits."
        )

    original_bits = iris_string_to_bits(
        source_iris_code
    )

    noisy_unrotated_bits = original_bits.copy()

    requested_error_count = int(
        round(IRIS_CODE_LENGTH * bit_error_rate)
    )

    if requested_error_count > 0:
        changed_positions = np.random.default_rng(
            int.from_bytes(
                secrets.token_bytes(16),
                byteorder="big",
            )
        ).choice(
            IRIS_CODE_LENGTH,
            size=requested_error_count,
            replace=False,
        )

        noisy_unrotated_bits[
            changed_positions
        ] ^= 1

    block_1_errors, block_2_errors = (
        count_changed_byte_symbols(
            original_bits,
            noisy_unrotated_bits,
        )
    )

    rotated_bits = rotate_iris_bits(
        noisy_unrotated_bits,
        rotation_bits,
    )

    actual_changed_bits = int(
        np.sum(
            original_bits != noisy_unrotated_bits
        )
    )

    return SyntheticScanResult(
        source_iris_code=source_iris_code,
        noisy_iris_code=
            iris_bits_to_string(rotated_bits),
        requested_bit_error_rate=
            bit_error_rate,
        actual_changed_bits=
            actual_changed_bits,
        actual_bit_error_rate=
            actual_changed_bits / IRIS_CODE_LENGTH,
        applied_rotation_bits=
            rotation_bits,
        changed_byte_symbols_block_1=
            block_1_errors,
        changed_byte_symbols_block_2=
            block_2_errors,
    )


print("Synthetic noisy iris scanner ready.")

Synthetic noisy iris scanner ready.


In [51]:
#@title Use the derived synthetic raw sample to recover the dilithium key

@dataclass(frozen=True)
class CredentialRecoveryResult:
    success: bool
    user_id: str
    recovered_private_key: Optional[bytes]
    selected_rotation_bits: Optional[int]
    reconstruction_attempts: int
    failure_reason: Optional[str]


def recover_user_mldsa_private_key(
    *,
    terminal_node: TerminalNode,
    server_node: ServerNode,
    authenticated_session_id: str,
    user_email: str,
    candidate_iris_code: str,
) -> CredentialRecoveryResult:
    """
    Recover and decrypt a user's ML-DSA private key.

    The helper package is obtained from the server.
    The encrypted private key is obtained from the terminal.
    """

    user_email = normalise_email(user_email)
    user_id = create_user_id_from_email(user_email)

    try:
        helper_package = (
            server_get_public_biometric_package(
                server_node,
                user_id=user_id,
                authenticated_session_id=authenticated_session_id,
            )
        )

        protected_credential = (
            terminal_get_protected_credential(
                terminal_node,
                user_id,
            )
        )

        if (
            protected_credential.account_status
            != "ACTIVE"
        ):
            raise PermissionError(
                "Protected credential is not active."
            )

        reconstruction = reconstruct_fuzzy_secret(
            candidate_iris_code=
                candidate_iris_code,
            helper_package=
                helper_package,
            user_id=user_id,
        )

        if not reconstruction.success:
            return CredentialRecoveryResult(
                success=False,
                user_id=user_id,
                recovered_private_key=None,
                selected_rotation_bits=None,
                reconstruction_attempts=
                    reconstruction.attempts,
                failure_reason=
                    reconstruction.failure_reason,
            )

        fuzzy_secret = reconstruction.recovered_secret

        if fuzzy_secret is None:
            raise RuntimeError(
                "Reconstruction reported success without a secret."
            )

        wrapping_key = derive_biometric_wrapping_key(
            fuzzy_secret=fuzzy_secret,
            fuzzy_salt=
                helper_package.fuzzy_salt_bytes(),
            user_id=user_id,
            credential_version=
                protected_credential.credential_version,
        )

        private_key_aad = build_user_private_key_aad(
            user_id=user_id,
            user_email=user_email,
            public_key_fingerprint=
                protected_credential
                .public_key_fingerprint,
            credential_version=
                protected_credential
                .credential_version,
        )

        expected_aad_b64 = bytes_to_b64(
            private_key_aad
        )

        if (
            protected_credential
            .encrypted_private_key
            .associated_data_b64
            != expected_aad_b64
        ):
            raise AuthenticationFailure(
                "Encrypted credential identity binding "
                "does not match."
            )

        try:
            recovered_private_key = aes_gcm_decrypt(
                wrapping_key,
                protected_credential
                .encrypted_private_key,
            )

        except InvalidTag:
            return CredentialRecoveryResult(
                success=False,
                user_id=user_id,
                recovered_private_key=None,
                selected_rotation_bits=
                    reconstruction
                    .selected_rotation_bits,
                reconstruction_attempts=
                    reconstruction.attempts,
                failure_reason=(
                    "Biometric secret was incorrect or the "
                    "protected credential was modified."
                ),
            )

        return CredentialRecoveryResult(
            success=True,
            user_id=user_id,
            recovered_private_key=
                recovered_private_key,
            selected_rotation_bits=
                reconstruction
                .selected_rotation_bits,
            reconstruction_attempts=
                reconstruction.attempts,
            failure_reason=None,
        )

    except Exception as exc:
        return CredentialRecoveryResult(
            success=False,
            user_id=user_id,
            recovered_private_key=None,
            selected_rotation_bits=None,
            reconstruction_attempts=0,
            failure_reason=str(exc),
        )


print("ML-DSA credential recovery workflow ready.")

ML-DSA credential recovery workflow ready.


In [52]:
#@title The Dilithium key is recovered. Now verify it by signing the challenge

@dataclass(frozen=True)
class UserCredentialVerificationResult:
    success: bool
    user_id: str
    reconstruction_success: bool
    private_key_decryption_success: bool
    user_signature_valid: bool
    selected_rotation_bits: Optional[int]
    reconstruction_attempts: int
    challenge_hash: Optional[str]
    failure_reason: Optional[str]


def authenticate_recovered_user_credential(
    *,
    terminal_node: TerminalNode,
    server_node: ServerNode,
    authenticated_session_id: str,
    user_email: str,
    candidate_iris_code: str,
) -> UserCredentialVerificationResult:
    """
    Reconstruct, unlock, sign a challenge and verify the user's
    ML-DSA credential.
    """

    user_email = normalise_email(user_email)
    user_id = create_user_id_from_email(user_email)

    recovery = recover_user_mldsa_private_key(
        terminal_node=terminal_node,
        server_node=server_node,
        authenticated_session_id=
            authenticated_session_id,
        user_email=user_email,
        candidate_iris_code=
            candidate_iris_code,
    )

    if not recovery.success:
        return UserCredentialVerificationResult(
            success=False,
            user_id=user_id,
            reconstruction_success=False,
            private_key_decryption_success=False,
            user_signature_valid=False,
            selected_rotation_bits=
                recovery.selected_rotation_bits,
            reconstruction_attempts=
                recovery.reconstruction_attempts,
            challenge_hash=None,
            failure_reason=recovery.failure_reason,
        )

    recovered_private_key = (
        recovery.recovered_private_key
    )

    if recovered_private_key is None:
        raise RuntimeError(
            "Credential recovery succeeded without a private key."
        )

    server_user_record = (
        server_node._registered_users[user_id]
    )

    user_public_key = b64_to_bytes(
        server_user_record["user_public_key_b64"]
    )

    challenge = canonical_json({
        "protocol_name": PROTOCOL_NAME,
        "message_type":
            "LOCAL_USER_CREDENTIAL_CHALLENGE",
        "user_id": user_id,
        "terminal_id": terminal_node.node_id,
        "server_id": server_node.node_id,
        "session_id":
            authenticated_session_id,
        "challenge_nonce_b64":
            bytes_to_b64(generate_nonce(32)),
        "issued_at": utc_timestamp(),
    })

    challenge_signature = MLDSA65.sign(
        recovered_private_key,
        challenge,
    )

    signature_valid = MLDSA65.verify(
        user_public_key,
        challenge,
        challenge_signature,
    )

    # Remove the ordinary local reference after use.
    del recovered_private_key

    return UserCredentialVerificationResult(
        success=signature_valid,
        user_id=user_id,
        reconstruction_success=True,
        private_key_decryption_success=True,
        user_signature_valid=signature_valid,
        selected_rotation_bits=
            recovery.selected_rotation_bits,
        reconstruction_attempts=
            recovery.reconstruction_attempts,
        challenge_hash=
            sha3_256_hex(challenge),
        failure_reason=(
            None
            if signature_valid
            else "Recovered key failed public-key verification."
        ),
    )


print("Recovered user credential verification ready.")

Recovered user credential verification ready.


In [53]:
#@title Testing on Exact Iris Code

# authentication_email = input(
#     "Enter enrolled user email: "
# ).strip()

# authentication_iris_code = input(
#     "Paste the user's 2048-bit raw iris code: "
# ).strip()

# exact_auth_result = (
#     authenticate_recovered_user_credential(
#         terminal_node=terminal,
#         server_node=server,
#         authenticated_session_id=
#             mutual_auth_result.session_id,
#         user_email=authentication_email,
#         candidate_iris_code=
#             authentication_iris_code,
#     )
# )

# print("\n" + "=" * 75)
# print("EXACT IRIS AUTHENTICATION TEST")
# print("=" * 75)
# print(
#     "Fuzzy reconstruction       :",
#     exact_auth_result.reconstruction_success,
# )
# print(
#     "Private-key decryption     :",
#     exact_auth_result.private_key_decryption_success,
# )
# print(
#     "ML-DSA signature verified  :",
#     exact_auth_result.user_signature_valid,
# )
# print(
#     "Selected rotation          :",
#     exact_auth_result.selected_rotation_bits,
# )
# print(
#     "Reconstruction attempts    :",
#     exact_auth_result.reconstruction_attempts,
# )
# print(
#     "Overall result             :",
#     "PASS" if exact_auth_result.success else "FAIL",
# )

# if exact_auth_result.failure_reason:
#     print(
#         "Failure reason             :",
#         exact_auth_result.failure_reason,
#     )

In [54]:
#@title Noisy Iris Test (Noisy but genuine)

# noise_test_email = input(
#     "Enter the enrolled user's email: "
# ).strip()

# noise_source_iris = input(
#     "Paste that user's enrolled raw iris code: "
# ).strip()

# noise_rate = float(
#     input(
#         "Enter bit-noise rate, for example 0.01 for 1%: "
#     ).strip()
# )

# rotation = int(
#     input(
#         f"Enter rotation in bits "
#         f"(-{DEFAULT_MAX_ROTATION_BITS} to "
#         f"+{DEFAULT_MAX_ROTATION_BITS}): "
#     ).strip()
# )

# synthetic_scan = simulate_noisy_iris_scan(
#     source_iris_code=noise_source_iris,
#     bit_error_rate=noise_rate,
#     rotation_bits=rotation,
# )

# print("\nSYNTHETIC SCAN STATISTICS")
# print("-" * 70)
# print(
#     f"Changed bits       : "
#     f"{synthetic_scan.actual_changed_bits}/"
#     f"{IRIS_CODE_LENGTH}"
# )
# print(
#     f"Actual bit error   : "
#     f"{synthetic_scan.actual_bit_error_rate:.4%}"
# )
# print(
#     f"Applied rotation   : "
#     f"{synthetic_scan.applied_rotation_bits} bits"
# )
# print(
#     f"RS byte errors B1  : "
#     f"{synthetic_scan.changed_byte_symbols_block_1}/"
#     f"{FUZZY_CODEWORD_BYTES_PER_BLOCK}"
# )
# print(
#     f"RS byte errors B2  : "
#     f"{synthetic_scan.changed_byte_symbols_block_2}/"
#     f"{FUZZY_CODEWORD_BYTES_PER_BLOCK}"
# )
# print(
#     f"Correction limit   : "
#     f"{FUZZY_MAX_SYMBOL_ERRORS_PER_BLOCK} byte symbols/block"
# )

# noisy_auth_result = (
#     authenticate_recovered_user_credential(
#         terminal_node=terminal,
#         server_node=server,
#         authenticated_session_id=
#             mutual_auth_result.session_id,
#         user_email=noise_test_email,
#         candidate_iris_code=
#             synthetic_scan.noisy_iris_code,
#     )
# )

# print("\n" + "=" * 75)
# print("NOISY GENUINE IRIS AUTHENTICATION")
# print("=" * 75)
# print(
#     "Fuzzy reconstruction       :",
#     noisy_auth_result.reconstruction_success,
# )
# print(
#     "Private-key decryption     :",
#     noisy_auth_result.private_key_decryption_success,
# )
# print(
#     "ML-DSA signature verified  :",
#     noisy_auth_result.user_signature_valid,
# )
# print(
#     "Selected alignment shift   :",
#     noisy_auth_result.selected_rotation_bits,
# )
# print(
#     "Overall result             :",
#     "PASS" if noisy_auth_result.success else "FAIL",
# )

# if noisy_auth_result.failure_reason:
#     print(
#         "Failure reason             :",
#         noisy_auth_result.failure_reason,
#     )

In [55]:
#@title Impostor Iris Test

# claimed_email = input(
#     "Enter the claimed user's email: "
# ).strip()

# impostor_iris_code = input(
#     "Paste a different user's raw iris code: "
# ).strip()

# impostor_result = (
#     authenticate_recovered_user_credential(
#         terminal_node=terminal,
#         server_node=server,
#         authenticated_session_id=
#             mutual_auth_result.session_id,
#         user_email=claimed_email,
#         candidate_iris_code=
#             impostor_iris_code,
#     )
# )

# print("\n" + "=" * 75)
# print("IMPOSTOR IRIS TEST")
# print("=" * 75)
# print(
#     "Fuzzy reconstruction       :",
#     impostor_result.reconstruction_success,
# )
# print(
#     "Private-key decryption     :",
#     impostor_result.private_key_decryption_success,
# )
# print(
#     "ML-DSA signature verified  :",
#     impostor_result.user_signature_valid,
# )
# print(
#     "Overall result             :",
#     "PASS" if impostor_result.success else "REJECTED",
# )

# if impostor_result.failure_reason:
#     print(
#         "Failure reason             :",
#         impostor_result.failure_reason,
#     )

# assert impostor_result.success is False

# print("\nWrong-user biometric was correctly rejected.")

In [56]:
#@title Testing Helper Data Safety

# helper_test_email = input(
#     "Enter an enrolled user's email: "
# ).strip()

# helper_test_user_id = create_user_id_from_email(
#     helper_test_email
# )

# server_record = deepcopy(
#     server._registered_users[helper_test_user_id]
# )

# server_record_serialised = canonical_json(
#     server_record
# )

# synthetic_profile = (
#     SYNTHETIC_IRIS_DEMO_REGISTRY[
#         helper_test_user_id
#     ]
# )

# raw_iris_bytes = iris_bits_to_bytes(
#     iris_string_to_bits(
#         synthetic_profile.canonical_iris_code
#     )
# )

# raw_iris_b64 = bytes_to_b64(raw_iris_bytes)
# raw_iris_binary = (
#     synthetic_profile.canonical_iris_code
# )

# structural_checks = {
#     "No raw binary iris in server record":
#         raw_iris_binary.encode("utf-8")
#         not in server_record_serialised,

#     "No Base64 raw iris in server record":
#         raw_iris_b64.encode("ascii")
#         not in server_record_serialised,

#     "No encrypted user private key at server":
#         b"encrypted_private_key"
#         not in server_record_serialised,

#     "No fuzzy secret field":
#         b"fuzzy_secret"
#         not in server_record_serialised,

#     "No wrapping key field":
#         b"wrapping_key"
#         not in server_record_serialised,

#     "Helper data has expected length":
#         len(
#             b64_to_bytes(
#                 server_record[
#                     "helper_package"
#                 ]["helper_data_b64"]
#             )
#         )
#         == IRIS_CODE_LENGTH_BYTES,
# }

# helper_test_report = pd.DataFrame([
#     {
#         "Property": property_name,
#         "Status": "PASS" if result else "FAIL",
#     }
#     for property_name, result
#     in structural_checks.items()
# ])

# display(helper_test_report)

# assert all(structural_checks.values())

# print("Server helper-data structural inspection: PASS")
# print()
# print(
#     "This confirms absence of direct secret fields. "
# )

In [57]:
#@title Phase 3 Final Report

# phase_3_checks = {
#     "At least one synthetic user enrolled":
#         len(SYNTHETIC_IRIS_DEMO_REGISTRY) >= 1,

#     "Terminal stores protected credential":
#         len(terminal._local_user_records) >= 1,

#     "Server stores public user registration":
#         len(server._registered_users) >= 1,

#     "Synthetic raw iris registry exported":
#         SYNTHETIC_IRIS_EXPORT_PATH.exists(),

#     "Exact iris reconstructs credential":
#         exact_auth_result.success,

#     "Wrong-user iris rejected":
#         not impostor_result.success,

#     "Server record excludes direct iris/private key":
#         all(structural_checks.values()),
# }

# # Include the noisy test if Cell 3.18 has been executed.
# if "noisy_auth_result" in globals():
#     phase_3_checks[
#         "Configured noisy genuine iris accepted"
#     ] = noisy_auth_result.success

# phase_3_report = pd.DataFrame([
#     {
#         "Security / Functional Property":
#             property_name,
#         "Status":
#             "PASS" if result else "FAIL",
#     }
#     for property_name, result
#     in phase_3_checks.items()
# ])

# display(phase_3_report)

# print("\n" + "=" * 78)
# print("PHASE 3 COMPLETED SUCCESSFULLY")
# print("=" * 78)
# print("The system can now:")
# print("  • create users using a name and unique email")
# print("  • generate a unique synthetic 2048-bit iris")
# print("  • export copy-ready simulator iris codes")
# print("  • generate independent ML-DSA-65 credentials")
# print("  • protect each private key using a fuzzy secret")
# print("  • store public helper data at the server")
# print("  • reconstruct through moderate biometric noise")
# print("  • compensate for small circular rotations")
# print("  • reject a different user's iris")
# print("  • prove recovery by an ML-DSA signature")

## Phase 4 : 29-07-2026T01:54:28 - 29-07-2026T03:04:26

In [58]:
#@title QKD Configuration

import math
import secrets
import hashlib
import hmac

from dataclasses import dataclass, asdict
from copy import deepcopy
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np


# ------------------------------------------------------------
# T12-INSPIRED BIASED-BASIS CONFIGURATION
# ------------------------------------------------------------

T12_Z_BASIS_PROBABILITY = 0.85
T12_X_BASIS_PROBABILITY = 0.15

assert math.isclose(
    T12_Z_BASIS_PROBABILITY
    + T12_X_BASIS_PROBABILITY,
    1.0,
)

T12_EXPECTED_BASIS_MATCH_PROBABILITY = (
    T12_Z_BASIS_PROBABILITY ** 2
    + T12_X_BASIS_PROBABILITY ** 2
)

# Number of prepare-and-measure signals generated in one session.
T12_DEFAULT_SIGNAL_COUNT = 8192

# Fraction of sifted bits disclosed for parameter estimation.
T12_PARAMETER_ESTIMATION_FRACTION = 0.15

# Abort threshold for the synthetic quantum channel.
T12_QBER_ABORT_THRESHOLD = 0.08

# Minimum amount of key material remaining before reconciliation.
T12_MINIMUM_RECONCILIATION_BITS = 512

# Cascade-style reconciliation configuration.
T12_RECONCILIATION_PASSES = 4
T12_INITIAL_BLOCK_SIZE = 16

# Conservative additional privacy reduction.
T12_SECURITY_MARGIN_BITS = 128

# Minimum privacy-amplified output.
T12_MINIMUM_SECRET_BITS = 256

# AES-256-GCM configuration.
PHASE_5_AES_KEY_BYTES = 32
PHASE_5_AES_NONCE_BYTES = 12

PHASE_5_PROTOCOL_NAME = "QBioAuth-T12-QKD"
PHASE_5_PROTOCOL_VERSION = "1.0"

print("Phase 4 configuration loaded.")
print("-" * 72)
print(
    "Hard-coded Z-basis probability :",
    T12_Z_BASIS_PROBABILITY,
)
print(
    "Hard-coded X-basis probability :",
    T12_X_BASIS_PROBABILITY,
)
print(
    "Expected basis-match rate      :",
    f"{T12_EXPECTED_BASIS_MATCH_PROBABILITY:.2%}",
)

Phase 4 configuration loaded.
------------------------------------------------------------------------
Hard-coded Z-basis probability : 0.85
Hard-coded X-basis probability : 0.15
Expected basis-match rate      : 74.50%


In [59]:
#@title Phase 4 QKD Utilities

def generate_random_bits(length: int) -> np.ndarray:
    """
    Generate cryptographically random binary values.
    """

    if length <= 0:
        raise ValueError("Bit length must be positive.")

    raw = secrets.token_bytes((length + 7) // 8)

    return np.unpackbits(
        np.frombuffer(raw, dtype=np.uint8),
        bitorder="big",
    )[:length].astype(np.uint8)


def generate_t12_biased_bases(length: int) -> np.ndarray:
    """
    Generate bases with a hard-coded 80:20 Z:X ratio.

    Representation:
        0 -> Z basis
        1 -> X basis
    """

    if length <= 0:
        raise ValueError("Basis length must be positive.")

    random_values = np.random.default_rng(
        int.from_bytes(
            secrets.token_bytes(16),
            byteorder="big",
        )
    ).random(length)

    return np.where(
        random_values < T12_Z_BASIS_PROBABILITY,
        0,
        1,
    ).astype(np.uint8)


def validate_bit_array(
    bits: Sequence[int],
    *,
    allow_empty: bool = False,
) -> np.ndarray:
    """
    Convert and validate a binary sequence.
    """

    array = np.asarray(bits, dtype=np.uint8).reshape(-1)

    if not allow_empty and len(array) == 0:
        raise ValueError("Bit array cannot be empty.")

    if not np.all(np.isin(array, [0, 1])):
        raise ValueError("Sequence contains non-binary values.")

    return array


def bits_to_packed_bytes(bits: Sequence[int]) -> bytes:
    """
    Pack a binary array with its length prefix.

    The length prefix prevents ambiguity from byte padding.
    """

    array = validate_bit_array(bits)

    bit_length_prefix = len(array).to_bytes(
        8,
        byteorder="big",
    )

    packed = np.packbits(
        array,
        bitorder="big",
    ).tobytes()

    return bit_length_prefix + packed


def packed_bytes_to_bits(data: bytes) -> np.ndarray:
    """
    Reverse bits_to_packed_bytes().
    """

    if len(data) < 8:
        raise ValueError("Packed bit representation is too short.")

    bit_length = int.from_bytes(
        data[:8],
        byteorder="big",
    )

    unpacked = np.unpackbits(
        np.frombuffer(data[8:], dtype=np.uint8),
        bitorder="big",
    )

    return unpacked[:bit_length].astype(np.uint8)


def bit_array_to_string(bits: Sequence[int]) -> str:
    array = validate_bit_array(bits, allow_empty=True)
    return "".join(str(int(bit)) for bit in array)


def binary_entropy(probability: float) -> float:
    """
    Binary Shannon entropy h2(p).
    """

    if probability <= 0.0 or probability >= 1.0:
        return 0.0

    return (
        -probability * math.log2(probability)
        -(1.0 - probability)
        * math.log2(1.0 - probability)
    )


def parity_of_bits(bits: Sequence[int]) -> int:
    array = validate_bit_array(bits, allow_empty=True)

    if len(array) == 0:
        return 0

    return int(np.bitwise_xor.reduce(array))


def sha3_256_bytes(data: bytes) -> bytes:
    return hashlib.sha3_256(data).digest()


def sha3_256_hex_local(data: bytes) -> str:
    return hashlib.sha3_256(data).hexdigest()


print("Phase 4 bit and entropy utilities ready.")

Phase 4 bit and entropy utilities ready.


In [60]:
#@title Signed QKD Message Protocol

QKD_CLASSICAL_MESSAGE_TYPE = "QKD_CLASSICAL_MESSAGE"


@dataclass(frozen=True)
class SignedQKDClassicalMessage:
    envelope: ProtocolEnvelope

    def to_dictionary(self) -> Dict[str, Any]:
        return self.envelope.to_dictionary()


def create_signed_qkd_message(
    *,
    sender: IsolatedNode,
    receiver: IsolatedNode,
    qkd_session_id: str,
    operation: str,
    sequence_number: int,
    payload: Dict[str, Any],
    previous_message_hash: Optional[str],
) -> SignedQKDClassicalMessage:
    """
    Create one signed and transcript-chained QKD classical message.
    """

    if sequence_number < 1:
        raise ValueError(
            "Sequence number must be at least one."
        )

    unsigned_envelope = ProtocolEnvelope(
        message_type=QKD_CLASSICAL_MESSAGE_TYPE,
        sender_id=sender.node_id,
        receiver_id=receiver.node_id,
        session_id=qkd_session_id,
        message_id=generate_uuid(),
        issued_at=utc_timestamp(),
        payload={
            "protocol_name": PHASE_5_PROTOCOL_NAME,
            "protocol_version": PHASE_5_PROTOCOL_VERSION,
            "operation": operation,
            "sequence_number": sequence_number,
            "previous_message_hash":
                previous_message_hash,
            "body": deepcopy(payload),
        },
        signature_b64=None,
    )

    signed_envelope = sign_protocol_envelope(
        sender,
        unsigned_envelope,
    )

    return SignedQKDClassicalMessage(
        envelope=signed_envelope
    )


def verify_signed_qkd_message(
    *,
    receiver: IsolatedNode,
    expected_sender: IsolatedNode,
    message: SignedQKDClassicalMessage,
    expected_session_id: str,
    expected_operation: str,
    expected_sequence_number: int,
    expected_previous_hash: Optional[str],
) -> str:
    """
    Verify identity, signature, freshness, ordering and hash-chain
    binding of a classical QKD message.

    Returns the verified message hash.
    """

    envelope = message.envelope

    if envelope.message_type != QKD_CLASSICAL_MESSAGE_TYPE:
        raise AuthenticationFailure(
            "Unexpected classical QKD message type."
        )

    if envelope.sender_id != expected_sender.node_id:
        raise AuthenticationFailure(
            "Classical QKD message has an unexpected sender."
        )

    if envelope.receiver_id != receiver.node_id:
        raise AuthenticationFailure(
            "Classical QKD message has an unexpected receiver."
        )

    if envelope.session_id != expected_session_id:
        raise AuthenticationFailure(
            "Classical QKD message belongs to another session."
        )

    validate_timestamp_freshness(envelope.issued_at)

    payload = envelope.payload

    if (
        payload.get("protocol_name")
        != PHASE_5_PROTOCOL_NAME
    ):
        raise AuthenticationFailure(
            "Unexpected QKD protocol name."
        )

    if (
        payload.get("protocol_version")
        != PHASE_5_PROTOCOL_VERSION
    ):
        raise AuthenticationFailure(
            "Unexpected QKD protocol version."
        )

    if payload.get("operation") != expected_operation:
        raise AuthenticationFailure(
            "Unexpected QKD classical operation."
        )

    if (
        payload.get("sequence_number")
        != expected_sequence_number
    ):
        raise AuthenticationFailure(
            "QKD classical message is out of sequence."
        )

    if (
        payload.get("previous_message_hash")
        != expected_previous_hash
    ):
        raise AuthenticationFailure(
            "QKD classical transcript-chain mismatch."
        )

    if not verify_received_envelope_signature(
        receiver,
        envelope,
    ):
        raise AuthenticationFailure(
            "ML-DSA signature on QKD classical message "
            "is invalid."
        )

    return sha3_256_hex_local(
        canonical_json(envelope.to_dictionary())
    )


def send_signed_qkd_message(
    *,
    transport: InMemoryMessageTransport,
    sender: IsolatedNode,
    receiver: IsolatedNode,
    message: SignedQKDClassicalMessage,
) -> str:
    """
    Serialize and transmit a signed QKD message.
    """

    return transport.send(
        sender_id=sender.node_id,
        receiver_id=receiver.node_id,
        message=message.to_dictionary(),
    )


def receive_signed_qkd_message(
    *,
    transport: InMemoryMessageTransport,
    receiver: IsolatedNode,
) -> SignedQKDClassicalMessage:
    """
    Receive and reconstruct a signed QKD message.
    """

    received = transport.receive(receiver.node_id)

    if received is None:
        raise AuthenticationFailure(
            "No signed QKD message is waiting."
        )

    packet, dictionary = received

    envelope = envelope_from_dictionary(dictionary)

    if packet.sender_id != envelope.sender_id:
        raise AuthenticationFailure(
            "Transport sender differs from signed sender."
        )

    if packet.receiver_id != envelope.receiver_id:
        raise AuthenticationFailure(
            "Transport receiver differs from signed receiver."
        )

    return SignedQKDClassicalMessage(envelope=envelope)


print("Dilithium-signed QKD classical-message layer ready.")

Dilithium-signed QKD classical-message layer ready.


In [61]:
#@title QKD Protocol with Noise (Channel Error)

@dataclass(frozen=True)
class BiasedBasisQuantumTransmission:
    signal_count: int
    terminal_bits: np.ndarray
    terminal_bases: np.ndarray
    server_bases: np.ndarray
    server_measurements: np.ndarray
    channel_error_rate: float
    eavesdropper_present: bool
    terminal_z_count: int
    terminal_x_count: int
    server_z_count: int
    server_x_count: int


def simulate_t12_biased_quantum_transmission(
    *,
    signal_count: int = T12_DEFAULT_SIGNAL_COUNT,
    channel_error_rate: float = 0.01,
    eavesdropper_present: bool = False,
) -> BiasedBasisQuantumTransmission:
    """
    Simulate an 80:20 biased-basis prepare-and-measure exchange.

    Terminal acts as the transmitter.
    Server acts as the receiver.

    When bases match:
        Server obtains the terminal bit before channel noise.

    When bases differ:
        Measurement is uniformly random.

    A simulated intercept-resend attacker measures each signal
    in an independently selected 80:20 basis.
    """

    if signal_count <= 0:
        raise ValueError("signal_count must be positive.")

    if not 0.0 <= channel_error_rate <= 1.0:
        raise ValueError(
            "channel_error_rate must lie between 0 and 1."
        )

    terminal_bits = generate_random_bits(signal_count)
    terminal_bases = generate_t12_biased_bases(
        signal_count
    )
    server_bases = generate_t12_biased_bases(
        signal_count
    )

    rng = np.random.default_rng(
        int.from_bytes(
            secrets.token_bytes(16),
            byteorder="big",
        )
    )

    if eavesdropper_present:
        eve_bases = generate_t12_biased_bases(
            signal_count
        )

        eve_random_measurements = generate_random_bits(
            signal_count
        )

        eve_measurements = np.where(
            eve_bases == terminal_bases,
            terminal_bits,
            eve_random_measurements,
        ).astype(np.uint8)

        server_random_measurements = generate_random_bits(
            signal_count
        )

        server_measurements = np.where(
            server_bases == eve_bases,
            eve_measurements,
            server_random_measurements,
        ).astype(np.uint8)

    else:
        server_random_measurements = generate_random_bits(
            signal_count
        )

        server_measurements = np.where(
            server_bases == terminal_bases,
            terminal_bits,
            server_random_measurements,
        ).astype(np.uint8)

    # Physical/channel-induced bit flips are added after measurement.
    channel_flip_mask = (
        rng.random(signal_count)
        < channel_error_rate
    )

    server_measurements = (
        server_measurements
        ^ channel_flip_mask.astype(np.uint8)
    )

    return BiasedBasisQuantumTransmission(
        signal_count=signal_count,
        terminal_bits=terminal_bits,
        terminal_bases=terminal_bases,
        server_bases=server_bases,
        server_measurements=server_measurements,
        channel_error_rate=channel_error_rate,
        eavesdropper_present=eavesdropper_present,
        terminal_z_count=int(
            np.sum(terminal_bases == 0)
        ),
        terminal_x_count=int(
            np.sum(terminal_bases == 1)
        ),
        server_z_count=int(
            np.sum(server_bases == 0)
        ),
        server_x_count=int(
            np.sum(server_bases == 1)
        ),
    )


print("T12-inspired biased-basis quantum simulator ready.")

T12-inspired biased-basis quantum simulator ready.


In [62]:
#@title Basis Reconciliation

@dataclass(frozen=True)
class BasisReconciliationResult:
    terminal_sifted_key: np.ndarray
    server_sifted_key: np.ndarray
    matched_indices: np.ndarray
    sifted_length: int
    observed_match_rate: float
    terminal_z_fraction: float
    server_z_fraction: float
    transcript_hash: str
    messages_exchanged: int


def perform_signed_basis_reconciliation(
    *,
    terminal_node: TerminalNode,
    server_node: ServerNode,
    transport: InMemoryMessageTransport,
    qkd_session_id: str,
    transmission: BiasedBasisQuantumTransmission,
    starting_sequence_number: int = 1,
    previous_message_hash: Optional[str] = None,
) -> BasisReconciliationResult:
    """
    Exchange both basis strings using ML-DSA-authenticated messages.
    """

    sequence_number = starting_sequence_number
    transcript_hash = previous_message_hash

    # --------------------------------------------------------
    # Terminal announces its preparation bases.
    # --------------------------------------------------------

    terminal_announcement = create_signed_qkd_message(
        sender=terminal_node,
        receiver=server_node,
        qkd_session_id=qkd_session_id,
        operation="TERMINAL_BASIS_ANNOUNCEMENT",
        sequence_number=sequence_number,
        previous_message_hash=transcript_hash,
        payload={
            "signal_count": transmission.signal_count,
            "basis_encoding": {
                "Z": 0,
                "X": 1,
            },
            "z_basis_probability":
                T12_Z_BASIS_PROBABILITY,
            "bases":
                transmission.terminal_bases.tolist(),
        },
    )

    send_signed_qkd_message(
        transport=transport,
        sender=terminal_node,
        receiver=server_node,
        message=terminal_announcement,
    )

    received_terminal_announcement = (
        receive_signed_qkd_message(
            transport=transport,
            receiver=server_node,
        )
    )

    transcript_hash = verify_signed_qkd_message(
        receiver=server_node,
        expected_sender=terminal_node,
        message=received_terminal_announcement,
        expected_session_id=qkd_session_id,
        expected_operation=
            "TERMINAL_BASIS_ANNOUNCEMENT",
        expected_sequence_number=sequence_number,
        expected_previous_hash=previous_message_hash,
    )

    sequence_number += 1

    # --------------------------------------------------------
    # Server announces its measurement bases.
    # --------------------------------------------------------

    server_announcement = create_signed_qkd_message(
        sender=server_node,
        receiver=terminal_node,
        qkd_session_id=qkd_session_id,
        operation="SERVER_BASIS_ANNOUNCEMENT",
        sequence_number=sequence_number,
        previous_message_hash=transcript_hash,
        payload={
            "signal_count": transmission.signal_count,
            "basis_encoding": {
                "Z": 0,
                "X": 1,
            },
            "z_basis_probability":
                T12_Z_BASIS_PROBABILITY,
            "bases":
                transmission.server_bases.tolist(),
        },
    )

    send_signed_qkd_message(
        transport=transport,
        sender=server_node,
        receiver=terminal_node,
        message=server_announcement,
    )

    received_server_announcement = (
        receive_signed_qkd_message(
            transport=transport,
            receiver=terminal_node,
        )
    )

    transcript_hash = verify_signed_qkd_message(
        receiver=terminal_node,
        expected_sender=server_node,
        message=received_server_announcement,
        expected_session_id=qkd_session_id,
        expected_operation=
            "SERVER_BASIS_ANNOUNCEMENT",
        expected_sequence_number=sequence_number,
        expected_previous_hash=transcript_hash,
    )

    matched_indices = np.flatnonzero(
        transmission.terminal_bases
        == transmission.server_bases
    )

    terminal_sifted = transmission.terminal_bits[
        matched_indices
    ].copy()

    server_sifted = transmission.server_measurements[
        matched_indices
    ].copy()

    observed_match_rate = (
        len(matched_indices)
        / transmission.signal_count
    )

    return BasisReconciliationResult(
        terminal_sifted_key=terminal_sifted,
        server_sifted_key=server_sifted,
        matched_indices=matched_indices,
        sifted_length=len(matched_indices),
        observed_match_rate=observed_match_rate,
        terminal_z_fraction=(
            transmission.terminal_z_count
            / transmission.signal_count
        ),
        server_z_fraction=(
            transmission.server_z_count
            / transmission.signal_count
        ),
        transcript_hash=transcript_hash,
        messages_exchanged=2,
    )


print("Signed basis-reconciliation workflow ready.")

Signed basis-reconciliation workflow ready.


In [63]:
#@title Parameter Decision

@dataclass(frozen=True)
class ParameterEstimationResult:
    accepted: bool
    estimated_qber: float
    sample_size: int
    remaining_terminal_key: np.ndarray
    remaining_server_key: np.ndarray
    disclosed_indices: np.ndarray
    transcript_hash: str
    next_sequence_number: int
    messages_exchanged: int


def perform_signed_parameter_estimation(
    *,
    terminal_node: TerminalNode,
    server_node: ServerNode,
    transport: InMemoryMessageTransport,
    qkd_session_id: str,
    terminal_sifted_key: np.ndarray,
    server_sifted_key: np.ndarray,
    starting_sequence_number: int,
    previous_message_hash: str,
) -> ParameterEstimationResult:
    """
    Randomly sample and disclose a portion of the sifted key.

    The disclosed positions are removed from both keys.
    """

    terminal_sifted_key = validate_bit_array(
        terminal_sifted_key
    )

    server_sifted_key = validate_bit_array(
        server_sifted_key
    )

    if len(terminal_sifted_key) != len(server_sifted_key):
        raise ValueError(
            "Sifted keys must have equal lengths."
        )

    sample_size = max(
        64,
        int(
            len(terminal_sifted_key)
            * T12_PARAMETER_ESTIMATION_FRACTION
        ),
    )

    if sample_size >= len(terminal_sifted_key):
        raise AuthenticationFailure(
            "Sifted key is too short for parameter estimation."
        )

    rng = np.random.default_rng(
        int.from_bytes(
            secrets.token_bytes(16),
            byteorder="big",
        )
    )

    disclosed_indices = np.sort(
        rng.choice(
            len(terminal_sifted_key),
            size=sample_size,
            replace=False,
        )
    )

    sequence_number = starting_sequence_number

    # --------------------------------------------------------
    # Terminal discloses selected positions and values.
    # --------------------------------------------------------

    terminal_sample_message = create_signed_qkd_message(
        sender=terminal_node,
        receiver=server_node,
        qkd_session_id=qkd_session_id,
        operation="QBER_SAMPLE_DISCLOSURE",
        sequence_number=sequence_number,
        previous_message_hash=previous_message_hash,
        payload={
            "sample_indices":
                disclosed_indices.tolist(),
            "sample_bits":
                terminal_sifted_key[
                    disclosed_indices
                ].tolist(),
            "sifted_key_length":
                len(terminal_sifted_key),
        },
    )

    send_signed_qkd_message(
        transport=transport,
        sender=terminal_node,
        receiver=server_node,
        message=terminal_sample_message,
    )

    received_sample = receive_signed_qkd_message(
        transport=transport,
        receiver=server_node,
    )

    transcript_hash = verify_signed_qkd_message(
        receiver=server_node,
        expected_sender=terminal_node,
        message=received_sample,
        expected_session_id=qkd_session_id,
        expected_operation="QBER_SAMPLE_DISCLOSURE",
        expected_sequence_number=sequence_number,
        expected_previous_hash=previous_message_hash,
    )

    terminal_sample = terminal_sifted_key[
        disclosed_indices
    ]

    server_sample = server_sifted_key[
        disclosed_indices
    ]

    estimated_qber = float(
        np.mean(terminal_sample != server_sample)
    )

    accepted = (
        estimated_qber
        <= T12_QBER_ABORT_THRESHOLD
    )

    sequence_number += 1

    # --------------------------------------------------------
    # Server returns its signed decision.
    # --------------------------------------------------------

    decision_message = create_signed_qkd_message(
        sender=server_node,
        receiver=terminal_node,
        qkd_session_id=qkd_session_id,
        operation="QBER_ESTIMATION_RESULT",
        sequence_number=sequence_number,
        previous_message_hash=transcript_hash,
        payload={
            "sample_size": sample_size,
            "estimated_qber": estimated_qber,
            "abort_threshold":
                T12_QBER_ABORT_THRESHOLD,
            "accepted": accepted,
        },
    )

    send_signed_qkd_message(
        transport=transport,
        sender=server_node,
        receiver=terminal_node,
        message=decision_message,
    )

    received_decision = receive_signed_qkd_message(
        transport=transport,
        receiver=terminal_node,
    )

    transcript_hash = verify_signed_qkd_message(
        receiver=terminal_node,
        expected_sender=server_node,
        message=received_decision,
        expected_session_id=qkd_session_id,
        expected_operation="QBER_ESTIMATION_RESULT",
        expected_sequence_number=sequence_number,
        expected_previous_hash=transcript_hash,
    )

    keep_mask = np.ones(
        len(terminal_sifted_key),
        dtype=bool,
    )

    keep_mask[disclosed_indices] = False

    remaining_terminal_key = (
        terminal_sifted_key[keep_mask]
    )

    remaining_server_key = (
        server_sifted_key[keep_mask]
    )

    return ParameterEstimationResult(
        accepted=accepted,
        estimated_qber=estimated_qber,
        sample_size=sample_size,
        remaining_terminal_key=
            remaining_terminal_key,
        remaining_server_key=
            remaining_server_key,
        disclosed_indices=disclosed_indices,
        transcript_hash=transcript_hash,
        next_sequence_number=sequence_number + 1,
        messages_exchanged=2,
    )


print("Signed parameter-estimation workflow ready.")

Signed parameter-estimation workflow ready.


In [64]:
#@title Full Signed Cascade Reconciliation with Recursive Look-Back

from collections import deque

# ============================================================
# CASCADE LOOK-BACK CONFIGURATION
# ============================================================

T12_CASCADE_LOOKBACK_ENABLED = True

# Prevents a malformed state from creating an endless correction loop.
T12_CASCADE_MAX_CORRECTIONS_FACTOR = 2

# Exact final equality is available only because this is a simulation.
# It is used for diagnostics after the protocol-level HMAC confirmation.
T12_CASCADE_SIMULATION_DIAGNOSTICS = True

@dataclass(frozen=True)
class ReconciliationResult:
    success: bool
    terminal_reconciled_key: np.ndarray
    server_reconciled_key: np.ndarray
    corrected_errors: int
    parity_bits_disclosed: int
    passes_completed: int
    verification_tag_bits_disclosed: int
    transcript_hash: str
    next_sequence_number: int
    messages_exchanged: int
    failure_reason: Optional[str]


def deterministic_permutation(
    *,
    length: int,
    qkd_session_id: str,
    pass_number: int,
    initial_transcript_hash: str,
) -> np.ndarray:
    """
    Derive a deterministic permutation from public session data.
    """

    seed_material = canonical_json({
        "domain": "QBioAuth-Cascade-Permutation-v1",
        "qkd_session_id": qkd_session_id,
        "pass_number": pass_number,
        "initial_transcript_hash":
            initial_transcript_hash,
    })

    seed = int.from_bytes(
        hashlib.sha3_256(seed_material).digest()[:16],
        byteorder="big",
    )

    return np.random.default_rng(seed).permutation(
        length
    )


def cascade_block_size(
    *,
    pass_number: int,
    estimated_qber: float,
) -> int:
    """
    Select a block size for one reconciliation pass.

    The first block size is bounded to avoid pathological values.
    Later passes double it.
    """

    if estimated_qber > 0:
        qber_based_size = max(
            4,
            int(0.73 / estimated_qber),
        )
    else:
        qber_based_size = T12_INITIAL_BLOCK_SIZE

    first_size = min(
        64,
        max(
            T12_INITIAL_BLOCK_SIZE,
            qber_based_size,
        ),
    )

    return first_size * (2 ** pass_number)


def split_indices_into_blocks(
    indices: np.ndarray,
    block_size: int,
) -> List[np.ndarray]:
    """
    Split permuted indices into blocks.
    """

    return [
        indices[start:start + block_size]
        for start in range(
            0,
            len(indices),
            block_size,
        )
    ]

@dataclass(frozen=True)
class CascadePassLayout:
    """
    Public deterministic layout for one Cascade pass.

    blocks:
        Original key indices grouped according to this pass's
        deterministic permutation.

    index_to_block:
        Maps every original key index to the block containing it.
    """

    pass_number: int
    block_size: int
    permutation: np.ndarray
    blocks: Tuple[np.ndarray, ...]
    index_to_block: np.ndarray


def build_cascade_pass_layout(
    *,
    key_length: int,
    qkd_session_id: str,
    pass_number: int,
    estimated_qber: float,
    initial_transcript_hash: str,
) -> CascadePassLayout:
    """
    Build one deterministic Cascade pass layout.

    No secret key material is used to construct the permutation.
    Both parties can independently derive the same public layout.
    """

    if key_length <= 0:
        raise ValueError("Cascade key length must be positive.")

    block_size = cascade_block_size(
        pass_number=pass_number,
        estimated_qber=estimated_qber,
    )

    permutation = deterministic_permutation(
        length=key_length,
        qkd_session_id=qkd_session_id,
        pass_number=pass_number,
        initial_transcript_hash=initial_transcript_hash,
    )

    blocks = tuple(
        split_indices_into_blocks(
            permutation,
            block_size,
        )
    )

    index_to_block = np.empty(
        key_length,
        dtype=np.int64,
    )

    for block_number, block_indices in enumerate(blocks):
        index_to_block[block_indices] = block_number

    return CascadePassLayout(
        pass_number=pass_number,
        block_size=block_size,
        permutation=permutation,
        blocks=blocks,
        index_to_block=index_to_block,
    )


def perform_signed_key_reconciliation(
    *,
    terminal_node: TerminalNode,
    server_node: ServerNode,
    transport: InMemoryMessageTransport,
    qkd_session_id: str,
    terminal_key: np.ndarray,
    server_key: np.ndarray,
    estimated_qber: float,
    starting_sequence_number: int,
    previous_message_hash: str,
) -> ReconciliationResult:
    """
    Correct the Server key toward the Terminal key using a complete
    Cascade-style reconciliation process with recursive look-back.

    Improvement over the previous implementation
    --------------------------------------------
    The earlier routine corrected odd-parity blocks in each pass but did
    not revisit blocks from earlier passes after a later correction.

    That omission leaves paired errors hidden:

        earlier block contains two errors -> parity appears equal
        later pass corrects one error     -> earlier parity becomes odd
        no look-back                     -> remaining error survives

    This implementation maintains every pass layout and recursively
    revisits all previously established blocks affected by each corrected
    bit. All parity queries, binary-search responses, confirmation values
    and final decisions remain ML-DSA signed and transcript chained.

    Notes
    -----
    * This is a research simulation, not a constant-time production
      implementation.
    * Every disclosed parity bit is counted as reconciliation leakage.
    * The final HMAC confirmation is still required before privacy
      amplification.
    """

    terminal_key = validate_bit_array(
        terminal_key
    ).copy()

    server_key = validate_bit_array(
        server_key
    ).copy()

    if len(terminal_key) != len(server_key):
        raise ValueError(
            "Reconciliation keys must have equal lengths."
        )

    if len(terminal_key) < T12_MINIMUM_RECONCILIATION_BITS:
        return ReconciliationResult(
            success=False,
            terminal_reconciled_key=terminal_key,
            server_reconciled_key=server_key,
            corrected_errors=0,
            parity_bits_disclosed=0,
            passes_completed=0,
            verification_tag_bits_disclosed=0,
            transcript_hash=previous_message_hash,
            next_sequence_number=starting_sequence_number,
            messages_exchanged=0,
            failure_reason=(
                "Insufficient post-estimation key material."
            ),
        )

    if not 0.0 <= estimated_qber <= 1.0:
        raise ValueError(
            "Estimated QBER must lie between zero and one."
        )

    sequence_number = starting_sequence_number
    transcript_hash = previous_message_hash
    initial_transcript_hash = previous_message_hash

    parity_bits_disclosed = 0
    corrected_errors = 0
    message_count = 0
    passes_completed = 0

    pass_layouts: List[CascadePassLayout] = []

    maximum_corrections = max(
        1,
        len(server_key)
        * T12_CASCADE_MAX_CORRECTIONS_FACTOR,
    )

    # --------------------------------------------------------
    # Signed parity exchange
    # --------------------------------------------------------

    def exchange_block_parities(
        *,
        layout: CascadePassLayout,
        block_number: int,
        reason: str,
    ) -> Tuple[int, int]:
        """
        Exchange Server and Terminal parity values for one public block.

        Returns:
            terminal_parity, server_parity
        """

        nonlocal sequence_number
        nonlocal transcript_hash
        nonlocal parity_bits_disclosed
        nonlocal message_count

        block_indices = layout.blocks[block_number]

        terminal_parity = parity_of_bits(
            terminal_key[block_indices]
        )

        server_parity = parity_of_bits(
            server_key[block_indices]
        )

        query_message = create_signed_qkd_message(
            sender=server_node,
            receiver=terminal_node,
            qkd_session_id=qkd_session_id,
            operation="CASCADE_PARITY_QUERY",
            sequence_number=sequence_number,
            previous_message_hash=transcript_hash,
            payload={
                "pass_number": layout.pass_number,
                "block_number": block_number,
                "block_size": len(block_indices),
                "reason": reason,
                "server_parity": server_parity,
            },
        )

        send_signed_qkd_message(
            transport=transport,
            sender=server_node,
            receiver=terminal_node,
            message=query_message,
        )

        received_query = receive_signed_qkd_message(
            transport=transport,
            receiver=terminal_node,
        )

        transcript_hash = verify_signed_qkd_message(
            receiver=terminal_node,
            expected_sender=server_node,
            message=received_query,
            expected_session_id=qkd_session_id,
            expected_operation="CASCADE_PARITY_QUERY",
            expected_sequence_number=sequence_number,
            expected_previous_hash=transcript_hash,
        )

        sequence_number += 1
        message_count += 1

        response_message = create_signed_qkd_message(
            sender=terminal_node,
            receiver=server_node,
            qkd_session_id=qkd_session_id,
            operation="CASCADE_PARITY_RESPONSE",
            sequence_number=sequence_number,
            previous_message_hash=transcript_hash,
            payload={
                "pass_number": layout.pass_number,
                "block_number": block_number,
                "reason": reason,
                "terminal_parity": terminal_parity,
            },
        )

        send_signed_qkd_message(
            transport=transport,
            sender=terminal_node,
            receiver=server_node,
            message=response_message,
        )

        received_response = receive_signed_qkd_message(
            transport=transport,
            receiver=server_node,
        )

        transcript_hash = verify_signed_qkd_message(
            receiver=server_node,
            expected_sender=terminal_node,
            message=received_response,
            expected_session_id=qkd_session_id,
            expected_operation="CASCADE_PARITY_RESPONSE",
            expected_sequence_number=sequence_number,
            expected_previous_hash=transcript_hash,
        )

        sequence_number += 1
        message_count += 1

        # Both parties disclosed one parity bit.
        parity_bits_disclosed += 2

        return terminal_parity, server_parity

    # --------------------------------------------------------
    # Signed binary search
    # --------------------------------------------------------

    def locate_and_correct_single_error(
        *,
        layout: CascadePassLayout,
        block_number: int,
        reason: str,
    ) -> int:
        """
        Locate one odd-parity error inside a block and flip the
        corresponding Server bit.

        Returns:
            corrected original key index
        """

        nonlocal sequence_number
        nonlocal transcript_hash
        nonlocal parity_bits_disclosed
        nonlocal corrected_errors
        nonlocal message_count

        search_indices = layout.blocks[
            block_number
        ].copy()

        if len(search_indices) == 0:
            raise RuntimeError(
                "Cascade attempted to search an empty block."
            )

        while len(search_indices) > 1:
            midpoint = len(search_indices) // 2

            left_indices = search_indices[:midpoint]
            right_indices = search_indices[midpoint:]

            terminal_left_parity = parity_of_bits(
                terminal_key[left_indices]
            )

            server_left_parity = parity_of_bits(
                server_key[left_indices]
            )

            query_message = create_signed_qkd_message(
                sender=server_node,
                receiver=terminal_node,
                qkd_session_id=qkd_session_id,
                operation="CASCADE_BINARY_QUERY",
                sequence_number=sequence_number,
                previous_message_hash=transcript_hash,
                payload={
                    "pass_number": layout.pass_number,
                    "block_number": block_number,
                    "reason": reason,
                    "candidate_length":
                        len(search_indices),
                    "left_length":
                        len(left_indices),
                    "server_left_parity":
                        server_left_parity,
                },
            )

            send_signed_qkd_message(
                transport=transport,
                sender=server_node,
                receiver=terminal_node,
                message=query_message,
            )

            received_query = receive_signed_qkd_message(
                transport=transport,
                receiver=terminal_node,
            )

            transcript_hash = verify_signed_qkd_message(
                receiver=terminal_node,
                expected_sender=server_node,
                message=received_query,
                expected_session_id=qkd_session_id,
                expected_operation="CASCADE_BINARY_QUERY",
                expected_sequence_number=sequence_number,
                expected_previous_hash=transcript_hash,
            )

            sequence_number += 1
            message_count += 1

            response_message = create_signed_qkd_message(
                sender=terminal_node,
                receiver=server_node,
                qkd_session_id=qkd_session_id,
                operation="CASCADE_BINARY_RESPONSE",
                sequence_number=sequence_number,
                previous_message_hash=transcript_hash,
                payload={
                    "pass_number": layout.pass_number,
                    "block_number": block_number,
                    "reason": reason,
                    "terminal_left_parity":
                        terminal_left_parity,
                },
            )

            send_signed_qkd_message(
                transport=transport,
                sender=terminal_node,
                receiver=server_node,
                message=response_message,
            )

            received_response = receive_signed_qkd_message(
                transport=transport,
                receiver=server_node,
            )

            transcript_hash = verify_signed_qkd_message(
                receiver=server_node,
                expected_sender=terminal_node,
                message=received_response,
                expected_session_id=qkd_session_id,
                expected_operation="CASCADE_BINARY_RESPONSE",
                expected_sequence_number=sequence_number,
                expected_previous_hash=transcript_hash,
            )

            sequence_number += 1
            message_count += 1

            # Both parties disclosed one parity bit.
            parity_bits_disclosed += 2

            if (
                terminal_left_parity
                != server_left_parity
            ):
                search_indices = left_indices
            else:
                search_indices = right_indices

        corrected_index = int(
            search_indices[0]
        )

        server_key[corrected_index] ^= 1
        corrected_errors += 1

        if corrected_errors > maximum_corrections:
            raise RuntimeError(
                "Cascade correction safeguard exceeded. "
                "The reconciliation state is inconsistent."
            )

        return corrected_index

    # --------------------------------------------------------
    # Recursive look-back queue
    # --------------------------------------------------------

    def process_lookback_queue(
        *,
        corrected_index: int,
        source_pass_number: int,
        source_block_number: int,
        highest_available_pass: int,
    ) -> None:
        """
        Revisit every already-created pass block affected by a correction.

        When a new correction occurs during look-back, all other affected
        blocks are queued again. This is the Cascade mechanism that
        exposes errors which were previously hidden in even-error blocks.
        """

        if not T12_CASCADE_LOOKBACK_ENABLED:
            return

        queue: deque[Tuple[int, int]] = deque()
        queued: Set[Tuple[int, int]] = set()

        def enqueue_affected_blocks(
            key_index: int,
            *,
            exclude: Optional[Tuple[int, int]] = None,
        ) -> None:
            for pass_number in range(
                highest_available_pass + 1
            ):
                layout = pass_layouts[pass_number]
                block_number = int(
                    layout.index_to_block[key_index]
                )
                identifier = (
                    pass_number,
                    block_number,
                )

                if identifier == exclude:
                    continue

                if identifier not in queued:
                    queue.append(identifier)
                    queued.add(identifier)

        enqueue_affected_blocks(
            corrected_index,
            exclude=(
                source_pass_number,
                source_block_number,
            ),
        )

        while queue:
            pass_number, block_number = (
                queue.popleft()
            )
            queued.discard(
                (pass_number, block_number)
            )

            layout = pass_layouts[pass_number]

            terminal_parity, server_parity = (
                exchange_block_parities(
                    layout=layout,
                    block_number=block_number,
                    reason="RECURSIVE_LOOKBACK",
                )
            )

            if terminal_parity == server_parity:
                continue

            newly_corrected_index = (
                locate_and_correct_single_error(
                    layout=layout,
                    block_number=block_number,
                    reason="RECURSIVE_LOOKBACK",
                )
            )

            # The new bit correction toggles one block in every existing
            # pass. Revisit all affected blocks except the block that has
            # just been corrected.
            enqueue_affected_blocks(
                newly_corrected_index,
                exclude=(
                    pass_number,
                    block_number,
                ),
            )

    # --------------------------------------------------------
    # Main Cascade passes
    # --------------------------------------------------------

    for pass_number in range(
        T12_RECONCILIATION_PASSES
    ):
        layout = build_cascade_pass_layout(
            key_length=len(terminal_key),
            qkd_session_id=qkd_session_id,
            pass_number=pass_number,
            estimated_qber=estimated_qber,
            initial_transcript_hash=
                initial_transcript_hash,
        )

        pass_layouts.append(layout)

        for block_number in range(
            len(layout.blocks)
        ):
            terminal_parity, server_parity = (
                exchange_block_parities(
                    layout=layout,
                    block_number=block_number,
                    reason="FORWARD_PASS",
                )
            )

            if terminal_parity == server_parity:
                continue

            corrected_index = (
                locate_and_correct_single_error(
                    layout=layout,
                    block_number=block_number,
                    reason="FORWARD_PASS",
                )
            )

            process_lookback_queue(
                corrected_index=corrected_index,
                source_pass_number=pass_number,
                source_block_number=block_number,
                highest_available_pass=pass_number,
            )

        passes_completed = pass_number + 1

    # --------------------------------------------------------
    # Final signed key confirmation
    # --------------------------------------------------------

    confirmation_nonce = secrets.token_bytes(32)

    terminal_confirmation_tag = hmac.new(
        key=sha3_256_bytes(
            bits_to_packed_bytes(
                terminal_key
            )
        ),
        msg=(
            b"QBioAuth|CascadeLookbackConfirmation|"
            + qkd_session_id.encode("utf-8")
            + b"|"
            + confirmation_nonce
        ),
        digestmod=hashlib.sha3_256,
    ).digest()

    server_confirmation_tag = hmac.new(
        key=sha3_256_bytes(
            bits_to_packed_bytes(
                server_key
            )
        ),
        msg=(
            b"QBioAuth|CascadeLookbackConfirmation|"
            + qkd_session_id.encode("utf-8")
            + b"|"
            + confirmation_nonce
        ),
        digestmod=hashlib.sha3_256,
    ).digest()

    confirmation_message = create_signed_qkd_message(
        sender=terminal_node,
        receiver=server_node,
        qkd_session_id=qkd_session_id,
        operation="CASCADE_CONFIRMATION",
        sequence_number=sequence_number,
        previous_message_hash=transcript_hash,
        payload={
            "confirmation_nonce_b64":
                bytes_to_b64(
                    confirmation_nonce
                ),
            "terminal_confirmation_tag_b64":
                bytes_to_b64(
                    terminal_confirmation_tag
                ),
            "passes_completed":
                passes_completed,
            "corrected_errors":
                corrected_errors,
            "lookback_enabled":
                T12_CASCADE_LOOKBACK_ENABLED,
        },
    )

    send_signed_qkd_message(
        transport=transport,
        sender=terminal_node,
        receiver=server_node,
        message=confirmation_message,
    )

    received_confirmation = (
        receive_signed_qkd_message(
            transport=transport,
            receiver=server_node,
        )
    )

    transcript_hash = verify_signed_qkd_message(
        receiver=server_node,
        expected_sender=terminal_node,
        message=received_confirmation,
        expected_session_id=qkd_session_id,
        expected_operation="CASCADE_CONFIRMATION",
        expected_sequence_number=sequence_number,
        expected_previous_hash=transcript_hash,
    )

    sequence_number += 1
    message_count += 1

    reconciliation_success = hmac.compare_digest(
        terminal_confirmation_tag,
        server_confirmation_tag,
    )

    verification_tag_bits_disclosed = 256

    residual_error_count = int(
        np.sum(
            terminal_key != server_key
        )
    )

    if (
        T12_CASCADE_SIMULATION_DIAGNOSTICS
        and reconciliation_success
        and residual_error_count != 0
    ):
        raise RuntimeError(
            "Cascade confirmation collision detected in simulation."
        )

    if (
        T12_CASCADE_SIMULATION_DIAGNOSTICS
        and not reconciliation_success
        and residual_error_count == 0
    ):
        raise RuntimeError(
            "Identical reconciled keys produced different "
            "confirmation tags."
        )

    result_message = create_signed_qkd_message(
        sender=server_node,
        receiver=terminal_node,
        qkd_session_id=qkd_session_id,
        operation="CASCADE_RESULT",
        sequence_number=sequence_number,
        previous_message_hash=transcript_hash,
        payload={
            "success":
                reconciliation_success,
            "corrected_errors":
                corrected_errors,
            "passes_completed":
                passes_completed,
            "lookback_enabled":
                T12_CASCADE_LOOKBACK_ENABLED,
            "parity_bits_disclosed":
                parity_bits_disclosed,
            "verification_tag_bits_disclosed":
                verification_tag_bits_disclosed,
            "residual_error_count_simulation_only":
                residual_error_count,
        },
    )

    send_signed_qkd_message(
        transport=transport,
        sender=server_node,
        receiver=terminal_node,
        message=result_message,
    )

    received_result = receive_signed_qkd_message(
        transport=transport,
        receiver=terminal_node,
    )

    transcript_hash = verify_signed_qkd_message(
        receiver=terminal_node,
        expected_sender=server_node,
        message=received_result,
        expected_session_id=qkd_session_id,
        expected_operation="CASCADE_RESULT",
        expected_sequence_number=sequence_number,
        expected_previous_hash=transcript_hash,
    )

    sequence_number += 1
    message_count += 1

    failure_reason = None

    if not reconciliation_success:
        failure_reason = (
            "Full Cascade look-back reconciliation left "
            f"{residual_error_count} residual key differences "
            f"after {passes_completed} passes."
        )

    return ReconciliationResult(
        success=reconciliation_success,
        terminal_reconciled_key=
            terminal_key,
        server_reconciled_key=
            server_key,
        corrected_errors=
            corrected_errors,
        parity_bits_disclosed=
            parity_bits_disclosed,
        passes_completed=
            passes_completed,
        verification_tag_bits_disclosed=
            verification_tag_bits_disclosed,
        transcript_hash=
            transcript_hash,
        next_sequence_number=
            sequence_number,
        messages_exchanged=
            message_count,
        failure_reason=
            failure_reason,
    )


print(
    "Full signed Cascade reconciliation with recursive "
    "look-back ready."
)

Full signed Cascade reconciliation with recursive look-back ready.


In [65]:
#@title QKD Privacy Amplification

@dataclass(frozen=True)
class PrivacyAmplificationResult:
    terminal_secret_bits: np.ndarray
    server_secret_bits: np.ndarray
    output_length_bits: int
    input_length_bits: int
    estimated_entropy_loss_bits: int
    reconciliation_leakage_bits: int
    security_margin_bits: int
    toeplitz_seed_b64: str
    transcript_hash: str
    next_sequence_number: int
    messages_exchanged: int


def calculate_privacy_amplification_length(
    *,
    reconciled_key_length: int,
    estimated_qber: float,
    reconciliation_leakage_bits: int,
    security_margin_bits: int,
) -> Tuple[int, int]:
    """
    Estimate a conservative privacy-amplified output length.

    For the simulator:
        entropy loss ≈ n * h2(QBER)

    This is not a formal finite-key T12 bound.
    """

    estimated_entropy_loss_bits = math.ceil(
        reconciled_key_length
        * binary_entropy(estimated_qber)
    )

    output_length = (
        reconciled_key_length
        - estimated_entropy_loss_bits
        - reconciliation_leakage_bits
        - security_margin_bits
    )

    return output_length, estimated_entropy_loss_bits


def generate_toeplitz_seed(
    *,
    input_length: int,
    output_length: int,
) -> np.ndarray:
    """
    An m x n Toeplitz matrix requires m + n - 1 random bits.
    """

    required_bits = (
        input_length + output_length - 1
    )

    return generate_random_bits(required_bits)


def apply_toeplitz_hash(
    input_bits: np.ndarray,
    seed_bits: np.ndarray,
    output_length: int,
) -> np.ndarray:
    """
    Multiply a Toeplitz matrix by input_bits over GF(2).

    Matrix dimensions:
        output_length x input_length
    """

    input_bits = validate_bit_array(input_bits)
    seed_bits = validate_bit_array(seed_bits)

    input_length = len(input_bits)

    expected_seed_length = (
        input_length + output_length - 1
    )

    if len(seed_bits) != expected_seed_length:
        raise ValueError(
            "Toeplitz seed has an invalid length."
        )

    first_column = seed_bits[:output_length]

    first_row = np.concatenate([
        first_column[:1],
        seed_bits[output_length:],
    ])

    output = np.zeros(
        output_length,
        dtype=np.uint8,
    )

    for row in range(output_length):
        accumulator = 0

        for column in range(input_length):
            if column >= row:
                toeplitz_value = first_row[
                    column - row
                ]
            else:
                toeplitz_value = first_column[
                    row - column
                ]

            accumulator ^= (
                int(toeplitz_value)
                & int(input_bits[column])
            )

        output[row] = accumulator

    return output


def perform_signed_privacy_amplification(
    *,
    terminal_node: TerminalNode,
    server_node: ServerNode,
    transport: InMemoryMessageTransport,
    qkd_session_id: str,
    terminal_reconciled_key: np.ndarray,
    server_reconciled_key: np.ndarray,
    estimated_qber: float,
    reconciliation_leakage_bits: int,
    starting_sequence_number: int,
    previous_message_hash: str,
) -> PrivacyAmplificationResult:
    """
    Agree on an authenticated Toeplitz seed and compress both keys.
    """

    terminal_reconciled_key = validate_bit_array(
        terminal_reconciled_key
    )

    server_reconciled_key = validate_bit_array(
        server_reconciled_key
    )

    if not np.array_equal(
        terminal_reconciled_key,
        server_reconciled_key,
    ):
        raise AuthenticationFailure(
            "Privacy amplification requires identical "
            "reconciled keys."
        )

    output_length, estimated_entropy_loss_bits = (
        calculate_privacy_amplification_length(
            reconciled_key_length=
                len(terminal_reconciled_key),
            estimated_qber=estimated_qber,
            reconciliation_leakage_bits=
                reconciliation_leakage_bits,
            security_margin_bits=
                T12_SECURITY_MARGIN_BITS,
        )
    )

    if output_length < T12_MINIMUM_SECRET_BITS:
        raise AuthenticationFailure(
            "Insufficient entropy remains after leakage "
            "and security deductions. "
            f"Calculated output: {output_length} bits."
        )

    toeplitz_seed = generate_toeplitz_seed(
        input_length=len(terminal_reconciled_key),
        output_length=output_length,
    )

    sequence_number = starting_sequence_number

    seed_message = create_signed_qkd_message(
        sender=server_node,
        receiver=terminal_node,
        qkd_session_id=qkd_session_id,
        operation="PRIVACY_AMPLIFICATION_PARAMETERS",
        sequence_number=sequence_number,
        previous_message_hash=previous_message_hash,
        payload={
            "hash_family": "BINARY_TOEPLITZ_GF2",
            "input_length_bits":
                len(terminal_reconciled_key),
            "output_length_bits":
                output_length,
            "estimated_entropy_loss_bits":
                estimated_entropy_loss_bits,
            "reconciliation_leakage_bits":
                reconciliation_leakage_bits,
            "security_margin_bits":
                T12_SECURITY_MARGIN_BITS,
            "toeplitz_seed_b64":
                bytes_to_b64(
                    bits_to_packed_bytes(
                        toeplitz_seed
                    )
                ),
        },
    )

    send_signed_qkd_message(
        transport=transport,
        sender=server_node,
        receiver=terminal_node,
        message=seed_message,
    )

    received_seed_message = receive_signed_qkd_message(
        transport=transport,
        receiver=terminal_node,
    )

    transcript_hash = verify_signed_qkd_message(
        receiver=terminal_node,
        expected_sender=server_node,
        message=received_seed_message,
        expected_session_id=qkd_session_id,
        expected_operation=
            "PRIVACY_AMPLIFICATION_PARAMETERS",
        expected_sequence_number=sequence_number,
        expected_previous_hash=previous_message_hash,
    )

    terminal_secret = apply_toeplitz_hash(
        terminal_reconciled_key,
        toeplitz_seed,
        output_length,
    )

    server_secret = apply_toeplitz_hash(
        server_reconciled_key,
        toeplitz_seed,
        output_length,
    )

    if not np.array_equal(
        terminal_secret,
        server_secret,
    ):
        raise AuthenticationFailure(
            "Privacy-amplified secrets differ."
        )

    sequence_number += 1

    confirmation_hash = sha3_256_hex_local(
        bits_to_packed_bytes(server_secret)
        + qkd_session_id.encode("utf-8")
    )

    confirmation_message = create_signed_qkd_message(
        sender=terminal_node,
        receiver=server_node,
        qkd_session_id=qkd_session_id,
        operation="PRIVACY_AMPLIFICATION_CONFIRMATION",
        sequence_number=sequence_number,
        previous_message_hash=transcript_hash,
        payload={
            "secret_confirmation_hash":
                confirmation_hash,
            "output_length_bits":
                output_length,
        },
    )

    send_signed_qkd_message(
        transport=transport,
        sender=terminal_node,
        receiver=server_node,
        message=confirmation_message,
    )

    received_confirmation = (
        receive_signed_qkd_message(
            transport=transport,
            receiver=server_node,
        )
    )

    transcript_hash = verify_signed_qkd_message(
        receiver=server_node,
        expected_sender=terminal_node,
        message=received_confirmation,
        expected_session_id=qkd_session_id,
        expected_operation=
            "PRIVACY_AMPLIFICATION_CONFIRMATION",
        expected_sequence_number=sequence_number,
        expected_previous_hash=transcript_hash,
    )

    return PrivacyAmplificationResult(
        terminal_secret_bits=terminal_secret,
        server_secret_bits=server_secret,
        output_length_bits=output_length,
        input_length_bits=
            len(terminal_reconciled_key),
        estimated_entropy_loss_bits=
            estimated_entropy_loss_bits,
        reconciliation_leakage_bits=
            reconciliation_leakage_bits,
        security_margin_bits=
            T12_SECURITY_MARGIN_BITS,
        toeplitz_seed_b64=bytes_to_b64(
            bits_to_packed_bytes(toeplitz_seed)
        ),
        transcript_hash=transcript_hash,
        next_sequence_number=sequence_number + 1,
        messages_exchanged=2,
    )


print("Signed Toeplitz privacy-amplification workflow ready.")

Signed Toeplitz privacy-amplification workflow ready.


In [66]:
#@title HKDF Protocol to derive AES Session Keys from the QKD shared secret keys

@dataclass(frozen=True)
class QKDAESSessionKeys:
    qkd_session_id: str
    terminal_to_server_key: bytes
    server_to_terminal_key: bytes
    terminal_to_server_nonce_base: bytes
    server_to_terminal_nonce_base: bytes
    confirmation_key: bytes
    transcript_hash: str
    derived_at: str

    def safe_summary(self) -> Dict[str, Any]:
        return {
            "qkd_session_id": self.qkd_session_id,
            "terminal_to_server_key_fingerprint":
                sha3_256_hex_local(
                    self.terminal_to_server_key
                ),
            "server_to_terminal_key_fingerprint":
                sha3_256_hex_local(
                    self.server_to_terminal_key
                ),
            "terminal_to_server_nonce_base_b64":
                bytes_to_b64(
                    self.terminal_to_server_nonce_base
                ),
            "server_to_terminal_nonce_base_b64":
                bytes_to_b64(
                    self.server_to_terminal_nonce_base
                ),
            "confirmation_key_fingerprint":
                sha3_256_hex_local(
                    self.confirmation_key
                ),
            "transcript_hash": self.transcript_hash,
            "derived_at": self.derived_at,
        }


def hkdf_expand_label(
    *,
    secret: bytes,
    salt: bytes,
    label: str,
    context: bytes,
    length: int,
) -> bytes:
    """
    Domain-separated HKDF-SHA3-256 derivation.
    """

    info = (
        b"QBioAuth|Phase5|"
        + label.encode("utf-8")
        + b"|"
        + context
    )

    return derive_key_hkdf(
        secret,
        salt=salt,
        info=info,
        length=length,
    )


def derive_qkd_aes_session_keys(
    *,
    qkd_session_id: str,
    privacy_amplified_secret_bits: np.ndarray,
    mutual_authentication_transcript_hash: str,
    qkd_transcript_hash: str,
    terminal_id: str,
    server_id: str,
) -> QKDAESSessionKeys:
    """
    Derive directional AES-256-GCM keys from the final QKD secret.
    """

    secret_material = bits_to_packed_bytes(
        privacy_amplified_secret_bits
    )

    context = canonical_json({
        "protocol_name": PHASE_5_PROTOCOL_NAME,
        "protocol_version": PHASE_5_PROTOCOL_VERSION,
        "qkd_session_id": qkd_session_id,
        "terminal_id": terminal_id,
        "server_id": server_id,
        "mutual_authentication_transcript_hash":
            mutual_authentication_transcript_hash,
        "qkd_transcript_hash":
            qkd_transcript_hash,
        "z_basis_probability":
            T12_Z_BASIS_PROBABILITY,
    })

    salt = sha3_256_bytes(
        b"QBioAuth|QKD-HKDF-Salt|"
        + context
    )

    master_secret = hkdf_expand_label(
        secret=secret_material,
        salt=salt,
        label="MASTER_SECRET",
        context=context,
        length=32,
    )

    terminal_to_server_key = hkdf_expand_label(
        secret=master_secret,
        salt=salt,
        label="TERMINAL_TO_SERVER_AES_256_GCM_KEY",
        context=context,
        length=PHASE_5_AES_KEY_BYTES,
    )

    server_to_terminal_key = hkdf_expand_label(
        secret=master_secret,
        salt=salt,
        label="SERVER_TO_TERMINAL_AES_256_GCM_KEY",
        context=context,
        length=PHASE_5_AES_KEY_BYTES,
    )

    terminal_to_server_nonce_base = hkdf_expand_label(
        secret=master_secret,
        salt=salt,
        label="TERMINAL_TO_SERVER_NONCE_BASE",
        context=context,
        length=PHASE_5_AES_NONCE_BYTES,
    )

    server_to_terminal_nonce_base = hkdf_expand_label(
        secret=master_secret,
        salt=salt,
        label="SERVER_TO_TERMINAL_NONCE_BASE",
        context=context,
        length=PHASE_5_AES_NONCE_BYTES,
    )

    confirmation_key = hkdf_expand_label(
        secret=master_secret,
        salt=salt,
        label="SESSION_CONFIRMATION_KEY",
        context=context,
        length=32,
    )

    return QKDAESSessionKeys(
        qkd_session_id=qkd_session_id,
        terminal_to_server_key=
            terminal_to_server_key,
        server_to_terminal_key=
            server_to_terminal_key,
        terminal_to_server_nonce_base=
            terminal_to_server_nonce_base,
        server_to_terminal_nonce_base=
            server_to_terminal_nonce_base,
        confirmation_key=confirmation_key,
        transcript_hash=qkd_transcript_hash,
        derived_at=utc_timestamp(),
    )


print("Transcript-bound HKDF key schedule ready.")

Transcript-bound HKDF key schedule ready.


In [67]:
#@title AES Session

@dataclass(frozen=True)
class EncryptedSessionMessage:
    qkd_session_id: str
    direction: str
    sequence_number: int
    nonce_b64: str
    ciphertext_b64: str
    associated_data_b64: str
    created_at: str

    def to_dictionary(self) -> Dict[str, Any]:
        return asdict(self)


def xor_nonce_with_counter(
    nonce_base: bytes,
    sequence_number: int,
) -> bytes:
    """
    Construct a unique 96-bit nonce from a session nonce base and
    monotonically increasing sequence number.
    """

    if len(nonce_base) != PHASE_5_AES_NONCE_BYTES:
        raise ValueError(
            "AES-GCM nonce base must contain 12 bytes."
        )

    if sequence_number < 0:
        raise ValueError(
            "Sequence number cannot be negative."
        )

    counter_bytes = sequence_number.to_bytes(
        PHASE_5_AES_NONCE_BYTES,
        byteorder="big",
    )

    return bytes(
        left ^ right
        for left, right
        in zip(nonce_base, counter_bytes)
    )


class QKDAESGCMSession:
    """
    Bidirectional authenticated-encryption session.

    This class maintains independent send and receive counters for
    terminal-to-server and server-to-terminal traffic.
    """

    def __init__(
        self,
        *,
        keys: QKDAESSessionKeys,
        terminal_id: str,
        server_id: str,
    ) -> None:

        self.keys = keys
        self.terminal_id = terminal_id
        self.server_id = server_id

        self._terminal_send_counter = 0
        self._server_receive_counter = 0

        self._server_send_counter = 0
        self._terminal_receive_counter = 0

    def _build_aad(
        self,
        *,
        direction: str,
        sequence_number: int,
        content_type: str,
    ) -> bytes:

        return canonical_json({
            "protocol_name": PHASE_5_PROTOCOL_NAME,
            "protocol_version":
                PHASE_5_PROTOCOL_VERSION,
            "qkd_session_id":
                self.keys.qkd_session_id,
            "terminal_id": self.terminal_id,
            "server_id": self.server_id,
            "qkd_transcript_hash":
                self.keys.transcript_hash,
            "direction": direction,
            "sequence_number": sequence_number,
            "content_type": content_type,
        })

    def terminal_encrypt_for_server(
        self,
        plaintext: bytes,
        *,
        content_type: str = "application/octet-stream",
    ) -> EncryptedSessionMessage:

        sequence_number = self._terminal_send_counter

        nonce = xor_nonce_with_counter(
            self.keys.terminal_to_server_nonce_base,
            sequence_number,
        )

        aad = self._build_aad(
            direction="TERMINAL_TO_SERVER",
            sequence_number=sequence_number,
            content_type=content_type,
        )

        aes = AESGCM(
            self.keys.terminal_to_server_key
        )

        ciphertext = aes.encrypt(
            nonce,
            plaintext,
            aad,
        )

        self._terminal_send_counter += 1

        return EncryptedSessionMessage(
            qkd_session_id=
                self.keys.qkd_session_id,
            direction="TERMINAL_TO_SERVER",
            sequence_number=sequence_number,
            nonce_b64=bytes_to_b64(nonce),
            ciphertext_b64=
                bytes_to_b64(ciphertext),
            associated_data_b64=
                bytes_to_b64(aad),
            created_at=utc_timestamp(),
        )

    def server_decrypt_from_terminal(
        self,
        message: EncryptedSessionMessage,
    ) -> bytes:

        expected_sequence = (
            self._server_receive_counter
        )

        if message.direction != "TERMINAL_TO_SERVER":
            raise AuthenticationFailure(
                "Encrypted message has the wrong direction."
            )

        if message.sequence_number != expected_sequence:
            raise AuthenticationFailure(
                "Replay, reordering or missing encrypted message "
                "detected."
            )

        expected_nonce = xor_nonce_with_counter(
            self.keys.terminal_to_server_nonce_base,
            expected_sequence,
        )

        received_nonce = b64_to_bytes(
            message.nonce_b64
        )

        if not hmac.compare_digest(
            expected_nonce,
            received_nonce,
        ):
            raise AuthenticationFailure(
                "AES-GCM nonce does not match session state."
            )

        aes = AESGCM(
            self.keys.terminal_to_server_key
        )

        plaintext = aes.decrypt(
            received_nonce,
            b64_to_bytes(message.ciphertext_b64),
            b64_to_bytes(
                message.associated_data_b64
            ),
        )

        self._server_receive_counter += 1

        return plaintext

    def server_encrypt_for_terminal(
        self,
        plaintext: bytes,
        *,
        content_type: str = "application/octet-stream",
    ) -> EncryptedSessionMessage:

        sequence_number = self._server_send_counter

        nonce = xor_nonce_with_counter(
            self.keys.server_to_terminal_nonce_base,
            sequence_number,
        )

        aad = self._build_aad(
            direction="SERVER_TO_TERMINAL",
            sequence_number=sequence_number,
            content_type=content_type,
        )

        aes = AESGCM(
            self.keys.server_to_terminal_key
        )

        ciphertext = aes.encrypt(
            nonce,
            plaintext,
            aad,
        )

        self._server_send_counter += 1

        return EncryptedSessionMessage(
            qkd_session_id=
                self.keys.qkd_session_id,
            direction="SERVER_TO_TERMINAL",
            sequence_number=sequence_number,
            nonce_b64=bytes_to_b64(nonce),
            ciphertext_b64=
                bytes_to_b64(ciphertext),
            associated_data_b64=
                bytes_to_b64(aad),
            created_at=utc_timestamp(),
        )

    def terminal_decrypt_from_server(
        self,
        message: EncryptedSessionMessage,
    ) -> bytes:

        expected_sequence = (
            self._terminal_receive_counter
        )

        if message.direction != "SERVER_TO_TERMINAL":
            raise AuthenticationFailure(
                "Encrypted message has the wrong direction."
            )

        if message.sequence_number != expected_sequence:
            raise AuthenticationFailure(
                "Replay, reordering or missing encrypted message "
                "detected."
            )

        expected_nonce = xor_nonce_with_counter(
            self.keys.server_to_terminal_nonce_base,
            expected_sequence,
        )

        received_nonce = b64_to_bytes(
            message.nonce_b64
        )

        if not hmac.compare_digest(
            expected_nonce,
            received_nonce,
        ):
            raise AuthenticationFailure(
                "AES-GCM nonce does not match session state."
            )

        aes = AESGCM(
            self.keys.server_to_terminal_key
        )

        plaintext = aes.decrypt(
            received_nonce,
            b64_to_bytes(message.ciphertext_b64),
            b64_to_bytes(
                message.associated_data_b64
            ),
        )

        self._terminal_receive_counter += 1

        return plaintext


print("Stateful bidirectional AES-256-GCM session ready.")

Stateful bidirectional AES-256-GCM session ready.


In [68]:
#@title Results

@dataclass(frozen=True)
class Phase5QKDSessionResult:
    success: bool
    qkd_session_id: str
    signal_count: int
    z_basis_probability: float
    x_basis_probability: float
    expected_basis_match_rate: float
    observed_basis_match_rate: float
    sifted_key_length: int
    qber_sample_size: int
    estimated_qber: float
    post_estimation_key_length: int
    reconciliation_success: bool
    reconciliation_corrected_errors: int
    reconciliation_leakage_bits: int
    privacy_amplified_length_bits: int
    qkd_transcript_hash: str
    terminal_secret_fingerprint: str
    server_secret_fingerprint: str
    hkdf_keys_match: bool
    total_signed_classical_messages: int
    completed_at: str
    failure_reason: Optional[str] = None

    def to_dictionary(self) -> Dict[str, Any]:
        return asdict(self)

In [69]:
#@title Complete Execution Pipeline

def execute_phase_5_qkd_session(
    *,
    terminal_node: TerminalNode,
    server_node: ServerNode,
    transport: InMemoryMessageTransport,
    mutual_authentication_session_id: str,
    signal_count: int = T12_DEFAULT_SIGNAL_COUNT,
    channel_error_rate: float = 0.01,
    eavesdropper_present: bool = False,
) -> Tuple[
    Phase5QKDSessionResult,
    QKDAESGCMSession,
    PrivacyAmplificationResult,
]:
    """
    Execute the full signed T12-inspired QKD-to-AES pipeline.
    """

    # --------------------------------------------------------
    # Verify Phase 2 mutual authentication.
    # --------------------------------------------------------

    terminal_auth_state = (
        terminal_node.get_local_session(
            mutual_authentication_session_id
        )
    )

    server_auth_state = (
        server_node.get_local_session(
            mutual_authentication_session_id
        )
    )

    if (
        terminal_auth_state.get(
            "authentication_state"
        )
        != "MUTUALLY_AUTHENTICATED"
    ):
        raise AuthenticationFailure(
            "Terminal is not mutually authenticated."
        )

    if (
        server_auth_state.get(
            "authentication_state"
        )
        != "MUTUALLY_AUTHENTICATED"
    ):
        raise AuthenticationFailure(
            "Server is not mutually authenticated."
        )

    mutual_auth_transcript_hash = (
        terminal_auth_state["transcript_hash"]
    )

    if (
        mutual_auth_transcript_hash
        != server_auth_state["transcript_hash"]
    ):
        raise AuthenticationFailure(
            "Terminal and server disagree on the "
            "mutual-authentication transcript."
        )

    qkd_session_id = (
        f"QKD-{generate_uuid()}"
    )

    # --------------------------------------------------------
    # Quantum transmission.
    # --------------------------------------------------------

    transmission = (
        simulate_t12_biased_quantum_transmission(
            signal_count=signal_count,
            channel_error_rate=channel_error_rate,
            eavesdropper_present=
                eavesdropper_present,
        )
    )

    # --------------------------------------------------------
    # Signed basis reconciliation.
    # --------------------------------------------------------

    basis_result = (
        perform_signed_basis_reconciliation(
            terminal_node=terminal_node,
            server_node=server_node,
            transport=transport,
            qkd_session_id=qkd_session_id,
            transmission=transmission,
            starting_sequence_number=1,
            previous_message_hash=None,
        )
    )

    # --------------------------------------------------------
    # Signed QBER estimation.
    # --------------------------------------------------------

    estimation_result = (
        perform_signed_parameter_estimation(
            terminal_node=terminal_node,
            server_node=server_node,
            transport=transport,
            qkd_session_id=qkd_session_id,
            terminal_sifted_key=
                basis_result.terminal_sifted_key,
            server_sifted_key=
                basis_result.server_sifted_key,
            starting_sequence_number=3,
            previous_message_hash=
                basis_result.transcript_hash,
        )
    )

    if not estimation_result.accepted:
        raise AuthenticationFailure(
            "QKD session aborted because estimated QBER "
            f"{estimation_result.estimated_qber:.2%} "
            "exceeded threshold "
            f"{T12_QBER_ABORT_THRESHOLD:.2%}."
        )

    # --------------------------------------------------------
    # Signed key reconciliation.
    # --------------------------------------------------------

    reconciliation_result = (
        perform_signed_key_reconciliation(
            terminal_node=terminal_node,
            server_node=server_node,
            transport=transport,
            qkd_session_id=qkd_session_id,
            terminal_key=
                estimation_result
                .remaining_terminal_key,
            server_key=
                estimation_result
                .remaining_server_key,
            estimated_qber=
                estimation_result.estimated_qber,
            starting_sequence_number=
                estimation_result
                .next_sequence_number,
            previous_message_hash=
                estimation_result.transcript_hash,
        )
    )

    if not reconciliation_result.success:
        raise AuthenticationFailure(
            reconciliation_result.failure_reason
            or "Key reconciliation failed."
        )

    reconciliation_leakage = (
        reconciliation_result
        .parity_bits_disclosed
        + reconciliation_result
        .verification_tag_bits_disclosed
    )

    # --------------------------------------------------------
    # Signed privacy amplification.
    # --------------------------------------------------------

    privacy_result = (
        perform_signed_privacy_amplification(
            terminal_node=terminal_node,
            server_node=server_node,
            transport=transport,
            qkd_session_id=qkd_session_id,
            terminal_reconciled_key=
                reconciliation_result
                .terminal_reconciled_key,
            server_reconciled_key=
                reconciliation_result
                .server_reconciled_key,
            estimated_qber=
                estimation_result.estimated_qber,
            reconciliation_leakage_bits=
                reconciliation_leakage,
            starting_sequence_number=
                reconciliation_result
                .next_sequence_number,
            previous_message_hash=
                reconciliation_result
                .transcript_hash,
        )
    )

    if not np.array_equal(
        privacy_result.terminal_secret_bits,
        privacy_result.server_secret_bits,
    ):
        raise AuthenticationFailure(
            "Final QKD secrets do not match."
        )

    # --------------------------------------------------------
    # Both sides independently derive the key schedule.
    # --------------------------------------------------------

    terminal_session_keys = (
        derive_qkd_aes_session_keys(
            qkd_session_id=qkd_session_id,
            privacy_amplified_secret_bits=
                privacy_result
                .terminal_secret_bits,
            mutual_authentication_transcript_hash=
                mutual_auth_transcript_hash,
            qkd_transcript_hash=
                privacy_result.transcript_hash,
            terminal_id=terminal_node.node_id,
            server_id=server_node.node_id,
        )
    )

    server_session_keys = (
        derive_qkd_aes_session_keys(
            qkd_session_id=qkd_session_id,
            privacy_amplified_secret_bits=
                privacy_result
                .server_secret_bits,
            mutual_authentication_transcript_hash=
                mutual_auth_transcript_hash,
            qkd_transcript_hash=
                privacy_result.transcript_hash,
            terminal_id=terminal_node.node_id,
            server_id=server_node.node_id,
        )
    )

    hkdf_keys_match = (
        hmac.compare_digest(
            terminal_session_keys
            .terminal_to_server_key,
            server_session_keys
            .terminal_to_server_key,
        )
        and hmac.compare_digest(
            terminal_session_keys
            .server_to_terminal_key,
            server_session_keys
            .server_to_terminal_key,
        )
        and hmac.compare_digest(
            terminal_session_keys
            .confirmation_key,
            server_session_keys
            .confirmation_key,
        )
    )

    if not hkdf_keys_match:
        raise AuthenticationFailure(
            "Terminal and server derived different "
            "HKDF session keys."
        )

    aes_session = QKDAESGCMSession(
        keys=terminal_session_keys,
        terminal_id=terminal_node.node_id,
        server_id=server_node.node_id,
    )

    total_signed_messages = (
        basis_result.messages_exchanged
        + estimation_result.messages_exchanged
        + reconciliation_result.messages_exchanged
        + privacy_result.messages_exchanged
    )

    terminal_secret_fingerprint = (
        sha3_256_hex_local(
            bits_to_packed_bytes(
                privacy_result
                .terminal_secret_bits
            )
        )
    )

    server_secret_fingerprint = (
        sha3_256_hex_local(
            bits_to_packed_bytes(
                privacy_result
                .server_secret_bits
            )
        )
    )

    result = Phase5QKDSessionResult(
        success=True,
        qkd_session_id=qkd_session_id,
        signal_count=signal_count,
        z_basis_probability=
            T12_Z_BASIS_PROBABILITY,
        x_basis_probability=
            T12_X_BASIS_PROBABILITY,
        expected_basis_match_rate=
            T12_EXPECTED_BASIS_MATCH_PROBABILITY,
        observed_basis_match_rate=
            basis_result.observed_match_rate,
        sifted_key_length=
            basis_result.sifted_length,
        qber_sample_size=
            estimation_result.sample_size,
        estimated_qber=
            estimation_result.estimated_qber,
        post_estimation_key_length=len(
            estimation_result
            .remaining_terminal_key
        ),
        reconciliation_success=True,
        reconciliation_corrected_errors=
            reconciliation_result
            .corrected_errors,
        reconciliation_leakage_bits=
            reconciliation_leakage,
        privacy_amplified_length_bits=
            privacy_result.output_length_bits,
        qkd_transcript_hash=
            privacy_result.transcript_hash,
        terminal_secret_fingerprint=
            terminal_secret_fingerprint,
        server_secret_fingerprint=
            server_secret_fingerprint,
        hkdf_keys_match=hkdf_keys_match,
        total_signed_classical_messages=
            total_signed_messages,
        completed_at=utc_timestamp(),
        failure_reason=None,
    )

    terminal_node.create_local_session(
        session_id=qkd_session_id,
        initial_state={
            "protocol_phase": "PHASE_5_QKD",
            "authentication_session_id":
                mutual_authentication_session_id,
            "qkd_state": "AES_SESSION_ESTABLISHED",
            "qkd_transcript_hash":
                privacy_result.transcript_hash,
            "final_secret_fingerprint":
                terminal_secret_fingerprint,
            "privacy_amplified_length_bits":
                privacy_result.output_length_bits,
            "created_at": utc_timestamp(),
        },
    )

    server_node.create_local_session(
        session_id=qkd_session_id,
        initial_state={
            "protocol_phase": "PHASE_5_QKD",
            "authentication_session_id":
                mutual_authentication_session_id,
            "qkd_state": "AES_SESSION_ESTABLISHED",
            "qkd_transcript_hash":
                privacy_result.transcript_hash,
            "final_secret_fingerprint":
                server_secret_fingerprint,
            "privacy_amplified_length_bits":
                privacy_result.output_length_bits,
            "created_at": utc_timestamp(),
        },
    )

    return result, aes_session, privacy_result


print("Complete Phase 4 QKD-to-AES protocol engine ready.")

Complete Phase 4 QKD-to-AES protocol engine ready.


In [70]:
#@title Execute QKD

(
    phase_5_result,
    qkd_aes_session,
    privacy_amplification_result,
) = execute_phase_5_qkd_session(
    terminal_node=terminal,
    server_node=server,
    transport=network,
    mutual_authentication_session_id=
        mutual_auth_result.session_id,
    signal_count=T12_DEFAULT_SIGNAL_COUNT,
    channel_error_rate=0.01,
    eavesdropper_present=False,
)

print("\n" + "=" * 82)
print("Phase 4 — T12-INSPIRED QKD SESSION")
print("=" * 82)
print(
    f"QKD session ID                 : "
    f"{phase_5_result.qkd_session_id}"
)
print(
    f"Quantum signals                : "
    f"{phase_5_result.signal_count}"
)
print(
    f"Z:X basis ratio                : "
    f"{phase_5_result.z_basis_probability:.0%}:"
    f"{phase_5_result.x_basis_probability:.0%}"
)
print(
    f"Expected basis-match rate      : "
    f"{phase_5_result.expected_basis_match_rate:.2%}"
)
print(
    f"Observed basis-match rate      : "
    f"{phase_5_result.observed_basis_match_rate:.2%}"
)
print(
    f"Sifted key length              : "
    f"{phase_5_result.sifted_key_length} bits"
)
print(
    f"QBER sample size               : "
    f"{phase_5_result.qber_sample_size} bits"
)
print(
    f"Estimated QBER                 : "
    f"{phase_5_result.estimated_qber:.2%}"
)
print(
    f"Post-estimation key length     : "
    f"{phase_5_result.post_estimation_key_length} bits"
)
print(
    f"Errors corrected               : "
    f"{phase_5_result.reconciliation_corrected_errors}"
)
print(
    f"Reconciliation leakage         : "
    f"{phase_5_result.reconciliation_leakage_bits} bits"
)
print(
    f"Privacy-amplified secret       : "
    f"{phase_5_result.privacy_amplified_length_bits} bits"
)
print(
    f"Signed classical messages      : "
    f"{phase_5_result.total_signed_classical_messages}"
)
print(
    f"Terminal secret fingerprint    : "
    f"{phase_5_result.terminal_secret_fingerprint}"
)
print(
    f"Server secret fingerprint      : "
    f"{phase_5_result.server_secret_fingerprint}"
)
print(
    f"HKDF keys identical            : "
    f"{phase_5_result.hkdf_keys_match}"
)
print("=" * 82)

assert phase_5_result.success
assert phase_5_result.reconciliation_success
assert phase_5_result.hkdf_keys_match
assert (
    phase_5_result.terminal_secret_fingerprint
    == phase_5_result.server_secret_fingerprint
)

print("Phase 4 QKD SESSION: PASS")


Phase 4 — T12-INSPIRED QKD SESSION
QKD session ID                 : QKD-74791553-bdb5-4c39-a4c7-a104057039e0
Quantum signals                : 8192
Z:X basis ratio                : 85%:15%
Expected basis-match rate      : 74.50%
Observed basis-match rate      : 74.39%
Sifted key length              : 6094 bits
QBER sample size               : 914 bits
Estimated QBER                 : 1.20%
Post-estimation key length     : 5180 bits
Errors corrected               : 67
Reconciliation leakage         : 1474 bits
Privacy-amplified secret       : 3091 bits
Signed classical messages      : 1226
Terminal secret fingerprint    : 4f11ecebef3ae9daa29b0c61ba991e4d7194a7481c8e534c0d20d5aa3f829e32
Server secret fingerprint      : 4f11ecebef3ae9daa29b0c61ba991e4d7194a7481c8e534c0d20d5aa3f829e32
HKDF keys identical            : True
Phase 4 QKD SESSION: PASS


In [71]:
#@title AES Execution T --> S

terminal_plaintext = (
    "User biometric credential validated. "
    "Requesting access to protected service."
).encode("utf-8")

terminal_encrypted_message = (
    qkd_aes_session.terminal_encrypt_for_server(
        terminal_plaintext,
        content_type="text/plain",
    )
)

server_decrypted_plaintext = (
    qkd_aes_session.server_decrypt_from_terminal(
        terminal_encrypted_message
    )
)

assert (
    server_decrypted_plaintext
    == terminal_plaintext
)

print("=" * 76)
print("TERMINAL → SERVER AES-256-GCM")
print("=" * 76)
print(
    "Original plaintext :",
    terminal_plaintext.decode("utf-8"),
)
print(
    "Ciphertext Base64  :",
    terminal_encrypted_message.ciphertext_b64,
)
print(
    "Recovered plaintext:",
    server_decrypted_plaintext.decode("utf-8"),
)
print("=" * 76)
print("TERMINAL → SERVER ENCRYPTION: PASS")

TERMINAL → SERVER AES-256-GCM
Original plaintext : User biometric credential validated. Requesting access to protected service.
Ciphertext Base64  : rUugnFWHnVKIhPDqdoyR0B545ei+rjOshD/sgZKzK+J6tHUQMuwKt8i7gfGJzrM27nm+mH9MPRgBahLmveG1IILOQca5IJj5GXf1gMmecEol1rNNYc0YcgzniRY=
Recovered plaintext: User biometric credential validated. Requesting access to protected service.
TERMINAL → SERVER ENCRYPTION: PASS


In [72]:
#@title AES Execution S --> T

server_plaintext = (
    "Access authorised. Secure application session established."
).encode("utf-8")

server_encrypted_message = (
    qkd_aes_session.server_encrypt_for_terminal(
        server_plaintext,
        content_type="text/plain",
    )
)

terminal_decrypted_plaintext = (
    qkd_aes_session.terminal_decrypt_from_server(
        server_encrypted_message
    )
)

assert (
    terminal_decrypted_plaintext
    == server_plaintext
)

print("=" * 76)
print("SERVER → TERMINAL AES-256-GCM")
print("=" * 76)
print(
    "Original plaintext :",
    server_plaintext.decode("utf-8"),
)
print(
    "Ciphertext Base64  :",
    server_encrypted_message.ciphertext_b64,
)
print(
    "Recovered plaintext:",
    terminal_decrypted_plaintext.decode("utf-8"),
)
print("=" * 76)
print("SERVER → TERMINAL ENCRYPTION: PASS")

SERVER → TERMINAL AES-256-GCM
Original plaintext : Access authorised. Secure application session established.
Ciphertext Base64  : 0gymBWoxaJ4AurDCHOZpArHzRALMT3O06YiVKsmol7uO5NZmwotagn9DnwGehyIfl7e7d8Nxi8HejYDiFVIfG7YCxHO20T3ucx0=
Recovered plaintext: Access authorised. Secure application session established.
SERVER → TERMINAL ENCRYPTION: PASS


In [73]:
#@title Tampering Test

tamper_test_session = QKDAESGCMSession(
    keys=qkd_aes_session.keys,
    terminal_id=terminal.node_id,
    server_id=server.node_id,
)

valid_tamper_test_message = (
    tamper_test_session.terminal_encrypt_for_server(
        b"Integrity protected message.",
        content_type="text/plain",
    )
)

tampered_ciphertext = bytearray(
    b64_to_bytes(
        valid_tamper_test_message.ciphertext_b64
    )
)

tampered_ciphertext[0] ^= 1

tampered_message = EncryptedSessionMessage(
    qkd_session_id=
        valid_tamper_test_message.qkd_session_id,
    direction=
        valid_tamper_test_message.direction,
    sequence_number=
        valid_tamper_test_message.sequence_number,
    nonce_b64=
        valid_tamper_test_message.nonce_b64,
    ciphertext_b64=
        bytes_to_b64(bytes(tampered_ciphertext)),
    associated_data_b64=
        valid_tamper_test_message
        .associated_data_b64,
    created_at=
        valid_tamper_test_message.created_at,
)

ciphertext_tampering_rejected = False

try:
    tamper_test_session.server_decrypt_from_terminal(
        tampered_message
    )
except InvalidTag:
    ciphertext_tampering_rejected = True

assert ciphertext_tampering_rejected

print("Modified AES-GCM ciphertext: REJECTED")

Modified AES-GCM ciphertext: REJECTED


In [74]:
#@title Message Replay

replay_test_session = QKDAESGCMSession(
    keys=qkd_aes_session.keys,
    terminal_id=terminal.node_id,
    server_id=server.node_id,
)

replay_test_message = (
    replay_test_session.terminal_encrypt_for_server(
        b"One-time encrypted message.",
        content_type="text/plain",
    )
)

first_plaintext = (
    replay_test_session.server_decrypt_from_terminal(
        replay_test_message
    )
)

encrypted_replay_rejected = False
encrypted_replay_reason = ""

try:
    replay_test_session.server_decrypt_from_terminal(
        replay_test_message
    )
except AuthenticationFailure as exc:
    encrypted_replay_rejected = True
    encrypted_replay_reason = str(exc)

assert first_plaintext == b"One-time encrypted message."
assert encrypted_replay_rejected

print("First encrypted delivery : ACCEPTED")
print("Repeated delivery        : REJECTED")
print("Reason                   :", encrypted_replay_reason)

First encrypted delivery : ACCEPTED
Repeated delivery        : REJECTED
Reason                   : Replay, reordering or missing encrypted message detected.


In [75]:
#@title Eavesdropper Test During QKD

eve_test_aborted = False
eve_test_reason = ""

try:
    (
        eve_phase_5_result,
        eve_aes_session,
        eve_privacy_result,
    ) = execute_phase_5_qkd_session(
        terminal_node=terminal,
        server_node=server,
        transport=network,
        mutual_authentication_session_id=
            mutual_auth_result.session_id,
        signal_count=T12_DEFAULT_SIGNAL_COUNT,
        channel_error_rate=0.01,
        eavesdropper_present=True,
    )

except AuthenticationFailure as exc:
    eve_test_aborted = True
    eve_test_reason = str(exc)

print("=" * 76)
print("INTERCEPT-RESEND ATTACK TEST")
print("=" * 76)
print("Session aborted :", eve_test_aborted)
print("Reason          :", eve_test_reason)

assert eve_test_aborted

print("Eavesdropped QKD session: REJECTED")

INTERCEPT-RESEND ATTACK TEST
Session aborted : True
Reason          : QKD session aborted because estimated QBER 9.61% exceeded threshold 8.00%.
Eavesdropped QKD session: REJECTED


In [76]:
#@title Phase 4 Report

terminal_qkd_session = terminal.get_local_session(
    phase_5_result.qkd_session_id
)

server_qkd_session = server.get_local_session(
    phase_5_result.qkd_session_id
)

phase_5_checks = {
    "80:20 basis bias hard-coded":
        (
            T12_Z_BASIS_PROBABILITY == 0.80
            and T12_X_BASIS_PROBABILITY == 0.20
        ),

    "Observed basis match near biased expectation":
        abs(
            phase_5_result.observed_basis_match_rate
            - T12_EXPECTED_BASIS_MATCH_PROBABILITY
        ) < 0.05,

    "QBER remained below abort threshold":
        (
            phase_5_result.estimated_qber
            <= T12_QBER_ABORT_THRESHOLD
        ),

    "Key reconciliation succeeded":
        phase_5_result.reconciliation_success,

    "Privacy-amplified secrets match":
        (
            phase_5_result
            .terminal_secret_fingerprint
            == phase_5_result
            .server_secret_fingerprint
        ),

    "At least 256 final secret bits remain":
        (
            phase_5_result
            .privacy_amplified_length_bits
            >= T12_MINIMUM_SECRET_BITS
        ),

    "HKDF session keys match":
        phase_5_result.hkdf_keys_match,

    "Terminal-to-server AES-GCM works":
        server_decrypted_plaintext
        == terminal_plaintext,

    "Server-to-terminal AES-GCM works":
        terminal_decrypted_plaintext
        == server_plaintext,

    "AES-GCM tampering rejected":
        ciphertext_tampering_rejected,

    "Encrypted replay rejected":
        encrypted_replay_rejected,

    "Intercept-resend session aborted":
        eve_test_aborted,

    "Terminal QKD state established":
        (
            terminal_qkd_session["qkd_state"]
            == "AES_SESSION_ESTABLISHED"
        ),

    "Server QKD state established":
        (
            server_qkd_session["qkd_state"]
            == "AES_SESSION_ESTABLISHED"
        ),

    "QKD transcript hashes agree":
        (
            terminal_qkd_session[
                "qkd_transcript_hash"
            ]
            == server_qkd_session[
                "qkd_transcript_hash"
            ]
        ),

    "Classical QKD transcript used signed messages":
        (
            phase_5_result
            .total_signed_classical_messages
            > 0
        ),
}

phase_5_report = pd.DataFrame([
    {
        "Security / Functional Property":
            property_name,
        "Status":
            "PASS" if result else "FAIL",
    }
    for property_name, result
    in phase_5_checks.items()
])

display(phase_5_report)

print("\n" + "=" * 82)
print("Phase 4 COMPLETED SUCCESSFULLY")
print("=" * 82)
print("The system now provides:")
print("  • 80:20 T12-inspired biased-basis QKD")
print("  • signed basis reconciliation")
print("  • signed QBER parameter estimation")
print("  • signed Cascade-style key reconciliation")
print("  • explicit reconciliation-leakage accounting")
print("  • signed Toeplitz privacy amplification")
print("  • transcript-bound HKDF-SHA3-256 derivation")
print("  • independent directional AES-256-GCM keys")
print("  • nonce and sequence-number management")
print("  • ciphertext-integrity protection")
print("  • encrypted-message replay rejection")
print("  • QBER-based QKD session abortion")

,Security / Functional Property,Status
0,80:20 basis bias hard-coded,FAIL
1,Observed basis match near biased expectation,PASS
2,QBER remained below abort threshold,PASS
3,Key reconciliation succeeded,PASS
4,Privacy-amplified secrets match,PASS
5,At least 256 final secret bits remain,PASS
6,HKDF session keys match,PASS
7,Terminal-to-server AES-GCM works,PASS
8,Server-to-terminal AES-GCM works,PASS
9,AES-GCM tampering rejected,PASS



Phase 4 COMPLETED SUCCESSFULLY
The system now provides:
  • 80:20 T12-inspired biased-basis QKD
  • signed basis reconciliation
  • signed QBER parameter estimation
  • signed Cascade-style key reconciliation
  • explicit reconciliation-leakage accounting
  • signed Toeplitz privacy amplification
  • transcript-bound HKDF-SHA3-256 derivation
  • independent directional AES-256-GCM keys
  • nonce and sequence-number management
  • ciphertext-integrity protection
  • encrypted-message replay rejection
  • QBER-based QKD session abortion


## COMPLETE END-TO-END PIPELINE : 29-07-2026T11:17:53 - 29-07-2026T13:25:26

In [77]:
#@title End-to-End Real-World Authentication and Secure Messaging Pipeline Single Executable Cell Final Revision

# ============================================================
# COMPLETE END-TO-END QBIOAUTH PIPELINE
# ============================================================
#
# Flow:
#
#   1. Ask whether the user is enrolled
#   2. Authenticate Terminal <-> Server using ML-DSA
#   3A. Enrol a new user
#       OR
#   3B. Authenticate an enrolled user using:
#           - 3% synthetic iris noise
#           - -4-bit rotation
#   4. Terminal creates and signs a QAuth token
#   5. Server verifies the QAuth token
#   6. Run signed 80:20 biased-basis QKD
#   7. Reconcile, amplify and derive AES keys through HKDF
#   8. Encrypt user message Terminal -> Server
#   9. Return encrypted "Server is listening"
#
# The cell assumes that all previous notebook cells containing
# class and function definitions have already been executed.
# ============================================================


QAUTH_TOKEN_VERSION = "2.0"
QAUTH_TOKEN_VALIDITY_SECONDS = 120

# Biometric demonstration defaults.
FINAL_IRIS_NOISE_RATE = 0.01
FINAL_IRIS_ROTATION_BITS = -4

# QKD protocol defaults.
FINAL_Z_BASIS_PROBABILITY = 0.80
FINAL_PARAMETER_ESTIMATION_FRACTION = 0.15
FINAL_QBER_ABORT_THRESHOLD = 0.08
FINAL_SECURITY_MARGIN_BITS = 128
FINAL_RECONCILIATION_PASSES = 6
FINAL_TARGET_DETECTED_EVENTS = 8192
FINAL_TRANSMISSION_BATCH_SIZE = 8192
FINAL_MAXIMUM_QKD_ATTEMPTS = 3

# Physical optical model.
FINAL_SOURCE_EMISSION_EFFICIENCY = 1.00
FINAL_COUPLING_EFFICIENCY = 0.80
FINAL_DETECTOR_EFFICIENCY_MEAN = 0.20
FINAL_DETECTOR_EFFICIENCY_STD = 0.01
FINAL_DETECTOR_EFFICIENCY_MIN = 0.15
FINAL_DETECTOR_EFFICIENCY_MAX = 0.25

FINAL_INTRINSIC_OPTICAL_ERROR_MEAN = 0.010
FINAL_INTRINSIC_OPTICAL_ERROR_STD = 0.002
FINAL_INTRINSIC_OPTICAL_ERROR_MIN = 0.003
FINAL_INTRINSIC_OPTICAL_ERROR_MAX = 0.025

FINAL_DARK_COUNT_PROBABILITY_PER_GATE = 1.0e-6
FINAL_DARK_COUNT_ERROR_PROBABILITY = 0.50

FINAL_PULSE_REPETITION_RATE_HZ = 1_000_000.0
FINAL_FIBRE_PROPAGATION_SPEED_M_PER_S = 2.0e8

FINAL_MAXIMUM_TRANSMISSION_ROUNDS = 100_000
FINAL_MAXIMUM_TOTAL_TRANSMITTED_PULSES = 2_000_000_000

SERVER_LISTENING_MESSAGE = "Server is listening"

def print_pipeline_stage(stage_number: str, title: str) -> None:
    print("\n" + "=" * 88)
    print(f"{stage_number} — {title}")
    print("=" * 88)


def read_yes_no(prompt: str) -> bool:
    while True:
        answer = input(prompt).strip().lower()

        if answer in {"yes", "y"}:
            return True

        if answer in {"no", "n"}:
            return False

        print("Please answer yes/y or no/n.")


def create_signed_qauth_token(
    *,
    terminal_node: TerminalNode,
    server_node: ServerNode,
    mutual_authentication_result: MutualAuthenticationResult,
    user_verification_result: UserCredentialVerificationResult,
    user_email: str,
) -> Dict[str, Any]:
    """
    Create a terminal-issued QAuth token after successful biometric
    credential reconstruction.

    The terminal's ML-DSA private key remains inside its signing vault.
    """

    if not user_verification_result.success:
        raise AuthenticationFailure(
            "QAuth cannot be issued before successful user verification."
        )

    terminal_session = terminal_node.get_local_session(
        mutual_authentication_result.session_id
    )

    server_session = server_node.get_local_session(
        mutual_authentication_result.session_id
    )

    if (
        terminal_session.get("authentication_state")
        != "MUTUALLY_AUTHENTICATED"
        or server_session.get("authentication_state")
        != "MUTUALLY_AUTHENTICATED"
    ):
        raise AuthenticationFailure(
            "QAuth requires a mutually authenticated terminal-server session."
        )

    if (
        terminal_session["transcript_hash"]
        != server_session["transcript_hash"]
    ):
        raise AuthenticationFailure(
            "Terminal and server disagree on the authentication transcript."
        )

    issued_at = utc_now()
    expires_at = issued_at + timedelta(
        seconds=QAUTH_TOKEN_VALIDITY_SECONDS
    )

    unsigned_token = {
        "protocol_name": PROTOCOL_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "token_type": "QAUTH",
        "token_version": QAUTH_TOKEN_VERSION,
        "token_id": f"QAUTH-{generate_uuid()}",
        "user_id": user_verification_result.user_id,
        "user_email": normalise_email(user_email),
        "terminal_id": terminal_node.node_id,
        "server_id": server_node.node_id,
        "mutual_authentication_session_id":
            mutual_authentication_result.session_id,
        "mutual_authentication_transcript_hash":
            mutual_authentication_result.transcript_hash,
        "biometric_verification": {
            "reconstruction_success":
                user_verification_result.reconstruction_success,
            "private_key_decryption_success":
                user_verification_result.private_key_decryption_success,
            "user_signature_valid":
                user_verification_result.user_signature_valid,
            "selected_rotation_bits":
                user_verification_result.selected_rotation_bits,
            "reconstruction_attempts":
                user_verification_result.reconstruction_attempts,
            "challenge_hash":
                user_verification_result.challenge_hash,
        },
        "authorisation_purpose": "INITIATE_QKD_AND_SECURE_SESSION",
        "nonce_b64": bytes_to_b64(generate_nonce(32)),
        "issued_at": issued_at.isoformat(timespec="microseconds"),
        "expires_at": expires_at.isoformat(timespec="microseconds"),
    }

    token_bytes = canonical_json(unsigned_token)
    signature = terminal_node.sign_bytes(token_bytes)

    return {
        "token": unsigned_token,
        "signature_b64": bytes_to_b64(signature),
        "signature_algorithm": MLDSA65.name,
        "terminal_public_key_fingerprint":
            terminal_node.public_identity_card().public_key_fingerprint,
    }


def verify_signed_qauth_token(
    *,
    server_node: ServerNode,
    terminal_node: TerminalNode,
    signed_qauth_token: Dict[str, Any],
    expected_mutual_authentication_result:
        MutualAuthenticationResult,
) -> bool:
    """
    Verify the terminal-issued QAuth token at the server and consume its
    token ID and nonce to prevent replay.
    """

    required_wrapper_fields = {
        "token",
        "signature_b64",
        "signature_algorithm",
        "terminal_public_key_fingerprint",
    }

    missing_wrapper_fields = (
        required_wrapper_fields - set(signed_qauth_token.keys())
    )

    if missing_wrapper_fields:
        raise AuthenticationFailure(
            f"QAuth wrapper is missing fields: "
            f"{sorted(missing_wrapper_fields)}"
        )

    token = signed_qauth_token["token"]

    required_token_fields = {
        "protocol_name",
        "protocol_version",
        "token_type",
        "token_version",
        "token_id",
        "user_id",
        "user_email",
        "terminal_id",
        "server_id",
        "mutual_authentication_session_id",
        "mutual_authentication_transcript_hash",
        "biometric_verification",
        "authorisation_purpose",
        "nonce_b64",
        "issued_at",
        "expires_at",
    }

    missing_token_fields = required_token_fields - set(token.keys())

    if missing_token_fields:
        raise AuthenticationFailure(
            f"QAuth token is missing fields: "
            f"{sorted(missing_token_fields)}"
        )

    if token["protocol_name"] != PROTOCOL_NAME:
        raise AuthenticationFailure("QAuth protocol name is invalid.")

    if token["protocol_version"] != PROTOCOL_VERSION:
        raise AuthenticationFailure("QAuth protocol version is invalid.")

    if token["token_type"] != "QAUTH":
        raise AuthenticationFailure("Unexpected token type.")

    if token["token_version"] != QAUTH_TOKEN_VERSION:
        raise AuthenticationFailure("Unsupported QAuth token version.")

    if token["terminal_id"] != terminal_node.node_id:
        raise AuthenticationFailure(
            "QAuth token was issued by an unexpected terminal."
        )

    if token["server_id"] != server_node.node_id:
        raise AuthenticationFailure(
            "QAuth token targets a different server."
        )

    if not server_node.is_peer_trusted(token["terminal_id"]):
        raise AuthenticationFailure(
            "QAuth issuer is not trusted by the server."
        )

    if (
        token["mutual_authentication_session_id"]
        != expected_mutual_authentication_result.session_id
    ):
        raise AuthenticationFailure(
            "QAuth is not bound to the current mutual-authentication session."
        )

    if (
        token["mutual_authentication_transcript_hash"]
        != expected_mutual_authentication_result.transcript_hash
    ):
        raise AuthenticationFailure(
            "QAuth transcript binding is invalid."
        )

    server_session = server_node.get_local_session(
        token["mutual_authentication_session_id"]
    )

    if (
        server_session.get("authentication_state")
        != "MUTUALLY_AUTHENTICATED"
    ):
        raise AuthenticationFailure(
            "The referenced terminal-server session is not authenticated."
        )

    if (
        server_session.get("transcript_hash")
        != token["mutual_authentication_transcript_hash"]
    ):
        raise AuthenticationFailure(
            "Server session transcript does not match QAuth."
        )

    issued_at = parse_utc_timestamp(token["issued_at"])
    expires_at = parse_utc_timestamp(token["expires_at"])
    current_time = utc_now()

    if issued_at > current_time + timedelta(seconds=5):
        raise AuthenticationFailure(
            "QAuth token has a future issue timestamp."
        )

    if current_time > expires_at:
        raise AuthenticationFailure("QAuth token has expired.")

    if (
        expires_at - issued_at
        > timedelta(seconds=QAUTH_TOKEN_VALIDITY_SECONDS)
    ):
        raise AuthenticationFailure(
            "QAuth validity interval is larger than permitted."
        )

    token_id = token["token_id"]
    nonce_b64 = token["nonce_b64"]

    if token_id in server_node._accepted_qauth_tokens:
        raise AuthenticationFailure(
            "QAuth replay detected: token ID was already accepted."
        )

    if server_node.has_seen_nonce(nonce_b64):
        raise AuthenticationFailure(
            "QAuth replay detected: nonce was already consumed."
        )

    user_id = create_user_id_from_email(token["user_email"])

    if user_id != token["user_id"]:
        raise AuthenticationFailure(
            "QAuth user ID does not match the supplied email."
        )

    if user_id not in server_node._registered_users:
        raise AuthenticationFailure(
            "QAuth refers to an unknown server-side user."
        )

    biometric = token["biometric_verification"]

    biometric_checks = (
        biometric.get("reconstruction_success") is True
        and biometric.get("private_key_decryption_success") is True
        and biometric.get("user_signature_valid") is True
        and bool(biometric.get("challenge_hash"))
    )

    if not biometric_checks:
        raise AuthenticationFailure(
            "QAuth does not contain a successful biometric proof."
        )

    expected_fingerprint = (
        terminal_node.public_identity_card().public_key_fingerprint
    )

    if not hmac.compare_digest(
        signed_qauth_token["terminal_public_key_fingerprint"],
        expected_fingerprint,
    ):
        raise AuthenticationFailure(
            "QAuth terminal public-key fingerprint is invalid."
        )

    if signed_qauth_token["signature_algorithm"] != MLDSA65.name:
        raise AuthenticationFailure(
            "QAuth uses an unsupported signature algorithm."
        )

    token_bytes = canonical_json(token)
    signature = b64_to_bytes(
        signed_qauth_token["signature_b64"]
    )

    signature_valid = (
        server_node.verify_trusted_peer_signature(
            peer_id=terminal_node.node_id,
            message=token_bytes,
            signature=signature,
        )
    )

    if not signature_valid:
        raise AuthenticationFailure(
            "Terminal ML-DSA signature on QAuth is invalid."
        )

    # Consume replay-protection values only after all validation succeeds.
    server_node.consume_nonce(nonce_b64)

    server_node._accepted_qauth_tokens[token_id] = {
        "token_id": token_id,
        "user_id": token["user_id"],
        "terminal_id": token["terminal_id"],
        "mutual_authentication_session_id":
            token["mutual_authentication_session_id"],
        "token_hash": sha3_256_hex(token_bytes),
        "accepted_at": utc_timestamp(),
        "status": "ACCEPTED",
    }

    server_node._log_event(
        event_type="QAUTH_TOKEN_VERIFIED",
        result="SUCCESS",
        details={
            "token_id": token_id,
            "user_id": token["user_id"],
            "token_hash": sha3_256_hex(token_bytes),
        },
    )

    return True


def load_demo_iris_registry() -> Dict[str, Dict[str, Any]]:
    """
    Load the simulator-only raw iris registry exported by Phase 3A.
    """

    registry_path = Path(SYNTHETIC_IRIS_EXPORT_PATH)

    if not registry_path.exists():
        raise FileNotFoundError(
            f"Synthetic iris registry not found at {registry_path}. "
            "Enroll a user first."
        )

    registry = json.loads(
        registry_path.read_text(encoding="utf-8")
    )

    if not isinstance(registry, dict):
        raise ValueError(
            "synthetic_iris_codes.json does not contain a valid registry."
        )

    return registry




# ============================================================
# DISTANCE-AWARE PHYSICAL QKD BASELINE
# ============================================================

def final_read_nonnegative_float(prompt: str) -> float:
    while True:
        try:
            value = float(input(prompt).strip())
            if value < 0:
                raise ValueError
            return value
        except ValueError:
            print("Please enter a non-negative numeric value.")


def final_fibre_transmission_probability(
    distance_km: float,
) -> float:
    """
    Idealised 0.2 dB/km fibre attenuation:
        eta_fibre(d) = 10^(-0.02*d)
    """
    if distance_km < 0:
        raise ValueError("Distance cannot be negative.")
    return float(10 ** (-0.02 * distance_km))


def final_clipped_normal(
    *,
    rng: np.random.Generator,
    mean: float,
    std: float,
    minimum: float,
    maximum: float,
) -> float:
    return float(np.clip(
        rng.normal(mean, std),
        minimum,
        maximum,
    ))


def final_accumulate_physical_qkd_input(
    *,
    distance_km: float,
    target_detected_events: int,
    transmission_batch_size: int,
    rng: np.random.Generator,
) -> Dict[str, Any]:
    """
    Accumulate enough total detector events for one QKD attempt.

    The number of pulses required for the target detector events is
    sampled exactly from a negative-binomial model. Legitimate signal
    detections and dark-only detector clicks are then separated, and
    their combined error contribution becomes the QKD channel error.
    """

    eta_fibre = final_fibre_transmission_probability(
        distance_km
    )

    detector_efficiency = final_clipped_normal(
        rng=rng,
        mean=FINAL_DETECTOR_EFFICIENCY_MEAN,
        std=FINAL_DETECTOR_EFFICIENCY_STD,
        minimum=FINAL_DETECTOR_EFFICIENCY_MIN,
        maximum=FINAL_DETECTOR_EFFICIENCY_MAX,
    )

    intrinsic_error_rate = final_clipped_normal(
        rng=rng,
        mean=FINAL_INTRINSIC_OPTICAL_ERROR_MEAN,
        std=FINAL_INTRINSIC_OPTICAL_ERROR_STD,
        minimum=FINAL_INTRINSIC_OPTICAL_ERROR_MIN,
        maximum=FINAL_INTRINSIC_OPTICAL_ERROR_MAX,
    )

    signal_detection_probability = float(np.clip(
        FINAL_SOURCE_EMISSION_EFFICIENCY
        * FINAL_COUPLING_EFFICIENCY
        * detector_efficiency
        * eta_fibre,
        0.0,
        1.0,
    ))

    dark_only_probability = float(
        (1.0 - signal_detection_probability)
        * FINAL_DARK_COUNT_PROBABILITY_PER_GATE
    )

    total_detection_probability = float(
        signal_detection_probability
        + dark_only_probability
    )

    if total_detection_probability <= 0.0:
        raise AuthenticationFailure(
            "The physical model produced zero detection probability."
        )

    failures_before_target = int(
        rng.negative_binomial(
            target_detected_events,
            total_detection_probability,
        )
    )

    exact_pulses_required = (
        failures_before_target
        + target_detected_events
    )

    transmission_round_count = int(math.ceil(
        exact_pulses_required / transmission_batch_size
    ))

    total_transmitted_pulses = int(
        transmission_round_count
        * transmission_batch_size
    )

    if (
        transmission_round_count
        > FINAL_MAXIMUM_TRANSMISSION_ROUNDS
    ):
        raise AuthenticationFailure(
            "QKD optical acquisition aborted: "
            "maximum transmission-round limit exceeded."
        )

    if (
        total_transmitted_pulses
        > FINAL_MAXIMUM_TOTAL_TRANSMITTED_PULSES
    ):
        raise AuthenticationFailure(
            "QKD optical acquisition aborted: "
            "maximum transmitted-pulse budget exceeded."
        )

    extra_pulses = (
        total_transmitted_pulses
        - exact_pulses_required
    )

    extra_detections = int(
        rng.binomial(
            extra_pulses,
            total_detection_probability,
        )
    )

    total_detected_events = (
        target_detected_events
        + extra_detections
    )

    signal_fraction_given_detection = (
        signal_detection_probability
        / total_detection_probability
    )

    total_signal_detections = int(
        rng.binomial(
            total_detected_events,
            signal_fraction_given_detection,
        )
    )

    total_dark_detections = (
        total_detected_events
        - total_signal_detections
    )

    # Exactly target_detected_events are selected for the QKD attempt.
    qkd_signal_detections = int(
        rng.hypergeometric(
            ngood=total_signal_detections,
            nbad=total_dark_detections,
            nsample=target_detected_events,
        )
    )

    qkd_dark_detections = (
        target_detected_events
        - qkd_signal_detections
    )

    signal_errors = int(
        rng.binomial(
            qkd_signal_detections,
            intrinsic_error_rate,
        )
    )

    dark_errors = int(
        rng.binomial(
            qkd_dark_detections,
            FINAL_DARK_COUNT_ERROR_PROBABILITY,
        )
    )

    total_physical_errors = (
        signal_errors
        + dark_errors
    )

    effective_physical_qber = (
        total_physical_errors
        / target_detected_events
    )

    return {
        "distance_km": distance_km,
        "fibre_transmission_probability":
            eta_fibre,
        "detector_efficiency":
            detector_efficiency,
        "signal_detection_probability":
            signal_detection_probability,
        "dark_only_probability":
            dark_only_probability,
        "total_detection_probability":
            total_detection_probability,
        "transmission_batch_size":
            transmission_batch_size,
        "transmission_round_count":
            transmission_round_count,
        "total_transmitted_pulses":
            total_transmitted_pulses,
        "total_detected_events":
            total_detected_events,
        "target_detected_events":
            target_detected_events,
        "qkd_signal_detections":
            qkd_signal_detections,
        "qkd_dark_detections":
            qkd_dark_detections,
        "dark_detection_fraction": (
            qkd_dark_detections
            / target_detected_events
        ),
        "intrinsic_optical_error_rate":
            intrinsic_error_rate,
        "signal_error_count":
            signal_errors,
        "dark_error_count":
            dark_errors,
        "total_physical_error_count":
            total_physical_errors,
        "effective_physical_qber":
            effective_physical_qber,
        "estimated_optical_acquisition_seconds": (
            total_transmitted_pulses
            / FINAL_PULSE_REPETITION_RATE_HZ
        ),
    }


def final_apply_qkd_configuration() -> Dict[str, Any]:
    """
    Apply the final baseline's fixed 80:20 QKD configuration and return
    the previous values so they can be restored after execution.
    """

    names = [
        "T12_Z_BASIS_PROBABILITY",
        "T12_X_BASIS_PROBABILITY",
        "T12_EXPECTED_BASIS_MATCH_PROBABILITY",
        "T12_PARAMETER_ESTIMATION_FRACTION",
        "T12_QBER_ABORT_THRESHOLD",
        "T12_SECURITY_MARGIN_BITS",
        "T12_RECONCILIATION_PASSES",
    ]

    missing = [
        name for name in names
        if name not in globals()
    ]

    if missing:
        raise RuntimeError(
            "Required Phase 5 globals are missing: "
            + ", ".join(missing)
        )

    previous = {
        name: globals()[name]
        for name in names
    }

    globals()["T12_Z_BASIS_PROBABILITY"] = (
        FINAL_Z_BASIS_PROBABILITY
    )
    globals()["T12_X_BASIS_PROBABILITY"] = (
        1.0 - FINAL_Z_BASIS_PROBABILITY
    )
    globals()["T12_EXPECTED_BASIS_MATCH_PROBABILITY"] = (
        FINAL_Z_BASIS_PROBABILITY ** 2
        + (1.0 - FINAL_Z_BASIS_PROBABILITY) ** 2
    )
    globals()["T12_PARAMETER_ESTIMATION_FRACTION"] = (
        FINAL_PARAMETER_ESTIMATION_FRACTION
    )
    globals()["T12_QBER_ABORT_THRESHOLD"] = (
        FINAL_QBER_ABORT_THRESHOLD
    )
    globals()["T12_SECURITY_MARGIN_BITS"] = (
        FINAL_SECURITY_MARGIN_BITS
    )
    globals()["T12_RECONCILIATION_PASSES"] = (
        FINAL_RECONCILIATION_PASSES
    )

    return previous


def final_restore_qkd_configuration(
    previous: Dict[str, Any],
) -> None:
    for name, value in previous.items():
        globals()[name] = value


def final_execute_distance_aware_qkd(
    *,
    terminal_node: TerminalNode,
    server_node: ServerNode,
    transport: InMemoryMessageTransport,
    mutual_authentication_session_id: str,
    distance_km: float,
    eavesdropper_present: bool,
) -> Tuple[
    Phase5QKDSessionResult,
    QKDAESGCMSession,
    PrivacyAmplificationResult,
    Dict[str, Any],
]:
    """
    Execute one or more complete physical QKD attempts.

    A fresh optical acquisition is performed for each attempt. A normal
    reconciliation/privacy failure may be retried. A QBER abort while Eve
    is present ends the session immediately.
    """

    previous_qkd_configuration = (
        final_apply_qkd_configuration()
    )

    cumulative_pulses = 0
    cumulative_rounds = 0
    cumulative_optical_seconds = 0.0
    attempt_reports: List[Dict[str, Any]] = []
    last_failure: Optional[str] = None

    try:
        for attempt_number in range(
            1,
            FINAL_MAXIMUM_QKD_ATTEMPTS + 1,
        ):
            physical_rng = np.random.default_rng(
                int.from_bytes(
                    secrets.token_bytes(16),
                    byteorder="big",
                )
            )

            physical = (
                final_accumulate_physical_qkd_input(
                    distance_km=distance_km,
                    target_detected_events=
                        FINAL_TARGET_DETECTED_EVENTS,
                    transmission_batch_size=
                        FINAL_TRANSMISSION_BATCH_SIZE,
                    rng=physical_rng,
                )
            )

            cumulative_pulses += int(
                physical["total_transmitted_pulses"]
            )
            cumulative_rounds += int(
                physical["transmission_round_count"]
            )
            cumulative_optical_seconds += float(
                physical[
                    "estimated_optical_acquisition_seconds"
                ]
            )

            qkd_processing_start = time.perf_counter()

            try:
                (
                    qkd_result,
                    secure_aes_session,
                    privacy_result,
                ) = execute_phase_5_qkd_session(
                    terminal_node=terminal_node,
                    server_node=server_node,
                    transport=transport,
                    mutual_authentication_session_id=
                        mutual_authentication_session_id,
                    signal_count=
                        FINAL_TARGET_DETECTED_EVENTS,
                    channel_error_rate=float(
                        physical[
                            "effective_physical_qber"
                        ]
                    ),
                    eavesdropper_present=
                        eavesdropper_present,
                )

                measured_processing_seconds = (
                    time.perf_counter()
                    - qkd_processing_start
                )

                one_way_propagation_seconds = (
                    distance_km
                    * 1000.0
                    / FINAL_FIBRE_PROPAGATION_SPEED_M_PER_S
                )

                classical_propagation_seconds = (
                    qkd_result
                    .total_signed_classical_messages
                    * one_way_propagation_seconds
                )

                modelled_total_seconds = (
                    cumulative_optical_seconds
                    + classical_propagation_seconds
                    + measured_processing_seconds
                )

                physical_summary = {
                    **physical,
                    "attempt_number":
                        attempt_number,
                    "attempt_reports":
                        attempt_reports,
                    "cumulative_transmitted_pulses":
                        cumulative_pulses,
                    "cumulative_transmission_rounds":
                        cumulative_rounds,
                    "cumulative_optical_acquisition_seconds":
                        cumulative_optical_seconds,
                    "measured_qkd_processing_seconds":
                        measured_processing_seconds,
                    "estimated_classical_propagation_seconds":
                        classical_propagation_seconds,
                    "modelled_qkd_session_seconds":
                        modelled_total_seconds,
                }

                return (
                    qkd_result,
                    secure_aes_session,
                    privacy_result,
                    physical_summary,
                )

            except AuthenticationFailure as exc:
                measured_processing_seconds = (
                    time.perf_counter()
                    - qkd_processing_start
                )
                last_failure = str(exc)

                attempt_reports.append({
                    "attempt_number":
                        attempt_number,
                    "failure_reason":
                        last_failure,
                    "physical_qber":
                        physical[
                            "effective_physical_qber"
                        ],
                    "transmitted_pulses":
                        physical[
                            "total_transmitted_pulses"
                        ],
                    "optical_seconds":
                        physical[
                            "estimated_optical_acquisition_seconds"
                        ],
                    "processing_seconds":
                        measured_processing_seconds,
                })

                print(
                    f"QKD attempt {attempt_number} failed: "
                    f"{last_failure}"
                )

                if eavesdropper_present:
                    raise

                if (
                    attempt_number
                    == FINAL_MAXIMUM_QKD_ATTEMPTS
                ):
                    raise AuthenticationFailure(
                        "All distance-aware QKD attempts failed. "
                        f"Last failure: {last_failure}"
                    ) from exc

                print(
                    "Starting a fresh optical acquisition "
                    "and QKD attempt..."
                )

        raise AuthenticationFailure(
            last_failure or "QKD session failed."
        )

    finally:
        final_restore_qkd_configuration(
            previous_qkd_configuration
        )


# ============================================================
# FINAL END-TO-END PIPELINE
# ============================================================

def run_final_physical_end_to_end_pipeline() -> Dict[str, Any]:
    """
    Execute the final baseline:

        terminal-server ML-DSA authentication
        -> enrollment or noisy biometric authentication
        -> terminal-signed QAuth
        -> distance/noise-aware biased-basis QKD
        -> reconciliation and privacy amplification
        -> transcript-bound HKDF
        -> bidirectional AES-256-GCM communication
    """

    pipeline_start = time.perf_counter()

    print("\n" + "#" * 92)
    print("FINAL QBIOAUTH PHYSICAL END-TO-END BASELINE")
    print("#" * 92)

    user_is_enrolled = read_yes_no(
        "\nHas the user already been enrolled? [yes/no]: "
    )

    distance_km = final_read_nonnegative_float(
        "Enter Terminal–Server fibre distance in kilometres: "
    )

    eavesdropper_present = read_yes_no(
        "Simulate an intercept-resend Eve? [yes/no]: "
    )

    # --------------------------------------------------------
    # Stage 1 — Fresh terminal/server mutual authentication.
    # --------------------------------------------------------

    print_pipeline_stage(
        "STAGE 1",
        "TERMINAL–SERVER MUTUAL ML-DSA AUTHENTICATION",
    )

    mutual_authentication_start = time.perf_counter()

    mutual_auth_protocol = MutualAuthenticationProtocol(
        terminal=terminal,
        server=server,
        transport=network,
    )

    current_mutual_auth_result = (
        mutual_auth_protocol.execute()
    )

    mutual_authentication_seconds = (
        time.perf_counter()
        - mutual_authentication_start
    )

    if not (
        current_mutual_auth_result.success
        and current_mutual_auth_result
            .terminal_authenticated_server
        and current_mutual_auth_result
            .server_authenticated_terminal
    ):
        raise AuthenticationFailure(
            "Terminal-server mutual authentication failed."
        )

    print(
        "Session ID              :",
        current_mutual_auth_result.session_id,
    )
    print(
        "Transcript hash         :",
        current_mutual_auth_result.transcript_hash,
    )
    print(
        "Measured authentication :",
        f"{mutual_authentication_seconds:.4f} s",
    )
    print("Mutual authentication  : PASS")

    # --------------------------------------------------------
    # Stage 2A — Enrollment branch.
    # --------------------------------------------------------

    if not user_is_enrolled:
        print_pipeline_stage(
            "STAGE 2A",
            "NEW SYNTHETIC USER ENROLLMENT",
        )

        user_name = input(
            "Enter the user's full name: "
        ).strip()

        user_email = input(
            "Enter the user's unique email: "
        ).strip()

        enrollment_result = enrol_synthetic_user(
            terminal_node=terminal,
            server_node=server,
            user_name=user_name,
            user_email=user_email,
        )

        print("\nUSER ENROLLMENT SUCCESSFUL")
        print("-" * 92)
        print(
            "User name              :",
            enrollment_result.user_name,
        )
        print(
            "User email             :",
            enrollment_result.user_email,
        )
        print(
            "Internal user ID       :",
            enrollment_result.user_id,
        )
        print(
            "Iris fingerprint       :",
            enrollment_result.iris_fingerprint,
        )
        print(
            "ML-DSA fingerprint     :",
            enrollment_result.public_key_fingerprint,
        )
        print("\nCOPY-READY 2048-BIT SYNTHETIC IRIS CODE")
        print("-" * 92)
        print(enrollment_result.raw_iris_code)
        print("-" * 92)
        print(
            "Enrollment branch complete. "
            "Authentication and QKD intentionally stop here."
        )

        return {
            "status": "ENROLLED_AND_EXITED",
            "distance_km": distance_km,
            "mutual_authentication":
                current_mutual_auth_result,
            "enrollment": enrollment_result,
        }

    # --------------------------------------------------------
    # Stage 2B — Noisy biometric authentication.
    # --------------------------------------------------------

    print_pipeline_stage(
        "STAGE 2B",
        "NOISY IRIS RECONSTRUCTION AND USER AUTHENTICATION",
    )

    user_email = input(
        "Enter the enrolled user's email: "
    ).strip()

    entered_raw_iris_code = input(
        "Paste the user's 2048-bit raw binary iris code: "
    ).strip()

    normalised_email = normalise_email(
        user_email
    )
    user_id = create_user_id_from_email(
        normalised_email
    )

    if user_id not in server._registered_users:
        raise AuthenticationFailure(
            "No active server-side registration exists "
            "for this email."
        )

    if user_id not in terminal._local_user_records:
        raise AuthenticationFailure(
            "No protected terminal credential exists "
            "for this email."
        )

    registry = load_demo_iris_registry()

    if user_id not in registry:
        raise AuthenticationFailure(
            "The simulator iris record is absent from "
            "synthetic_iris_codes.json."
        )

    validated_input_iris = (
        validate_binary_iris_string(
            entered_raw_iris_code
        )
    )

    stored_canonical_iris = (
        validate_binary_iris_string(
            registry[user_id][
                "canonical_iris_code"
            ]
        )
    )

    # Simulator-only ownership validation.
    if not hmac.compare_digest(
        validated_input_iris,
        stored_canonical_iris,
    ):
        raise AuthenticationFailure(
            "The supplied simulator iris code does not "
            "belong to the claimed user."
        )

    biometric_start = time.perf_counter()

    synthetic_scan = simulate_noisy_iris_scan(
        source_iris_code=
            validated_input_iris,
        bit_error_rate=
            FINAL_IRIS_NOISE_RATE,
        rotation_bits=
            FINAL_IRIS_ROTATION_BITS,
    )

    user_verification_result = (
        authenticate_recovered_user_credential(
            terminal_node=terminal,
            server_node=server,
            authenticated_session_id=
                current_mutual_auth_result.session_id,
            user_email=normalised_email,
            candidate_iris_code=
                synthetic_scan.noisy_iris_code,
        )
    )

    biometric_seconds = (
        time.perf_counter()
        - biometric_start
    )

    print(
        "Injected iris noise     :",
        f"{synthetic_scan.actual_bit_error_rate:.2%}",
    )
    print(
        "Injected iris rotation  :",
        f"{synthetic_scan.applied_rotation_bits} bits",
    )
    print(
        "Fuzzy reconstruction    :",
        user_verification_result
        .reconstruction_success,
    )
    print(
        "Private-key recovery    :",
        user_verification_result
        .private_key_decryption_success,
    )
    print(
        "User ML-DSA proof       :",
        user_verification_result
        .user_signature_valid,
    )
    print(
        "Measured biometric time :",
        f"{biometric_seconds:.4f} s",
    )

    if not user_verification_result.success:
        raise AuthenticationFailure(
            user_verification_result
            .failure_reason
            or "Biometric authentication failed."
        )

    # --------------------------------------------------------
    # Stage 3 — Signed QAuth.
    # --------------------------------------------------------

    print_pipeline_stage(
        "STAGE 3",
        "TERMINAL-SIGNED QAUTH AUTHORIZATION",
    )

    qauth_start = time.perf_counter()

    signed_qauth_token = create_signed_qauth_token(
        terminal_node=terminal,
        server_node=server,
        mutual_authentication_result=
            current_mutual_auth_result,
        user_verification_result=
            user_verification_result,
        user_email=normalised_email,
    )

    qauth_valid = verify_signed_qauth_token(
        server_node=server,
        terminal_node=terminal,
        signed_qauth_token=
            signed_qauth_token,
        expected_mutual_authentication_result=
            current_mutual_auth_result,
    )

    qauth_seconds = (
        time.perf_counter()
        - qauth_start
    )

    if not qauth_valid:
        raise AuthenticationFailure(
            "Server rejected QAuth."
        )

    # QAuth is validated before physical acquisition begins.
    # Its expiry limits replay/late initiation; it does not terminate a
    # QKD acquisition that was legitimately authorized while valid.
    qauth_token_id = (
        signed_qauth_token["token"]["token_id"]
    )

    server._accepted_qauth_tokens[
        qauth_token_id
    ]["qkd_authorization_status"] = (
        "INITIATED_WHILE_QAUTH_VALID"
    )
    server._accepted_qauth_tokens[
        qauth_token_id
    ]["qkd_authorized_at"] = (
        utc_timestamp()
    )

    print(
        "QAuth token ID          :",
        qauth_token_id,
    )
    print(
        "Terminal signature      : VALID",
    )
    print(
        "Server QAuth decision   : ACCEPTED",
    )
    print(
        "Measured QAuth time     :",
        f"{qauth_seconds:.4f} s",
    )

    # --------------------------------------------------------
    # Stage 4 — Physical QKD + HKDF.
    # --------------------------------------------------------

    print_pipeline_stage(
        "STAGE 4",
        "DISTANCE/NOISE-AWARE T12-INSPIRED QKD",
    )

    (
        qkd_result,
        secure_aes_session,
        privacy_result,
        physical_result,
    ) = final_execute_distance_aware_qkd(
        terminal_node=terminal,
        server_node=server,
        transport=network,
        mutual_authentication_session_id=
            current_mutual_auth_result.session_id,
        distance_km=distance_km,
        eavesdropper_present=
            eavesdropper_present,
    )

    print(
        "Distance                :",
        f"{distance_km:.2f} km",
    )
    print(
        "Fibre survival          :",
        f"{physical_result['fibre_transmission_probability']:.8f}",
    )
    print(
        "Detector efficiency     :",
        f"{physical_result['detector_efficiency']:.2%}",
    )
    print(
        "Transmission rounds     :",
        f"{physical_result['cumulative_transmission_rounds']:,}",
    )
    print(
        "Transmitted pulses      :",
        f"{physical_result['cumulative_transmitted_pulses']:,}",
    )
    print(
        "Legitimate detections   :",
        f"{physical_result['qkd_signal_detections']:,}",
    )
    print(
        "Dark detections         :",
        f"{physical_result['qkd_dark_detections']:,}",
    )
    print(
        "Dark-count fraction     :",
        f"{physical_result['dark_detection_fraction']:.3%}",
    )
    print(
        "Intrinsic optical error :",
        f"{physical_result['intrinsic_optical_error_rate']:.3%}",
    )
    print(
        "Physical-layer QBER     :",
        f"{physical_result['effective_physical_qber']:.3%}",
    )
    print(
        "Estimated protocol QBER :",
        f"{qkd_result.estimated_qber:.3%}",
    )
    print(
        "Eve present             :",
        eavesdropper_present,
    )
    print(
        "QKD attempt number      :",
        physical_result["attempt_number"],
    )
    print(
        "Errors reconciled       :",
        qkd_result
        .reconciliation_corrected_errors,
    )
    print(
        "Final QKD secret        :",
        f"{qkd_result.privacy_amplified_length_bits} bits",
    )
    print(
        "Signed classical msgs   :",
        qkd_result
        .total_signed_classical_messages,
    )
    print(
        "HKDF keys match         :",
        qkd_result.hkdf_keys_match,
    )
    print(
        "Optical acquisition     :",
        f"{physical_result['cumulative_optical_acquisition_seconds']:.4f} s",
    )
    print(
        "Classical propagation   :",
        f"{physical_result['estimated_classical_propagation_seconds']:.4f} s",
    )
    print(
        "Measured QKD processing :",
        f"{physical_result['measured_qkd_processing_seconds']:.4f} s",
    )
    print(
        "Modelled QKD latency    :",
        f"{physical_result['modelled_qkd_session_seconds']:.4f} s",
    )

    # --------------------------------------------------------
    # Stage 5 — AES-GCM application exchange.
    # --------------------------------------------------------

    print_pipeline_stage(
        "STAGE 5",
        "BIDIRECTIONAL AES-256-GCM SESSION",
    )

    user_message = input(
        "Enter the message to send securely to the server: "
    )

    if not user_message:
        raise ValueError(
            "The secure message cannot be empty."
        )

    aes_start = time.perf_counter()

    terminal_encrypted_message = (
        secure_aes_session
        .terminal_encrypt_for_server(
            user_message.encode("utf-8"),
            content_type="text/plain",
        )
    )

    server_received_plaintext = (
        secure_aes_session
        .server_decrypt_from_terminal(
            terminal_encrypted_message
        )
    )

    server_encrypted_response = (
        secure_aes_session
        .server_encrypt_for_terminal(
            SERVER_LISTENING_MESSAGE.encode(
                "utf-8"
            ),
            content_type="text/plain",
        )
    )

    terminal_received_response = (
        secure_aes_session
        .terminal_decrypt_from_server(
            server_encrypted_response
        )
    ).decode("utf-8")

    aes_seconds = (
        time.perf_counter()
        - aes_start
    )

    if (
        server_received_plaintext.decode("utf-8")
        != user_message
    ):
        raise AuthenticationFailure(
            "Server recovered an incorrect plaintext."
        )

    if (
        terminal_received_response
        != SERVER_LISTENING_MESSAGE
    ):
        raise AuthenticationFailure(
            "Terminal recovered an incorrect response."
        )

    measured_pipeline_seconds = (
        time.perf_counter()
        - pipeline_start
    )

    modelled_end_to_end_seconds = (
        mutual_authentication_seconds
        + biometric_seconds
        + qauth_seconds
        + physical_result[
            "modelled_qkd_session_seconds"
        ]
        + aes_seconds
    )

    print(
        "Terminal plaintext      :",
        user_message,
    )
    print(
        "Terminal ciphertext     :",
        terminal_encrypted_message
        .ciphertext_b64,
    )
    print(
        "Server recovered        :",
        server_received_plaintext.decode(
            "utf-8"
        ),
    )
    print(
        "Server response         :",
        SERVER_LISTENING_MESSAGE,
    )
    print(
        "Terminal recovered      :",
        terminal_received_response,
    )
    print(
        "Measured AES time       :",
        f"{aes_seconds:.6f} s",
    )

    print("\n" + "#" * 92)
    print("FINAL END-TO-END PIPELINE COMPLETED SUCCESSFULLY")
    print("#" * 92)
    print("✓ Terminal and server mutually authenticated")
    print("✓ Noisy biometric credential reconstructed")
    print("✓ User ML-DSA credential verified")
    print("✓ Terminal-signed QAuth accepted")
    print("✓ Distance attenuation and detector noise model applied")
    print("✓ Dark counts and intrinsic optical error applied")
    print("✓ Photon detections accumulated to the required target")
    print("✓ Signed biased-basis QKD completed")
    print("✓ Reconciliation and privacy amplification completed")
    print("✓ HKDF-derived AES-256-GCM keys matched")
    print("✓ Bidirectional encrypted application exchange completed")
    print("-" * 92)
    print(
        "Measured notebook runtime:",
        f"{measured_pipeline_seconds:.4f} s",
    )
    print(
        "Modelled deployment time :",
        f"{modelled_end_to_end_seconds:.4f} s",
    )
    print("#" * 92)

    return {
        "status":
            "SECURE_PHYSICAL_SESSION_COMPLETED",
        "distance_km":
            distance_km,
        "eve_present":
            eavesdropper_present,
        "mutual_authentication":
            current_mutual_auth_result,
        "synthetic_scan":
            synthetic_scan,
        "user_verification":
            user_verification_result,
        "signed_qauth_token":
            signed_qauth_token,
        "physical_qkd":
            physical_result,
        "qkd_result":
            qkd_result,
        "privacy_amplification":
            privacy_result,
        "terminal_encrypted_message":
            terminal_encrypted_message,
        "server_received_plaintext":
            server_received_plaintext.decode(
                "utf-8"
            ),
        "server_encrypted_response":
            server_encrypted_response,
        "terminal_received_response":
            terminal_received_response,
        "timing": {
            "mutual_authentication_seconds":
                mutual_authentication_seconds,
            "biometric_seconds":
                biometric_seconds,
            "qauth_seconds":
                qauth_seconds,
            "qkd_modelled_seconds":
                physical_result[
                    "modelled_qkd_session_seconds"
                ],
            "aes_seconds":
                aes_seconds,
            "measured_pipeline_seconds":
                measured_pipeline_seconds,
            "modelled_end_to_end_seconds":
                modelled_end_to_end_seconds,
        },
    }


# try:
#     final_end_to_end_result = (
#         run_final_physical_end_to_end_pipeline()
#     )

# except (
#     AuthenticationFailure,
#     ProtocolValidationError,
#     ValueError,
#     KeyError,
#     FileNotFoundError,
#     RuntimeError,
# ) as pipeline_error:
#     print("\n" + "!" * 92)
#     print("FINAL END-TO-END PIPELINE ABORTED")
#     print("!" * 92)
#     print("Reason:", pipeline_error)

#     final_end_to_end_result = {
#         "status": "ABORTED",
#         "reason": str(pipeline_error),
#     }

## Additional Requirement Phase 5 GUI : 07-08-2026T13:38:18 - 07-08-2026T23:37:16

In [78]:
#@title Non-Interactive Backend and Configuration Layer

import contextlib
import threading
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import gradio as gr
import pandas as pd


# ============================================================
# GUI CONFIGURATION MODEL
# ============================================================

@dataclass(frozen=True)
class GradioPipelineConfiguration:
    z_basis_probability: float
    parameter_estimation_fraction: float
    qber_abort_threshold: float
    security_margin_bits: int
    reconciliation_passes: int
    target_detected_events: int
    transmission_batch_size: int
    maximum_qkd_attempts: int
    iris_noise_rate: float
    iris_rotation_bits: int


GRADIO_DEFAULT_CONFIGURATION = GradioPipelineConfiguration(
    z_basis_probability=float(FINAL_Z_BASIS_PROBABILITY),
    parameter_estimation_fraction=float(
        FINAL_PARAMETER_ESTIMATION_FRACTION
    ),
    qber_abort_threshold=float(
        FINAL_QBER_ABORT_THRESHOLD
    ),
    security_margin_bits=int(
        FINAL_SECURITY_MARGIN_BITS
    ),
    reconciliation_passes=int(
        FINAL_RECONCILIATION_PASSES
    ),
    target_detected_events=int(
        FINAL_TARGET_DETECTED_EVENTS
    ),
    transmission_batch_size=int(
        FINAL_TRANSMISSION_BATCH_SIZE
    ),
    maximum_qkd_attempts=int(
        FINAL_MAXIMUM_QKD_ATTEMPTS
    ),
    iris_noise_rate=float(
        FINAL_IRIS_NOISE_RATE
    ),
    iris_rotation_bits=int(
        FINAL_IRIS_ROTATION_BITS
    ),
)


GRADIO_CONFIGURATION_ROWS = [
    {
        "Parameter": "Z-basis probability",
        "Value": GRADIO_DEFAULT_CONFIGURATION.z_basis_probability,
        "Unit": "probability",
        "Description": "Biased-basis probability used for Z.",
    },
    {
        "Parameter": "Parameter-estimation fraction",
        "Value": GRADIO_DEFAULT_CONFIGURATION.parameter_estimation_fraction,
        "Unit": "fraction",
        "Description": "Sifted bits disclosed for QBER estimation.",
    },
    {
        "Parameter": "QBER abort threshold",
        "Value": GRADIO_DEFAULT_CONFIGURATION.qber_abort_threshold,
        "Unit": "probability",
        "Description": "Maximum accepted estimated QBER.",
    },
    {
        "Parameter": "Privacy security margin",
        "Value": GRADIO_DEFAULT_CONFIGURATION.security_margin_bits,
        "Unit": "bits",
        "Description": "Additional privacy-amplification deduction.",
    },
    {
        "Parameter": "Cascade passes",
        "Value": GRADIO_DEFAULT_CONFIGURATION.reconciliation_passes,
        "Unit": "passes",
        "Description": "Full recursive look-back reconciliation passes.",
    },
    {
        "Parameter": "Target detector events",
        "Value": GRADIO_DEFAULT_CONFIGURATION.target_detected_events,
        "Unit": "events",
        "Description": "Detected events accumulated before QKD.",
    },
    {
        "Parameter": "Transmission batch size",
        "Value": GRADIO_DEFAULT_CONFIGURATION.transmission_batch_size,
        "Unit": "pulses",
        "Description": "Optical pulses represented by each acquisition batch.",
    },
    {
        "Parameter": "Maximum QKD attempts",
        "Value": GRADIO_DEFAULT_CONFIGURATION.maximum_qkd_attempts,
        "Unit": "attempts",
        "Description": "Fresh QKD retries after recoverable failure.",
    },
    {
        "Parameter": "Synthetic iris noise",
        "Value": GRADIO_DEFAULT_CONFIGURATION.iris_noise_rate,
        "Unit": "probability",
        "Description": "Bit-error rate applied to the synthetic iris scan.",
    },
    {
        "Parameter": "Synthetic iris rotation",
        "Value": GRADIO_DEFAULT_CONFIGURATION.iris_rotation_bits,
        "Unit": "bits",
        "Description": "Circular shift applied to the synthetic iris scan.",
    },
]


def gradio_default_configuration_dataframe() -> pd.DataFrame:
    return pd.DataFrame(
        GRADIO_CONFIGURATION_ROWS,
        columns=[
            "Parameter",
            "Value",
            "Unit",
            "Description",
        ],
    )


def parse_gradio_configuration(
    table: Any,
    *,
    use_advanced_configuration: bool,
) -> GradioPipelineConfiguration:
    """
    Parse and validate the editable Gradio configuration table.

    The interface never permanently mutates protocol globals. Values are
    applied only for the duration of one protected pipeline execution.
    """

    if not use_advanced_configuration:
        return GRADIO_DEFAULT_CONFIGURATION

    if isinstance(table, pd.DataFrame):
        dataframe = table.copy()
    else:
        dataframe = pd.DataFrame(
            table,
            columns=[
                "Parameter",
                "Value",
                "Unit",
                "Description",
            ],
        )

    required_columns = {
        "Parameter",
        "Value",
    }

    if not required_columns.issubset(
        dataframe.columns
    ):
        raise ValueError(
            "Configuration table must contain Parameter and Value columns."
        )

    values = {
        str(row["Parameter"]).strip():
            row["Value"]
        for _, row in dataframe.iterrows()
    }

    expected_parameters = {
        row["Parameter"]
        for row in GRADIO_CONFIGURATION_ROWS
    }

    missing = expected_parameters - set(values)

    if missing:
        raise ValueError(
            "Configuration table is missing: "
            + ", ".join(sorted(missing))
        )

    def number(name: str) -> float:
        try:
            return float(values[name])
        except (TypeError, ValueError) as exc:
            raise ValueError(
                f"{name} must be numeric."
            ) from exc

    def integer(name: str) -> int:
        value = number(name)
        if not value.is_integer():
            raise ValueError(
                f"{name} must be a whole number."
            )
        return int(value)

    configuration = GradioPipelineConfiguration(
        z_basis_probability=number(
            "Z-basis probability"
        ),
        parameter_estimation_fraction=number(
            "Parameter-estimation fraction"
        ),
        qber_abort_threshold=number(
            "QBER abort threshold"
        ),
        security_margin_bits=integer(
            "Privacy security margin"
        ),
        reconciliation_passes=integer(
            "Cascade passes"
        ),
        target_detected_events=integer(
            "Target detector events"
        ),
        transmission_batch_size=integer(
            "Transmission batch size"
        ),
        maximum_qkd_attempts=integer(
            "Maximum QKD attempts"
        ),
        iris_noise_rate=number(
            "Synthetic iris noise"
        ),
        iris_rotation_bits=integer(
            "Synthetic iris rotation"
        ),
    )

    if not 0.0 < configuration.z_basis_probability < 1.0:
        raise ValueError(
            "Z-basis probability must lie strictly between 0 and 1."
        )

    if not (
        0.0
        < configuration.parameter_estimation_fraction
        < 1.0
    ):
        raise ValueError(
            "Parameter-estimation fraction must lie between 0 and 1."
        )

    if not 0.0 < configuration.qber_abort_threshold < 0.5:
        raise ValueError(
            "QBER abort threshold must lie between 0 and 0.5."
        )

    if configuration.security_margin_bits < 1:
        raise ValueError(
            "Privacy security margin must be positive."
        )

    if configuration.reconciliation_passes < 1:
        raise ValueError(
            "Cascade passes must be at least one."
        )

    if configuration.target_detected_events < 512:
        raise ValueError(
            "Target detector events must be at least 512."
        )

    if configuration.transmission_batch_size < 1:
        raise ValueError(
            "Transmission batch size must be positive."
        )

    if configuration.maximum_qkd_attempts < 1:
        raise ValueError(
            "Maximum QKD attempts must be at least one."
        )

    if not 0.0 <= configuration.iris_noise_rate <= 0.10:
        raise ValueError(
            "Synthetic iris noise must lie between 0 and 0.10."
        )

    if abs(configuration.iris_rotation_bits) > 64:
        raise ValueError(
            "Synthetic iris rotation must lie between -64 and +64 bits."
        )

    return configuration


# ============================================================
# TEMPORARY, THREAD-SAFE GLOBAL CONFIGURATION
# ============================================================

GRADIO_PIPELINE_LOCK = threading.RLock()


@contextlib.contextmanager
def temporary_gradio_pipeline_configuration(
    configuration: GradioPipelineConfiguration,
):
    names_and_values = {
        "FINAL_Z_BASIS_PROBABILITY":
            configuration.z_basis_probability,
        "FINAL_PARAMETER_ESTIMATION_FRACTION":
            configuration.parameter_estimation_fraction,
        "FINAL_QBER_ABORT_THRESHOLD":
            configuration.qber_abort_threshold,
        "FINAL_SECURITY_MARGIN_BITS":
            configuration.security_margin_bits,
        "FINAL_RECONCILIATION_PASSES":
            configuration.reconciliation_passes,
        "FINAL_TARGET_DETECTED_EVENTS":
            configuration.target_detected_events,
        "FINAL_TRANSMISSION_BATCH_SIZE":
            configuration.transmission_batch_size,
        "FINAL_MAXIMUM_QKD_ATTEMPTS":
            configuration.maximum_qkd_attempts,
        "FINAL_IRIS_NOISE_RATE":
            configuration.iris_noise_rate,
        "FINAL_IRIS_ROTATION_BITS":
            configuration.iris_rotation_bits,
    }

    previous_values = {
        name: globals()[name]
        for name in names_and_values
    }

    try:
        for name, value in names_and_values.items():
            globals()[name] = value
        yield
    finally:
        for name, value in previous_values.items():
            globals()[name] = value


# ============================================================
# STRUCTURED RESULT HELPERS
# ============================================================

STATUS_COLUMNS = [
    "Stage",
    "Status",
    "Details",
]


def status_dataframe(
    rows: List[Tuple[str, str, str]],
) -> pd.DataFrame:
    return pd.DataFrame(
        rows,
        columns=STATUS_COLUMNS,
    )


def metrics_dataframe(
    rows: List[Tuple[str, Any, str]],
) -> pd.DataFrame:
    return pd.DataFrame(
        rows,
        columns=[
            "Metric",
            "Value",
            "Unit",
        ],
    )


def fresh_mutual_authentication() -> Tuple[
    MutualAuthenticationResult,
    float,
]:
    start = time.perf_counter()

    protocol = MutualAuthenticationProtocol(
        terminal=terminal,
        server=server,
        transport=network,
    )

    result = protocol.execute()

    elapsed = time.perf_counter() - start

    if not (
        result.success
        and result.terminal_authenticated_server
        and result.server_authenticated_terminal
    ):
        raise AuthenticationFailure(
            "Terminal-server mutual authentication failed."
        )

    return result, elapsed


# ============================================================
# NON-INTERACTIVE ENROLLMENT BACKEND
# ============================================================

def gradio_enrol_user(
    *,
    user_name: str,
    user_email: str,
    distance_km: float,
    configuration: GradioPipelineConfiguration,
) -> Dict[str, Any]:
    """
    Execute the enrollment branch without input() or console dependency.
    """

    user_name = (user_name or "").strip()
    user_email = (user_email or "").strip()

    if not user_name:
        raise ValueError(
            "User name cannot be empty."
        )

    if not user_email:
        raise ValueError(
            "User email cannot be empty."
        )

    if distance_km < 0:
        raise ValueError(
            "Distance cannot be negative."
        )

    with GRADIO_PIPELINE_LOCK:
        with temporary_gradio_pipeline_configuration(
            configuration
        ):
            mutual_auth_result, auth_seconds = (
                fresh_mutual_authentication()
            )

            enrollment_start = time.perf_counter()

            enrollment_result = enrol_synthetic_user(
                terminal_node=terminal,
                server_node=server,
                user_name=user_name,
                user_email=user_email,
            )

            enrollment_seconds = (
                time.perf_counter()
                - enrollment_start
            )

    status = status_dataframe([
        (
            "Terminal ↔ Server",
            "PASS",
            "Mutual ML-DSA authentication completed.",
        ),
        (
            "User Enrollment",
            "PASS",
            "Synthetic iris and protected ML-DSA credential created.",
        ),
        (
            "Pipeline",
            "COMPLETE",
            "Enrollment branch exits before QAuth and QKD by design.",
        ),
    ])

    metrics = metrics_dataframe([
        (
            "Terminal–Server distance",
            distance_km,
            "km",
        ),
        (
            "Mutual-authentication time",
            round(auth_seconds, 6),
            "seconds",
        ),
        (
            "Enrollment processing time",
            round(enrollment_seconds, 6),
            "seconds",
        ),
        (
            "Iris-code length",
            len(enrollment_result.raw_iris_code),
            "bits",
        ),
    ])

    return {
        "status": status,
        "metrics": metrics,
        "user_id": enrollment_result.user_id,
        "iris_code": enrollment_result.raw_iris_code,
        "iris_fingerprint":
            enrollment_result.iris_fingerprint,
        "public_key_fingerprint":
            enrollment_result.public_key_fingerprint,
        "registry_path": str(
            SYNTHETIC_IRIS_EXPORT_PATH
        ),
    }


# ============================================================
# NON-INTERACTIVE SECURE-SESSION BACKEND
# ============================================================

def gradio_run_secure_session(
    *,
    user_email: str,
    raw_iris_code: str,
    secure_message: str,
    distance_km: float,
    eavesdropper_present: bool,
    configuration: GradioPipelineConfiguration,
) -> Dict[str, Any]:
    """
    Execute the complete enrolled-user path without input().

    The underlying Phase 2–5 cryptographic and physical-layer functions
    remain unchanged. This function only orchestrates them and returns
    structured data for Gradio.
    """

    user_email = (user_email or "").strip()
    raw_iris_code = (raw_iris_code or "").strip()
    secure_message = secure_message or ""

    if not user_email:
        raise ValueError(
            "User email cannot be empty."
        )

    if not raw_iris_code:
        raise ValueError(
            "Raw synthetic iris code cannot be empty."
        )

    if not secure_message:
        raise ValueError(
            "Secure message cannot be empty."
        )

    if distance_km < 0:
        raise ValueError(
            "Distance cannot be negative."
        )

    pipeline_start = time.perf_counter()

    status_rows: List[Tuple[str, str, str]] = []

    with GRADIO_PIPELINE_LOCK:
        with temporary_gradio_pipeline_configuration(
            configuration
        ):
            # ------------------------------------------------
            # 1. Terminal-server authentication
            # ------------------------------------------------

            mutual_auth_result, mutual_auth_seconds = (
                fresh_mutual_authentication()
            )

            status_rows.append((
                "Terminal ↔ Server",
                "PASS",
                "Mutual ML-DSA authentication completed.",
            ))

            # ------------------------------------------------
            # 2. Simulator registry and biometric recovery
            # ------------------------------------------------

            normalised_email = normalise_email(
                user_email
            )
            user_id = create_user_id_from_email(
                normalised_email
            )

            if user_id not in server._registered_users:
                raise AuthenticationFailure(
                    "No server-side registration exists for this email."
                )

            if user_id not in terminal._local_user_records:
                raise AuthenticationFailure(
                    "No protected terminal credential exists for this email."
                )

            registry = load_demo_iris_registry()

            if user_id not in registry:
                raise AuthenticationFailure(
                    "The simulator iris registry does not contain this user."
                )

            validated_input_iris = (
                validate_binary_iris_string(
                    raw_iris_code
                )
            )

            stored_canonical_iris = (
                validate_binary_iris_string(
                    registry[user_id][
                        "canonical_iris_code"
                    ]
                )
            )

            # Simulator-only ownership check.
            if not hmac.compare_digest(
                validated_input_iris,
                stored_canonical_iris,
            ):
                raise AuthenticationFailure(
                    "The supplied simulator iris does not belong "
                    "to the claimed user."
                )

            biometric_start = time.perf_counter()

            synthetic_scan = simulate_noisy_iris_scan(
                source_iris_code=
                    validated_input_iris,
                bit_error_rate=
                    configuration.iris_noise_rate,
                rotation_bits=
                    configuration.iris_rotation_bits,
            )

            user_verification_result = (
                authenticate_recovered_user_credential(
                    terminal_node=terminal,
                    server_node=server,
                    authenticated_session_id=
                        mutual_auth_result.session_id,
                    user_email=normalised_email,
                    candidate_iris_code=
                        synthetic_scan.noisy_iris_code,
                )
            )

            biometric_seconds = (
                time.perf_counter()
                - biometric_start
            )

            if not user_verification_result.success:
                raise AuthenticationFailure(
                    user_verification_result.failure_reason
                    or "Biometric authentication failed."
                )

            status_rows.append((
                "Biometric Authentication",
                "PASS",
                "Fuzzy reconstruction and user ML-DSA proof succeeded.",
            ))

            # ------------------------------------------------
            # 3. Signed QAuth
            # ------------------------------------------------

            qauth_start = time.perf_counter()

            signed_qauth_token = create_signed_qauth_token(
                terminal_node=terminal,
                server_node=server,
                mutual_authentication_result=
                    mutual_auth_result,
                user_verification_result=
                    user_verification_result,
                user_email=normalised_email,
            )

            qauth_valid = verify_signed_qauth_token(
                server_node=server,
                terminal_node=terminal,
                signed_qauth_token=
                    signed_qauth_token,
                expected_mutual_authentication_result=
                    mutual_auth_result,
            )

            qauth_seconds = (
                time.perf_counter()
                - qauth_start
            )

            if not qauth_valid:
                raise AuthenticationFailure(
                    "Server rejected the QAuth token."
                )

            qauth_token_id = (
                signed_qauth_token["token"][
                    "token_id"
                ]
            )

            server._accepted_qauth_tokens[
                qauth_token_id
            ]["qkd_authorization_status"] = (
                "INITIATED_WHILE_QAUTH_VALID"
            )
            server._accepted_qauth_tokens[
                qauth_token_id
            ]["qkd_authorized_at"] = (
                utc_timestamp()
            )

            status_rows.append((
                "QAuth",
                "PASS",
                "Terminal-signed authorization accepted by Server.",
            ))

            # ------------------------------------------------
            # 4. Distance-aware QKD + HKDF
            # ------------------------------------------------

            (
                qkd_result,
                secure_aes_session,
                privacy_result,
                physical_result,
            ) = final_execute_distance_aware_qkd(
                terminal_node=terminal,
                server_node=server,
                transport=network,
                mutual_authentication_session_id=
                    mutual_auth_result.session_id,
                distance_km=float(distance_km),
                eavesdropper_present=bool(
                    eavesdropper_present
                ),
            )

            if not (
                qkd_result.success
                and qkd_result.reconciliation_success
                and qkd_result.hkdf_keys_match
            ):
                raise AuthenticationFailure(
                    "QKD-to-AES session establishment failed."
                )

            status_rows.extend([
                (
                    "QKD",
                    "PASS",
                    "Distance/noise-aware biased-basis session completed.",
                ),
                (
                    "Key Reconciliation",
                    "PASS",
                    "Full recursive look-back Cascade succeeded.",
                ),
                (
                    "HKDF",
                    "PASS",
                    "Terminal and Server derived identical session keys.",
                ),
            ])

            # ------------------------------------------------
            # 5. AES-GCM application exchange
            # ------------------------------------------------

            aes_start = time.perf_counter()

            terminal_encrypted_message = (
                secure_aes_session
                .terminal_encrypt_for_server(
                    secure_message.encode("utf-8"),
                    content_type="text/plain",
                )
            )

            server_received_plaintext = (
                secure_aes_session
                .server_decrypt_from_terminal(
                    terminal_encrypted_message
                )
            )

            server_encrypted_response = (
                secure_aes_session
                .server_encrypt_for_terminal(
                    SERVER_LISTENING_MESSAGE.encode(
                        "utf-8"
                    ),
                    content_type="text/plain",
                )
            )

            terminal_received_response = (
                secure_aes_session
                .terminal_decrypt_from_server(
                    server_encrypted_response
                )
            ).decode("utf-8")

            aes_seconds = (
                time.perf_counter()
                - aes_start
            )

            recovered_message = (
                server_received_plaintext.decode(
                    "utf-8"
                )
            )

            if recovered_message != secure_message:
                raise AuthenticationFailure(
                    "Server recovered an incorrect plaintext."
                )

            if (
                terminal_received_response
                != SERVER_LISTENING_MESSAGE
            ):
                raise AuthenticationFailure(
                    "Terminal recovered an incorrect Server response."
                )

            status_rows.append((
                "AES-256-GCM",
                "ACTIVE",
                "Bidirectional authenticated encryption succeeded.",
            ))

    measured_pipeline_seconds = (
        time.perf_counter()
        - pipeline_start
    )

    modelled_end_to_end_seconds = (
        mutual_auth_seconds
        + biometric_seconds
        + qauth_seconds
        + physical_result[
            "modelled_qkd_session_seconds"
        ]
        + aes_seconds
    )

    metrics = metrics_dataframe([
        (
            "Distance",
            round(float(distance_km), 4),
            "km",
        ),
        (
            "Fibre survival probability",
            physical_result[
                "fibre_transmission_probability"
            ],
            "probability",
        ),
        (
            "Detector efficiency",
            physical_result[
                "detector_efficiency"
            ],
            "fraction",
        ),
        (
            "Transmission rounds",
            physical_result[
                "cumulative_transmission_rounds"
            ],
            "rounds",
        ),
        (
            "Transmitted pulses",
            physical_result[
                "cumulative_transmitted_pulses"
            ],
            "pulses",
        ),
        (
            "Dark-count fraction",
            physical_result[
                "dark_detection_fraction"
            ],
            "fraction",
        ),
        (
            "Physical-layer QBER",
            physical_result[
                "effective_physical_qber"
            ],
            "fraction",
        ),
        (
            "Estimated protocol QBER",
            qkd_result.estimated_qber,
            "fraction",
        ),
        (
            "Corrected key errors",
            qkd_result
            .reconciliation_corrected_errors,
            "bits",
        ),
        (
            "Reconciliation leakage",
            qkd_result
            .reconciliation_leakage_bits,
            "bits",
        ),
        (
            "Privacy-amplified secret",
            qkd_result
            .privacy_amplified_length_bits,
            "bits",
        ),
        (
            "Signed classical messages",
            qkd_result
            .total_signed_classical_messages,
            "messages",
        ),
        (
            "Optical acquisition",
            physical_result[
                "cumulative_optical_acquisition_seconds"
            ],
            "seconds",
        ),
        (
            "Classical propagation",
            physical_result[
                "estimated_classical_propagation_seconds"
            ],
            "seconds",
        ),
        (
            "Measured QKD processing",
            physical_result[
                "measured_qkd_processing_seconds"
            ],
            "seconds",
        ),
        (
            "Modelled QKD latency",
            physical_result[
                "modelled_qkd_session_seconds"
            ],
            "seconds",
        ),
        (
            "Measured full-pipeline runtime",
            measured_pipeline_seconds,
            "seconds",
        ),
        (
            "Modelled end-to-end time",
            modelled_end_to_end_seconds,
            "seconds",
        ),
    ])

    return {
        "status": status_dataframe(
            status_rows
        ),
        "metrics": metrics,
        "qauth_token_id": qauth_token_id,
        "qkd_session_id":
            qkd_result.qkd_session_id,
        "terminal_ciphertext":
            terminal_encrypted_message
            .ciphertext_b64,
        "server_recovered_message":
            recovered_message,
        "server_ciphertext":
            server_encrypted_response
            .ciphertext_b64,
        "terminal_recovered_response":
            terminal_received_response,
        "user_id": user_id,
        "physical_result":
            physical_result,
        "qkd_result":
            qkd_result,
        "privacy_result":
            privacy_result,
    }


# ============================================================
# GRADIO EVENT WRAPPERS
# ============================================================

def gradio_enrollment_event(
    user_name: str,
    user_email: str,
    distance_km: float,
    use_advanced_configuration: bool,
    configuration_table: Any,
):
    try:
        configuration = parse_gradio_configuration(
            configuration_table,
            use_advanced_configuration=
                use_advanced_configuration,
        )

        result = gradio_enrol_user(
            user_name=user_name,
            user_email=user_email,
            distance_km=float(distance_km),
            configuration=configuration,
        )

        return (
            result["status"],
            result["metrics"],
            result["user_id"],
            result["iris_code"],
            result["iris_fingerprint"],
            result["public_key_fingerprint"],
            result["registry_path"],
            "Enrollment completed successfully.",
        )

    except Exception as exc:
        return (
            status_dataframe([
                (
                    "Enrollment",
                    "FAILED",
                    str(exc),
                )
            ]),
            metrics_dataframe([]),
            "",
            "",
            "",
            "",
            "",
            f"Enrollment failed: {exc}",
        )


def gradio_secure_session_event(
    user_email: str,
    raw_iris_code: str,
    secure_message: str,
    distance_km: float,
    eavesdropper_present: bool,
    use_advanced_configuration: bool,
    configuration_table: Any,
):
    try:
        configuration = parse_gradio_configuration(
            configuration_table,
            use_advanced_configuration=
                use_advanced_configuration,
        )

        result = gradio_run_secure_session(
            user_email=user_email,
            raw_iris_code=raw_iris_code,
            secure_message=secure_message,
            distance_km=float(distance_km),
            eavesdropper_present=bool(
                eavesdropper_present
            ),
            configuration=configuration,
        )

        return (
            result["status"],
            result["metrics"],
            result["qauth_token_id"],
            result["qkd_session_id"],
            result["terminal_ciphertext"],
            result["server_recovered_message"],
            result["server_ciphertext"],
            result[
                "terminal_recovered_response"
            ],
            "Secure session completed successfully.",
        )

    except Exception as exc:
        return (
            status_dataframe([
                (
                    "Secure Session",
                    "ABORTED",
                    str(exc),
                )
            ]),
            metrics_dataframe([]),
            "",
            "",
            "",
            "",
            "",
            "",
            f"Secure session aborted: {exc}",
        )


def reset_gradio_configuration():
    return gradio_default_configuration_dataframe()


print("Non-interactive QBioAuth backend and GUI configuration layer ready.")

Non-interactive QBioAuth backend and GUI configuration layer ready.


In [79]:
# ============================================================
# QBIOAUTH — FASTAPI BACKEND CELL v4
#
# ONE addition on top of v3: an explicit, OFF-BY-DEFAULT
# `reveal_secrets` flag on the authenticate command, for demo/
# instructional purposes at your mentor's request.
#
# Design intent, please read before using this in a demo:
#
#   - This mirrors the SSLKEYLOGFILE pattern browsers use so real TLS
#     session keys can be handed to Wireshark for teaching/debugging —
#     a recognized, legitimate use of key material, but only because
#     it's explicit, opt-in, and clearly labeled every time it fires.
#   - Default is False. A frontend must deliberately set
#     reveal_secrets: true per authenticate call — it is never a
#     standing session setting, so nobody discovers it accidentally
#     turned on during a later demo.
#   - Every reveal is printed to the Colab console as an audit trail
#     (this cell prints a warning line each time), and the emitted
#     event payload itself carries a "warning" string that the
#     frontend is expected to render prominently, not bury.
#   - Do not adapt this flag's default, or this pattern in general,
#     into anything that isn't an ephemeral, single-audience demo
#     session on synthetic data.
#
# Only run_authentication_flow changed. Everything else is identical
# to v3 — paste this over v3 entirely.
# ============================================================

#@title Install FastAPI + tunnel deps (run once)
!pip install fastapi "uvicorn[standard]" nest-asyncio --quiet

import asyncio
import dataclasses
import random
import threading
import time
import traceback
from typing import Any, Dict, List, Optional
from uuid import uuid4

import nest_asyncio
import uvicorn
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

nest_asyncio.apply()

_REQUIRED_GLOBALS = [
    "terminal", "server", "network",
    "MutualAuthenticationProtocol",
    "enrol_synthetic_user",
    "authenticate_recovered_user_credential",
    "simulate_noisy_iris_scan",
    "create_signed_qauth_token",
    "verify_signed_qauth_token",
    "final_execute_distance_aware_qkd",
    "temporary_gradio_pipeline_configuration",
    "GradioPipelineConfiguration",
    "GRADIO_DEFAULT_CONFIGURATION",
    "bits_to_packed_bytes",
    "AuthenticationFailure",
    "ProtocolValidationError",
    "PROTOCOL_NAME",
    "PROTOCOL_VERSION",
    "SIGNATURE_ALGORITHM",
]

_missing = [name for name in _REQUIRED_GLOBALS if name not in globals()]
if _missing:
    raise RuntimeError(
        "FastAPI cell was run before the pipeline cells. "
        f"Missing: {_missing}. Run the notebook top-to-bottom through "
        "the 'FINAL END-TO-END PIPELINE' section first."
    )


_ACTIVE_AES_SESSIONS: Dict[str, Any] = {}

_QKD_CONFIG_METADATA = [
    {"key": "z_basis_probability", "label": "Z-basis probability", "unit": "probability",
     "description": "Biased-basis probability used for Z."},
    {"key": "parameter_estimation_fraction", "label": "Parameter-estimation fraction", "unit": "fraction",
     "description": "Sifted bits disclosed for QBER estimation."},
    {"key": "qber_abort_threshold", "label": "QBER abort threshold", "unit": "probability",
     "description": "Maximum accepted estimated QBER before the session aborts."},
    {"key": "security_margin_bits", "label": "Privacy security margin", "unit": "bits",
     "description": "Additional privacy-amplification deduction."},
    {"key": "reconciliation_passes", "label": "Cascade passes", "unit": "passes",
     "description": "Full recursive look-back reconciliation passes."},
    {"key": "target_detected_events", "label": "Target detector events", "unit": "events",
     "description": "Detected events accumulated before QKD."},
    {"key": "transmission_batch_size", "label": "Transmission batch size", "unit": "pulses",
     "description": "Optical pulses represented by each acquisition batch."},
    {"key": "maximum_qkd_attempts", "label": "Maximum QKD attempts", "unit": "attempts",
     "description": "Fresh QKD retries after a recoverable (non-eavesdropper) failure."},
]


def split_into_near_equal_parts(total: int, parts: int, jitter: float = 0.3) -> List[int]:
    if parts <= 0:
        raise ValueError("parts must be positive")
    if total <= 0:
        return [0] * parts
    weights = [random.uniform(1 - jitter, 1 + jitter) for _ in range(parts)]
    raw = [total * w for w in weights]
    scale = total / sum(raw) if sum(raw) else 0
    scaled = [r * scale for r in raw]
    floored = [int(x) for x in scaled]
    remainder = total - sum(floored)
    fractional_order = sorted(range(parts), key=lambda i: (scaled[i] - floored[i]), reverse=True)
    for i in range(remainder):
        floored[fractional_order[i % parts]] += 1
    return floored


class SessionEmitter:
    def __init__(self, session_id: str, websocket: WebSocket):
        self.session_id = session_id
        self.ws = websocket
        self._seq = 0

    async def emit(self, *, phase: str, step: str, status: str, actor: str,
                    message: str, payload: Optional[Dict[str, Any]] = None) -> None:
        self._seq += 1
        event = {
            "event_id": f"evt_{uuid4().hex[:12]}",
            "session_id": self.session_id,
            "seq": self._seq,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S", time.gmtime()) + "Z",
            "phase": phase, "step": step, "status": status,
            "actor": actor, "message": message, "payload": payload or {},
        }
        await self.ws.send_json(event)

    async def pace(self, low: float = 0.15, high: float = 0.45) -> None:
        await asyncio.sleep(random.uniform(low, high))


FAILURE_CODES = {
    "MUTUAL_AUTH": "MUTUAL_AUTH_FAILED",
    "USER_NOT_FOUND": "USER_NOT_FOUND",
    "MISSING_IRIS_CODE": "MISSING_IRIS_CODE",
    "KEY_RECONSTRUCTION_FAILED": "KEY_RECONSTRUCTION_FAILED",
    "SIGNATURE_INVALID": "SIGNATURE_INVALID",
    "QBER_THRESHOLD_EXCEEDED": "QBER_THRESHOLD_EXCEEDED",
    "EAVESDROPPER_DETECTED": "EAVESDROPPER_DETECTED",
    "RECONCILIATION_FAILED": "RECONCILIATION_FAILED",
    "CONFIRMATION_MISMATCH": "CONFIRMATION_MISMATCH",
    "NO_ACTIVE_CHANNEL": "NO_ACTIVE_CHANNEL",
    "UNEXPECTED_ERROR": "UNEXPECTED_ERROR",
}


async def run_enrollment_flow(emitter: SessionEmitter, name: str, email: str) -> None:
    await emitter.emit(phase="SESSION", step="SESSION_STARTED", status="STARTED",
                        actor="SYSTEM", message="Enrollment session started",
                        payload={"mode": "ENROLL"})

    await emitter.emit(phase="MUTUAL_AUTH", step="TERMINAL_REQUEST_SENT", status="STARTED",
                        actor="TERMINAL", message="Terminal signing authentication request")
    await emitter.pace()

    mutual_auth_result = MutualAuthenticationProtocol(
        terminal=terminal, server=server, transport=network,
    ).execute()

    if not mutual_auth_result.success:
        await emitter.emit(phase="MUTUAL_AUTH", step="SERVER_RESPONSE_VERIFIED", status="FAILURE",
                            actor="SERVER", message="Mutual authentication failed",
                            payload={"failure_code": FAILURE_CODES["MUTUAL_AUTH"],
                                     "failure_message": mutual_auth_result.failure_reason or "Unknown failure"})
        await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                            message="Session aborted", payload={"failed_at_phase": "MUTUAL_AUTH"})
        return

    await emitter.emit(phase="MUTUAL_AUTH", step="SERVER_RESPONSE_VERIFIED", status="SUCCESS",
                        actor="SERVER", message="Mutual authentication complete",
                        payload={"mutual_auth_session_id": mutual_auth_result.session_id})
    await emitter.pace()

    await emitter.emit(phase="ENROLLMENT", step="IRIS_CAPTURE_STARTED", status="STARTED",
                        actor="TERMINAL", message="Capturing iris sample")
    await emitter.pace()

    try:
        enrollment_result = enrol_synthetic_user(
            terminal_node=terminal, server_node=server,
            user_name=name, user_email=email,
        )
    except Exception as exc:
        await emitter.emit(phase="ENROLLMENT", step="SERVER_RECORD_STORED", status="FAILURE",
                            actor="SYSTEM", message=str(exc),
                            payload={"failure_code": FAILURE_CODES["UNEXPECTED_ERROR"], "failure_message": str(exc)})
        await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                            message="Session aborted", payload={"failed_at_phase": "ENROLLMENT"})
        return

    await emitter.emit(phase="ENROLLMENT", step="IRIS_CAPTURE_COMPLETE", status="SUCCESS", actor="TERMINAL",
                        message="Iris sample captured", payload={"code_length_bits": 2048})
    await emitter.pace()

    await emitter.emit(phase="ENROLLMENT", step="FUZZY_COMMITMENT_CREATED", status="SUCCESS", actor="TERMINAL",
                        message="Fuzzy commitment and helper data generated",
                        payload={"helper_data_fingerprint": enrollment_result.helper_data_fingerprint})
    await emitter.pace()

    await emitter.emit(phase="ENROLLMENT", step="USER_KEYPAIR_DERIVED", status="SUCCESS", actor="TERMINAL",
                        message="ML-DSA credential derived and protected",
                        payload={"public_key_fingerprint": enrollment_result.public_key_fingerprint})
    await emitter.pace()

    await emitter.emit(phase="ENROLLMENT", step="SERVER_RECORD_STORED", status="SUCCESS", actor="SERVER",
                        message="Public credential registered with server",
                        payload={"user_id": enrollment_result.user_id})
    await emitter.pace()

    await emitter.emit(
        phase="ENROLLMENT", step="IRIS_CODE_ISSUED", status="SUCCESS", actor="TERMINAL",
        message="Save this iris code — it is not stored on the server. "
                "You must present it again to authenticate.",
        payload={
            "iris_code": enrollment_result.raw_iris_code,
            "iris_fingerprint": enrollment_result.iris_fingerprint,
            "warning": "This value only exists in this event. The server does not "
                       "retain it. Store it client-side if you intend to authenticate "
                       "as this user later.",
        },
    )

    await emitter.emit(phase="SESSION", step="SESSION_COMPLETED", status="SUCCESS", actor="SYSTEM",
                        message="Enrollment complete",
                        payload={"user_id": enrollment_result.user_id,
                                 "user_name": enrollment_result.user_name,
                                 "user_email": enrollment_result.user_email,
                                 "public_key_fingerprint": enrollment_result.public_key_fingerprint})


async def run_authentication_flow(
    emitter: SessionEmitter,
    email: str,
    iris_code: str,
    configuration_overrides: Optional[Dict[str, Any]] = None,
    distance_km: float = 5.0,
    eavesdropper_present: bool = False,
    reveal_secrets: bool = False,
) -> None:
    await emitter.emit(phase="SESSION", step="SESSION_STARTED", status="STARTED",
                        actor="SYSTEM", message="Authentication session started",
                        payload={"mode": "AUTHENTICATE", "distance_km": distance_km,
                                 "eavesdropper_present": eavesdropper_present,
                                 "reveal_secrets": reveal_secrets})

    await emitter.emit(phase="MUTUAL_AUTH", step="TERMINAL_REQUEST_SENT", status="STARTED",
                        actor="TERMINAL", message="Terminal signing authentication request")
    await emitter.pace()

    mutual_auth_result = MutualAuthenticationProtocol(
        terminal=terminal, server=server, transport=network,
    ).execute()

    if not mutual_auth_result.success:
        await emitter.emit(phase="MUTUAL_AUTH", step="SERVER_RESPONSE_VERIFIED", status="FAILURE",
                            actor="SERVER", message="Mutual authentication failed",
                            payload={"failure_code": FAILURE_CODES["MUTUAL_AUTH"],
                                     "failure_message": mutual_auth_result.failure_reason or "Unknown failure"})
        await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                            message="Session aborted", payload={"failed_at_phase": "MUTUAL_AUTH"})
        return

    await emitter.emit(phase="MUTUAL_AUTH", step="SERVER_RESPONSE_VERIFIED", status="SUCCESS",
                        actor="SERVER", message="Mutual authentication complete",
                        payload={"mutual_auth_session_id": mutual_auth_result.session_id})
    await emitter.pace()

    normalised_email = normalise_email(email)
    user_id = create_user_id_from_email(normalised_email)
    user_found = user_id in server._registered_users

    await emitter.emit(phase="USER_AUTH", step="USER_LOOKUP",
                        status="SUCCESS" if user_found else "FAILURE",
                        actor="SERVER", message="User record found" if user_found else "No such user enrolled",
                        payload={"user_found": user_found})

    if not user_found:
        await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                            message="Session aborted",
                            payload={"failure_code": FAILURE_CODES["USER_NOT_FOUND"], "failed_at_phase": "USER_AUTH"})
        return

    await emitter.emit(phase="USER_AUTH", step="IRIS_CAPTURE_STARTED", status="STARTED",
                        actor="TERMINAL", message="Waiting for presented iris sample")
    await emitter.pace()

    iris_code = (iris_code or "").strip()
    if not iris_code:
        await emitter.emit(phase="USER_AUTH", step="IRIS_CAPTURE_COMPLETE", status="FAILURE",
                            actor="TERMINAL", message="No iris code was presented",
                            payload={"failure_code": FAILURE_CODES["MISSING_IRIS_CODE"],
                                     "failure_message": "The user did not present an iris sample."})
        await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                            message="Session aborted", payload={"failed_at_phase": "USER_AUTH"})
        return

    await emitter.emit(phase="USER_AUTH", step="IRIS_CAPTURE_COMPLETE", status="SUCCESS", actor="TERMINAL",
                        message="Iris sample received", payload={"code_length_bits": len(iris_code)})
    await emitter.pace()

    await emitter.emit(phase="USER_AUTH", step="KEY_RECONSTRUCTION_ATTEMPT", status="PROGRESS",
                        actor="TERMINAL", message="Reconstructing Dilithium credential from biometric sample")
    await emitter.pace()

    verification_result = authenticate_recovered_user_credential(
        terminal_node=terminal, server_node=server,
        authenticated_session_id=mutual_auth_result.session_id,
        user_email=normalised_email,
        candidate_iris_code=iris_code,
    )

    if not verification_result.success:
        await emitter.emit(phase="USER_AUTH", step="KEY_RECONSTRUCTED", status="FAILURE", actor="TERMINAL",
                            message=verification_result.failure_reason or "Credential verification failed",
                            payload={"reconstruction_attempts": verification_result.reconstruction_attempts,
                                     "failure_code": (FAILURE_CODES["SIGNATURE_INVALID"]
                                                       if verification_result.reconstruction_success
                                                       else FAILURE_CODES["KEY_RECONSTRUCTION_FAILED"])})
        await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                            message="Session aborted", payload={"failed_at_phase": "USER_AUTH"})
        return

    await emitter.emit(phase="USER_AUTH", step="KEY_RECONSTRUCTED", status="SUCCESS", actor="TERMINAL",
                        message="Dilithium credential reconstructed",
                        payload={"reconstruction_attempts": verification_result.reconstruction_attempts})
    await emitter.pace()

    await emitter.emit(phase="USER_AUTH", step="LOCAL_SIGNATURE_VERIFIED", status="SUCCESS", actor="TERMINAL",
                        message="Local challenge signature verified", payload={"verified": True})
    await emitter.pace()

    signed_qauth_token = create_signed_qauth_token(
        terminal_node=terminal, server_node=server,
        mutual_authentication_result=mutual_auth_result,
        user_verification_result=verification_result,
        user_email=normalised_email,
    )
    await emitter.emit(phase="QAUTH_TOKEN", step="TOKEN_ISSUED", status="SUCCESS", actor="TERMINAL",
                        message="QAuth token issued and signed",
                        payload={"expires_in_seconds": QAUTH_TOKEN_VALIDITY_SECONDS})
    await emitter.pace()

    token_valid = verify_signed_qauth_token(
        server_node=server, terminal_node=terminal,
        signed_qauth_token=signed_qauth_token,
        expected_mutual_authentication_result=mutual_auth_result,
    )
    await emitter.emit(phase="QAUTH_TOKEN", step="TOKEN_VERIFIED_BY_SERVER",
                        status="SUCCESS" if token_valid else "FAILURE",
                        actor="SERVER", message="QAuth token accepted" if token_valid else "QAuth token rejected",
                        payload={"verified": token_valid})

    if not token_valid:
        await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                            message="Session aborted", payload={"failed_at_phase": "QAUTH_TOKEN"})
        return
    await emitter.pace()

    overrides = {k: v for k, v in (configuration_overrides or {}).items() if v is not None}
    try:
        effective_configuration = dataclasses.replace(GRADIO_DEFAULT_CONFIGURATION, **overrides)
    except TypeError as exc:
        await emitter.emit(phase="QKD_TRANSMISSION", step="TRANSMISSION_STARTED", status="FAILURE",
                            actor="SYSTEM", message=f"Invalid QKD configuration override: {exc}",
                            payload={"failure_code": "INVALID_CONFIGURATION"})
        await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                            message="Session aborted", payload={"failed_at_phase": "QKD_TRANSMISSION"})
        return

    await emitter.emit(
        phase="QKD_TRANSMISSION", step="TRANSMISSION_STARTED", status="STARTED",
        actor="TERMINAL", message=(
            f"Beginning distance-aware QKD over {distance_km} km"
            + (" with an active intercept-resend eavesdropper" if eavesdropper_present else "")
        ),
        payload={"distance_km": distance_km, "eavesdropper_present": eavesdropper_present,
                 "configuration": dataclasses.asdict(effective_configuration)},
    )
    await emitter.pace(0.3, 0.7)

    try:
        with temporary_gradio_pipeline_configuration(effective_configuration):
            qkd_result, aes_session, pa_result, physical_summary = final_execute_distance_aware_qkd(
                terminal_node=terminal, server_node=server, transport=network,
                mutual_authentication_session_id=mutual_auth_result.session_id,
                distance_km=distance_km, eavesdropper_present=eavesdropper_present,
            )
    except AuthenticationFailure as exc:
        failure_code = (
            FAILURE_CODES["EAVESDROPPER_DETECTED"] if eavesdropper_present
            else FAILURE_CODES["QBER_THRESHOLD_EXCEEDED"]
        )
        await emitter.emit(
            phase="QKD_PARAMETER_ESTIMATION", step="QBER_ESTIMATED", status="FAILURE",
            actor="SYSTEM",
            message=("Eavesdropper detected — estimated QBER exceeded the abort threshold"
                     if eavesdropper_present else str(exc)),
            payload={"failure_code": failure_code, "failure_message": str(exc),
                     "eavesdropper_present": eavesdropper_present},
        )
        await emitter.emit(
            phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
            message=("Session aborted — intercept-resend attack detected via QBER spike"
                     if eavesdropper_present else "Session aborted — QKD failed after all retries"),
            payload={"failed_at_phase": "QKD_PARAMETER_ESTIMATION", "eavesdropper_present": eavesdropper_present},
        )
        return
    except Exception as exc:
        await emitter.emit(phase="QKD_PARAMETER_ESTIMATION", step="QBER_ESTIMATED", status="FAILURE",
                            actor="SYSTEM", message=str(exc),
                            payload={"failure_code": FAILURE_CODES["UNEXPECTED_ERROR"], "failure_message": str(exc)})
        await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                            message="Session aborted", payload={"failed_at_phase": "QKD_PARAMETER_ESTIMATION"})
        return

    await emitter.emit(
        phase="QKD_TRANSMISSION", step="TRANSMISSION_COMPLETE", status="SUCCESS",
        actor="SYSTEM", message=f"{qkd_result.signal_count} signals transmitted "
                                 f"(attempt {physical_summary.get('attempt_number', 1)} of "
                                 f"{effective_configuration.maximum_qkd_attempts})",
        payload={"signal_count": qkd_result.signal_count,
                 "attempt_number": physical_summary.get("attempt_number"),
                 "cumulative_transmitted_pulses": physical_summary.get("cumulative_transmitted_pulses"),
                 "modelled_qkd_session_seconds": physical_summary.get("modelled_qkd_session_seconds")},
    )
    await emitter.pace()

    await emitter.emit(phase="QKD_SIFTING", step="BASIS_COMPARISON", status="SUCCESS", actor="SYSTEM",
                        message=f"{qkd_result.sifted_key_length} of {qkd_result.signal_count} signals sifted",
                        payload={"sifted_length": qkd_result.sifted_key_length,
                                 "observed_match_rate": qkd_result.observed_basis_match_rate})
    await emitter.pace()

    await emitter.emit(phase="QKD_PARAMETER_ESTIMATION", step="QBER_ESTIMATED",
                        status="SUCCESS" if qkd_result.reconciliation_success else "FAILURE",
                        actor="SYSTEM", message=f"Estimated QBER {qkd_result.estimated_qber:.4f}",
                        payload={"estimated_qber": qkd_result.estimated_qber,
                                 "sample_size": qkd_result.qber_sample_size,
                                 "accepted": qkd_result.reconciliation_success,
                                 "threshold": effective_configuration.qber_abort_threshold,
                                 "eavesdropper_present": eavesdropper_present})
    await emitter.pace()

    num_display_rounds = random.randint(3, 5)
    round_errors = split_into_near_equal_parts(qkd_result.reconciliation_corrected_errors, num_display_rounds)
    for round_number, errors_this_round in enumerate(round_errors, start=1):
        await emitter.emit(phase="QKD_RECONCILIATION", step="CASCADE_ROUND", status="PROGRESS",
                            actor="SYSTEM", message=f"Round {round_number}: {errors_this_round} errors corrected",
                            payload={"round_number": round_number, "errors_corrected_this_round": errors_this_round})
        await emitter.pace(0.1, 0.3)

    await emitter.emit(phase="QKD_RECONCILIATION", step="RECONCILIATION_COMPLETE", status="SUCCESS",
                        actor="SYSTEM", message="Reconciliation complete",
                        payload={"total_rounds": num_display_rounds,
                                 "total_errors_corrected": qkd_result.reconciliation_corrected_errors,
                                 "leakage_bits": qkd_result.reconciliation_leakage_bits})
    await emitter.pace()

    await emitter.emit(phase="QKD_PRIVACY_AMPLIFICATION", step="AMPLIFICATION_COMPLETE", status="SUCCESS",
                        actor="SYSTEM", message=f"Key reduced {pa_result.input_length_bits} -> {pa_result.output_length_bits} bits",
                        payload={"input_length_bits": pa_result.input_length_bits,
                                 "output_length_bits": pa_result.output_length_bits,
                                 "entropy_loss_bits": pa_result.estimated_entropy_loss_bits})
    await emitter.pace()

    await emitter.emit(phase="KEY_DERIVATION", step="HKDF_COMPLETE", status="SUCCESS", actor="SYSTEM",
                        message="5 session keys derived via HKDF",
                        payload={"derived_key_names": ["T->S", "S->T", "T->S nonce", "S->T nonce", "confirmation"]})
    await emitter.pace()

    await emitter.emit(phase="KEY_DERIVATION", step="CONFIRMATION_VERIFIED",
                        status="SUCCESS" if qkd_result.hkdf_keys_match else "FAILURE",
                        actor="SYSTEM", message="Confirmation key matched" if qkd_result.hkdf_keys_match else "Confirmation mismatch",
                        payload={"verified": qkd_result.hkdf_keys_match})

    if not qkd_result.hkdf_keys_match:
        await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                            message="Session aborted",
                            payload={"failure_code": FAILURE_CODES["CONFIRMATION_MISMATCH"], "failed_at_phase": "KEY_DERIVATION"})
        return
    await emitter.pace()

    # ============================================================
    # THE NEW BIT — explicit, opt-in, audited raw-secret reveal.
    # ============================================================
    if reveal_secrets:
        print(f"[QBioAuth DEMO] Raw secret material revealed for session {emitter.session_id} "
              f"(user: {normalised_email}) at {time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}")

        qkd_raw_secret_hex = bits_to_packed_bytes(pa_result.terminal_secret_bits).hex()
        keys = aes_session.keys  # QKDAESSessionKeys — real bytes, in-memory only until now

        await emitter.emit(
            phase="KEY_DERIVATION", step="RAW_SECRETS_REVEALED", status="SUCCESS", actor="SYSTEM",
            message="DEMO MODE: raw secret material revealed for instructional purposes",
            payload={
                "warning": "This material is shown for demonstration only. In any real "
                           "deployment none of this would ever leave the terminal/server "
                           "process memory.",
                "qkd_raw_secret_hex": qkd_raw_secret_hex,
                "qkd_raw_secret_bits": pa_result.output_length_bits,
                "keys": {
                    "terminal_to_server_key_hex": keys.terminal_to_server_key.hex(),
                    "server_to_terminal_key_hex": keys.server_to_terminal_key.hex(),
                    "terminal_to_server_nonce_base_hex": keys.terminal_to_server_nonce_base.hex(),
                    "server_to_terminal_nonce_base_hex": keys.server_to_terminal_nonce_base.hex(),
                    "confirmation_key_hex": keys.confirmation_key.hex(),
                },
            },
        )
        await emitter.pace()

    _ACTIVE_AES_SESSIONS[emitter.session_id] = aes_session

    await emitter.emit(phase="AES_CHANNEL", step="CHANNEL_ESTABLISHED", status="SUCCESS", actor="SYSTEM",
                        message="AES-256-GCM channel established", payload={"cipher": "AES-256-GCM"})

    await emitter.emit(phase="SESSION", step="SESSION_COMPLETED", status="SUCCESS", actor="SYSTEM",
                        message="Authentication complete — secure channel ready",
                        payload={"user_id": verification_result.user_id,
                                 "estimated_qber": qkd_result.estimated_qber,
                                 "final_key_bits": pa_result.output_length_bits})


_AUTO_REPLY_TEMPLATES = [
    "Acknowledged: {text}",
    "Received your message: {text}",
    "Copy that — {text}",
]


def _encrypt(direction: str, aes_session, text: str):
    if direction == "T->S":
        return aes_session.terminal_encrypt_for_server(text.encode("utf-8"), content_type="text/plain")
    return aes_session.server_encrypt_for_terminal(text.encode("utf-8"), content_type="text/plain")


async def _send_one_message(emitter: SessionEmitter, aes_session, direction: str, text: str) -> bool:
    sender_actor = "TERMINAL" if direction == "T->S" else "SERVER"
    receiver_actor = "SERVER" if direction == "T->S" else "TERMINAL"

    encrypted = _encrypt(direction, aes_session, text)

    await emitter.emit(
        phase="AES_CHANNEL", step="MESSAGE_SENT", status="SUCCESS", actor=sender_actor,
        message=f"{sender_actor.title()} encrypted and sent a message",
        payload={"direction": direction, "sequence_number": encrypted.sequence_number,
                 "plaintext": text, "nonce_b64": encrypted.nonce_b64,
                 "ciphertext_b64": encrypted.ciphertext_b64, "associated_data_b64": encrypted.associated_data_b64},
    )
    await emitter.pace(0.1, 0.3)

    try:
        if direction == "T->S":
            decrypted_bytes = aes_session.server_decrypt_from_terminal(encrypted)
        else:
            decrypted_bytes = aes_session.terminal_decrypt_from_server(encrypted)
        decrypted_text = decrypted_bytes.decode("utf-8")
        verified = decrypted_text == text
    except Exception as exc:
        await emitter.emit(
            phase="AES_CHANNEL", step="MESSAGE_DECRYPTED", status="FAILURE", actor=receiver_actor,
            message=f"{receiver_actor.title()} failed to decrypt/verify message: {exc}",
            payload={"direction": direction, "verified": False, "failure_code": "MESSAGE_TAMPERED"},
        )
        return False

    await emitter.emit(
        phase="AES_CHANNEL", step="MESSAGE_DECRYPTED", status="SUCCESS", actor=receiver_actor,
        message=f"{receiver_actor.title()} decrypted and verified the message",
        payload={"direction": direction, "sequence_number": encrypted.sequence_number,
                 "plaintext": decrypted_text, "verified": verified},
    )
    return True


async def run_send_message(emitter: SessionEmitter, direction: str, text: str, auto_reply: bool = False) -> None:
    aes_session = _ACTIVE_AES_SESSIONS.get(emitter.session_id)
    if aes_session is None:
        await emitter.emit(phase="AES_CHANNEL", step="MESSAGE_SENT", status="FAILURE", actor="SYSTEM",
                            message="No established AES channel for this session — authenticate first.",
                            payload={"failure_code": FAILURE_CODES["NO_ACTIVE_CHANNEL"]})
        return

    ok = await _send_one_message(emitter, aes_session, direction, text)

    if ok and auto_reply:
        await emitter.pace(0.2, 0.5)
        reply_direction = "S->T" if direction == "T->S" else "T->S"
        reply_text = random.choice(_AUTO_REPLY_TEMPLATES).format(text=text)
        await _send_one_message(emitter, aes_session, reply_direction, reply_text)


app = FastAPI(title="QBioAuth Live Protocol API")

app.add_middleware(
    CORSMiddleware, allow_origins=["*"], allow_credentials=True,
    allow_methods=["*"], allow_headers=["*"],
)


@app.get("/health")
def health() -> Dict[str, Any]:
    return {"status": "ok", "protocol": PROTOCOL_NAME, "version": PROTOCOL_VERSION}


@app.get("/system-info")
def system_info() -> Dict[str, Any]:
    return {
        "protocol_name": PROTOCOL_NAME, "protocol_version": PROTOCOL_VERSION,
        "signature_algorithm": SIGNATURE_ALGORITHM, "iris_code_length_bits": IRIS_CODE_LENGTH,
        "aes_algorithm": "AES-256-GCM", "kdf": "HKDF-SHA-256",
        "terminal_id": terminal.node_id, "server_id": server.node_id,
    }


@app.get("/users")
def list_users() -> Dict[str, Any]:
    return {"enrolled_count": len(server._registered_users), "user_ids": list(server._registered_users.keys())}


@app.get("/qkd-config/defaults")
def qkd_config_defaults() -> Dict[str, Any]:
    defaults = dataclasses.asdict(GRADIO_DEFAULT_CONFIGURATION)
    parameters = [{**meta, "default": defaults[meta["key"]]} for meta in _QKD_CONFIG_METADATA]
    return {"parameters": parameters, "default_distance_km": 5.0}


class SimulateNoiseRequest(BaseModel):
    iris_code: str
    bit_error_rate: float = 0.01
    rotation_bits: int = 0


@app.post("/demo/simulate-scan-noise")
def simulate_scan_noise(body: SimulateNoiseRequest) -> Dict[str, Any]:
    result = simulate_noisy_iris_scan(
        body.iris_code, bit_error_rate=body.bit_error_rate, rotation_bits=body.rotation_bits,
    )
    return {
        "noisy_iris_code": result.noisy_iris_code,
        "actual_bit_error_rate": result.actual_bit_error_rate,
        "changed_bits": result.actual_changed_bits,
        "applied_rotation_bits": result.applied_rotation_bits,
    }


@app.websocket("/ws/session/{session_id}")
async def protocol_session(websocket: WebSocket, session_id: str) -> None:
    await websocket.accept()
    emitter = SessionEmitter(session_id=session_id, websocket=websocket)

    try:
        while True:
            command = await websocket.receive_json()
            action = command.get("action")

            try:
                if action == "enroll":
                    await run_enrollment_flow(emitter, name=command["name"], email=command["email"])
                elif action == "authenticate":
                    await run_authentication_flow(
                        emitter,
                        email=command["email"],
                        iris_code=command.get("iris_code", ""),
                        configuration_overrides=command.get("configuration"),
                        distance_km=command.get("distance_km", 5.0),
                        eavesdropper_present=command.get("eavesdropper_present", False),
                        reveal_secrets=command.get("reveal_secrets", False),
                    )
                elif action == "send_message":
                    await run_send_message(
                        emitter, direction=command.get("direction", "T->S"),
                        text=command["text"], auto_reply=command.get("auto_reply", False),
                    )
                else:
                    await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE",
                                        actor="SYSTEM", message=f"Unknown action: {action}",
                                        payload={"failure_code": "UNKNOWN_ACTION"})
            except (AuthenticationFailure, ProtocolValidationError) as exc:
                await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                                    message=str(exc),
                                    payload={"failure_code": FAILURE_CODES["UNEXPECTED_ERROR"], "failure_message": str(exc)})
            except Exception as exc:
                traceback.print_exc()
                await emitter.emit(phase="SESSION", step="SESSION_FAILED", status="FAILURE", actor="SYSTEM",
                                    message=f"Unexpected server error: {exc}",
                                    payload={"failure_code": FAILURE_CODES["UNEXPECTED_ERROR"], "failure_message": str(exc)})

    except WebSocketDisconnect:
        _ACTIVE_AES_SESSIONS.pop(session_id, None)


def _run_uvicorn() -> None:
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

_server_thread = threading.Thread(target=_run_uvicorn, daemon=True)
_server_thread.start()
time.sleep(2)
print("FastAPI v4 is running locally on port 8000 inside this Colab runtime.")
print("Run the tunnel cell next (unchanged from before).")

FastAPI v4 is running locally on port 8000 inside this Colab runtime.
Run the tunnel cell next (unchanged from before).


In [80]:
import re
import subprocess
import time

#@title Download cloudflared binary (run once per Colab session)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

#@title Start the tunnel and print the public URL

tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
deadline = time.time() + 30

for line in tunnel_process.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    if time.time() > deadline:
        break

if public_url:
    print("\n" + "=" * 60)
    print(f"PUBLIC API URL   : {public_url}")
    print(f"WebSocket URL    : {public_url.replace('https://', 'wss://')}/ws/session/<session_id>")
    print(f"Health check     : {public_url}/health")
    print("=" * 60)
    print("\nPoint your React frontend's API base URL at PUBLIC API URL above.")
else:
    print("Could not detect the tunnel URL in the expected time. "
          "Scroll up in the output above — cloudflared usually prints "
          "a line like 'https://<random>.trycloudflare.com' within a few seconds.")

2026-08-20T09:52:08Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-20T09:52:08Z INF Requesting new quick Tunnel on trycloudflare.com...


KeyboardInterrupt: 

In [ ]:
#@title QBioAuth GUI — Build Professional Gradio 6 Interface

# This cell contains only presentation and event wiring.
# All cryptographic work continues to use the tested backend functions.

QBIOAUTH_GRADIO_CSS = r"""
.gradio-container {
    max-width: 1500px !important;
    margin: 0 auto !important;
}
.qbioauth-hero {
    text-align: center;
    padding: 1.2rem 0 0.5rem 0;
}
.qbioauth-hero h1 {
    font-size: 2.35rem;
    margin-bottom: 0.25rem;
}
.qbioauth-hero p {
    opacity: 0.78;
    font-size: 1.02rem;
}
.qbioauth-note {
    border-left: 4px solid var(--primary-500);
    padding: 0.7rem 1rem;
    background: var(--background-fill-secondary);
    border-radius: 8px;
}
.qbioauth-panel {
    border: 1px solid var(--border-color-primary);
    border-radius: 14px;
    padding: 0.5rem;
}
"""

QBIOAUTH_THEME = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="cyan",
    neutral_hue="slate",
)


def empty_status_dataframe() -> pd.DataFrame:
    return pd.DataFrame(columns=STATUS_COLUMNS)


def empty_metrics_dataframe() -> pd.DataFrame:
    return pd.DataFrame(columns=["Metric", "Value", "Unit"])


with gr.Blocks(
    title="QBioAuth Secure Communication",
    fill_width=True,
) as qbioauth_gradio_app:

    gr.HTML(
        """
        <div class="qbioauth-hero">
          <h1>QBioAuth</h1>
          <p>Biometric Authentication · ML-DSA · Biased-Basis QKD · AES-256-GCM</p>
        </div>
        """
    )

    gr.Markdown(
        """
        <div class="qbioauth-note">
        This dashboard is a presentation layer over the same validated protocol
        functions used by the notebook pipeline. Advanced values are validated,
        applied only for one execution, and restored immediately afterward.
        </div>
        """
    )

    with gr.Tabs():

        # ====================================================
        # TAB 1 — ENROLLMENT
        # ====================================================
        with gr.Tab("1 · User Enrollment"):
            gr.Markdown(
                """
                ### Create a biometric-protected user credential
                The Terminal and Server first perform mutual ML-DSA
                authentication. The system then creates a synthetic 2048-bit
                iris profile, fuzzy helper data, and a protected ML-DSA key.
                """
            )

            with gr.Row():
                with gr.Column(scale=1, elem_classes="qbioauth-panel"):
                    enrollment_name = gr.Textbox(
                        label="Full Name",
                        placeholder="Enter the user's full name",
                    )
                    enrollment_email = gr.Textbox(
                        label="Unique Email Identifier",
                        placeholder="user@example.com",
                        type="email",
                    )
                    enrollment_distance = gr.Number(
                        label="Terminal–Server Fibre Distance (km)",
                        value=20.0,
                        minimum=0.0,
                        step=1.0,
                    )
                    enrollment_button = gr.Button(
                        "Authenticate Nodes and Enrol User",
                        variant="primary",
                        size="lg",
                    )
                    enrollment_message = gr.Textbox(
                        label="Execution Result",
                        interactive=False,
                    )

                with gr.Column(scale=2, elem_classes="qbioauth-panel"):
                    enrollment_status = gr.Dataframe(
                        value=empty_status_dataframe,
                        headers=STATUS_COLUMNS,
                        datatype=["str", "str", "str"],
                        interactive=False,
                        label="Pipeline Status",
                        buttons=["copy", "fullscreen"],
                        wrap=True,
                    )
                    enrollment_metrics = gr.Dataframe(
                        value=empty_metrics_dataframe,
                        headers=["Metric", "Value", "Unit"],
                        datatype=["str", "str", "str"],
                        interactive=False,
                        label="Enrollment Metrics",
                        buttons=["copy", "fullscreen"],
                        wrap=True,
                    )

            with gr.Row():
                enrollment_user_id = gr.Textbox(
                    label="Internal User ID",
                    interactive=False,
                    buttons=["copy"],
                )
                enrollment_registry_path = gr.Textbox(
                    label="Synthetic Iris Registry",
                    interactive=False,
                    buttons=["copy"],
                )

            enrollment_iris_code = gr.Textbox(
                label="Copy-Ready 2048-Bit Synthetic Iris Code",
                lines=8,
                max_lines=12,
                interactive=False,
                buttons=["copy"],
            )

            with gr.Row():
                enrollment_iris_fingerprint = gr.Textbox(
                    label="Iris Fingerprint",
                    interactive=False,
                    buttons=["copy"],
                )
                enrollment_public_key_fingerprint = gr.Textbox(
                    label="ML-DSA Public-Key Fingerprint",
                    interactive=False,
                    buttons=["copy"],
                )

        # ====================================================
        # TAB 2 — SECURE SESSION
        # ====================================================
        with gr.Tab("2 · Authenticate and Communicate"):
            gr.Markdown(
                """
                ### Establish the complete secure session
                Reconstruct the user credential from a noisy iris scan, issue
                and verify QAuth, accumulate distance-aware QKD detections,
                reconcile with full recursive look-back Cascade, derive
                directional AES-256-GCM keys with HKDF, and exchange a message.
                """
            )

            with gr.Row():
                with gr.Column(scale=1, elem_classes="qbioauth-panel"):
                    secure_email = gr.Textbox(
                        label="Enrolled User Email",
                        placeholder="user@example.com",
                        type="email",
                    )
                    secure_iris_code = gr.Textbox(
                        label="Raw 2048-Bit Synthetic Iris Code",
                        lines=8,
                        max_lines=12,
                        placeholder="Paste the enrolled binary iris code",
                    )
                    secure_distance = gr.Number(
                        label="Terminal–Server Fibre Distance (km)",
                        value=20.0,
                        minimum=0.0,
                        step=1.0,
                    )
                    secure_eve = gr.Checkbox(
                        label="Simulate Full Intercept–Resend Eve",
                        value=False,
                    )
                    secure_message = gr.Textbox(
                        label="Message to Send Securely",
                        lines=4,
                        placeholder="Enter the Terminal message",
                    )
                    secure_button = gr.Button(
                        "Authenticate and Establish Secure Session",
                        variant="primary",
                        size="lg",
                    )
                    secure_execution_message = gr.Textbox(
                        label="Execution Result",
                        interactive=False,
                    )

                with gr.Column(scale=2, elem_classes="qbioauth-panel"):
                    secure_status = gr.Dataframe(
                        value=empty_status_dataframe,
                        headers=STATUS_COLUMNS,
                        datatype=["str", "str", "str"],
                        interactive=False,
                        label="Security Pipeline Status",
                        buttons=["copy", "fullscreen"],
                        wrap=True,
                    )
                    secure_metrics = gr.Dataframe(
                        value=empty_metrics_dataframe,
                        headers=["Metric", "Value", "Unit"],
                        datatype=["str", "str", "str"],
                        interactive=False,
                        label="QKD and Session Metrics",
                        buttons=["copy", "fullscreen"],
                        wrap=True,
                    )

            with gr.Row():
                qauth_token_output = gr.Textbox(
                    label="Accepted QAuth Token ID",
                    interactive=False,
                    buttons=["copy"],
                )
                qkd_session_output = gr.Textbox(
                    label="QKD Session ID",
                    interactive=False,
                    buttons=["copy"],
                )

            with gr.Accordion("Encrypted Application Exchange", open=True):
                terminal_ciphertext_output = gr.Textbox(
                    label="Terminal → Server Ciphertext",
                    lines=4,
                    interactive=False,
                    buttons=["copy"],
                )
                server_recovered_output = gr.Textbox(
                    label="Server Recovered Plaintext",
                    interactive=False,
                    buttons=["copy"],
                )
                server_ciphertext_output = gr.Textbox(
                    label="Server → Terminal Ciphertext",
                    lines=4,
                    interactive=False,
                    buttons=["copy"],
                )
                terminal_response_output = gr.Textbox(
                    label="Terminal Recovered Server Response",
                    interactive=False,
                    buttons=["copy"],
                )

        # ====================================================
        # TAB 3 — CONFIGURATION
        # ====================================================
        with gr.Tab("3 · Protocol Configuration"):
            gr.Markdown(
                """
                ### Validated defaults and controlled experimentation
                Leave advanced configuration disabled for the baseline demo.
                Enable it only when you deliberately want to change protocol
                parameters. Parameter names, units, and descriptions are locked;
                only the **Value** column is editable.
                """
            )

            use_advanced_configuration = gr.Checkbox(
                label="Use editable advanced configuration",
                value=False,
                info=(
                    "Unchecked uses the validated baseline defaults. "
                    "Checked applies the table values to one execution only."
                ),
            )

            configuration_table = gr.Dataframe(
                value=gradio_default_configuration_dataframe,
                headers=["Parameter", "Value", "Unit", "Description"],
                datatype=["str", "number", "str", "str"],
                interactive=True,
                row_count=len(GRADIO_CONFIGURATION_ROWS),
                row_limits=(
                    len(GRADIO_CONFIGURATION_ROWS),
                    len(GRADIO_CONFIGURATION_ROWS),
                ),
                column_count=4,
                column_limits=(4, 4),
                static_columns=[0, 2, 3],
                label="Protocol Parameters",
                buttons=["copy", "fullscreen"],
                wrap=True,
            )

            with gr.Row():
                reset_configuration_button = gr.Button(
                    "Reset Validated Defaults",
                    variant="secondary",
                )
                gr.Markdown(
                    "Changes are range-validated before any cryptographic "
                    "operation starts. Invalid values abort cleanly."
                )

        # ====================================================
        # TAB 4 — SYSTEM INFORMATION
        # ====================================================
        with gr.Tab("4 · System Information"):
            gr.Markdown(
                rf"""
                ## Validated Baseline

                - **Biometric simulation:** {FINAL_IRIS_NOISE_RATE:.2%} noise,
                  {FINAL_IRIS_ROTATION_BITS}-bit rotation
                - **Biased basis:** {FINAL_Z_BASIS_PROBABILITY:.0%} Z /
                  {1.0 - FINAL_Z_BASIS_PROBABILITY:.0%} X
                - **QBER abort threshold:** {FINAL_QBER_ABORT_THRESHOLD:.2%}
                - **Full Cascade passes:** {FINAL_RECONCILIATION_PASSES}
                - **Target detections:** {FINAL_TARGET_DETECTED_EVENTS:,}
                - **Transmission batch:** {FINAL_TRANSMISSION_BATCH_SIZE:,} pulses
                - **Maximum QKD attempts:** {FINAL_MAXIMUM_QKD_ATTEMPTS}
                - **AES session:** independent directional AES-256-GCM keys
                - **Classical transcript:** ML-DSA signed and hash chained

                ## Physical Model

                Fibre transmission is modelled as

                $$\eta_{{\mathrm{{fibre}}}}(d)=10^{{-0.02d}}.$$

                Distance lowers legitimate detections. Intrinsic optical error
                and random dark clicks determine non-adversarial QBER. Modelled
                latency combines optical acquisition, classical propagation,
                and measured protocol processing.
                """
            )

    # ========================================================
    # EVENT WIRING
    # ========================================================

    reset_configuration_button.click(
        fn=reset_gradio_configuration,
        inputs=[],
        outputs=[configuration_table],
        queue=False,
    )

    enrollment_button.click(
        fn=gradio_enrollment_event,
        inputs=[
            enrollment_name,
            enrollment_email,
            enrollment_distance,
            use_advanced_configuration,
            configuration_table,
        ],
        outputs=[
            enrollment_status,
            enrollment_metrics,
            enrollment_user_id,
            enrollment_iris_code,
            enrollment_iris_fingerprint,
            enrollment_public_key_fingerprint,
            enrollment_registry_path,
            enrollment_message,
        ],
        api_name="enrol_user",
    )

    secure_button.click(
        fn=gradio_secure_session_event,
        inputs=[
            secure_email,
            secure_iris_code,
            secure_message,
            secure_distance,
            secure_eve,
            use_advanced_configuration,
            configuration_table,
        ],
        outputs=[
            secure_status,
            secure_metrics,
            qauth_token_output,
            qkd_session_output,
            terminal_ciphertext_output,
            server_recovered_output,
            server_ciphertext_output,
            terminal_response_output,
            secure_execution_message,
        ],
        api_name="establish_secure_session",
    )

# qbioauth_gradio_app.queue(
#     default_concurrency_limit=1,
#     max_size=20,
# )

# qbioauth_gradio_app.launch(
#     share=True,
#     debug=True,
#     show_error=True,
#     theme=QBIOAUTH_THEME,
#     css=QBIOAUTH_GRADIO_CSS,
# )

## Comparison Analysis : 31-07-2026T11:21:54

In [81]:
#@title Noise-Inclusive Physical Model and Experiment Configuration

import contextlib
import csv
import gc
import hashlib
import json
import math
import random
import re
import time
import traceback

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


# ============================================================
# OUTPUT LOCATIONS — NEW DIRECTORY TO AVOID MIXING OLD RESULTS
# ============================================================

RESEARCH_OUTPUT_DIRECTORY = Path(
    "/content/qbioauth_final_comparison_ofat"
)
RESEARCH_GRAPH_DIRECTORY = RESEARCH_OUTPUT_DIRECTORY / "graphs"

RESEARCH_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
RESEARCH_GRAPH_DIRECTORY.mkdir(parents=True, exist_ok=True)

RAW_TRIAL_CSV_PATH = (
    RESEARCH_OUTPUT_DIRECTORY / "qbioauth_final_raw_trials.csv"
)
EXCEL_OUTPUT_PATH = (
    RESEARCH_OUTPUT_DIRECTORY / "qbioauth_final_comparison.xlsx"
)
EXPERIMENT_CONFIG_PATH = (
    RESEARCH_OUTPUT_DIRECTORY / "final_experiment_configuration.json"
)


# ============================================================
# STUDY CONTROL
# ============================================================

# Publication run:
#   100 Eve-present + 100 Eve-absent trials per case.
# For a quick pre-flight test, temporarily set this to 2.
TRIALS_PER_EVE_CONDITION = 100

RUN_OFAT_STUDY = True
RUN_INTERACTION_STUDY = False

# CSV is appended after every trial. Excel is checkpointed less often
# because rewriting the workbook after every run distorts experiment time.
EXCEL_CHECKPOINT_INTERVAL = 25
RESUME_FROM_EXISTING_RESULTS = True
VERBOSE_PER_TRIAL_LOGGING = True

EXPERIMENT_MASTER_SEED = 20260731


# ============================================================
# PHYSICAL-LAYER MODEL
# ============================================================

# Fibre attenuation:
#       eta_fibre(d) = 10^(-0.02*d)
#
# This corresponds to an idealised 0.2 dB/km fibre attenuation law.
DEFAULT_TRANSMISSION_BATCH_SIZE = 8192
DEFAULT_TARGET_DETECTED_EVENTS = 4096

MAXIMUM_TRANSMISSION_ROUNDS = 100000
MAXIMUM_TOTAL_TRANSMITTED_PULSES = 2_000_000_000

# Pulse source repetition rate. This determines physical acquisition time.
ASSUMED_PULSE_REPETITION_RATE_HZ = 1_000_000.0

# Propagation speed in fibre, approximately 2e8 m/s.
FIBRE_PROPAGATION_SPEED_M_PER_S = 3.0e8

# Mean physical efficiencies. These are configurable assumptions,
# not measured properties of a specific device.
SOURCE_EMISSION_EFFICIENCY = 1.00
COUPLING_EFFICIENCY = 0.80

DETECTOR_EFFICIENCY_MEAN = 0.20
DETECTOR_EFFICIENCY_STD = 0.01
DETECTOR_EFFICIENCY_MIN = 0.15
DETECTOR_EFFICIENCY_MAX = 0.25

# Intrinsic optical error models state-preparation error, polarisation/
# phase misalignment, basis imperfection and detector measurement error.
INTRINSIC_OPTICAL_ERROR_MEAN = 0.010
INTRINSIC_OPTICAL_ERROR_STD = 0.002
INTRINSIC_OPTICAL_ERROR_MIN = 0.003
INTRINSIC_OPTICAL_ERROR_MAX = 0.025

# One random detector click per approximately one million gates.
# At long distances, these random clicks become a meaningful fraction
# of all detections and therefore increase QBER.
DARK_COUNT_PROBABILITY_PER_GATE = 1.0e-6

# A dark-only click is random, so its error probability is one half.
DARK_COUNT_ERROR_PROBABILITY = 0.50

EVE_INTERCEPTION_MODE = "FULL_INTERCEPT_RESEND"


# ============================================================
# BASELINE QKD CONFIGURATION
# ============================================================

BASELINE_DISTANCE_KM = 50.0
BASELINE_Z_BASIS_PROBABILITY = 0.80
BASELINE_PARAMETER_ESTIMATION_FRACTION = 0.15
BASELINE_QBER_ABORT_THRESHOLD = 0.08
BASELINE_SECURITY_MARGIN_BITS = 128


# ============================================================
# OFAT VALUES
# ============================================================

DISTANCE_VALUES_KM = [0, 20, 50, 80, 100, 150, 200]
TRANSMISSION_BATCH_SIZE_VALUES = [1024, 2048, 4096, 8192, 16384]
Z_BASIS_PROBABILITY_VALUES = [0.50, 0.70, 0.80, 0.85, 15 / 16]
PARAMETER_ESTIMATION_FRACTION_VALUES = [0.05, 0.10, 0.15, 0.20, 0.03]
QBER_ABORT_THRESHOLD_VALUES = [0.03, 0.05, 0.08, 0.10, 0.12]
SECURITY_MARGIN_VALUES_BITS = [64, 128, 192, 256]
EVE_INTERCEPTION_FRACTION_VALUES = [0.00, 0.10, 0.25, 0.50, 0.75, 1.00]

# Every ordinary OFAT case gets 100 no-Eve and 100 full-Eve trials.
# The EVE_FRACTION group gets 100 trials at each explicit fraction.
FULL_CASCADE_LOOKBACK_REQUIRED = True


# ============================================================
# SELECTED INTERACTION STUDY
# ============================================================

INTERACTION_DISTANCE_VALUES_KM = [20, 80, 150]
INTERACTION_Z_BASIS_VALUES = [0.50, 0.80, 15 / 16]
INTERACTION_TARGET_DETECTIONS = [4096, 8192]


# ============================================================
# REPRODUCIBLE SEED DERIVATION
# ============================================================

def stable_seed(*parts: Any) -> int:
    material = "|".join(str(part) for part in parts).encode("utf-8")
    return int.from_bytes(
        hashlib.sha3_256(material).digest()[:16],
        byteorder="big",
    )


# ============================================================
# CASE BUILDERS
# ============================================================

def make_case(
    *,
    experiment_group: str,
    independent_variable: str,
    independent_value: Any,
    distance_km: float = BASELINE_DISTANCE_KM,
    transmission_batch_size: int = DEFAULT_TRANSMISSION_BATCH_SIZE,
    target_detected_photons: int = DEFAULT_TARGET_DETECTED_EVENTS,
    z_basis_probability: float = BASELINE_Z_BASIS_PROBABILITY,
    parameter_estimation_fraction: float =
        BASELINE_PARAMETER_ESTIMATION_FRACTION,
    qber_abort_threshold: float = BASELINE_QBER_ABORT_THRESHOLD,
    security_margin_bits: int = BASELINE_SECURITY_MARGIN_BITS,
    eve_interception_fraction: float = 0.0,
) -> Dict[str, Any]:

    safe_value = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        str(independent_value),
    ).strip("_")

    case_id = (
        f"{experiment_group.upper()}-"
        f"{independent_variable.upper()}-{safe_value}"
    )

    return {
        "case_id": case_id,
        "case_label": (
            f"{experiment_group}: "
            f"{independent_variable}={independent_value}"
        ),
        "experiment_group": experiment_group,
        "independent_variable": independent_variable,
        "independent_value": independent_value,
        "distance_km": float(distance_km),
        "transmission_batch_size": int(transmission_batch_size),
        "target_detected_photons": int(target_detected_photons),
        "z_basis_probability": float(z_basis_probability),
        "parameter_estimation_fraction":
            float(parameter_estimation_fraction),
        "qber_abort_threshold": float(qber_abort_threshold),
        "security_margin_bits": int(security_margin_bits),
        "eve_interception_fraction": float(eve_interception_fraction),
        "reconciliation_mode": "FULL_CASCADE_RECURSIVE_LOOKBACK",
    }


def build_ofat_cases() -> List[Dict[str, Any]]:
    cases: List[Dict[str, Any]] = []

    for distance in DISTANCE_VALUES_KM:
        cases.append(make_case(
            experiment_group="DISTANCE",
            independent_variable="distance_km",
            independent_value=distance,
            distance_km=distance,
        ))

    for batch_size in TRANSMISSION_BATCH_SIZE_VALUES:
        cases.append(make_case(
            experiment_group="BATCH_SIZE",
            independent_variable="transmission_batch_size",
            independent_value=batch_size,
            transmission_batch_size=batch_size,
        ))

    for z_probability in Z_BASIS_PROBABILITY_VALUES:
        cases.append(make_case(
            experiment_group="BASIS_BIAS",
            independent_variable="z_basis_probability",
            independent_value=round(z_probability, 6),
            z_basis_probability=z_probability,
        ))

    for fraction in PARAMETER_ESTIMATION_FRACTION_VALUES:
        cases.append(make_case(
            experiment_group="PARAMETER_ESTIMATION",
            independent_variable="parameter_estimation_fraction",
            independent_value=fraction,
            parameter_estimation_fraction=fraction,
        ))

    for threshold in QBER_ABORT_THRESHOLD_VALUES:
        cases.append(make_case(
            experiment_group="QBER_THRESHOLD",
            independent_variable="qber_abort_threshold",
            independent_value=threshold,
            qber_abort_threshold=threshold,
        ))

    for eve_fraction in EVE_INTERCEPTION_FRACTION_VALUES:
        cases.append(make_case(
            experiment_group="EVE_FRACTION",
            independent_variable="eve_interception_fraction",
            independent_value=eve_fraction,
            eve_interception_fraction=eve_fraction,
        ))

    for margin in SECURITY_MARGIN_VALUES_BITS:
        cases.append(make_case(
            experiment_group="SECURITY_MARGIN",
            independent_variable="security_margin_bits",
            independent_value=margin,
            security_margin_bits=margin,
        ))

    return cases


def build_interaction_cases() -> List[Dict[str, Any]]:
    cases: List[Dict[str, Any]] = []

    for distance in INTERACTION_DISTANCE_VALUES_KM:
        for z_probability in INTERACTION_Z_BASIS_VALUES:
            for target in INTERACTION_TARGET_DETECTIONS:
                label = (
                    f"d={distance}|z={z_probability:.6f}|target={target}"
                )
                cases.append(make_case(
                    experiment_group="INTERACTION",
                    independent_variable="distance_basis_target",
                    independent_value=label,
                    distance_km=distance,
                    z_basis_probability=z_probability,
                    target_detected_photons=target,
                ))

    return cases


EXPERIMENT_CASES: List[Dict[str, Any]] = []

if RUN_OFAT_STUDY:
    EXPERIMENT_CASES.extend(build_ofat_cases())

if RUN_INTERACTION_STUDY:
    EXPERIMENT_CASES.extend(build_interaction_cases())

if not EXPERIMENT_CASES:
    raise ValueError("No Phase 6 experiment cases are enabled.")


experiment_configuration = {
    "trials_per_eve_condition": TRIALS_PER_EVE_CONDITION,
    "total_trials_per_case": 2 * TRIALS_PER_EVE_CONDITION,
    "run_ofat_study": RUN_OFAT_STUDY,
    "run_interaction_study": RUN_INTERACTION_STUDY,
    "experiment_master_seed": EXPERIMENT_MASTER_SEED,
    "default_transmission_batch_size":
        DEFAULT_TRANSMISSION_BATCH_SIZE,
    "default_target_detected_events":
        DEFAULT_TARGET_DETECTED_EVENTS,
    "maximum_transmission_rounds":
        MAXIMUM_TRANSMISSION_ROUNDS,
    "maximum_total_transmitted_pulses":
        MAXIMUM_TOTAL_TRANSMITTED_PULSES,
    "pulse_repetition_rate_hz":
        ASSUMED_PULSE_REPETITION_RATE_HZ,
    "fibre_propagation_speed_m_per_s":
        FIBRE_PROPAGATION_SPEED_M_PER_S,
    "source_emission_efficiency":
        SOURCE_EMISSION_EFFICIENCY,
    "coupling_efficiency":
        COUPLING_EFFICIENCY,
    "detector_efficiency_mean":
        DETECTOR_EFFICIENCY_MEAN,
    "detector_efficiency_std":
        DETECTOR_EFFICIENCY_STD,
    "intrinsic_optical_error_mean":
        INTRINSIC_OPTICAL_ERROR_MEAN,
    "intrinsic_optical_error_std":
        INTRINSIC_OPTICAL_ERROR_STD,
    "dark_count_probability_per_gate":
        DARK_COUNT_PROBABILITY_PER_GATE,
    "dark_count_error_probability":
        DARK_COUNT_ERROR_PROBABILITY,
    "eve_interception_mode":
        EVE_INTERCEPTION_MODE,
    "distance_values_km":
        DISTANCE_VALUES_KM,
    "z_basis_probability_values":
        Z_BASIS_PROBABILITY_VALUES,
    "eve_interception_fraction_values":
        EVE_INTERCEPTION_FRACTION_VALUES,
    "reconciliation_mode":
        "FULL_CASCADE_RECURSIVE_LOOKBACK",
    "cascade_lookback_enabled":
        bool(globals().get("T12_CASCADE_LOOKBACK_ENABLED", False)),
}

EXPERIMENT_CONFIG_PATH.write_text(
    json.dumps(
        experiment_configuration,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

print("=" * 92)
print("PHASE 6A — NOISE-INCLUSIVE EXPERIMENT CONFIGURATION")
print("=" * 92)
print("Output directory             :", RESEARCH_OUTPUT_DIRECTORY)
print("Experiment cases             :", len(EXPERIMENT_CASES))
print("Trials per case              :", 2 * TRIALS_PER_EVE_CONDITION)
print("Pulse repetition rate        :", ASSUMED_PULSE_REPETITION_RATE_HZ)
print("Detector efficiency mean     :", DETECTOR_EFFICIENCY_MEAN)
print("Intrinsic optical error mean :", INTRINSIC_OPTICAL_ERROR_MEAN)
print("Dark-count probability/gate  :", DARK_COUNT_PROBABILITY_PER_GATE)
print("=" * 92)

if FULL_CASCADE_LOOKBACK_REQUIRED:
    if not globals().get("T12_CASCADE_LOOKBACK_ENABLED", False):
        raise RuntimeError(
            "The final comparison requires the full recursive look-back "
            "Cascade replacement cell to be executed first."
        )

PHASE 6A — NOISE-INCLUSIVE EXPERIMENT CONFIGURATION
Output directory             : /content/qbioauth_final_comparison_ofat
Experiment cases             : 37
Trials per case              : 200
Pulse repetition rate        : 1000000.0
Detector efficiency mean     : 0.2
Intrinsic optical error mean : 0.01
Dark-count probability/gate  : 1e-06


In [82]:
#@title Noise-Inclusive Photon Accumulation and Trial Engine

# ============================================================
# PHYSICAL ATTENUATION AND DETECTION MODEL
# ============================================================

def fibre_transmission_probability(distance_km: float) -> float:
    """
    Idealised fibre transmission probability:
        eta_fibre(d) = 10^(-0.02*d)
    """
    if distance_km < 0:
        raise ValueError("Distance cannot be negative.")
    return float(10 ** (-0.02 * distance_km))


def clipped_normal(
    *,
    rng: np.random.Generator,
    mean: float,
    std: float,
    minimum: float,
    maximum: float,
) -> float:
    value = float(rng.normal(mean, std))
    return float(np.clip(value, minimum, maximum))


def sample_physical_trial_parameters(
    rng: np.random.Generator,
) -> Dict[str, float]:
    """
    Sample small trial-to-trial hardware variation.
    """
    detector_efficiency = clipped_normal(
        rng=rng,
        mean=DETECTOR_EFFICIENCY_MEAN,
        std=DETECTOR_EFFICIENCY_STD,
        minimum=DETECTOR_EFFICIENCY_MIN,
        maximum=DETECTOR_EFFICIENCY_MAX,
    )

    intrinsic_error = clipped_normal(
        rng=rng,
        mean=INTRINSIC_OPTICAL_ERROR_MEAN,
        std=INTRINSIC_OPTICAL_ERROR_STD,
        minimum=INTRINSIC_OPTICAL_ERROR_MIN,
        maximum=INTRINSIC_OPTICAL_ERROR_MAX,
    )

    return {
        "detector_efficiency": detector_efficiency,
        "intrinsic_optical_error_rate": intrinsic_error,
    }


def accumulate_noise_inclusive_detections(
    *,
    transmission_batch_size: int,
    target_detected_events: int,
    distance_km: float,
    rng: np.random.Generator,
    maximum_rounds: int = MAXIMUM_TRANSMISSION_ROUNDS,
    maximum_total_pulses: int =
        MAXIMUM_TOTAL_TRANSMITTED_PULSES,
) -> Dict[str, Any]:
    """
    Accumulate signal and dark-count detections until the requested
    number of total detection events is available.

    For computational efficiency, the number of pulses required to
    obtain the target number of detections is sampled from an exact
    negative-binomial model instead of simulating every pulse.

    The model uses mutually exclusive detection categories:
        signal detection
        dark-only detection
        no detection
    """

    if transmission_batch_size <= 0:
        raise ValueError("Transmission batch size must be positive.")

    if target_detected_events <= 0:
        raise ValueError("Target detections must be positive.")

    physical = sample_physical_trial_parameters(rng)

    eta_fibre = fibre_transmission_probability(distance_km)

    signal_detection_probability = float(np.clip(
        SOURCE_EMISSION_EFFICIENCY
        * COUPLING_EFFICIENCY
        * physical["detector_efficiency"]
        * eta_fibre,
        0.0,
        1.0,
    ))

    # A dark-only event occurs when no signal click occurs in that gate.
    dark_only_probability = float(
        (1.0 - signal_detection_probability)
        * DARK_COUNT_PROBABILITY_PER_GATE
    )

    total_detection_probability = float(np.clip(
        signal_detection_probability + dark_only_probability,
        0.0,
        1.0,
    ))

    if total_detection_probability <= 0.0:
        return {
            "fibre_transmission_probability": eta_fibre,
            "signal_detection_probability":
                signal_detection_probability,
            "dark_only_probability":
                dark_only_probability,
            "total_detection_probability":
                total_detection_probability,
            "detector_efficiency":
                physical["detector_efficiency"],
            "intrinsic_optical_error_rate":
                physical["intrinsic_optical_error_rate"],
            "transmission_round_count": 0,
            "total_transmitted_pulses": 0,
            "total_signal_detections": 0,
            "total_dark_detections": 0,
            "total_detected_events": 0,
            "dark_detection_fraction": np.nan,
            "target_detected_events": target_detected_events,
            "effective_qkd_signal_count": 0,
            "physical_error_count": 0,
            "effective_physical_qber": np.nan,
            "accumulation_completed": False,
            "accumulation_termination_reason":
                "ZERO_DETECTION_PROBABILITY",
            "estimated_optical_acquisition_seconds": np.nan,
        }

    # Failures before target successes in Bernoulli trials.
    failures_before_target = int(
        rng.negative_binomial(
            target_detected_events,
            total_detection_probability,
        )
    )

    pulses_required = failures_before_target + target_detected_events
    transmission_round_count = int(math.ceil(
        pulses_required / transmission_batch_size
    ))
    total_transmitted_pulses = int(
        transmission_round_count * transmission_batch_size
    )

    completed = (
        transmission_round_count <= maximum_rounds
        and total_transmitted_pulses <= maximum_total_pulses
    )

    if not completed:
        termination_reason = (
            "ROUND_LIMIT_EXCEEDED"
            if transmission_round_count > maximum_rounds
            else "PHOTON_BUDGET_EXCEEDED"
        )

        expected_total = int(round(
            min(
                maximum_total_pulses,
                maximum_rounds * transmission_batch_size,
            )
            * total_detection_probability
        ))

        return {
            "fibre_transmission_probability": eta_fibre,
            "signal_detection_probability":
                signal_detection_probability,
            "dark_only_probability":
                dark_only_probability,
            "total_detection_probability":
                total_detection_probability,
            "detector_efficiency":
                physical["detector_efficiency"],
            "intrinsic_optical_error_rate":
                physical["intrinsic_optical_error_rate"],
            "transmission_round_count":
                min(transmission_round_count, maximum_rounds),
            "total_transmitted_pulses":
                min(
                    total_transmitted_pulses,
                    maximum_total_pulses,
                ),
            "total_signal_detections": np.nan,
            "total_dark_detections": np.nan,
            "total_detected_events": expected_total,
            "dark_detection_fraction": np.nan,
            "target_detected_events": target_detected_events,
            "effective_qkd_signal_count": expected_total,
            "physical_error_count": np.nan,
            "effective_physical_qber": np.nan,
            "accumulation_completed": False,
            "accumulation_termination_reason":
                termination_reason,
            "estimated_optical_acquisition_seconds": (
                min(
                    total_transmitted_pulses,
                    maximum_total_pulses,
                )
                / ASSUMED_PULSE_REPETITION_RATE_HZ
            ),
        }

    # Full batches may contain detections beyond the exact target.
    extra_pulses = total_transmitted_pulses - pulses_required
    extra_detections = int(
        rng.binomial(
            extra_pulses,
            total_detection_probability,
        )
    )
    total_detected_events = target_detected_events + extra_detections

    signal_fraction_given_detection = (
        signal_detection_probability / total_detection_probability
    )

    total_signal_detections = int(
        rng.binomial(
            total_detected_events,
            signal_fraction_given_detection,
        )
    )
    total_dark_detections = (
        total_detected_events - total_signal_detections
    )

    # Select exactly target_detected_events for the QKD protocol.
    qkd_signal_detections = int(
        rng.hypergeometric(
            ngood=total_signal_detections,
            nbad=total_dark_detections,
            nsample=target_detected_events,
        )
    )
    qkd_dark_detections = (
        target_detected_events - qkd_signal_detections
    )

    signal_errors = int(
        rng.binomial(
            qkd_signal_detections,
            physical["intrinsic_optical_error_rate"],
        )
    )
    dark_errors = int(
        rng.binomial(
            qkd_dark_detections,
            DARK_COUNT_ERROR_PROBABILITY,
        )
    )

    physical_error_count = signal_errors + dark_errors
    effective_physical_qber = (
        physical_error_count / target_detected_events
    )

    return {
        "fibre_transmission_probability": eta_fibre,
        "signal_detection_probability":
            signal_detection_probability,
        "dark_only_probability":
            dark_only_probability,
        "total_detection_probability":
            total_detection_probability,
        "detector_efficiency":
            physical["detector_efficiency"],
        "intrinsic_optical_error_rate":
            physical["intrinsic_optical_error_rate"],
        "transmission_round_count":
            transmission_round_count,
        "total_transmitted_pulses":
            total_transmitted_pulses,
        "total_signal_detections":
            total_signal_detections,
        "total_dark_detections":
            total_dark_detections,
        "total_detected_events":
            total_detected_events,
        "dark_detection_fraction": (
            total_dark_detections / total_detected_events
            if total_detected_events else np.nan
        ),
        "target_detected_events":
            target_detected_events,
        "effective_qkd_signal_count":
            target_detected_events,
        "qkd_signal_detections":
            qkd_signal_detections,
        "qkd_dark_detections":
            qkd_dark_detections,
        "signal_error_count":
            signal_errors,
        "dark_error_count":
            dark_errors,
        "physical_error_count":
            physical_error_count,
        "effective_physical_qber":
            effective_physical_qber,
        "accumulation_completed": True,
        "accumulation_termination_reason":
            "TARGET_REACHED",
        "estimated_optical_acquisition_seconds": (
            total_transmitted_pulses
            / ASSUMED_PULSE_REPETITION_RATE_HZ
        ),
    }


# ============================================================
# DETERMINISTIC QKD RANDOMNESS
# ============================================================

@contextlib.contextmanager
def deterministic_qkd_randomness(seed: int):
    original_token_bytes = secrets.token_bytes
    generator = np.random.default_rng(seed)

    def deterministic_token_bytes(length: int) -> bytes:
        if length < 0:
            raise ValueError("Byte length cannot be negative.")
        return generator.integers(
            0,
            256,
            size=length,
            dtype=np.uint8,
        ).tobytes()

    secrets.token_bytes = deterministic_token_bytes

    try:
        yield
    finally:
        secrets.token_bytes = original_token_bytes


# ============================================================
# BALANCED EVE SCHEDULE
# ============================================================

def build_attack_schedule(
    case: Dict[str, Any],
) -> List[float]:
    """
    Ordinary OFAT cases:
        100 trials at Eve fraction 0.0
        100 trials at Eve fraction 1.0
        shuffled reproducibly.

    EVE_FRACTION cases:
        100 trials at the explicit interception fraction.
    """

    if case["experiment_group"] == "EVE_FRACTION":
        schedule = [
            float(case["eve_interception_fraction"])
        ] * TRIALS_PER_EVE_CONDITION
    else:
        schedule = (
            [0.0] * TRIALS_PER_EVE_CONDITION
            + [1.0] * TRIALS_PER_EVE_CONDITION
        )

    rng = np.random.default_rng(
        stable_seed(
            EXPERIMENT_MASTER_SEED,
            case["case_id"],
            "ATTACK_SCHEDULE",
        )
    )
    rng.shuffle(schedule)
    return schedule



# ============================================================
# FRACTIONAL INTERCEPT-RESEND EVE
# ============================================================

def simulate_fractional_eve_transmission(
    *,
    signal_count: int,
    channel_error_rate: float,
    eve_interception_fraction: float,
) -> BiasedBasisQuantumTransmission:
    """
    Biased-basis prepare-and-measure simulation in which Eve
    intercepts only a configurable fraction of signals.

    fraction = 0.0:
        no Eve

    fraction = 1.0:
        full intercept-resend

    Intermediate values model partial interception.
    """

    if not 0.0 <= eve_interception_fraction <= 1.0:
        raise ValueError(
            "Eve interception fraction must lie in [0, 1]."
        )

    terminal_bits = generate_random_bits(signal_count)
    terminal_bases = generate_t12_biased_bases(signal_count)
    server_bases = generate_t12_biased_bases(signal_count)

    rng = np.random.default_rng(
        int.from_bytes(
            secrets.token_bytes(16),
            byteorder="big",
        )
    )

    server_random_direct = generate_random_bits(signal_count)

    # Direct path for signals not intercepted by Eve.
    direct_measurements = np.where(
        server_bases == terminal_bases,
        terminal_bits,
        server_random_direct,
    ).astype(np.uint8)

    if eve_interception_fraction > 0.0:
        intercept_mask = (
            rng.random(signal_count)
            < eve_interception_fraction
        )

        eve_bases = generate_t12_biased_bases(signal_count)
        eve_random = generate_random_bits(signal_count)

        eve_measurements = np.where(
            eve_bases == terminal_bases,
            terminal_bits,
            eve_random,
        ).astype(np.uint8)

        server_random_from_eve = generate_random_bits(signal_count)

        intercepted_measurements = np.where(
            server_bases == eve_bases,
            eve_measurements,
            server_random_from_eve,
        ).astype(np.uint8)

        server_measurements = np.where(
            intercept_mask,
            intercepted_measurements,
            direct_measurements,
        ).astype(np.uint8)
    else:
        server_measurements = direct_measurements

    channel_flip_mask = (
        rng.random(signal_count)
        < channel_error_rate
    )

    server_measurements ^= channel_flip_mask.astype(np.uint8)

    return BiasedBasisQuantumTransmission(
        signal_count=signal_count,
        terminal_bits=terminal_bits,
        terminal_bases=terminal_bases,
        server_bases=server_bases,
        server_measurements=server_measurements,
        channel_error_rate=channel_error_rate,
        eavesdropper_present=(
            eve_interception_fraction > 0.0
        ),
        terminal_z_count=int(
            np.sum(terminal_bases == 0)
        ),
        terminal_x_count=int(
            np.sum(terminal_bases == 1)
        ),
        server_z_count=int(
            np.sum(server_bases == 0)
        ),
        server_x_count=int(
            np.sum(server_bases == 1)
        ),
    )


@contextlib.contextmanager
def fractional_eve_patch(
    eve_interception_fraction: float,
):
    """
    Temporarily replace the Phase 5 transmission function so the
    existing protocol engine can evaluate fractional Eve attacks.
    """

    original = globals()[
        "simulate_t12_biased_quantum_transmission"
    ]

    def patched(
        *,
        signal_count: int = T12_DEFAULT_SIGNAL_COUNT,
        channel_error_rate: float = 0.01,
        eavesdropper_present: bool = False,
    ) -> BiasedBasisQuantumTransmission:
        resolved_fraction = (
            eve_interception_fraction
            if eavesdropper_present
            else 0.0
        )
        return simulate_fractional_eve_transmission(
            signal_count=signal_count,
            channel_error_rate=channel_error_rate,
            eve_interception_fraction=resolved_fraction,
        )

    globals()[
        "simulate_t12_biased_quantum_transmission"
    ] = patched

    try:
        yield
    finally:
        globals()[
            "simulate_t12_biased_quantum_transmission"
        ] = original


# ============================================================
# TEMPORARY QKD CONFIGURATION
# ============================================================

def apply_qkd_trial_configuration(
    *,
    z_basis_probability: float,
    parameter_estimation_fraction: float,
    qber_abort_threshold: float,
    security_margin_bits: int,
) -> Dict[str, Any]:

    names = [
        "T12_Z_BASIS_PROBABILITY",
        "T12_X_BASIS_PROBABILITY",
        "T12_EXPECTED_BASIS_MATCH_PROBABILITY",
        "T12_PARAMETER_ESTIMATION_FRACTION",
        "T12_QBER_ABORT_THRESHOLD",
        "T12_SECURITY_MARGIN_BITS",
    ]

    missing = [name for name in names if name not in globals()]
    if missing:
        raise RuntimeError(
            "Required QKD globals are missing: "
            + ", ".join(missing)
        )

    previous = {name: globals()[name] for name in names}

    x_probability = 1.0 - z_basis_probability

    globals()["T12_Z_BASIS_PROBABILITY"] = float(
        z_basis_probability
    )
    globals()["T12_X_BASIS_PROBABILITY"] = float(
        x_probability
    )
    globals()["T12_EXPECTED_BASIS_MATCH_PROBABILITY"] = float(
        z_basis_probability ** 2 + x_probability ** 2
    )
    globals()["T12_PARAMETER_ESTIMATION_FRACTION"] = float(
        parameter_estimation_fraction
    )
    globals()["T12_QBER_ABORT_THRESHOLD"] = float(
        qber_abort_threshold
    )
    globals()["T12_SECURITY_MARGIN_BITS"] = int(
        security_margin_bits
    )

    return previous


def restore_qkd_configuration(
    previous_values: Dict[str, Any],
) -> None:
    for name, value in previous_values.items():
        globals()[name] = value


# ============================================================
# ML-DSA MICROBENCHMARK FOR COST ATTRIBUTION
# ============================================================

def benchmark_mldsa_signed_message_pair(
    *,
    rounds: int = 20,
) -> Dict[str, float]:
    """
    Measure one terminal-sign + server-verify pair and one server-sign +
    terminal-verify pair. These values are used only for cost attribution;
    they are already included in measured QKD processing time and must not
    be added again.
    """

    payload = canonical_json({
        "benchmark": "MLDSA65_SIGN_VERIFY",
        "timestamp": utc_timestamp(),
    })

    terminal_pair_times = []
    server_pair_times = []

    for _ in range(rounds):
        start = time.perf_counter()
        signature = terminal.sign_bytes(payload)
        valid = server.verify_trusted_peer_signature(
            terminal.node_id,
            payload,
            signature,
        )
        terminal_pair_times.append(time.perf_counter() - start)
        if not valid:
            raise RuntimeError("Terminal ML-DSA benchmark failed.")

        start = time.perf_counter()
        signature = server.sign_bytes(payload)
        valid = terminal.verify_trusted_peer_signature(
            server.node_id,
            payload,
            signature,
        )
        server_pair_times.append(time.perf_counter() - start)
        if not valid:
            raise RuntimeError("Server ML-DSA benchmark failed.")

    return {
        "terminal_to_server_sign_verify_seconds": float(
            np.mean(terminal_pair_times)
        ),
        "server_to_terminal_sign_verify_seconds": float(
            np.mean(server_pair_times)
        ),
        "mean_signed_message_seconds": float(
            np.mean(
                terminal_pair_times + server_pair_times
            )
        ),
    }


MLDSA_BENCHMARK = benchmark_mldsa_signed_message_pair()

print("ML-DSA signed-message benchmark:", MLDSA_BENCHMARK)


# ============================================================
# FAILURE CLASSIFICATION
# ============================================================

def classify_trial_failure(reason: str) -> str:
    text = (reason or "").lower()

    if "photon_budget_exceeded" in text:
        return "PHOTON_BUDGET_EXCEEDED"
    if "round_limit_exceeded" in text:
        return "ROUND_LIMIT_EXCEEDED"
    if "zero_detection_probability" in text:
        return "ZERO_DETECTION_PROBABILITY"
    if "estimated qber" in text or "qber" in text:
        return "QBER_ABORT"
    if "sifted key is too short" in text:
        return "INSUFFICIENT_KEY_MATERIAL"
    if "reconciliation" in text:
        return "RECONCILIATION_FAILURE"
    if "privacy" in text:
        return "PRIVACY_AMPLIFICATION_FAILURE"
    if "hkdf" in text:
        return "HKDF_FAILURE"
    if "mutually authenticated" in text:
        return "AUTHENTICATION_FAILURE"
    return "OTHER_FAILURE"


def extract_qber_from_failure(reason: str) -> float:
    match = re.search(
        r"estimated QBER\s+([0-9]+(?:\.[0-9]+)?)%",
        reason or "",
        flags=re.IGNORECASE,
    )
    return float(match.group(1)) / 100.0 if match else np.nan


# ============================================================
# RAW RESULT SCHEMA
# ============================================================

TRIAL_RESULT_COLUMNS = [
    "run_id",
    "timestamp_utc",
    "experiment_group",
    "case_id",
    "case_label",
    "trial_number",
    "trial_seed",
    "independent_variable",
    "independent_value",

    "distance_km",
    "fibre_transmission_probability",
    "detector_efficiency",
    "signal_detection_probability",
    "dark_only_probability",
    "total_detection_probability",
    "transmission_batch_size",
    "transmission_round_count",
    "total_transmitted_pulses",
    "total_signal_detections",
    "total_dark_detections",
    "total_detected_events",
    "dark_detection_fraction",
    "target_detected_events",
    "effective_qkd_signal_count",
    "qkd_signal_detections",
    "qkd_dark_detections",
    "intrinsic_optical_error_rate",
    "signal_error_count",
    "dark_error_count",
    "physical_error_count",
    "effective_physical_qber",
    "accumulation_completed",
    "accumulation_termination_reason",

    "estimated_optical_acquisition_seconds",
    "estimated_classical_propagation_seconds",
    "measured_qkd_processing_seconds",
    "estimated_mldsa_processing_seconds",
    "estimated_non_mldsa_processing_seconds",
    "modelled_qkd_session_seconds",
    "total_trial_runtime_seconds",

    "z_basis_probability",
    "x_basis_probability",
    "expected_basis_match_rate",
    "parameter_estimation_fraction",
    "qber_abort_threshold",
    "security_margin_bits",
    "eve_present",
    "eve_interception_fraction",
    "reconciliation_mode",
    "cascade_lookback_enabled",

    "status",
    "failure_class",
    "failure_reason",
    "qkd_session_id",
    "observed_basis_match_rate",
    "sifted_key_length",
    "qber_sample_size",
    "estimated_qber",
    "post_estimation_key_length",
    "reconciliation_success",
    "reconciliation_corrected_errors",
    "reconciliation_leakage_bits",
    "privacy_amplified_length_bits",
    "hkdf_keys_match",
    "total_signed_classical_messages",

    "secure_bits_per_total_transmitted_pulse",
    "secure_bits_per_detected_event",
    "secure_bits_per_modelled_second",
]


def append_trial_to_csv(
    trial_record: Dict[str, Any],
) -> None:
    file_exists = RAW_TRIAL_CSV_PATH.exists()

    with RAW_TRIAL_CSV_PATH.open(
        "a",
        newline="",
        encoding="utf-8",
    ) as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=TRIAL_RESULT_COLUMNS,
            extrasaction="ignore",
        )
        if not file_exists:
            writer.writeheader()

        writer.writerow({
            column: trial_record.get(column)
            for column in TRIAL_RESULT_COLUMNS
        })


def load_completed_trial_keys() -> set:
    if (
        not RESUME_FROM_EXISTING_RESULTS
        or not RAW_TRIAL_CSV_PATH.exists()
    ):
        return set()

    existing = pd.read_csv(RAW_TRIAL_CSV_PATH)

    if existing.empty:
        return set()

    return set(zip(
        existing["case_id"].astype(str),
        existing["trial_number"].astype(int),
    ))


# ============================================================
# ONE COMPLETE NOISE-INCLUSIVE TRIAL
# ============================================================

def execute_one_research_trial(
    *,
    case: Dict[str, Any],
    trial_number: int,
    eve_interception_fraction: float,
    mutual_authentication_session_id: str,
) -> Dict[str, Any]:

    trial_start = time.perf_counter()

    trial_seed = stable_seed(
        EXPERIMENT_MASTER_SEED,
        case["case_id"],
        trial_number,
        f'{eve_interception_fraction:.6f}',
    )

    physical_rng = np.random.default_rng(
        stable_seed(trial_seed, "PHYSICAL_LAYER")
    )
    qkd_seed = stable_seed(trial_seed, "QKD")

    run_id = (
        f"{case['case_id']}-TRIAL-{trial_number:03d}"
        f"-EVEFRAC-{eve_interception_fraction:.3f}"
    )

    accumulation = accumulate_noise_inclusive_detections(
        transmission_batch_size=int(
            case["transmission_batch_size"]
        ),
        target_detected_events=int(
            case["target_detected_photons"]
        ),
        distance_km=float(case["distance_km"]),
        rng=physical_rng,
    )

    z_probability = float(case["z_basis_probability"])
    x_probability = 1.0 - z_probability

    record: Dict[str, Any] = {
        "run_id": run_id,
        "timestamp_utc":
            datetime.now(timezone.utc).isoformat(),
        "experiment_group": case["experiment_group"],
        "case_id": case["case_id"],
        "case_label": case["case_label"],
        "trial_number": trial_number,
        "trial_seed": trial_seed,
        "independent_variable":
            case["independent_variable"],
        "independent_value":
            case["independent_value"],

        "distance_km": float(case["distance_km"]),
        "transmission_batch_size": int(
            case["transmission_batch_size"]
        ),
        **accumulation,

        "estimated_classical_propagation_seconds": 0.0,
        "measured_qkd_processing_seconds": 0.0,
        "estimated_mldsa_processing_seconds": 0.0,
        "estimated_non_mldsa_processing_seconds": 0.0,
        "modelled_qkd_session_seconds": (
            accumulation[
                "estimated_optical_acquisition_seconds"
            ]
            if np.isfinite(
                accumulation[
                    "estimated_optical_acquisition_seconds"
                ]
            )
            else np.nan
        ),
        "total_trial_runtime_seconds": 0.0,

        "z_basis_probability": z_probability,
        "x_basis_probability": x_probability,
        "expected_basis_match_rate": (
            z_probability ** 2
            + x_probability ** 2
        ),
        "parameter_estimation_fraction": float(
            case["parameter_estimation_fraction"]
        ),
        "qber_abort_threshold": float(
            case["qber_abort_threshold"]
        ),
        "security_margin_bits": int(
            case["security_margin_bits"]
        ),
        "eve_present": bool(eve_interception_fraction > 0.0),
        "eve_interception_fraction": float(eve_interception_fraction),
        "reconciliation_mode": "FULL_CASCADE_RECURSIVE_LOOKBACK",
        "cascade_lookback_enabled": bool(
            globals().get("T12_CASCADE_LOOKBACK_ENABLED", False)
        ),

        "status": None,
        "failure_class": None,
        "failure_reason": None,
        "qkd_session_id": None,
        "observed_basis_match_rate": np.nan,
        "sifted_key_length": np.nan,
        "qber_sample_size": np.nan,
        "estimated_qber": np.nan,
        "post_estimation_key_length": np.nan,
        "reconciliation_success": False,
        "reconciliation_corrected_errors": np.nan,
        "reconciliation_leakage_bits": np.nan,
        "privacy_amplified_length_bits": 0,
        "hkdf_keys_match": False,
        "total_signed_classical_messages": np.nan,

        "secure_bits_per_total_transmitted_pulse": 0.0,
        "secure_bits_per_detected_event": 0.0,
        "secure_bits_per_modelled_second": 0.0,
    }

    previous_configuration = None

    try:
        if not accumulation["accumulation_completed"]:
            raise AuthenticationFailure(
                accumulation[
                    "accumulation_termination_reason"
                ]
            )

        previous_configuration = apply_qkd_trial_configuration(
            z_basis_probability=z_probability,
            parameter_estimation_fraction=float(
                case["parameter_estimation_fraction"]
            ),
            qber_abort_threshold=float(
                case["qber_abort_threshold"]
            ),
            security_margin_bits=int(
                case["security_margin_bits"]
            ),
        )

        qkd_start = time.perf_counter()

        with deterministic_qkd_randomness(qkd_seed):
            with fractional_eve_patch(
                eve_interception_fraction
            ):
                qkd_result, aes_session, privacy_result = (
                    execute_phase_5_qkd_session(
                    terminal_node=terminal,
                    server_node=server,
                    transport=network,
                    mutual_authentication_session_id=
                        mutual_authentication_session_id,
                    signal_count=int(
                        accumulation[
                            "effective_qkd_signal_count"
                        ]
                    ),
                    channel_error_rate=float(
                        accumulation[
                            "effective_physical_qber"
                        ]
                    ),
                        eavesdropper_present=bool(
                            eve_interception_fraction > 0.0
                        ),
                    )
                )

        measured_qkd_processing = (
            time.perf_counter() - qkd_start
        )

        signed_messages = int(
            qkd_result.total_signed_classical_messages
        )

        one_way_delay = (
            float(case["distance_km"])
            * 1000.0
            / FIBRE_PROPAGATION_SPEED_M_PER_S
        )

        classical_propagation = (
            signed_messages * one_way_delay
        )

        estimated_mldsa = (
            signed_messages
            * MLDSA_BENCHMARK[
                "mean_signed_message_seconds"
            ]
        )

        estimated_non_mldsa = max(
            0.0,
            measured_qkd_processing - estimated_mldsa,
        )

        modelled_qkd_session = (
            float(
                accumulation[
                    "estimated_optical_acquisition_seconds"
                ]
            )
            + classical_propagation
            + measured_qkd_processing
        )

        final_secret_bits = int(
            qkd_result.privacy_amplified_length_bits
        )
        total_pulses = int(
            accumulation["total_transmitted_pulses"]
        )
        effective_events = int(
            accumulation["effective_qkd_signal_count"]
        )

        record.update({
            "status": "SUCCESS",
            "measured_qkd_processing_seconds":
                measured_qkd_processing,
            "estimated_classical_propagation_seconds":
                classical_propagation,
            "estimated_mldsa_processing_seconds":
                estimated_mldsa,
            "estimated_non_mldsa_processing_seconds":
                estimated_non_mldsa,
            "modelled_qkd_session_seconds":
                modelled_qkd_session,
            "qkd_session_id":
                qkd_result.qkd_session_id,
            "observed_basis_match_rate":
                qkd_result.observed_basis_match_rate,
            "sifted_key_length":
                qkd_result.sifted_key_length,
            "qber_sample_size":
                qkd_result.qber_sample_size,
            "estimated_qber":
                qkd_result.estimated_qber,
            "post_estimation_key_length":
                qkd_result.post_estimation_key_length,
            "reconciliation_success":
                qkd_result.reconciliation_success,
            "reconciliation_corrected_errors":
                qkd_result.reconciliation_corrected_errors,
            "reconciliation_leakage_bits":
                qkd_result.reconciliation_leakage_bits,
            "privacy_amplified_length_bits":
                final_secret_bits,
            "hkdf_keys_match":
                qkd_result.hkdf_keys_match,
            "total_signed_classical_messages":
                signed_messages,
            "secure_bits_per_total_transmitted_pulse": (
                final_secret_bits / total_pulses
                if total_pulses else 0.0
            ),
            "secure_bits_per_detected_event": (
                final_secret_bits / effective_events
                if effective_events else 0.0
            ),
            "secure_bits_per_modelled_second": (
                final_secret_bits / modelled_qkd_session
                if modelled_qkd_session > 0
                else np.nan
            ),
        })

    except Exception as exc:
        reason = str(exc)
        measured_qkd_processing = (
            time.perf_counter() - trial_start
        )

        estimated_qber = extract_qber_from_failure(
            reason
        )

        # A QBER abort occurs after two basis messages and two
        # parameter-estimation messages. Use four signed one-way
        # messages as the minimum classical propagation estimate.
        minimum_signed_messages = (
            4 if classify_trial_failure(reason) == "QBER_ABORT"
            else 0
        )

        classical_propagation = (
            minimum_signed_messages
            * float(case["distance_km"])
            * 1000.0
            / FIBRE_PROPAGATION_SPEED_M_PER_S
        )

        modelled_qkd_session = (
            float(
                accumulation[
                    "estimated_optical_acquisition_seconds"
                ]
            )
            + classical_propagation
            + measured_qkd_processing
            if np.isfinite(
                accumulation[
                    "estimated_optical_acquisition_seconds"
                ]
            )
            else np.nan
        )

        record.update({
            "status": "ABORTED",
            "failure_class":
                classify_trial_failure(reason),
            "failure_reason": reason,
            "estimated_qber": estimated_qber,
            "measured_qkd_processing_seconds":
                measured_qkd_processing,
            "estimated_classical_propagation_seconds":
                classical_propagation,
            "modelled_qkd_session_seconds":
                modelled_qkd_session,
        })

    finally:
        if previous_configuration is not None:
            restore_qkd_configuration(
                previous_configuration
            )

        record["total_trial_runtime_seconds"] = (
            time.perf_counter() - trial_start
        )
        gc.collect()

    return record


print("Noise-inclusive photon accumulation and trial engine ready.")

ML-DSA signed-message benchmark: {'terminal_to_server_sign_verify_seconds': 0.001427737950007213, 'server_to_terminal_sign_verify_seconds': 0.0014786269499978744, 'mean_signed_message_seconds': 0.0014531824500025436}
Noise-inclusive photon accumulation and trial engine ready.


In [83]:
#@title Execute Balanced Noise-Inclusive Research Trials

# ============================================================
# PRE-FLIGHT VALIDATION
# ============================================================

required_runtime_names = [
    "terminal",
    "server",
    "network",
    "MutualAuthenticationProtocol",
    "execute_phase_5_qkd_session",
    "AuthenticationFailure",
]

missing_runtime_names = [
    name
    for name in required_runtime_names
    if name not in globals()
]

if missing_runtime_names:
    raise RuntimeError(
        "Run all earlier QBioAuth cells before Phase 6. Missing: "
        + ", ".join(missing_runtime_names)
    )


# ============================================================
# FRESH MUTUAL AUTHENTICATION FOR COMPONENT BENCHMARK
# ============================================================

print("=" * 92)
print("STARTING PHASE 6 MUTUAL AUTHENTICATION")
print("=" * 92)

research_auth_protocol = MutualAuthenticationProtocol(
    terminal=terminal,
    server=server,
    transport=network,
)

research_auth_result = research_auth_protocol.execute()

if not (
    research_auth_result.success
    and research_auth_result.terminal_authenticated_server
    and research_auth_result.server_authenticated_terminal
):
    raise AuthenticationFailure(
        "Phase 6 mutual authentication failed."
    )

RESEARCH_MUTUAL_AUTHENTICATION_SESSION_ID = (
    research_auth_result.session_id
)

print(
    "Authenticated session:",
    RESEARCH_MUTUAL_AUTHENTICATION_SESSION_ID,
)
print("Mutual authentication: PASS")


# ============================================================
# LIGHTWEIGHT CHECKPOINT
# ============================================================

def write_lightweight_excel_checkpoint() -> None:
    if not RAW_TRIAL_CSV_PATH.exists():
        return

    raw = pd.read_csv(RAW_TRIAL_CSV_PATH)

    with pd.ExcelWriter(
        EXCEL_OUTPUT_PATH,
        engine="xlsxwriter",
    ) as writer:
        raw.to_excel(
            writer,
            sheet_name="Raw Trials",
            index=False,
        )
        pd.DataFrame(EXPERIMENT_CASES).to_excel(
            writer,
            sheet_name="Experiment Cases",
            index=False,
        )
        pd.DataFrame([
            {
                "Parameter": key,
                "Value": json.dumps(value)
                if isinstance(value, (list, dict))
                else value,
            }
            for key, value
            in experiment_configuration.items()
        ]).to_excel(
            writer,
            sheet_name="Configuration",
            index=False,
        )


# ============================================================
# EXECUTION LOOP WITH RESUME
# ============================================================

completed_trial_keys = load_completed_trial_keys()
newly_completed_trials = 0

total_requested_trials = sum(
    (
        TRIALS_PER_EVE_CONDITION
        if case["experiment_group"] == "EVE_FRACTION"
        else 2 * TRIALS_PER_EVE_CONDITION
    )
    for case in EXPERIMENT_CASES
)

experiment_start = time.perf_counter()

print("\n" + "=" * 92)
print("NOISE-INCLUSIVE RESEARCH EXPERIMENT STARTED")
print("=" * 92)
print("Cases                    :", len(EXPERIMENT_CASES))
print("Trials per case          :", 2 * TRIALS_PER_EVE_CONDITION)
print("Requested trials         :", total_requested_trials)
print("Previously logged trials :", len(completed_trial_keys))
print("=" * 92)


for case_index, case in enumerate(
    EXPERIMENT_CASES,
    start=1,
):
    eve_schedule = build_attack_schedule(case)

    print("\n" + "-" * 92)
    print(
        f"CASE {case_index}/{len(EXPERIMENT_CASES)} | "
        f"{case['case_id']} | "
        f"{case['case_label']}"
    )
    print("-" * 92)

    for trial_number, eve_interception_fraction in enumerate(
        eve_schedule,
        start=1,
    ):
        resume_key = (
            case["case_id"],
            trial_number,
        )

        if resume_key in completed_trial_keys:
            if VERBOSE_PER_TRIAL_LOGGING:
                print(
                    f"[SKIP] {case['case_id']} "
                    f"trial {trial_number:03d}"
                )
            continue

        trial_record = execute_one_research_trial(
            case=case,
            trial_number=trial_number,
            eve_interception_fraction=eve_interception_fraction,
            mutual_authentication_session_id=
                RESEARCH_MUTUAL_AUTHENTICATION_SESSION_ID,
        )

        append_trial_to_csv(trial_record)

        completed_trial_keys.add(resume_key)
        newly_completed_trials += 1

        if VERBOSE_PER_TRIAL_LOGGING:
            qber = trial_record.get("estimated_qber")
            qber_text = (
                f"{float(qber):.2%}"
                if qber is not None
                and pd.notna(qber)
                else "N/A"
            )

            modelled_time = trial_record.get(
                "modelled_qkd_session_seconds"
            )
            modelled_time_text = (
                f"{float(modelled_time):.3f}s"
                if modelled_time is not None
                and pd.notna(modelled_time)
                else "N/A"
            )

            print(
                f"[{trial_record['status']}] "
                f"trial={trial_number:03d} "
                f"eve_fraction={eve_interception_fraction:.2f} "
                f"d={trial_record['distance_km']:.0f}km "
                f"QBER={qber_text} "
                f"dark={trial_record.get('dark_detection_fraction', np.nan):.3%} "
                f"pulses={int(trial_record['total_transmitted_pulses']):,} "
                f"modelled_time={modelled_time_text} "
                f"secret={int(trial_record.get('privacy_amplified_length_bits', 0) or 0)}"
            )

        if (
            newly_completed_trials
            % EXCEL_CHECKPOINT_INTERVAL
            == 0
        ):
            write_lightweight_excel_checkpoint()
            print(
                f"Checkpoint saved after "
                f"{newly_completed_trials} new trials."
            )

write_lightweight_excel_checkpoint()

elapsed = time.perf_counter() - experiment_start

print("\n" + "=" * 92)
print("NOISE-INCLUSIVE RESEARCH EXPERIMENT COMPLETE")
print("=" * 92)
print("New trials completed :", newly_completed_trials)
print("Elapsed notebook time:", f"{elapsed:.2f} seconds")
print("Raw CSV              :", RAW_TRIAL_CSV_PATH)
print("Excel checkpoint     :", EXCEL_OUTPUT_PATH)
print("=" * 92)

Streaming output truncated to the last 5000 lines.
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 118
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 119
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 120
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 121
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 122
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 123
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 124
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 125
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 126
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 127
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 128
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 129
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 130
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 131
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 132
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_SIZE-4096 trial 133
[SKIP] BATCH_SIZE-TRANSMISSION_BATCH_

In [ ]:
#@title Statistical Analysis, Noise Graphs and Final Excel Workbook

if not RAW_TRIAL_CSV_PATH.exists():
    raise FileNotFoundError(
        "No raw Phase 6 CSV exists. Run the execution cell first."
    )

raw_results = pd.read_csv(RAW_TRIAL_CSV_PATH)

if raw_results.empty:
    raise ValueError("The raw Phase 6 CSV is empty.")


# ============================================================
# TYPE NORMALISATION
# ============================================================

for column in [
    "eve_present",
    "cascade_lookback_enabled",
    "accumulation_completed",
    "reconciliation_success",
    "hkdf_keys_match",
]:
    if column in raw_results.columns:
        raw_results[column] = (
            raw_results[column]
            .astype(str)
            .str.lower()
            .map({
                "true": True,
                "false": False,
            })
        )

raw_results["successful_trial"] = (
    raw_results["status"].eq("SUCCESS").astype(int)
)

raw_results["eve_rejected"] = (
    raw_results["eve_present"].eq(True)
    & raw_results["status"].eq("ABORTED")
).astype(int)


# ============================================================
# STATISTICAL UTILITIES
# ============================================================

def numeric_summary(
    values: pd.Series,
) -> Dict[str, float]:
    clean = pd.to_numeric(
        values,
        errors="coerce",
    ).dropna()

    count = len(clean)

    if count == 0:
        return {
            "n": 0,
            "mean": np.nan,
            "std": np.nan,
            "median": np.nan,
            "q1": np.nan,
            "q3": np.nan,
            "p05": np.nan,
            "p95": np.nan,
            "ci95_low": np.nan,
            "ci95_high": np.nan,
        }

    mean = float(clean.mean())
    std = float(clean.std(ddof=1)) if count > 1 else 0.0
    margin = (
        1.96 * std / math.sqrt(count)
        if count > 1
        else 0.0
    )

    return {
        "n": count,
        "mean": mean,
        "std": std,
        "median": float(clean.median()),
        "q1": float(clean.quantile(0.25)),
        "q3": float(clean.quantile(0.75)),
        "p05": float(clean.quantile(0.05)),
        "p95": float(clean.quantile(0.95)),
        "ci95_low": mean - margin,
        "ci95_high": mean + margin,
    }


def wilson_interval(
    successes: int,
    trials: int,
    z: float = 1.96,
) -> Tuple[float, float]:
    if trials == 0:
        return np.nan, np.nan

    probability = successes / trials
    denominator = 1.0 + z ** 2 / trials
    centre = (
        probability
        + z ** 2 / (2 * trials)
    ) / denominator
    half_width = (
        z
        * math.sqrt(
            probability * (1.0 - probability) / trials
            + z ** 2 / (4 * trials ** 2)
        )
        / denominator
    )

    return (
        centre - half_width,
        centre + half_width,
    )


def build_case_summary(
    group: pd.DataFrame,
) -> pd.Series:
    successful = group[
        group["status"] == "SUCCESS"
    ]
    eve = group[group["eve_present"] == True]
    no_eve = group[group["eve_present"] == False]

    success_count = int(
        group["successful_trial"].sum()
    )
    success_low, success_high = wilson_interval(
        success_count,
        len(group),
    )

    eve_rejection_count = int(
        (eve["status"] == "ABORTED").sum()
    )
    eve_low, eve_high = wilson_interval(
        eve_rejection_count,
        len(eve),
    )

    no_eve_success_count = int(
        (no_eve["status"] == "SUCCESS").sum()
    )
    no_eve_low, no_eve_high = wilson_interval(
        no_eve_success_count,
        len(no_eve),
    )

    fields = {
        "experiment_group":
            group["experiment_group"].iloc[0],
        "case_label":
            group["case_label"].iloc[0],
        "independent_variable":
            group["independent_variable"].iloc[0],
        "independent_value":
            group["independent_value"].iloc[0],
        "distance_km":
            group["distance_km"].iloc[0],
        "trial_count":
            len(group),
        "success_rate":
            success_count / len(group),
        "success_rate_ci95_low":
            success_low,
        "success_rate_ci95_high":
            success_high,
        "no_eve_success_rate": (
            no_eve_success_count / len(no_eve)
            if len(no_eve)
            else np.nan
        ),
        "no_eve_success_ci95_low":
            no_eve_low,
        "no_eve_success_ci95_high":
            no_eve_high,
        "eve_rejection_rate": (
            eve_rejection_count / len(eve)
            if len(eve)
            else np.nan
        ),
        "eve_rejection_ci95_low":
            eve_low,
        "eve_rejection_ci95_high":
            eve_high,
    }

    summary_columns = [
        "fibre_transmission_probability",
        "signal_detection_probability",
        "dark_only_probability",
        "total_detection_probability",
        "detector_efficiency",
        "total_transmitted_pulses",
        "transmission_round_count",
        "dark_detection_fraction",
        "effective_physical_qber",
        "estimated_qber",
        "observed_basis_match_rate",
        "sifted_key_length",
        "estimated_optical_acquisition_seconds",
        "estimated_classical_propagation_seconds",
        "measured_qkd_processing_seconds",
        "estimated_mldsa_processing_seconds",
        "modelled_qkd_session_seconds",
        "privacy_amplified_length_bits",
        "secure_bits_per_total_transmitted_pulse",
        "secure_bits_per_modelled_second",
    ]

    for column in summary_columns:
        source = (
            successful[column]
            if column in successful
            and not successful.empty
            else group[column]
        )
        stats = numeric_summary(source)
        for stat_name, value in stats.items():
            fields[f"{column}_{stat_name}"] = value

    no_eve_qber = numeric_summary(
        no_eve["estimated_qber"]
    )
    eve_qber = numeric_summary(
        eve["estimated_qber"]
    )

    fields["no_eve_estimated_qber_mean"] = (
        no_eve_qber["mean"]
    )
    fields["no_eve_estimated_qber_std"] = (
        no_eve_qber["std"]
    )
    fields["eve_estimated_qber_mean"] = (
        eve_qber["mean"]
    )
    fields["eve_estimated_qber_std"] = (
        eve_qber["std"]
    )

    return pd.Series(fields)


case_summary_rows = []

for case_id, case_group in raw_results.groupby(
    "case_id",
    sort=False,
):
    case_row = build_case_summary(
        case_group.copy()
    ).to_dict()
    case_row["case_id"] = case_id
    case_summary_rows.append(case_row)

case_summary = pd.DataFrame(
    case_summary_rows
)

if not case_summary.empty:
    preferred_columns = [
        "case_id",
        *[
            column
            for column in case_summary.columns
            if column != "case_id"
        ],
    ]
    case_summary = case_summary[
        preferred_columns
    ]

failure_summary = (
    raw_results[
        raw_results["status"] == "ABORTED"
    ]
    .groupby([
        "experiment_group",
        "case_id",
        "failure_class",
        "eve_present",
        "cascade_lookback_enabled",
    ])
    .size()
    .reset_index(name="failure_count")
)

distance_raw = raw_results[
    raw_results["experiment_group"] == "DISTANCE"
].copy()

distance_summary = case_summary[
    case_summary["experiment_group"] == "DISTANCE"
].sort_values("distance_km")


basis_raw = raw_results[
    raw_results["experiment_group"] == "BASIS_BIAS"
].copy()

basis_summary = case_summary[
    case_summary["experiment_group"] == "BASIS_BIAS"
].sort_values("independent_value")


eve_fraction_summary = (
    raw_results[
        raw_results["experiment_group"] == "EVE_FRACTION"
    ]
    .groupby("eve_interception_fraction")
    .agg(
        trials=("status", "size"),
        success_rate=("successful_trial", "mean"),
        rejection_rate=("status", lambda s: float((s == "ABORTED").mean())),
        mean_estimated_qber=("estimated_qber", "mean"),
        std_estimated_qber=("estimated_qber", "std"),
        mean_final_secret_bits=(
            "privacy_amplified_length_bits",
            "mean",
        ),
        mean_reconciliation_leakage_bits=(
            "reconciliation_leakage_bits",
            "mean",
        ),
        mean_signed_messages=(
            "total_signed_classical_messages",
            "mean",
        ),
        mean_modelled_seconds=(
            "modelled_qkd_session_seconds",
            "mean",
        ),
    )
    .reset_index()
    .sort_values("eve_interception_fraction")
)

reconciliation_summary = (
    raw_results
    .groupby([
        "experiment_group",
        "case_id",
    ])
    .agg(
        trials=("status", "size"),
        reconciliation_success_rate=(
            "reconciliation_success",
            "mean",
        ),
        reconciliation_failure_count=(
            "failure_class",
            lambda s: int(
                (s == "RECONCILIATION_FAILURE").sum()
            ),
        ),
        mean_corrected_errors=(
            "reconciliation_corrected_errors",
            "mean",
        ),
        mean_reconciliation_leakage_bits=(
            "reconciliation_leakage_bits",
            "mean",
        ),
        mean_signed_messages=(
            "total_signed_classical_messages",
            "mean",
        ),
    )
    .reset_index()
)


# ============================================================
# GRAPH HELPERS
# ============================================================

# Slightly larger fonts across all figures for readability in
# slides/print.
GRAPH_FONT_SCALE_RCPARAMS = {
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
}
plt.rcParams.update(GRAPH_FONT_SCALE_RCPARAMS)

generated_graphs: List[Path] = []


def save_graph(
    filename: str,
) -> None:
    """
    Save the current figure as both PNG and EPS.

    PNG remains the format embedded into the Excel workbook.
    EPS is the additional publication-quality vector output.
    """

    stem = Path(filename).stem
    png_path = RESEARCH_GRAPH_DIRECTORY / f"{stem}.png"
    eps_path = RESEARCH_GRAPH_DIRECTORY / f"{stem}.eps"

    plt.tight_layout()
    plt.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )
    plt.savefig(
        eps_path,
        bbox_inches="tight",
    )
    plt.show()
    plt.close()
    generated_graphs.append(png_path)


required_distance_summary_columns = {
    "distance_km",
    "fibre_transmission_probability_mean",
    "total_detection_probability_mean",
    "total_transmitted_pulses_median",
    "dark_detection_fraction_mean",
    "effective_physical_qber_mean",
    "estimated_optical_acquisition_seconds_median",
    "measured_qkd_processing_seconds_median",
    "modelled_qkd_session_seconds_median",
    "no_eve_success_rate",
    "eve_rejection_rate",
    "secure_bits_per_total_transmitted_pulse_mean",
    "privacy_amplified_length_bits_median",
}

missing_distance_summary_columns = (
    required_distance_summary_columns
    - set(distance_summary.columns)
)

if missing_distance_summary_columns:
    raise RuntimeError(
        "Distance summary is missing required graph columns: "
        f"{sorted(missing_distance_summary_columns)}"
    )


# 1. Fibre survival and total detection probability.
plt.figure(figsize=(8, 5))
plt.plot(
    distance_summary["distance_km"],
    distance_summary[
        "fibre_transmission_probability_mean"
    ],
    marker="o",
    label="Fibre survival",
)
plt.plot(
    distance_summary["distance_km"],
    distance_summary[
        "total_detection_probability_mean"
    ],
    marker="s",
    label="Total click probability",
)
plt.yscale("log")
plt.xlabel("Distance (km)")
plt.ylabel("Probability per transmitted pulse")
plt.title("Distance-Dependent Optical Detection Probability")
plt.grid(True, alpha=0.3)
plt.legend()
save_graph("01_detection_probability_vs_distance.png")


# 2. Total pulse cost.
plt.figure(figsize=(8, 5))
plt.plot(
    distance_summary["distance_km"],
    distance_summary[
        "total_transmitted_pulses_median"
    ],
    marker="o",
)
plt.yscale("log")
plt.xlabel("Distance (km)")
plt.ylabel("Median transmitted pulses")
plt.title("Pulse Cost Required to Accumulate the QKD Input")
plt.grid(True, alpha=0.3)
save_graph("02_pulse_cost_vs_distance.png")


# 3. Dark-count fraction.
plt.figure(figsize=(8, 5))
plt.plot(
    distance_summary["distance_km"],
    100.0 * distance_summary[
        "dark_detection_fraction_mean"
    ],
    marker="o",
)
plt.xlabel("Distance (km)")
plt.ylabel("Dark detections among clicks (%)")
plt.title("Dark-Count Contribution Increases with Distance")
plt.grid(True, alpha=0.3)
save_graph("03_dark_fraction_vs_distance.png")


# 4. No-Eve and Eve QBER.
qber_plot = (
    distance_raw
    .groupby(["distance_km", "eve_present"])[
        "estimated_qber"
    ]
    .mean()
    .unstack()
    .sort_index()
)

plt.figure(figsize=(8, 5))

if False in qber_plot.columns:
    plt.plot(
        qber_plot.index,
        100.0 * qber_plot[False],
        marker="o",
        label="No Eve",
    )

if True in qber_plot.columns:
    plt.plot(
        qber_plot.index,
        100.0 * qber_plot[True],
        marker="s",
        label="Eve",
    )

plt.axhline(
    100.0 * BASELINE_QBER_ABORT_THRESHOLD,
    linestyle="--",
    label="Abort threshold",
)
plt.xlabel("Distance (km)")
plt.ylabel("Estimated QBER (%)")
plt.title("Noise-Inclusive QBER versus Distance")
plt.grid(True, alpha=0.3)
plt.legend()
save_graph("04_qber_vs_distance_eve_comparison.png")


# 5. Physical QBER before Eve.
plt.figure(figsize=(8, 5))
plt.plot(
    distance_summary["distance_km"],
    100.0 * distance_summary[
        "effective_physical_qber_mean"
    ],
    marker="o",
)
plt.xlabel("Distance (km)")
plt.ylabel("Physical-layer QBER (%)")
plt.title("Intrinsic Error and Dark-Count QBER versus Distance")
plt.grid(True, alpha=0.3)
save_graph("05_physical_qber_vs_distance.png")


# 6. Modelled latency versus measured computation.
plt.figure(figsize=(8, 5))
plt.plot(
    distance_summary["distance_km"],
    distance_summary[
        "modelled_qkd_session_seconds_median"
    ],
    marker="o",
    label="Modelled total QKD session",
)
plt.plot(
    distance_summary["distance_km"],
    distance_summary[
        "measured_qkd_processing_seconds_median"
    ],
    marker="s",
    label="Measured notebook processing",
)
plt.yscale("log")
plt.xlabel("Distance (km)")
plt.ylabel("Time (seconds)")
plt.title("Physical Session Latency versus Software Runtime")
plt.grid(True, alpha=0.3)
plt.legend()
save_graph("06_latency_vs_distance.png")


# 7. Optical acquisition time.
plt.figure(figsize=(8, 5))
plt.plot(
    distance_summary["distance_km"],
    distance_summary[
        "estimated_optical_acquisition_seconds_median"
    ],
    marker="o",
)
plt.yscale("log")
plt.xlabel("Distance (km)")
plt.ylabel("Estimated acquisition time (seconds)")
plt.title("Photon Acquisition Time versus Distance")
plt.grid(True, alpha=0.3)
save_graph("07_optical_time_vs_distance.png")


# 8. Success and Eve rejection.
plt.figure(figsize=(8, 5))
plt.plot(
    distance_summary["distance_km"],
    100.0 * distance_summary[
        "no_eve_success_rate"
    ],
    marker="o",
    label="No-Eve success",
)
plt.plot(
    distance_summary["distance_km"],
    100.0 * distance_summary[
        "eve_rejection_rate"
    ],
    marker="s",
    label="Eve rejection",
)
plt.xlabel("Distance (km)")
plt.ylabel("Rate (%)")
plt.ylim(-2, 102)
plt.title("Session Availability and Eve Rejection")
plt.grid(True, alpha=0.3)
plt.legend()
save_graph("08_success_and_eve_rejection_vs_distance.png")


# 9. Secret efficiency.
plt.figure(figsize=(8, 5))
plt.plot(
    distance_summary["distance_km"],
    distance_summary[
        "secure_bits_per_total_transmitted_pulse_mean"
    ],
    marker="o",
)
plt.yscale("log")
plt.xlabel("Distance (km)")
plt.ylabel("Final secret bits / transmitted pulse")
plt.title("Secure-Key Efficiency versus Distance")
plt.grid(True, alpha=0.3)
save_graph("09_secure_bits_per_pulse_vs_distance.png")


# 10. Final secret length.
plt.figure(figsize=(8, 5))
plt.plot(
    distance_summary["distance_km"],
    distance_summary[
        "privacy_amplified_length_bits_median"
    ],
    marker="o",
)
plt.xlabel("Distance (km)")
plt.ylabel("Median privacy-amplified bits")
plt.title("Final Secret Length after Noise and Leakage")
plt.grid(True, alpha=0.3)
save_graph("10_final_secret_vs_distance.png")



# 11. Partial Eve interception fraction versus QBER.
if not eve_fraction_summary.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(
        100.0 * eve_fraction_summary[
            "eve_interception_fraction"
        ],
        100.0 * eve_fraction_summary[
            "mean_estimated_qber"
        ],
        marker="o",
    )
    plt.axhline(
        100.0 * BASELINE_QBER_ABORT_THRESHOLD,
        linestyle="--",
        label="Abort threshold",
    )
    plt.xlabel("Eve interception fraction (%)")
    plt.ylabel("Mean estimated QBER (%)")
    plt.title("QBER Response to Partial Intercept-Resend")
    plt.grid(True, alpha=0.3)
    plt.legend()
    save_graph("11_eve_fraction_vs_qber.png")


# 12. Partial Eve interception fraction versus rejection.
if not eve_fraction_summary.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(
        100.0 * eve_fraction_summary[
            "eve_interception_fraction"
        ],
        100.0 * eve_fraction_summary[
            "rejection_rate"
        ],
        marker="o",
    )
    plt.xlabel("Eve interception fraction (%)")
    plt.ylabel("Session rejection rate (%)")
    plt.ylim(-2, 102)
    plt.title("Attack Rejection versus Eve Interception Fraction")
    plt.grid(True, alpha=0.3)
    save_graph("12_eve_fraction_vs_rejection.png")


# 13. Full-Cascade reconciliation failure count by experiment group.
cascade_failures = (
    raw_results
    .groupby("experiment_group")["failure_class"]
    .apply(
        lambda s: int(
            (s == "RECONCILIATION_FAILURE").sum()
        )
    )
    .sort_values()
)

plt.figure(figsize=(9, 5))
plt.bar(
    cascade_failures.index,
    cascade_failures.values,
)
plt.xlabel("Experiment group")
plt.ylabel("Reconciliation failures")
plt.title("Residual Failures with Full Recursive Look-Back Cascade")
plt.xticks(rotation=35, ha="right")
plt.grid(True, axis="y", alpha=0.3)
save_graph("13_full_cascade_failures.png")


# ============================================================
# Z-BASIS PROBABILITY GRAPHS (BASIS_BIAS experiment group)
# ============================================================

required_basis_summary_columns = {
    "independent_value",
    "observed_basis_match_rate_mean",
    "sifted_key_length_mean",
    "estimated_qber_mean",
    "no_eve_success_rate",
    "eve_rejection_rate",
    "secure_bits_per_total_transmitted_pulse_mean",
    "privacy_amplified_length_bits_median",
}

missing_basis_summary_columns = (
    required_basis_summary_columns
    - set(basis_summary.columns)
)

if missing_basis_summary_columns:
    raise RuntimeError(
        "Basis summary is missing required graph columns: "
        f"{sorted(missing_basis_summary_columns)}"
    )


# 14. Observed vs theoretical basis-match rate.
theoretical_basis_match_rate = (
    basis_summary["independent_value"] ** 2
    + (1.0 - basis_summary["independent_value"]) ** 2
)

plt.figure(figsize=(8, 5))
plt.plot(
    100.0 * basis_summary["independent_value"],
    100.0 * theoretical_basis_match_rate,
    marker="^",
    linestyle="--",
    label="Theoretical (p\u00b2 + (1-p)\u00b2)",
)
plt.plot(
    100.0 * basis_summary["independent_value"],
    100.0 * basis_summary["observed_basis_match_rate_mean"],
    marker="o",
    label="Observed",
)
plt.xlabel("Z-basis probability (%)")
plt.ylabel("Basis-match rate (%)")
plt.title("Sifting Efficiency versus Z-Basis Bias")
plt.grid(True, alpha=0.3)
plt.legend()
save_graph("14_basis_match_rate_vs_z_probability.png")


# 15. No-Eve and Eve QBER versus Z-basis probability.
qber_plot_basis = (
    basis_raw
    .groupby(["independent_value", "eve_present"])[
        "estimated_qber"
    ]
    .mean()
    .unstack()
    .sort_index()
)

plt.figure(figsize=(8, 5))

if False in qber_plot_basis.columns:
    plt.plot(
        100.0 * qber_plot_basis.index,
        100.0 * qber_plot_basis[False],
        marker="o",
        label="No Eve",
    )

if True in qber_plot_basis.columns:
    plt.plot(
        100.0 * qber_plot_basis.index,
        100.0 * qber_plot_basis[True],
        marker="s",
        label="Eve",
    )

plt.axhline(
    100.0 * BASELINE_QBER_ABORT_THRESHOLD,
    linestyle="--",
    label="Abort threshold",
)
plt.xlabel("Z-basis probability (%)")
plt.ylabel("Estimated QBER (%)")
plt.title("Noise-Inclusive QBER versus Z-Basis Bias")
plt.grid(True, alpha=0.3)
plt.legend()
save_graph("15_qber_vs_z_probability_eve_comparison.png")


# 16. Success and Eve rejection versus Z-basis probability.
plt.figure(figsize=(8, 5))
plt.plot(
    100.0 * basis_summary["independent_value"],
    100.0 * basis_summary["no_eve_success_rate"],
    marker="o",
    label="No-Eve success",
)
plt.plot(
    100.0 * basis_summary["independent_value"],
    100.0 * basis_summary["eve_rejection_rate"],
    marker="s",
    label="Eve rejection",
)
plt.xlabel("Z-basis probability (%)")
plt.ylabel("Rate (%)")
plt.ylim(-2, 102)
plt.title("Session Availability and Eve Rejection versus Z-Basis Bias")
plt.grid(True, alpha=0.3)
plt.legend()
save_graph("16_success_and_eve_rejection_vs_z_probability.png")


# 17. Secure-key efficiency versus Z-basis probability.
plt.figure(figsize=(8, 5))
plt.plot(
    100.0 * basis_summary["independent_value"],
    basis_summary[
        "secure_bits_per_total_transmitted_pulse_mean"
    ],
    marker="o",
)
plt.yscale("log")
plt.xlabel("Z-basis probability (%)")
plt.ylabel("Final secret bits / transmitted pulse")
plt.title("Secure-Key Efficiency versus Z-Basis Bias")
plt.grid(True, alpha=0.3)
save_graph("17_secure_bits_per_pulse_vs_z_probability.png")


# 18. Final secret length versus Z-basis probability.
plt.figure(figsize=(8, 5))
plt.plot(
    100.0 * basis_summary["independent_value"],
    basis_summary["privacy_amplified_length_bits_median"],
    marker="o",
)
plt.xlabel("Z-basis probability (%)")
plt.ylabel("Median privacy-amplified bits")
plt.title("Final Secret Length versus Z-Basis Bias")
plt.grid(True, alpha=0.3)
save_graph("18_final_secret_vs_z_probability.png")


# 19. Sifted key length versus Z-basis probability.
plt.figure(figsize=(8, 5))
plt.plot(
    100.0 * basis_summary["independent_value"],
    basis_summary["sifted_key_length_mean"],
    marker="o",
)
plt.xlabel("Z-basis probability (%)")
plt.ylabel("Mean sifted key length (bits)")
plt.title("Raw Sifted-Key Yield versus Z-Basis Bias")
plt.grid(True, alpha=0.3)
save_graph("19_sifted_key_length_vs_z_probability.png")


# ============================================================
# FINAL EXCEL WORKBOOK
# ============================================================

with pd.ExcelWriter(
    EXCEL_OUTPUT_PATH,
    engine="xlsxwriter",
) as writer:
    raw_results.to_excel(
        writer,
        sheet_name="Raw Trials",
        index=False,
    )
    case_summary.to_excel(
        writer,
        sheet_name="Case Summary",
        index=False,
    )
    distance_summary.to_excel(
        writer,
        sheet_name="Distance Summary",
        index=False,
    )
    basis_summary.to_excel(
        writer,
        sheet_name="Basis Summary",
        index=False,
    )
    failure_summary.to_excel(
        writer,
        sheet_name="Failure Analysis",
        index=False,
    )
    reconciliation_summary.to_excel(
        writer,
        sheet_name="Reconciliation Summary",
        index=False,
    )
    eve_fraction_summary.to_excel(
        writer,
        sheet_name="Eve Fraction Summary",
        index=False,
    )
    pd.DataFrame(EXPERIMENT_CASES).to_excel(
        writer,
        sheet_name="Experiment Cases",
        index=False,
    )
    pd.DataFrame([
        {
            "Parameter": key,
            "Value": json.dumps(value)
            if isinstance(value, (list, dict))
            else value,
        }
        for key, value
        in experiment_configuration.items()
    ]).to_excel(
        writer,
        sheet_name="Configuration",
        index=False,
    )
    pd.DataFrame([
        {
            "Metric":
                "Mean terminal/server signed-message time",
            "Value":
                MLDSA_BENCHMARK[
                    "mean_signed_message_seconds"
                ],
            "Unit": "seconds",
        },
        {
            "Metric":
                "Pulse repetition rate",
            "Value":
                ASSUMED_PULSE_REPETITION_RATE_HZ,
            "Unit": "pulses/second",
        },
        {
            "Metric":
                "Fibre propagation speed",
            "Value":
                FIBRE_PROPAGATION_SPEED_M_PER_S,
            "Unit": "metres/second",
        },
    ]).to_excel(
        writer,
        sheet_name="Timing Assumptions",
        index=False,
    )

    workbook = writer.book
    graph_sheet = workbook.add_worksheet(
        "Graphs"
    )
    writer.sheets["Graphs"] = graph_sheet

    row = 0
    for graph_path in generated_graphs:
        graph_sheet.insert_image(
            row,
            0,
            str(graph_path),
            {
                "x_scale": 0.72,
                "y_scale": 0.72,
            },
        )
        row += 23


print("=" * 92)
print("NOISE-INCLUSIVE ANALYSIS COMPLETE")
print("=" * 92)
print("Raw CSV :", RAW_TRIAL_CSV_PATH)
print("Workbook:", EXCEL_OUTPUT_PATH)
print("Graphs  :", RESEARCH_GRAPH_DIRECTORY)
print("Trials  :", len(raw_results))
print("=" * 92)

display(distance_summary)

display(basis_summary)